# Module 4 - Model Exploitation Techniques
---

<details>

<summary><b>Module Overview — Click to Expand</b></summary>

<br>

This module explores **Model Exploitation Techniques**, which target trained machine learning models, their interfaces, their outputs, and the information they have learned.

Earlier modules focused heavily on threats involving training data and the AI supply chain. In this module, the focus shifts toward attacks that can occur against a model once it has been trained or made available for use.

A deployed machine learning model can become an attack surface even when an adversary cannot directly access its source code, architecture, weights, or training dataset. By interacting with a model and observing its behavior, attackers may be able to manipulate predictions, approximate the model's functionality, infer information about its training data, reconstruct sensitive information, or consume the resources required to operate the system.

Throughout this module, you will examine several important forms of model exploitation, including **adversarial evasion, model extraction, membership inference, model inversion, transfer attacks, surrogate models, backdoors, API exploitation, and model denial of service**.

You will also examine how these techniques can be connected. An attacker may use information gained during one attack to support another. For example:

    Prediction API
          ↓
    Model Extraction
          ↓
     Surrogate Model
          ↓
    Adversarial Example
          ↓
      Transfer Attack
          ↓
      Target Evasion

This demonstrates why model exploitation should not always be viewed as a collection of completely independent attacks.

Throughout the module, you will use small hands-on demonstrations to observe how these attacks work and how their effects can be measured. These activities are intended to emphasize the **security concepts behind the attacks** rather than simply running attack code.

You will also examine defensive techniques such as adversarial training, differential privacy, output restriction, query monitoring, access controls, rate limiting, and other methods for reducing model exploitation risk.

By the end of the module, you should be able to examine a deployed machine learning system from both perspectives:

    ATTACKER
       ↓
    What information or behavior can be exploited?

and:

    DEFENDER
       ↓
    What controls can reduce the attack surface?

The central idea of this module is:

> **Protecting the training dataset is not enough. Once a machine learning model is trained and deployed, the model itself and the interfaces used to access it become security-relevant assets.**

</details>

<details>

<summary><b>Learning Objectives — Click to Expand</b></summary>

<br>

By the end of this module, you will be able to:

- **Explain model exploitation** and describe how attackers can target trained machine learning models without directly compromising the original training dataset.

- **Distinguish between white-box, gray-box, and black-box attacks** based on the information and access available to an attacker.

- **Explain adversarial evasion** and demonstrate how intentionally modified inputs can cause a trained model to produce incorrect predictions.

- **Describe common adversarial techniques**, including concepts associated with FGSM, PGD, and Carlini-Wagner attacks.

- **Explain model extraction or model stealing** and demonstrate how prediction queries can be used to approximate the behavior of a target model.

- **Explain membership inference attacks** and identify how model behavior may reveal whether a particular record was part of the training dataset.

- **Describe model inversion attacks** and explain how model outputs may reveal sensitive or representative information associated with learned data.

- **Explain transferability and surrogate models**, including how an attacker may construct or use a substitute model to develop attacks against a black-box target.

- **Describe model backdoors and Trojan behavior**, including the relationship between normal model behavior, hidden triggers, and attacker-selected predictions.

- **Explain how APIs and prediction interfaces create attack surfaces** through repeated queries, confidence information, and observable model responses.

- **Describe model denial-of-service and resource-exhaustion attacks** and explain how AI systems can be targeted through the computational resources required for inference.

- **Analyze attack chains** in which one model exploitation technique provides information or capabilities that enable another attack.

- **Evaluate defensive techniques** such as adversarial training, differential privacy, output minimization, access control, query monitoring, and rate limiting.

- **Measure attack and defense effectiveness** using appropriate evidence rather than assuming that an attack or mitigation succeeded simply because the code executed.

</details>

---

<details>

<summary><b>Section 1 — What Is Model Exploitation? — Click to Expand</b></summary>

<br>

Earlier modules focused heavily on the information used to build machine learning systems. You examined how manipulating training data, labels, distributions, and other parts of the AI supply chain can influence what a model learns.

This module shifts the focus.

Instead of asking:

> **How can an attacker manipulate the information used to train a model?**

we begin asking:

> **How can an attacker exploit a trained model, its behavior, or the interfaces used to access it?**

This is the basic idea behind **model exploitation**.

A machine learning model can become a valuable security target because it contains learned behavior created from its training process. Even when an attacker cannot directly access the original training dataset or source code, interactions with the trained model may reveal useful information or provide opportunities to manipulate its behavior.

Model exploitation therefore expands the AI attack surface beyond the training pipeline.

<br>

<details>

<summary><b>1.1 — From Data Attacks to Model Attacks</b></summary>

<br>

It is useful to distinguish the attacks from the previous module from the attacks you will study here.

A data-level attack generally attempts to influence the information used during training:

    Training Data
          ↓
       ATTACK
          ↓
    Model Training
          ↓
    Altered Model

Examples include label flipping, clean-label poisoning, duplication, and distribution manipulation.

Model exploitation focuses more directly on the trained model and the interfaces surrounding it:

    Trained Model
          ↓
    Model Interface
          ↓
       ATTACKER
          ↓
    Queries / Inputs
          ↓
    Observe Behavior
          ↓
    Exploit Information
    or Model Behavior

The attacker may not need permission to modify the original training dataset.

Instead, the attacker may attempt to manipulate inputs, observe predictions, collect confidence scores, reconstruct information, approximate the model, trigger hidden behavior, or consume computational resources.

This means that protecting the training pipeline is necessary, but it does not eliminate the security risks associated with a deployed model.

</details>

<br>

<details>

<summary><b>1.2 — The Model as a Security Asset</b></summary>

<br>

A trained model should be considered a security-relevant asset.

Consider what may be represented by a model.

The organization may have invested significant time and money collecting data, cleaning information, selecting features, developing architectures, training models, tuning parameters, and evaluating performance.

The resulting model may therefore represent valuable intellectual property.

A model may also indirectly contain information about the data used during training.

Finally, the model may control or influence important decisions made by an application.

Examples could include:

- Identifying malicious or legitimate activity
- Classifying suspicious messages
- Detecting fraudulent transactions
- Recognizing images or objects
- Prioritizing security alerts
- Making recommendations
- Supporting automated decisions

If an attacker can manipulate, copy, infer information from, or disrupt that model, the attacker may affect the larger system that depends on it.

The asset being protected is therefore not simply:

    model.pkl

or:

    model.pt

The defender must consider the complete environment:

    Model
      +
    Model Parameters
      +
    Prediction Interface
      +
    Outputs
      +
    Training Information
      +
    Computational Resources
      +
    Application Using the Model

</details>

<br>

<details>

<summary><b>1.3 — The Prediction Interface as an Attack Surface</b></summary>

<br>

Many machine learning models are accessed through an application or API.

A simplified prediction service might operate like this:

    User Input
         ↓
       API
         ↓
    ML Model
         ↓
    Prediction
         ↓
       User

For normal users, this interface provides the functionality of the system.

For an attacker, the same interface may provide an opportunity to study the model.

An attacker might repeatedly submit inputs and observe:

    Predicted Class

    Confidence Score

    Probability Distribution

    Response Time

    Error Messages

    Differences Between Responses

Each interaction may reveal a small amount of information.

A single prediction might reveal very little.

However, repeated interaction can sometimes allow an attacker to learn significantly more about the model.

Conceptually:

    Query 1 → Response
    Query 2 → Response
    Query 3 → Response
    Query 4 → Response
       ...
    Query N → Response
              ↓
        Collected Behavior
              ↓
       Attacker Analysis

This is why a prediction API should not automatically be treated as harmless simply because users cannot directly download the underlying model.

The **behavior exposed by the model** can itself become valuable information.

</details>

<br>

<details>

<summary><b>1.4 — Attacker Goals</b></summary>

<br>

Model exploitation does not describe one specific attack.

Different attackers may have very different objectives.

One attacker may want to cause incorrect predictions.

Another may want to copy the functionality of a valuable proprietary model.

Another may be interested in determining whether sensitive information was used during training.

Another may attempt to reconstruct information associated with the training data.

An attacker could also attempt to trigger hidden behavior or consume enough computational resources to reduce the availability of the system.

Throughout this module, these goals will appear in several attack categories.

### Manipulate Predictions

The attacker attempts to cause the model to make an incorrect or attacker-desired prediction.

This is central to **adversarial evasion**.

### Learn the Model

The attacker repeatedly observes the target model and attempts to construct another model that behaves similarly.

This is associated with **model extraction** or **model stealing**.

### Learn About Training Membership

The attacker attempts to determine whether a particular record was included in the model's training dataset.

This is known as **membership inference**.

### Recover Information

The attacker attempts to reconstruct or infer information associated with what the model learned.

This is associated with **model inversion**.

### Trigger Hidden Behavior

The attacker supplies a particular trigger that activates attacker-selected behavior embedded within the model.

This is associated with **backdoors and Trojans**.

### Exhaust Resources

The attacker attempts to cause unusually expensive model processing or generate enough requests to reduce availability.

This is associated with **model denial of service and resource exhaustion**.

These attacks have different goals, but they share an important idea:

> **The attacker is treating the trained model and its observable behavior as part of the attack surface.**

</details>

<br>

<details>

<summary><b>1.5 — Attacker Knowledge and Access</b></summary>

<br>

Not every attacker has the same level of access to a machine learning system.

Attack scenarios are often described using three general categories:

    White-Box

    Gray-Box

    Black-Box

Understanding these categories is important because the information available to the attacker can significantly affect which attacks are practical.

</details>

<br>

<details>

<summary><b>1.6 — White-Box Attacks</b></summary>

<br>

In a **white-box** scenario, the attacker has extensive knowledge of the target model.

Depending on the system, this might include access to:

- Model architecture
- Model parameters or weights
- Gradients
- Feature information
- Training configuration
- Learning algorithm

Conceptually:

    ATTACKER
       ↓
    Architecture ✓
    Parameters   ✓
    Gradients    ✓
    Outputs      ✓

This level of knowledge can make some attacks easier because the attacker can directly examine how the model responds internally.

For example, gradient information can be extremely useful when constructing adversarial examples.

However, white-box access represents a relatively strong attacker assumption.

A remote attacker interacting only with a public prediction API may not have this level of access.

</details>

<br>

<details>

<summary><b>1.7 — Black-Box Attacks</b></summary>

<br>

A **black-box** attacker does not have direct access to the internal details of the model.

Instead, the attacker interacts with the system through its exposed interface.

The attacker may know:

    Input
      ↓
    [ UNKNOWN MODEL ]
      ↓
    Output

but not:

    Internal Architecture

    Model Weights

    Gradients

    Training Process

This may initially appear much safer.

However, black-box access does not mean that exploitation is impossible.

The attacker may repeatedly query the model and observe its responses.

For example:

    Input A → Class 0

    Input B → Class 1

    Input C → Class 1

    Input D → Class 2

Over many interactions, the attacker may begin learning how the model behaves.

Some attacks in this module are especially important because they demonstrate that **observable outputs can reveal useful information even when the internal model remains hidden**.

</details>

<br>

<details>

<summary><b>1.8 — Gray-Box Attacks</b></summary>

<br>

A **gray-box** scenario falls between white-box and black-box access.

The attacker knows some information about the system but does not have complete internal access.

For example, the attacker might know:

    Model Type ✓

    Input Features ✓

    General Training Dataset ✓

but not:

    Exact Parameters ✗

    Exact Training Records ✗

    Internal Gradients ✗

Gray-box access is useful to consider because real systems often expose some information.

Documentation may reveal the model's purpose.

An API may describe expected input features.

Research papers may describe the general architecture.

Software packages may reveal implementation details.

An attacker may therefore know more than a completely blind black-box attacker while still lacking direct access to the deployed model.

</details>

<br>

<details>

<summary><b>1.9 — Comparing Access Levels</b></summary>

<br>

The three access levels can be summarized as:

| Access Level | Internal Knowledge | Typical Interaction |
|---|---|---|
| White-Box | Extensive | Direct model access |
| Gray-Box | Partial | Some system knowledge plus interaction |
| Black-Box | Limited | Queries and observable outputs |

These categories describe **attacker knowledge**, not attack success.

A black-box attacker is not automatically harmless.

A white-box attacker is not automatically successful.

The attack still needs to be demonstrated through evidence.

However, greater access may provide the attacker with additional information that can make certain exploitation techniques easier.

</details>

<br>

<details>

<summary><b>1.10 — Information Leakage Through Model Outputs</b></summary>

<br>

The amount of information returned by a model can influence the attack surface.

Consider two prediction APIs.

### API A

Returns:

    "Spam"

### API B

Returns:

    Spam:       0.91
    Legitimate: 0.09

Both APIs provide a prediction.

However, API B reveals additional information about the model's confidence.

Now imagine an API that returns probabilities for many possible classes.

The attacker receives even more information with every query.

This does not mean that confidence scores should never be provided. They may be useful or necessary for legitimate applications.

Instead, defenders should recognize an important principle:

> **Every piece of information exposed by a prediction interface should have a reason to be exposed.**

If detailed outputs are unnecessary, limiting them may reduce the information available to an attacker.

This idea will become especially important when you study model extraction, membership inference, and model inversion.

</details>

<br>

<details>

<summary><b>1.11 — Model Exploitation Can Be an Iterative Process</b></summary>

<br>

An attacker may not immediately know how to exploit a model.

Instead, exploitation can involve repeated experimentation.

For example:

    Submit Input
         ↓
    Observe Output
         ↓
    Modify Input
         ↓
    Submit Again
         ↓
    Compare Response
         ↓
    Learn More
         ↓
       Repeat

This creates an important security consideration.

A single query may look completely legitimate.

The suspicious behavior may only become visible when defenders examine patterns across many queries.

This is one reason why **query monitoring, rate limiting, logging, and behavioral analysis** can become useful defensive controls.

The security team may need to ask not only:

> "Was this individual request allowed?"

but also:

> "What pattern of behavior is this user producing over time?"

</details>

<br>

<details>

<summary><b>1.12 — Exploitation Techniques Can Be Chained</b></summary>

<br>

The attacks in this module should not always be viewed independently.

Information gained from one technique may help enable another.

For example:

    Target Model
         ↓
    Repeated Queries
         ↓
    Model Extraction
         ↓
    Surrogate Model
         ↓
    Adversarial Example
         ↓
    Transfer Attack
         ↓
    Target Model Evasion

In this example, the attacker does not need complete white-box access to the original target.

Instead, the attacker attempts to learn enough about the target to build a substitute model.

The substitute may then provide additional information, such as gradients, that the attacker did not have from the original black-box system.

This creates a progression:

    Black-Box Access
          ↓
    Learn Target Behavior
          ↓
    Build Surrogate
          ↓
    Gain White-Box Access
    to the Surrogate
          ↓
    Develop New Attack

Later in the module, you will examine these **attack chains** more directly.

</details>

<br>

<details>

<summary><b>1.13 — Defender Perspective</b></summary>

<br>

From the defender's perspective, model exploitation requires thinking beyond traditional protection of files and servers.

The defender should consider questions such as:

    Who can access the model?

    What inputs can users submit?

    What outputs are returned?

    Are confidence scores necessary?

    How many queries can a user make?

    Are unusual query patterns monitored?

    Can the model's behavior reveal training information?

    Is model performance monitored?

    Are computational resources protected?

    Are model artifacts protected from unauthorized access?

Different exploitation techniques require different defenses.

For example, rate limiting may help reduce large-scale automated querying but does not make a model resistant to adversarial examples.

Adversarial training may improve robustness against certain adversarial inputs but does not automatically prevent model extraction.

Differential privacy may reduce some privacy leakage but does not solve every model-security problem.

This again leads to **defense in depth**.

</details>

<br>

<details>

<summary><b>1.14 — Measuring Model Exploitation</b></summary>

<br>

Throughout this module, you will continue using the same evidence-based approach used in the previous module.

We should not say:

> "The attack code ran, therefore the attack worked."

Instead, each attack needs an appropriate measurement.

For adversarial evasion:

    Did prediction accuracy decrease?

    Did the target sample change classification?

For model extraction:

    How closely does the stolen model reproduce the target's predictions?

For membership inference:

    Can membership be predicted better than an appropriate baseline?

For model inversion:

    What information was actually reconstructed?

For resource exhaustion:

    Did processing time or resource consumption measurably increase?

Similarly, defensive effectiveness must also be measured.

The general process remains:

    Establish Baseline
           ↓
      Perform Attack
           ↓
      Measure Impact
           ↓
       Apply Defense
           ↓
    Measure Improvement

This approach prevents us from claiming success simply because an attack or defensive technique executed without an error.

</details>

<br>

<details>

<summary><b>Section 1 — Key Takeaways</b></summary>

<br>

Model exploitation focuses on attacks against **trained models, their behavior, and the interfaces used to access them**.

A trained model can represent valuable intellectual property, expose information about training data, influence important application decisions, and consume significant computational resources.

Attackers may attempt to:

- Manipulate model predictions
- Copy model functionality
- Infer training membership
- Reconstruct information
- Trigger hidden behavior
- Exhaust computational resources

The amount of information available to an attacker is commonly described using **white-box, gray-box, and black-box** access models.

Importantly, black-box access does not mean that a model is safe. Repeated queries and observable outputs may still reveal useful information.

Model outputs should therefore be treated as part of the attack surface.

Finally, model exploitation techniques can sometimes be combined into larger attack chains.

The central lesson is:

> **A deployed model does not need to be directly compromised for an attacker to exploit its behavior or learn useful information from it.**

</details>

</details>

In [ ]:
# @title Section 1 — Model Exploitation Visualizations { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from IPython.display import display, HTML
from io import BytesIO
import base64
import html

# ============================================================
# CONFIGURATION
# ============================================================

rng = np.random.default_rng(42)

BLUE = "#2563EB"
RED = "#DC2626"
GREEN = "#16A34A"
ORANGE = "#EA580C"
PURPLE = "#7C3AED"

plt.rcParams.update({
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# ============================================================
# COLLAPSIBLE VISUALIZATION HELPER
# ============================================================

def show_collapsible_figure(title, fig, explanation):

    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=130,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    buffer.close()

    display(HTML(f"""
    <details style="
        margin:12px 0;
        border:1px solid #cbd5e1;
        border-radius:9px;
        overflow:hidden;
    ">

        <summary style="
            padding:15px;
            background:#eff6ff;
            color:#1e3a8a;
            cursor:pointer;
            font-weight:bold;
            font-size:16px;
        ">
            {html.escape(title)} — Click to Expand
        </summary>

        <div style="
            padding:12px;
            text-align:center;
            background:white;
        ">

            <img
                src="data:image/png;base64,{encoded}"
                style="max-width:100%;width:900px;height:auto;"
            >

            <div style="
                text-align:left;
                background:#f8fafc;
                padding:14px;
                line-height:1.7;
                color:#334155;
                border-radius:8px;
                margin-top:12px;
            ">
                {explanation}
            </div>

        </div>

    </details>
    """))


# ============================================================
# VISUALIZATION 1 — MODEL EXPLOITATION ATTACK SURFACE
# ============================================================

stages = [
    "Deployed Model and Prediction Interface",
    "Attacker Access: White-Box, Gray-Box, or Black-Box",
    "Queries, Inputs, and Observable Outputs",
    "Possible Goals: Evasion, Extraction, Privacy Leakage",
    "Defender Controls and Measured Impact"
]

fig, ax = plt.subplots(figsize=(11, 8))

ax.set_xlim(0, 10)
ax.set_ylim(0, 7.5)
ax.axis("off")

ax.set_title(
    "Model Exploitation: Attack Surface",
    fontsize=17,
    fontweight="bold",
    pad=15
)

for i, label in enumerate(stages):

    y = (len(stages) - i - 1) * 1.25 + 0.4

    rect = FancyBboxPatch(
        (1.4, y),
        7.2,
        0.88,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        facecolor="#EFF6FF" if i % 2 == 0 else "#F5F3FF",
        edgecolor=BLUE,
        linewidth=1.5
    )

    ax.add_patch(rect)

    ax.text(
        5,
        y + 0.44,
        label,
        ha="center",
        va="center",
        fontweight="bold",
        fontsize=11
    )

    if i < len(stages) - 1:

        ax.annotate(
            "",
            xy=(5, y - 0.30),
            xytext=(5, y - 0.04),
            arrowprops={
                "arrowstyle": "-|>",
                "color": "#64748B",
                "lw": 1.7
            }
        )

plt.tight_layout()

show_collapsible_figure(
    "1.1 — Understanding the Model Exploitation Attack Surface",
    fig,
    """
    Model exploitation focuses on attacking a machine learning
    model's behavior, outputs, privacy, or availability.

    <br><br>

    An attacker may interact with a deployed model through
    prediction requests, API calls, or direct access to
    model components.

    <br><br>

    Different attacks attempt to achieve different objectives.
    For example, adversarial evasion attempts to manipulate
    predictions, while model extraction attempts to reproduce
    a model's functionality.

    <br><br>

    Privacy attacks may attempt to reveal information about
    training data, while denial-of-service attacks target
    the model's availability.

    <br><br>

    <b>Key takeaway:</b> Protecting a deployed model requires
    understanding what attackers can access, what information
    they can observe, and which security properties they
    may attempt to compromise.
    """
)


# ============================================================
# VISUALIZATION 2 — ATTACKER ACCESS LEVELS
# ============================================================

access_levels = [
    "Black-Box",
    "Gray-Box",
    "White-Box"
]

# Conceptual access levels, not measured attack success.
access_values = [1, 2, 3]

colors = [BLUE, ORANGE, RED]

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(
    access_levels,
    access_values,
    color=colors,
    width=0.58
)

ax.set_title(
    "Attacker Knowledge by Access Level",
    fontsize=17,
    fontweight="bold",
    pad=15
)

ax.set_ylabel("Conceptual Access Level")

ax.set_ylim(0, 3.7)

ax.set_yticks([1, 2, 3])

ax.set_yticklabels([
    "Limited",
    "Partial",
    "Full"
])

ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

for bar, value in zip(bars, access_values):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.08,
        str(value),
        ha="center",
        fontsize=12,
        fontweight="bold"
    )

plt.tight_layout()

show_collapsible_figure(
    "1.2 — Black-Box, Gray-Box, and White-Box Access",
    fig,
    """
    Attackers do not always have the same level of
    access to a machine learning system.

    <br><br>

    <b>Black-box access:</b> The attacker can submit
    inputs and observe model responses but cannot
    directly inspect the model's internal parameters.

    <br><br>

    <b>Gray-box access:</b> The attacker has partial
    knowledge of the system, such as its architecture,
    training approach, or certain implementation details.

    <br><br>

    <b>White-box access:</b> The attacker has extensive
    access to internal model information, potentially
    including architecture, parameters, and gradients.

    <br><br>

    The numerical values in this visualization represent
    conceptual levels of access, not measured attack
    effectiveness or probabilities of success.

    <br><br>

    <b>Key takeaway:</b> The information available to an
    attacker influences which exploitation techniques
    are practical. However, limited access does not
    necessarily prevent an attack.
    """
)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print("Section 1 visualizations are ready.")
print("Two collapsible visualizations have been generated.")

In [ ]:
# @title ▶ Section 1 Checkpoint — Model Exploitation Fundamentals { display-mode: "form" }

questions = [
    {
        "question":
        "1. What is the primary focus of model exploitation?",
        "options": [
            "A. Only manipulating training labels",
            "B. Exploiting trained models, their behavior, or their interfaces",
            "C. Only stealing training computers",
            "D. Increasing model accuracy"
        ],
        "answer": "B",
        "explanation":
        "Model exploitation focuses on the trained model, its observable behavior, and the interfaces surrounding it."
    },

    {
        "question":
        "2. Which attacker has extensive knowledge of model architecture, parameters, or gradients?",
        "options": [
            "A. Black-box attacker",
            "B. Gray-box attacker",
            "C. White-box attacker",
            "D. External user"
        ],
        "answer": "C",
        "explanation":
        "White-box access assumes extensive internal knowledge or access to the target model."
    },

    {
        "question":
        "3. A user can submit inputs to an API and observe predictions but cannot access the model internals. What type of access is this?",
        "options": [
            "A. White-box",
            "B. Black-box",
            "C. Full-access",
            "D. Training-box"
        ],
        "answer": "B",
        "explanation":
        "Black-box attackers interact with a model through observable inputs and outputs without direct internal access."
    },

    {
        "question":
        "4. What BEST describes gray-box access?",
        "options": [
            "A. The attacker knows absolutely nothing",
            "B. The attacker has complete access to everything",
            "C. The attacker has partial knowledge about the target system",
            "D. The attacker can only modify labels"
        ],
        "answer": "C",
        "explanation":
        "Gray-box scenarios fall between white-box and black-box access and provide the attacker with some system knowledge."
    },

    {
        "question":
        "5. Why can prediction confidence scores increase the attack surface?",
        "options": [
            "A. They reveal additional information about model behavior",
            "B. They prevent predictions",
            "C. They automatically expose model weights",
            "D. They delete training information"
        ],
        "answer": "A",
        "explanation":
        "Confidence scores can provide more information about how the model behaves than a class label alone."
    },

    {
        "question":
        "6. Which attack attempts to approximate or copy the functionality of a target model?",
        "options": [
            "A. Model extraction",
            "B. Label flipping",
            "C. Data duplication",
            "D. Distribution manipulation"
        ],
        "answer": "A",
        "explanation":
        "Model extraction or model stealing attempts to reproduce the behavior or functionality of a target model."
    },

    {
        "question":
        "7. Which attack asks whether a particular record was part of the training dataset?",
        "options": [
            "A. Model inversion",
            "B. Membership inference",
            "C. Resource exhaustion",
            "D. Label flipping"
        ],
        "answer": "B",
        "explanation":
        "Membership inference attempts to determine whether a particular record was included in model training."
    },

    {
        "question":
        "8. Why might defenders monitor patterns across many model queries?",
        "options": [
            "A. A single legitimate-looking request may be part of a larger exploitation attempt",
            "B. Every model query is automatically malicious",
            "C. Models cannot process individual queries",
            "D. Monitoring automatically prevents every attack"
        ],
        "answer": "A",
        "explanation":
        "Some exploitation techniques rely on repeated interaction, making patterns across many queries security-relevant."
    },

    {
        "question":
        "9. What is an example of an attack chain?",
        "options": [
            "A. Query a target, build a surrogate, create adversarial inputs, and attempt transfer",
            "B. Train the same clean model twice",
            "C. Change the notebook title",
            "D. Calculate model accuracy"
        ],
        "answer": "A",
        "explanation":
        "One exploitation technique can provide information or capabilities that support another attack."
    },

    {
        "question":
        "10. What is the BEST way to determine whether a model exploitation attack succeeded?",
        "options": [
            "A. Check whether the attack code ran without errors",
            "B. Measure the attack using evidence appropriate to its objective",
            "C. Assume every attack succeeds",
            "D. Count the number of attack code cells"
        ],
        "answer": "B",
        "explanation":
        "Attack success should be demonstrated through measurable evidence rather than assumed from successful code execution."
    }
]

score = 0

print("=" * 70)
print("SECTION 1 CHECKPOINT")
print("Model Exploitation Fundamentals")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:

        print("✓ Correct!")
        score += 1

    else:

        print(
            f"✗ Incorrect. "
            f"Correct answer: {q['answer']}"
        )

    print(
        "Explanation:",
        q["explanation"]
    )

percentage = (
    score / len(questions)
) * 100

print("\n" + "=" * 70)
print("SECTION 1 RESULTS")
print("=" * 70)

print(
    f"\nScore: "
    f"{score}/{len(questions)}"
)

print(
    f"Percentage: "
    f"{percentage:.1f}%"
)

if percentage >= 90:

    print(
        "\nExcellent! You have a strong understanding "
        "of model exploitation fundamentals."
    )

elif percentage >= 80:

    print(
        "\nGreat work! You are ready to continue "
        "to the attack sections."
    )

else:

    print(
        "\nReview Section 1 before continuing. "
        "Pay particular attention to attacker access "
        "levels, attacker goals, and model interfaces."
    )

print("=" * 70)

---

<details>

<summary><b>Section 2 — Adversarial Attacks & Evasion — Click to Expand</b></summary>

<br>

A machine learning model can perform extremely well on normal test data and still behave unexpectedly when an attacker intentionally manipulates its inputs.

This creates an important distinction between **accuracy** and **robustness**.

A model with high accuracy has learned to correctly classify normal examples.

A robust model should also continue behaving appropriately when inputs are deliberately manipulated.

An **adversarial attack** attempts to create an input that causes a machine learning model to make an incorrect or attacker-desired prediction.

When this manipulation occurs during inference so that an input avoids the model's intended decision, it is commonly described as **adversarial evasion**.

The basic process is:

    Legitimate Input
          ↓
    Correct Prediction
          ↓
    Attacker Modifies Input
          ↓
    Adversarial Input
          ↓
    Incorrect Prediction

Unlike training-data poisoning, the attacker does not necessarily modify the model or its training dataset.

The attacker manipulates what the trained model receives.

<br>

<details>

<summary><b>2.1 — Evasion at Inference Time</b></summary>

<br>

Adversarial evasion normally occurs after the model has already been trained.

Consider a simplified security classifier:

    Input
      ↓
    ML Classifier
      ↓
    Malicious

An attacker wants the same underlying object to be classified as:

    Input + Manipulation
            ↓
       ML Classifier
            ↓
         Benign

The attack therefore targets the model's **decision-making process at inference time**.

The attacker is not necessarily trying to permanently change the model.

Instead, the objective is to find an input that the model handles incorrectly.

This distinction separates evasion from poisoning:

    POISONING
       ↓
    Manipulate Training
       ↓
    Influence Learning

versus:

    EVASION
       ↓
    Manipulate Input
       ↓
    Influence Prediction

Both can produce incorrect predictions, but they attack different stages of the machine learning lifecycle.

</details>

<br>

<details>

<summary><b>2.2 — Adversarial Examples</b></summary>

<br>

An input intentionally modified to cause incorrect model behavior is commonly called an **adversarial example**.

Conceptually, an attacker begins with a legitimate input:

    x

and adds a carefully selected perturbation:

    δ

creating:

    x_adv = x + δ

The goal is to make the model respond differently to `x_adv`.

The important word is **carefully**.

An adversarial attack is generally more interesting than simply destroying the input.

If an image of a handwritten number is completely replaced with random noise, it would not be surprising if the model failed.

Instead, adversarial attacks attempt to find modifications that influence the model while keeping the input sufficiently similar or valid for the attack scenario.

The definition of an acceptable modification depends on the application.

For an image classifier, the change might involve pixel values.

For a malware classifier, the attacker may need to preserve the program's functionality.

For a network classifier, manipulated traffic may still need to follow valid protocol behavior.

Therefore:

> **An adversarial perturbation must be considered in the context of the system being attacked.**

</details>

<br>

<details>

<summary><b>2.3 — Why Small Changes Can Matter</b></summary>

<br>

Machine learning models do not necessarily interpret inputs in the same way humans do.

A human may look at two inputs and consider them almost identical while a model's internal calculations place them on different sides of a decision boundary.

Imagine a simplified classifier:

                Class A
                   ●
                ●  ●
              ●
    -------------------------- Decision Boundary
                         ○
                      ○  ○
                         Class B

A sample close to the boundary may require only a relatively small change to move into another classification region.

Conceptually:

    Original Sample
          ●
          |
          | small movement
          ↓
          ○
    Different Region

Adversarial attacks attempt to determine **which direction of change is useful** for producing this behavior.

This is why random noise and adversarial perturbations are not necessarily equivalent.

Random noise changes an input without considering the model.

An adversarial perturbation is chosen specifically to influence the model's decision.

</details>

<br>

<details>

<summary><b>2.4 — Untargeted and Targeted Attacks</b></summary>

<br>

Adversarial attacks can have different objectives.

### Untargeted Attack

An untargeted attacker wants the model to stop predicting the correct class.

For example:

    Correct:
    Class 7
       ↓
    Attack
       ↓
    Anything Except 7

The attacker does not necessarily care which incorrect class is produced.

### Targeted Attack

A targeted attacker wants a specific attacker-selected result.

For example:

    Correct:
    Malicious
       ↓
    Attack
       ↓
    Benign

The distinction is important because a targeted attack usually has a more specific objective.

In cybersecurity, an attacker may not simply want a detection system to be wrong.

The attacker may specifically want malicious activity to be classified as legitimate.

</details>

<br>

<details>

<summary><b>2.5 — White-Box Adversarial Attacks</b></summary>

<br>

Many well-known adversarial attack techniques assume **white-box access**.

Recall from Section 1 that white-box access may provide information such as:

    Model Architecture

    Parameters

    Gradients

A gradient describes how changes in an input influence the model's loss.

For an attacker, this can provide useful directional information.

Instead of randomly modifying an input, the attacker can ask:

> **Which direction should I change the input to increase the model's error?**

This leads to gradient-based adversarial attacks.

The mathematics can become complex, but the core security idea is straightforward:

    Input
      ↓
    Model
      ↓
    Calculate Direction
    That Increases Error
      ↓
    Modify Input
      ↓
    Test Again

You will observe this process in the mini lab.

</details>

<br>

<details>

<summary><b>2.6 — Fast Gradient Sign Method (FGSM)</b></summary>

<br>

The **Fast Gradient Sign Method (FGSM)** is a widely used introductory adversarial attack.

FGSM uses information from the model's gradient to determine how the input should be changed to increase prediction error.

A simplified representation is:

    adversarial input
           =
    original input
           +
    controlled perturbation

A parameter commonly called **epsilon (ε)** controls the strength of that perturbation.

Conceptually:

    Small ε
       ↓
    Smaller Change

    Large ε
       ↓
    Stronger Change

Increasing epsilon may make an attack more effective, but it also makes the modification larger.

This creates an important tradeoff:

    Attack Strength
          ↕
    Perturbation Size

FGSM is particularly useful for teaching because it demonstrates that an attacker does not need to repeatedly search for a perturbation.

The gradient provides a direction, and FGSM applies a single perturbation step.

</details>

<br>

<details>

<summary><b>2.7 — Projected Gradient Descent (PGD)</b></summary>

<br>

**Projected Gradient Descent (PGD)** extends the basic idea of a gradient-based attack by using multiple smaller steps.

FGSM can be viewed conceptually as:

    Original Input
          ↓
      One Step
          ↓
    Adversarial Input

PGD instead performs:

    Original Input
          ↓
       Step 1
          ↓
       Step 2
          ↓
       Step 3
          ↓
         ...
          ↓
    Adversarial Input

After each step, the attack keeps the manipulated input within an allowed perturbation region.

This prevents the attack from simply making unlimited changes.

PGD is often used when evaluating adversarial robustness because repeated optimization can produce stronger adversarial examples than a single-step method.

For this introductory section, you only need to understand the main distinction:

> **FGSM uses one gradient-based step, while PGD repeatedly applies smaller gradient-based steps within a defined perturbation limit.**

</details>

<br>

<details>

<summary><b>2.8 — Carlini-Wagner Attacks</b></summary>

<br>

The **Carlini-Wagner (CW)** family of attacks approaches adversarial example generation as an optimization problem.

Instead of simply asking how to increase model error, the attacker attempts to balance objectives such as:

    Cause Misclassification

while also:

    Keep the Modification Small

Conceptually:

    Find a Perturbation
           ↓
    Small Enough to Limit Change
           +
    Strong Enough to Fool Model

CW attacks are historically important because they demonstrated that defenses should be evaluated against strong adaptive attacks rather than assuming that resistance to one simple attack means a model is secure.

You will not implement CW in this introductory mini lab.

The important takeaway is that adversarial attacks vary in complexity and strength.

</details>

<br>

<details>

<summary><b>2.9 — Attack Strength and Robustness</b></summary>

<br>

Suppose a model achieves:

    Clean Accuracy = 97%

That number alone does not tell us how the model behaves under adversarial manipulation.

We might evaluate the same model at several perturbation strengths:

    ε = 0.00 → 97%

    ε = 0.05 → 92%

    ε = 0.10 → 76%

    ε = 0.20 → 41%

This provides a much clearer picture of robustness.

Instead of asking only:

> "How accurate is the model?"

we can ask:

> "How quickly does performance degrade as adversarial strength increases?"

A robust model should ideally maintain useful performance even when inputs are manipulated within realistic limits.

</details>

<br>

<details>

<summary><b>2.10 — Why Adversarial Evasion Matters in Cybersecurity</b></summary>

<br>

Adversarial evasion becomes especially important when machine learning is used to make security decisions.

Consider a hypothetical malware classifier.

Normal behavior:

    Malicious File
          ↓
      Classifier
          ↓
       MALWARE
          ↓
        Block

An attacker may attempt:

    Malicious File
          +
    Carefully Selected Modification
          ↓
      Classifier
          ↓
        BENIGN
          ↓
        Allow

The exact modifications that are realistic depend on how the classifier works and what features it examines.

Similar concerns can appear in phishing detection, network intrusion detection, fraud detection, image recognition, biometric systems, and other ML-assisted security applications.

The important point is that model performance on normal test data does not automatically tell us how the model behaves when someone is deliberately trying to fool it.

</details>

<br>

<details>

<summary><b>2.11 — Evaluating an Adversarial Attack</b></summary>

<br>

As with the previous module, we need measurable evidence that an attack actually worked.

A useful experiment follows:

    Train Model
         ↓
    Evaluate Clean Data
         ↓
    Record Baseline
         ↓
    Create Adversarial Inputs
         ↓
    Evaluate Again
         ↓
    Compare Results

For example:

    Clean Accuracy
          =
        0.96

    Adversarial Accuracy
          =
        0.54

This provides evidence that the adversarial inputs affected model behavior.

We can also measure:

    Number of Predictions Changed

    Attack Success Rate

    Accuracy at Different Epsilon Values

    Confusion Matrix Changes

The important rule remains:

> **An adversarial attack should be demonstrated through measurable changes in model behavior, not simply by successfully generating modified inputs.**

</details>

<br>

<details>

<summary><b>Section 2 — Key Takeaways</b></summary>

<br>

Adversarial evasion attacks manipulate model inputs at inference time in an attempt to produce incorrect or attacker-selected predictions.

Unlike poisoning, the attacker does not necessarily modify the training dataset.

An **adversarial example** is an intentionally manipulated input designed to influence model behavior.

Attacks may be:

- **Untargeted**, where any incorrect prediction is useful.
- **Targeted**, where the attacker wants a specific incorrect prediction.

Gradient-based attacks use information about how input changes affect model error.

Three important techniques are:

- **FGSM** — a single gradient-based perturbation step.
- **PGD** — multiple smaller gradient-based steps within a perturbation limit.
- **Carlini-Wagner** — an optimization-based approach designed to balance attack success and perturbation size.

The most important lesson is:

> **High clean accuracy does not automatically mean that a machine learning model is robust against intentionally manipulated inputs.**

</details>

</details>

In [ ]:
# @title Section 2 — Adversarial Attacks & Evasion Visualizations { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Circle
from matplotlib.colors import ListedColormap
from IPython.display import display, HTML
from io import BytesIO
import base64
import html

# ============================================================
# CONFIGURATION
# ============================================================

rng = np.random.default_rng(42)

BLUE = "#2563EB"
RED = "#DC2626"
GREEN = "#16A34A"
ORANGE = "#EA580C"
PURPLE = "#7C3AED"
GRAY = "#64748B"
DARK = "#0F172A"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# ============================================================
# COLLAPSIBLE VISUALIZATION HELPER
# ============================================================

def show_collapsible_figure(title, fig, explanation, color=BLUE):

    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=135,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    image_data = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    buffer.close()

    display(HTML(f"""
    <details style="
        margin:12px 0;
        border:1px solid #cbd5e1;
        border-radius:10px;
        overflow:hidden;
    ">

        <summary style="
            cursor:pointer;
            padding:16px;
            font-size:16px;
            font-weight:bold;
            background:#eff6ff;
            color:#1e3a8a;
            border-left:5px solid {color};
        ">
            {html.escape(title)} — Click to Expand
        </summary>

        <div style="
            padding:15px;
            background:white;
            text-align:center;
        ">

            <img
                src="data:image/png;base64,{image_data}"
                style="width:100%; max-width:950px; height:auto;"
            >

            <div style="
                max-width:850px;
                margin:15px auto;
                padding:16px;
                background:#f8fafc;
                border-radius:8px;
                color:#334155;
                font-size:14px;
                line-height:1.7;
                text-align:left;
            ">
                {explanation}
            </div>

        </div>

    </details>
    """))


# ============================================================
# HELPER — DECISION BOUNDARY
# ============================================================

def draw_boundary(ax):

    x = np.linspace(0, 10, 300)

    # Illustrative nonlinear decision boundary
    boundary = 5 + 0.35 * np.sin(x * 0.8)

    ax.plot(
        x,
        boundary,
        color=DARK,
        linewidth=2,
        label="Decision Boundary"
    )

    ax.fill_between(
        x,
        0,
        boundary,
        color=BLUE,
        alpha=0.10
    )

    ax.fill_between(
        x,
        boundary,
        10,
        color=RED,
        alpha=0.10
    )

    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)

    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")

    ax.grid(alpha=0.15)


# ============================================================
# VISUALIZATION 1 — ADVERSARIAL EVASION
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fig.suptitle(
    "Adversarial Evasion: Small Changes, Different Predictions",
    fontsize=16,
    fontweight="bold"
)

# Synthetic image representing a simple pattern
original = np.zeros((12, 12))

original[3:9, 5:7] = 0.85
original[3:5, 3:9] = 0.85

# Small perturbation
noise = rng.uniform(
    -0.12,
    0.12,
    original.shape
)

adversarial = np.clip(
    original + noise,
    0,
    1
)

axes[0].imshow(
    original,
    cmap="gray",
    vmin=0,
    vmax=1
)

axes[0].set_title(
    "Original Input\nIllustrative Prediction: Class A",
    color=GREEN
)

axes[1].imshow(
    adversarial,
    cmap="gray",
    vmin=0,
    vmax=1
)

axes[1].set_title(
    "Modified Input\nPossible Prediction: Class B",
    color=RED
)

for ax in axes:
    ax.axis("off")

plt.tight_layout()

show_collapsible_figure(
    "2.1 — Understanding Adversarial Evasion",
    fig,
    """
    Adversarial evasion attempts to change a model's
    prediction by modifying the input it receives.

    <br><br>

    In image classification, an attacker may introduce
    small changes to pixel values that are difficult
    for a person to notice.

    <br><br>

    The original and modified inputs may appear
    nearly identical while causing different
    model predictions.

    <br><br>

    This visualization uses a synthetic image
    and random perturbation to illustrate
    what a modified input can look like.

    <br><br>

    <b>Important:</b> No classifier is evaluated here.
    The prediction labels are hypothetical, and the
    random perturbation is not a demonstrated
    successful adversarial attack.
    """
)


# ============================================================
# VISUALIZATION 2 — DECISION BOUNDARY CROSSING
# ============================================================

fig, ax = plt.subplots(figsize=(9, 6))

draw_boundary(ax)

original_point = (4.5, 4.5)
adversarial_point = (4.6, 5.6)

ax.scatter(
    *original_point,
    color=GREEN,
    s=170,
    label="Original Input",
    zorder=5
)

ax.scatter(
    *adversarial_point,
    color=RED,
    marker="X",
    s=190,
    label="Adversarial Input",
    zorder=5
)

ax.annotate(
    "",
    xy=adversarial_point,
    xytext=original_point,
    arrowprops={
        "arrowstyle": "->",
        "color": ORANGE,
        "lw": 2.5
    }
)

ax.text(
    1.0, 2.0,
    "Class A Region",
    color=BLUE,
    fontsize=12,
    fontweight="bold"
)

ax.text(
    1.0, 8.0,
    "Class B Region",
    color=RED,
    fontsize=12,
    fontweight="bold"
)

ax.set_title(
    "Crossing a Model Decision Boundary",
    fontsize=16,
    fontweight="bold"
)

ax.legend()

plt.tight_layout()

show_collapsible_figure(
    "2.2 — Adversarial Inputs and Decision Boundaries",
    fig,
    """
    A decision boundary separates regions
    where a classifier predicts different classes.

    <br><br>

    An adversarial attack attempts to move an
    input across that boundary while keeping
    the modification sufficiently small.

    <br><br>

    The green point represents the original input.

    <br><br>

    The red point represents a modified input
    that lies on the opposite side of the boundary.

    <br><br>

    The model's predicted class changes because
    the input moved into another classification region.

    <br><br>

    <b>Key takeaway:</b> An attacker does not
    necessarily need to make an input look
    completely different to change a prediction.
    """
)


# ============================================================
# VISUALIZATION 3 — FGSM
# ============================================================

fig, ax = plt.subplots(figsize=(9, 6))

draw_boundary(ax)

start = np.array([4.5, 4.4])

# Illustrative gradient direction
gradient = np.array([0.4, 1.0])

epsilon = 1.2

fgsm_point = start + epsilon * np.sign(gradient)

ax.scatter(
    *start,
    color=GREEN,
    s=180,
    label="Original Input",
    zorder=5
)

ax.scatter(
    *fgsm_point,
    color=RED,
    marker="X",
    s=180,
    label="FGSM Example",
    zorder=5
)

ax.annotate(
    "",
    xy=fgsm_point,
    xytext=start,
    arrowprops={
        "arrowstyle": "->",
        "color": ORANGE,
        "lw": 3
    }
)

ax.text(
    5.8, 6.4,
    "Single Gradient-Sign Step",
    color=ORANGE,
    fontsize=11,
    fontweight="bold"
)

ax.set_title(
    "Fast Gradient Sign Method (FGSM)",
    fontsize=16,
    fontweight="bold"
)

ax.legend()

plt.tight_layout()

show_collapsible_figure(
    "2.3 — Fast Gradient Sign Method (FGSM)",
    fig,
    """
    FGSM is a gradient-based adversarial attack.

    <br><br>

    The attack calculates the gradient of the
    model's loss with respect to the input.

    <br><br>

    It then uses the sign of that gradient
    to determine the direction in which
    input values should be modified.

    <br><br>

    FGSM performs this modification in
    a single step.

    <br><br>

    The parameter <b>epsilon</b> controls
    the maximum perturbation applied
    to each input feature under an
    L-infinity constraint.

    <br><br>

    This diagram uses an illustrative gradient
    direction rather than a gradient computed
    from a trained model.

    <br><br>

    <b>Key takeaway:</b> FGSM is computationally
    efficient because it requires only one
    gradient-based update.
    """,
    color=ORANGE
)


# ============================================================
# VISUALIZATION 4 — PGD
# ============================================================

fig, ax = plt.subplots(figsize=(9, 6))

draw_boundary(ax)

start = np.array([4.4, 4.3])

# Illustrative iterative attack path
path = np.array([
    [4.4, 4.3],
    [4.5, 4.55],
    [4.7, 4.85],
    [4.8, 5.15],
    [5.0, 5.45],
    [5.2, 5.7]
])

ax.plot(
    path[:, 0],
    path[:, 1],
    color=ORANGE,
    linewidth=2.5,
    marker="o",
    markersize=7,
    label="Iterative Updates"
)

ax.scatter(
    *path[0],
    color=GREEN,
    s=170,
    label="Original Input",
    zorder=5
)

ax.scatter(
    *path[-1],
    color=RED,
    marker="X",
    s=180,
    label="Final Adversarial Input",
    zorder=5
)

# Conceptual perturbation constraint region
epsilon_box = plt.Rectangle(
    (start[0] - 1.0, start[1] - 1.0),
    2.0,
    2.0,
    fill=False,
    linestyle="--",
    linewidth=2,
    edgecolor=PURPLE,
    label="Illustrative Constraint Region"
)

ax.add_patch(epsilon_box)

ax.set_title(
    "Projected Gradient Descent (PGD)",
    fontsize=16,
    fontweight="bold"
)

ax.legend(loc="upper left")

plt.tight_layout()

show_collapsible_figure(
    "2.4 — Projected Gradient Descent (PGD)",
    fig,
    """
    PGD extends gradient-based attacks by
    performing multiple small updates.

    <br><br>

    Rather than modifying the input once,
    PGD repeatedly moves the input in
    a direction intended to increase
    the model's loss.

    <br><br>

    After each update, the modified input
    is projected back into the allowed
    perturbation region.

    <br><br>

    This prevents the attack from moving
    beyond the selected perturbation limit.

    <br><br>

    The orange path illustrates repeated
    updates toward an adversarial example.

    <br><br>

    <b>Important:</b> This is a conceptual path,
    not the output of a trained-model PGD attack.

    <br><br>

    <b>Key takeaway:</b> PGD generally requires
    more computation than FGSM because
    it performs multiple gradient updates.
    """,
    color=PURPLE
)


# ============================================================
# VISUALIZATION 5 — CARLINI & WAGNER
# ============================================================

# Illustrative optimization tradeoff
perturbation = np.linspace(0.01, 2.5, 200)

distortion_cost = perturbation**2

# Conceptual penalty for failing to achieve
# the attack objective
attack_penalty = 5 * np.exp(-2.5 * perturbation)

combined_objective = distortion_cost + attack_penalty

best_index = np.argmin(combined_objective)

fig, ax = plt.subplots(figsize=(10, 5.5))

ax.plot(
    perturbation,
    distortion_cost,
    color=BLUE,
    linewidth=2,
    label="Perturbation Cost"
)

ax.plot(
    perturbation,
    attack_penalty,
    color=RED,
    linewidth=2,
    label="Attack Objective Penalty"
)

ax.plot(
    perturbation,
    combined_objective,
    color=PURPLE,
    linewidth=3,
    label="Combined Objective"
)

ax.scatter(
    perturbation[best_index],
    combined_objective[best_index],
    color=GREEN,
    s=130,
    zorder=5,
    label="Illustrative Optimum"
)

ax.set_title(
    "Carlini & Wagner: Optimization Tradeoff",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Perturbation Magnitude")
ax.set_ylabel("Illustrative Objective Value")

ax.grid(alpha=0.2)
ax.legend()

plt.tight_layout()

show_collapsible_figure(
    "2.5 — Carlini & Wagner (C&W) Optimization",
    fig,
    """
    Carlini & Wagner attacks formulate adversarial
    example generation as an optimization problem.

    <br><br>

    The objective typically balances two goals:

    <br><br>

    <b>1.</b> Make the model produce an
    attacker-desired prediction.

    <br>

    <b>2.</b> Keep the modification to the
    original input as small as possible.

    <br><br>

    The blue curve represents a conceptual
    penalty for larger perturbations.

    <br><br>

    The red curve represents a conceptual
    penalty for not achieving the attack objective.

    <br><br>

    The purple curve combines these competing
    objectives.

    <br><br>

    These curves are illustrative and do not
    represent an actual C&W optimization run.

    <br><br>

    <b>Key takeaway:</b> C&W attacks explicitly
    optimize the relationship between attack
    effectiveness and perturbation size.
    """,
    color=PURPLE
)


# ============================================================
# VISUALIZATION 6 — FGSM VS PGD VS C&W
# ============================================================

fig, ax = plt.subplots(figsize=(11, 5.5))

methods = [
    "FGSM",
    "PGD",
    "C&W"
]

# Conceptual comparison, not measured values.
updates = [1, 5, 8]

colors = [
    ORANGE,
    PURPLE,
    BLUE
]

bars = ax.bar(
    methods,
    updates,
    color=colors,
    width=0.55
)

ax.set_ylim(0, 10)

ax.set_ylabel("Illustrative Optimization Steps")

ax.set_title(
    "Comparing Adversarial Attack Approaches",
    fontsize=16,
    fontweight="bold"
)

ax.grid(axis="y", alpha=0.2)

for bar, value in zip(bars, updates):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.2,
        str(value),
        ha="center",
        fontsize=12,
        fontweight="bold"
    )

plt.tight_layout()

show_collapsible_figure(
    "2.6 — Comparing FGSM, PGD, and C&W",
    fig,
    """
    These attacks differ primarily in
    how they generate adversarial inputs.

    <br><br>

    <b>FGSM:</b> Uses one gradient-sign
    update to create a modified input.

    <br><br>

    <b>PGD:</b> Uses multiple gradient-based
    updates with projection into an
    allowed perturbation region.

    <br><br>

    <b>C&W:</b> Uses optimization to balance
    the attack objective against the
    size of the perturbation.

    <br><br>

    The bars illustrate different optimization
    approaches and are not measured runtimes,
    fixed iteration counts, or attack success rates.

    <br><br>

    PGD and C&W can both use many optimization
    iterations. Their relative computational
    costs depend on implementation and settings.

    <br><br>

    <b>Key takeaway:</b> Different attacks
    make different tradeoffs between
    simplicity, computational effort,
    and perturbation constraints.
    """
)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print("Section 2 visualizations are ready.")
print("Six collapsible visualizations have been generated.")
print("Expand any visualization above to review its explanation.")

<details>

<summary><b>Section 2 Lab — Click to Expand</b></summary>

<br>

In this activity, you will perform a simple **Fast Gradient Sign Method (FGSM)** adversarial attack against an image classifier.

The purpose of this lab is not to turn you into an expert in adversarial mathematics.

Instead, you should understand the attack process:

    Clean Image
         ↓
    Model Prediction
         ↓
    Calculate Gradient
         ↓
    Create Perturbation
         ↓
    Adversarial Image
         ↓
    Model Prediction
         ↓
    Compare Performance

You will use the **MNIST handwritten-digit dataset** because it is small, easy to understand, and quick enough for repeated experiments.

First, you will train a small neural network.

Then you will measure its accuracy on clean test images.

Next, FGSM will use the model's gradients to modify the test images in a direction intended to increase prediction error.

Finally, you will compare:

    CLEAN ACCURACY

with:

    ADVERSARIAL ACCURACY

The attack should cause adversarial performance to be **worse than the clean baseline**.

You will also be able to adjust **epsilon**, which controls perturbation strength.

Try to focus on the relationship:

    Larger Epsilon
          ↓
    Stronger Perturbation
          ↓
    Usually Greater Attack Impact

Do not assume that every individual image must change classification.

The important result is the measurable effect across the evaluated dataset.

</details>

In [ ]:
# @title ▶ Part 1 — Load the Dataset { display-mode: "form" }

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

(X_train, y_train), (X_test, y_test) = (
    tf.keras.datasets.mnist.load_data()
)

# Normalize pixels from 0–255 to 0–1
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Add channel dimension
X_train = np.expand_dims(X_train, -1)
X_test = np.expand_dims(X_test, -1)

print("=" * 55)
print("MNIST DATASET")
print("=" * 55)

print(
    f"\nTraining Samples: "
    f"{len(X_train)}"
)

print(
    f"Testing Samples:  "
    f"{len(X_test)}"
)

print(
    f"Image Shape:      "
    f"{X_train.shape[1:]}"
)

print(
    f"Classes:          "
    f"{len(np.unique(y_train))}"
)

<details>

<summary><b>Part 1 Explanation — Click to Expand</b></summary>

<br>

MNIST contains grayscale images of handwritten digits from **0 through 9**.

Each image is:

    28 × 28 pixels

Pixel values originally range from:

    0 → 255

The code converts them to:

    0 → 1

This normalization makes the values easier for the neural network and for the adversarial attack to work with.

Nothing has been attacked yet.

At this stage, we are simply preparing clean training and testing information.

</details>

In [ ]:
# @title ▶ Part 2 — Train the Clean Model { display-mode: "form" }

model = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(28, 28, 1)
    ),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        128,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        10
    )
])

model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=["accuracy"]
)

model.fit(
    X_train,
    y_train,
    epochs=2,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

print("\n✓ Model training complete.")

<details>

<summary><b>Part 2 Explanation — Click to Expand</b></summary>

<br>

This code creates a small neural network for recognizing handwritten digits.

The model first flattens each 28 × 28 image into a sequence of values.

Those values pass through a hidden layer before reaching the final layer containing **10 outputs**, one for each possible digit.

Only two training epochs are used.

The goal is not to build the best possible MNIST classifier.

We want a model that:

    Trains Quickly

    Performs Well on Clean Data

    Can Be Attacked Quickly

This keeps the activity short enough to rerun while experimenting with different attack settings.

</details>

In [ ]:
# @title ▶ Part 3 — Measure Clean Performance { display-mode: "form" }

TEST_SAMPLES = 2000

X_eval = X_test[:TEST_SAMPLES]
y_eval = y_test[:TEST_SAMPLES]

clean_logits = model.predict(
    X_eval,
    verbose=0
)

clean_predictions = np.argmax(
    clean_logits,
    axis=1
)

clean_accuracy = np.mean(
    clean_predictions == y_eval
)

print("=" * 60)
print("CLEAN BASELINE")
print("=" * 60)

print(
    f"\nImages Evaluated: "
    f"{TEST_SAMPLES}"
)

print(
    f"Clean Accuracy:   "
    f"{clean_accuracy:.3f}"
)

print(
    f"Correct:          "
    f"{np.sum(clean_predictions == y_eval)}"
)

print(
    f"Incorrect:        "
    f"{np.sum(clean_predictions != y_eval)}"
)

<details>

<summary><b>Part 3 Explanation — Click to Expand</b></summary>

<br>

Before attacking the model, we establish its **clean baseline**.

The model is evaluated on 2,000 normal MNIST test images.

This gives us a trusted comparison point.

Our experiment will therefore follow:

    Clean Accuracy
          ↓
    Generate Attack
          ↓
    Adversarial Accuracy
          ↓
       Compare

The attack has not succeeded merely because adversarial images are generated.

We need to show that the model performs worse on those adversarial inputs than it does on the clean inputs.

Record the clean accuracy shown by your notebook.

</details>

In [ ]:
# @title ▶ Part 4 — Run the FGSM Attack { display-mode: "form" }

# ==========================================
# STUDENT-EDITABLE ATTACK SETTING
# ==========================================

EPSILON = 0.15

# ==========================================

X_tensor = tf.convert_to_tensor(
    X_eval
)

y_tensor = tf.convert_to_tensor(
    y_eval
)

with tf.GradientTape() as tape:

    tape.watch(X_tensor)

    predictions = model(
        X_tensor,
        training=False
    )

    loss = tf.keras.losses.sparse_categorical_crossentropy(
        y_tensor,
        predictions,
        from_logits=True
    )

gradient = tape.gradient(
    loss,
    X_tensor
)

signed_gradient = tf.sign(
    gradient
)

X_adversarial = (
    X_tensor
    + EPSILON * signed_gradient
)

# Keep valid pixel range
X_adversarial = tf.clip_by_value(
    X_adversarial,
    0.0,
    1.0
)

print("=" * 60)
print("FGSM ATTACK GENERATED")
print("=" * 60)

print(
    f"\nEpsilon:          "
    f"{EPSILON}"
)

print(
    f"Images Attacked:  "
    f"{len(X_adversarial)}"
)

average_change = tf.reduce_mean(
    tf.abs(
        X_adversarial
        - X_tensor
    )
).numpy()

print(
    f"Average Pixel Change: "
    f"{average_change:.4f}"
)

print(
    "\n✓ Adversarial examples created."
)

<details>

<summary><b>Part 4 Explanation — Click to Expand</b></summary>

<br>

This cell performs the FGSM attack.

The most important part is:

    gradient
        ↓
    sign of gradient
        ↓
    multiply by epsilon
        ↓
    add to original input

The gradient tells us how changing each input value affects the model's error.

FGSM takes the **sign** of that gradient to determine the direction each pixel should move.

Then:

    EPSILON = 0.15

controls how strongly the pixels are changed.

The operation:

    tf.clip_by_value(...)

ensures that pixel values remain within the valid normalized range:

    0 ≤ pixel ≤ 1

This creates adversarial versions of the same test images.

The next step determines whether those modifications actually affected the model.

</details>

In [ ]:
# @title ▶ Part 5 — Evaluate the Adversarial Images { display-mode: "form" }

adversarial_logits = model.predict(
    X_adversarial,
    verbose=0
)

adversarial_predictions = np.argmax(
    adversarial_logits,
    axis=1
)

adversarial_accuracy = np.mean(
    adversarial_predictions == y_eval
)

accuracy_drop = (
    clean_accuracy
    - adversarial_accuracy
)

changed_predictions = np.sum(
    clean_predictions
    != adversarial_predictions
)

print("=" * 65)
print("FGSM ATTACK RESULTS")
print("=" * 65)

print(
    f"\nClean Accuracy:       "
    f"{clean_accuracy:.3f}"
)

print(
    f"Adversarial Accuracy: "
    f"{adversarial_accuracy:.3f}"
)

print(
    f"Accuracy Drop:        "
    f"{accuracy_drop:.3f}"
)

print(
    f"\nPredictions Changed:  "
    f"{changed_predictions}/{TEST_SAMPLES}"
)

print("\nATTACK VALIDATION")

if adversarial_accuracy < clean_accuracy:

    print(
        "✓ Attack successfully degraded "
        "model performance."
    )

else:

    print(
        "⚠ Expected degradation was not observed."
    )

<details>

<summary><b>Part 5 Explanation — Click to Expand</b></summary>

<br>

This is the most important step in the experiment.

We now compare:

    CLEAN ACCURACY

against:

    ADVERSARIAL ACCURACY

The expected relationship is:

    Adversarial Accuracy
             <
       Clean Accuracy

If this occurs, we have measurable evidence that the adversarial manipulation affected model behavior.

The notebook also reports how many predictions changed.

Notice that these measurements answer slightly different questions.

**Accuracy drop** asks:

> How much worse did the model become?

**Predictions changed** asks:

> How often did the model produce a different decision after the perturbation?

This is stronger evidence than simply showing that the attack code generated modified images.

</details>

In [ ]:
# @title ▶ Part 6 — Compare Clean and Adversarial Images { display-mode: "form" }

sample_index = 0

# Find an example where the attack changed
# a previously correct prediction.
for i in range(TEST_SAMPLES):

    if (
        clean_predictions[i] == y_eval[i]
        and
        adversarial_predictions[i]
        != y_eval[i]
    ):

        sample_index = i
        break

plt.figure(figsize=(4, 4))

plt.imshow(
    X_eval[
        sample_index
    ].squeeze(),
    cmap="gray"
)

plt.title(
    f"Clean\n"
    f"True: {y_eval[sample_index]} | "
    f"Predicted: {clean_predictions[sample_index]}"
)

plt.axis("off")
plt.show()

plt.figure(figsize=(4, 4))

plt.imshow(
    X_adversarial[
        sample_index
    ].numpy().squeeze(),
    cmap="gray"
)

plt.title(
    f"Adversarial (ε={EPSILON})\n"
    f"True: {y_eval[sample_index]} | "
    f"Predicted: {adversarial_predictions[sample_index]}"
)

plt.axis("off")
plt.show()

print(
    "Prediction changed from "
    f"{clean_predictions[sample_index]} "
    "to "
    f"{adversarial_predictions[sample_index]}"
)

<details>

<summary><b>Part 6 Explanation — Click to Expand</b></summary>

<br>

The code searches for an image that the model originally classified correctly but misclassified after FGSM manipulation.

You then see:

    CLEAN IMAGE

followed by:

    ADVERSARIAL IMAGE

Compare the two visually.

Depending on epsilon, the adversarial image may still clearly resemble the original digit even though the model now produces a different prediction.

This demonstrates an important feature of adversarial examples:

> **A modification that appears relatively small to a human can still produce a meaningful change in model behavior.**

At larger epsilon values, however, the perturbation may become increasingly visible.

This illustrates the tradeoff between attack strength and perturbation size.

</details>

<details>

<summary><b>Part 7 — Student Experiment — Click to Expand</b></summary>

<br>

Return to **Part 4** and change:

    EPSILON

Try the following values:

    0.00

    0.05

    0.10

    0.15

    0.25

For each value, rerun:

    Part 4
       ↓
    Part 5
       ↓
    Part 6

Watch:

    Adversarial Accuracy

    Accuracy Drop

    Predictions Changed

    Appearance of the Image

At:

    EPSILON = 0.00

there is no adversarial perturbation, so adversarial performance should match the clean baseline.

As epsilon increases, the attack generally becomes stronger.

However, the modification also becomes larger.

Think about the security tradeoff:

    Small Perturbation
          ↓
    Harder to Notice
          ↓
    May Have Less Impact

versus:

    Large Perturbation
          ↓
    Greater Model Impact
          ↓
    Easier to Notice

There is no single epsilon value that represents every real-world adversarial attack.

The appropriate perturbation constraints depend on the system being attacked.

</details>

In [ ]:
# @title ▶ Part 8 — FGSM Lab Check { display-mode: "form" }

questions = [
    {
        "question":
        "1. What information did FGSM use to choose the direction of the perturbation?",
        "options": [
            "A. Model gradients",
            "B. Random labels",
            "C. Dataset duplicates",
            "D. File hashes"
        ],
        "answer": "A"
    },

    {
        "question":
        "2. What did epsilon control?",
        "options": [
            "A. Number of model classes",
            "B. Perturbation strength",
            "C. Training dataset size",
            "D. Number of hidden layers"
        ],
        "answer": "B"
    },

    {
        "question":
        "3. Which result provides evidence that the attack affected robustness?",
        "options": [
            "A. The code executed without errors",
            "B. Adversarial accuracy was lower than clean accuracy",
            "C. The dataset loaded",
            "D. The model contained 10 outputs"
        ],
        "answer": "B"
    },

    {
        "question":
        "4. What should happen when EPSILON = 0?",
        "options": [
            "A. The input should receive no FGSM perturbation",
            "B. Every prediction should fail",
            "C. The training data should be deleted",
            "D. The model should retrain itself"
        ],
        "answer": "A"
    },

    {
        "question":
        "5. Why might extremely large epsilon values be less realistic?",
        "options": [
            "A. The changes can become increasingly noticeable or violate realistic input constraints",
            "B. Large epsilon values remove the model",
            "C. Epsilon controls dataset labels",
            "D. Large epsilon prevents all computation"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 65)
print("FGSM MINI LAB CHECK")
print("=" * 65)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:

        print("✓ Correct!")
        score += 1

    else:

        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = (
    score / len(questions)
) * 100

print("\n" + "=" * 65)

print(
    f"Lab Score: "
    f"{score}/{len(questions)}"
)

print(
    f"Percentage: "
    f"{percentage:.1f}%"
)

if percentage >= 80:

    print(
        "\n✓ Mini lab complete."
    )

else:

    print(
        "\nReview the FGSM experiment and "
        "try the checkpoint again."
    )

print("=" * 65)

<details>

<summary><b>Section 2 — Practical Takeaways — Click to Expand</b></summary>

<br>

In this section, you moved from learning about model exploitation conceptually to performing your first model-level attack.

You established a clean baseline and then used FGSM to deliberately manipulate model inputs.

The experiment followed:

    CLEAN MODEL
         ↓
    CLEAN INPUTS
         ↓
    BASELINE ACCURACY
         ↓
    CALCULATE GRADIENTS
         ↓
    CREATE PERTURBATIONS
         ↓
    ADVERSARIAL INPUTS
         ↓
    MEASURE PERFORMANCE

The important result was not simply that the images changed.

The important result was:

    Adversarial Accuracy
             <
       Clean Accuracy

This provided measurable evidence that the attack reduced model robustness.

You also experimented with **epsilon** and observed that attack strength exists on a spectrum.

Increasing the perturbation may increase attack effectiveness, but it can also make the manipulation more noticeable or less realistic.

FGSM demonstrated a **single-step gradient attack**.

Later techniques can make this process more sophisticated. PGD repeatedly applies smaller gradient-based changes, while optimization-based attacks such as Carlini-Wagner attempt to balance successful misclassification against perturbation size.

The central lesson is:

> **A model that performs well on normal test data may still behave very differently when inputs are intentionally designed to exploit its decision boundaries.**

This is why security evaluation should consider both:

    CLEAN PERFORMANCE

and:

    ADVERSARIAL ROBUSTNESS

</details>

In [ ]:
# @title ▶ Section 2 Checkpoint — Adversarial Attacks & Evasion { display-mode: "form" }

questions = [
    {
        "question":
        "1. When does adversarial evasion normally target a model?",
        "options": [
            "A. During inference",
            "B. Only during data collection",
            "C. Only before training begins",
            "D. During dataset labeling"
        ],
        "answer": "A",
        "explanation":
        "Evasion attacks manipulate inputs presented to an already-trained model."
    },

    {
        "question":
        "2. What is an adversarial example?",
        "options": [
            "A. A corrupted model file",
            "B. An intentionally manipulated input designed to influence model behavior",
            "C. Any incorrectly labeled training sample",
            "D. A duplicate dataset"
        ],
        "answer": "B",
        "explanation":
        "Adversarial examples are crafted inputs intended to cause incorrect or attacker-desired model behavior."
    },

    {
        "question":
        "3. What is the main difference between poisoning and evasion?",
        "options": [
            "A. Poisoning targets training while evasion targets inference inputs",
            "B. They are identical",
            "C. Evasion always changes labels",
            "D. Poisoning only affects APIs"
        ],
        "answer": "A",
        "explanation":
        "Poisoning attempts to influence learning, while evasion manipulates inputs presented to the trained model."
    },

    {
        "question":
        "4. What does epsilon generally control in FGSM?",
        "options": [
            "A. Number of classes",
            "B. Perturbation strength",
            "C. Dataset size",
            "D. Number of training labels"
        ],
        "answer": "B",
        "explanation":
        "Epsilon controls how strongly FGSM changes the input."
    },

    {
        "question":
        "5. How does PGD differ conceptually from FGSM?",
        "options": [
            "A. PGD uses repeated perturbation steps",
            "B. PGD modifies training labels",
            "C. PGD cannot use gradients",
            "D. PGD only performs model extraction"
        ],
        "answer": "A",
        "explanation":
        "FGSM uses a single step, while PGD repeatedly applies smaller gradient-based changes."
    },

    {
        "question":
        "6. What does a targeted adversarial attack attempt to achieve?",
        "options": [
            "A. Any incorrect result",
            "B. A specific attacker-selected prediction",
            "C. Faster model training",
            "D. More training samples"
        ],
        "answer": "B",
        "explanation":
        "A targeted attack attempts to force the model toward a particular attacker-selected output."
    },

    {
        "question":
        "7. Why is clean accuracy alone insufficient for evaluating security?",
        "options": [
            "A. It does not show how the model behaves under adversarial manipulation",
            "B. Accuracy cannot measure models",
            "C. Clean data is always malicious",
            "D. Robustness and accuracy are always identical"
        ],
        "answer": "A",
        "explanation":
        "A model can perform well on normal inputs while being highly vulnerable to deliberately crafted inputs."
    }
]

score = 0

print("=" * 70)
print("SECTION 2 CHECKPOINT")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

    print(
        "Explanation:",
        q["explanation"]
    )

percentage = (
    score / len(questions)
) * 100

print("\n" + "=" * 70)
print("CHECKPOINT RESULTS")
print("=" * 70)

print(
    f"Score: {score}/{len(questions)}"
)

print(
    f"Percentage: {percentage:.1f}%"
)

if percentage >= 80:
    print(
        "\n✓ Checkpoint complete. "
        "Continue to the FGSM mini lab."
    )
else:
    print(
        "\nReview Section 2 before beginning "
        "the mini lab."
    )

---

<details>

<summary><b>Section 3 — Model Extraction / Model Stealing — Click to Expand</b></summary>

<br>

In Section 2, the attacker attempted to manipulate the predictions made by a trained model.

Model extraction has a different objective.

Instead of immediately trying to fool the target model, the attacker attempts to **learn how the target behaves**.

A **Model Extraction Attack**, sometimes called **Model Stealing**, occurs when an attacker interacts with a machine learning model and uses the information returned by that model to construct another model that approximates its behavior.

A simplified attack looks like:

    Attacker
       ↓
    Send Queries
       ↓
    Target Model
       ↓
    Observe Responses
       ↓
    Build Dataset
       ↓
    Train Surrogate
       ↓
    Approximate Target

The attacker may never directly access the target model's weights, architecture, source code, or original training dataset.

Instead, the attacker treats the model as a source of information.

<br>

<details>

<summary><b>3.1 — Why Would Someone Steal a Model?</b></summary>

<br>

Training a useful machine learning model can require significant resources.

An organization may invest in:

- Collecting and cleaning data
- Selecting useful features
- Developing model architectures
- Training and tuning models
- Evaluating performance
- Maintaining computational infrastructure

The resulting model can therefore represent valuable intellectual property.

An attacker who can approximate the target's functionality may gain some of that value without repeating the original development process.

However, intellectual property theft is not the only concern.

A stolen or surrogate model may also help an attacker understand how the target makes decisions.

This can support later attacks.

For example:

    MODEL EXTRACTION
           ↓
    SURROGATE MODEL
           ↓
    STUDY DECISION BEHAVIOR
           ↓
    CREATE ADVERSARIAL INPUTS
           ↓
    ATTACK ORIGINAL TARGET

Model extraction can therefore be both an attack by itself and one stage of a larger attack chain.

</details>

<br>

<details>

<summary><b>3.2 — Black-Box Model Extraction</b></summary>

<br>

Model extraction is particularly interesting because it can be performed in a **black-box environment**.

Recall that a black-box attacker may have:

    Input Access ✓

    Output Access ✓

but not:

    Model Weights ✗

    Architecture ✗

    Gradients ✗

    Training Dataset ✗

Suppose an attacker can access a prediction API.

The attacker submits:

    Query 1

and receives:

    Class A

The attacker then submits:

    Query 2

and receives:

    Class B

This process continues:

    Query 1 → Class A
    Query 2 → Class B
    Query 3 → Class A
    Query 4 → Class C
       ...
    Query N → Class B

The attacker can store these interactions.

Eventually, they have created a new dataset:

    QUERY                  TARGET RESPONSE

    x₁                     Class A
    x₂                     Class B
    x₃                     Class A
    x₄                     Class C
    ...                    ...
    xₙ                     Class B

The attacker can then train another machine learning model using this collected information.

The target model has effectively provided the labels for the attacker's dataset.

</details>

<br>

<details>

<summary><b>3.3 — The Target Model as a Labeling Oracle</b></summary>

<br>

A useful way to understand model extraction is to think of the target model as a **labeling oracle**.

Normally, supervised machine learning requires labeled information:

    Input + Correct Label
              ↓
           Training
              ↓
             Model

An attacker may not possess the target's original labeled dataset.

However, if the attacker can query the target, the target itself can provide labels.

For example:

    Unknown Sample
          ↓
    Target Model
          ↓
       "Class 2"

The attacker stores:

    Sample → Class 2

After enough queries:

    Query Dataset
          +
    Target Predictions
          ↓
    Surrogate Training

This creates an important security insight:

> **Every prediction returned by a model may provide information that helps another model learn its behavior.**

One prediction usually reveals very little.

Thousands or millions of carefully selected predictions may reveal considerably more.

</details>

<br>

<details>

<summary><b>3.4 — Surrogate Models</b></summary>

<br>

The model trained by the attacker is commonly called a **surrogate model**, **substitute model**, or **extracted model**.

The surrogate does not necessarily need to be identical to the target.

For example:

    TARGET MODEL
    Neural Network

while:

    SURROGATE MODEL
    Random Forest

The attacker may not even know the architecture of the target.

What matters is whether the surrogate can reproduce the target's **observable behavior** sufficiently well.

This gives us an important distinction.

The attacker may not recover:

    Exact Architecture

    Exact Weights

    Exact Training Dataset

Instead, the attacker may recover:

    Similar Input → Output Behavior

This is sometimes described in terms of **functional similarity** or **fidelity**.

</details>

<br>

<details>

<summary><b>3.5 — Accuracy vs. Fidelity</b></summary>

<br>

When evaluating model extraction, students should understand the difference between **accuracy** and **fidelity**.

Accuracy asks:

> How often is the surrogate correct compared with the true labels?

Fidelity asks:

> How often does the surrogate agree with the target model?

Suppose:

    True Label = Class 1

but:

    Target Prediction = Class 2

and:

    Surrogate Prediction = Class 2

The surrogate is incorrect according to the true label.

However, it perfectly copied the target's behavior for that sample.

Therefore:

    Accuracy → compares to TRUE LABELS

while:

    Fidelity → compares to TARGET OUTPUTS

For model extraction, fidelity is especially important because the attacker's goal is often to reproduce what the target does.

A successful extracted model should behave similarly to the target across inputs the attacker cares about.

</details>

<br>

<details>

<summary><b>3.6 — What Information Does the API Return?</b></summary>

<br>

The information returned by the target can influence how useful each query is.

Consider two APIs.

### API A

Returns only:

    Class 3

### API B

Returns:

    Class 0: 0.02
    Class 1: 0.04
    Class 2: 0.11
    Class 3: 0.79
    Class 4: 0.04

API B provides considerably more information.

Instead of learning only the final decision, the attacker learns something about the model's relative confidence across classes.

This can provide additional information about the target's decision behavior.

The principle from Section 1 therefore becomes important again:

> **The more information a model exposes, the more information each query may provide to an attacker.**

This does not mean probability or confidence information should never be exposed.

Legitimate applications may require it.

However, defenders should determine whether detailed outputs are actually necessary.

</details>

<br>

<details>

<summary><b>3.7 — Query Selection</b></summary>

<br>

An attacker also needs to decide **what to query**.

Simply generating random inputs may produce poor information.

For example, if most queries fall into the same region of the target's decision space, the attacker may learn very little about other regions.

More useful queries may explore:

    Different Classes

    Different Feature Values

    Decision Boundaries

    Uncertain Regions

    Rare Inputs

This leads to an important concept:

> **The value of a model extraction attack depends not only on the number of queries, but also on the information contained in those queries.**

An attacker may therefore attempt to strategically select inputs that reveal more about the target.

In our introductory lab, we will keep query selection simple so that you can focus on the extraction process itself.

</details>

<br>

<details>

<summary><b>3.8 — Query Budget</b></summary>

<br>

Attackers may not have unlimited access to a model.

An API may enforce:

    Rate Limits

    Query Limits

    Usage Costs

    Authentication

    Monitoring

Therefore, model extraction can involve a **query budget**.

For example:

    50 Queries

    100 Queries

    500 Queries

    1,000 Queries

One useful experiment is to ask:

> **How does surrogate fidelity change as the attacker receives more target predictions?**

Conceptually, we might observe:

    Small Query Budget
           ↓
    Limited Target Knowledge
           ↓
    Lower Fidelity

versus:

    Larger Query Budget
           ↓
    More Target Behavior Observed
           ↓
    Potentially Higher Fidelity

This relationship is not guaranteed to increase perfectly with every additional query, but query access is an important resource for an extraction attacker.

</details>

<br>

<details>

<summary><b>3.9 — Model Extraction as Reconnaissance</b></summary>

<br>

Even an imperfect surrogate can be valuable.

Suppose an attacker cannot calculate gradients from the original black-box model.

After extraction:

    Original Target
        BLACK BOX

but:

    Surrogate
        WHITE BOX

The attacker owns the surrogate.

Therefore, they may be able to inspect:

    Architecture

    Parameters

    Gradients

    Decision Behavior

The attacker can experiment against the surrogate without continuously interacting with the real target.

This creates the possibility of an attack chain:

    BLACK-BOX TARGET
           ↓
       Query Target
           ↓
      Collect Outputs
           ↓
     Train Surrogate
           ↓
    WHITE-BOX SURROGATE
           ↓
    Develop Attack Locally
           ↓
     Test Against Target

This concept will become especially important when you study **transfer attacks** later in the module.

</details>

<br>

<details>

<summary><b>3.10 — Defensive Considerations</b></summary>

<br>

There is no single defense that completely eliminates model extraction risk while still allowing legitimate users to interact with a model.

Instead, defenders can attempt to reduce how much information attackers can collect or detect suspicious extraction behavior.

Possible controls include:

- Authentication and authorization
- Rate limiting
- Query budgets
- Usage monitoring
- Output minimization
- Restricting unnecessary confidence information
- Detecting unusual query patterns

For example, a normal user may make:

    5–20 predictions

while an automated extraction attempt might make:

    Thousands of systematically varied predictions

The exact pattern depends on the application, so a high number of queries does not automatically prove an attack.

This is why monitoring should consider context.

A security control should not simply ask:

> "Did this user make many requests?"

It may also consider:

> "Are these requests systematically exploring the model's decision behavior?"

</details>

<br>

<details>

<summary><b>3.11 — Measuring Extraction Success</b></summary>

<br>

Just as we measured adversarial attacks in Section 2, model extraction requires evidence.

The attack has not succeeded simply because:

    Surrogate Model Trained Successfully

Instead, we need to compare the surrogate with the target.

One useful measurement is:

    Fidelity
       =
    Percentage of Inputs Where
    Surrogate Prediction
       ==
    Target Prediction

For example:

    Target Predictions:
    0 1 2 1 0 2 1 0

    Surrogate Predictions:
    0 1 2 1 0 1 1 0

The models agree on:

    7 of 8 predictions

giving:

    Fidelity = 87.5%

Higher fidelity indicates that the surrogate is reproducing more of the target's observable behavior.

In the mini lab, you will measure both:

    Surrogate Accuracy

and:

    Surrogate Fidelity

so that you can see why the two measurements are not identical concepts.

</details>

<br>

<details>

<summary><b>Section 3 — Key Takeaways</b></summary>

<br>

Model extraction attempts to approximate the behavior or functionality of a target machine learning model.

An attacker may perform extraction without directly accessing:

    Model Weights

    Architecture

    Gradients

    Original Training Dataset

Instead, the attacker can query the target and collect its outputs.

Those interactions can be converted into a new training dataset:

    Query Inputs
          +
    Target Predictions
          ↓
    Surrogate Model

The extracted model does not need to be internally identical to the target.

The important measurement is often **fidelity**, which describes how frequently the surrogate reproduces the target's predictions.

Model extraction can also support later attacks because the attacker gains a model they control and can analyze locally.

The central lesson is:

> **A prediction interface does more than provide a service. Every response may also reveal a small amount of information about the model behind that service.**

</details>

</details>

In [ ]:
# @title Section 3 — Model Extraction / Model Stealing Visualizations { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from IPython.display import display, HTML
from io import BytesIO
import base64
import html

# ============================================================
# CONFIGURATION
# ============================================================

rng = np.random.default_rng(42)

BLUE = "#2563EB"
RED = "#DC2626"
GREEN = "#16A34A"
ORANGE = "#EA580C"
PURPLE = "#7C3AED"
GRAY = "#64748B"
DARK = "#0F172A"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# ============================================================
# COLLAPSIBLE VISUALIZATION HELPER
# ============================================================

def show_collapsible_figure(title, fig, explanation, color=BLUE):

    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=135,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    buffer.close()

    display(HTML(f"""
    <details style="
        margin:12px 0;
        border:1px solid #cbd5e1;
        border-radius:10px;
        overflow:hidden;
    ">

        <summary style="
            cursor:pointer;
            padding:16px;
            font-size:16px;
            font-weight:bold;
            background:#eff6ff;
            color:#1e3a8a;
            border-left:5px solid {color};
        ">
            {html.escape(title)} — Click to Expand
        </summary>

        <div style="
            padding:15px;
            background:white;
            text-align:center;
        ">

            <img
                src="data:image/png;base64,{encoded}"
                style="width:100%;max-width:950px;height:auto;"
            >

            <div style="
                max-width:850px;
                margin:15px auto;
                padding:16px;
                background:#f8fafc;
                border-radius:8px;
                color:#334155;
                font-size:14px;
                line-height:1.7;
                text-align:left;
            ">
                {explanation}
            </div>

        </div>
    </details>
    """))


# ============================================================
# DIAGRAM HELPERS
# ============================================================

def add_card(ax, x, y, w, h, title, subtitle="",
             accent=BLUE, background="#EFF6FF"):

    rect = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        facecolor=background,
        edgecolor=accent,
        linewidth=1.7
    )

    ax.add_patch(rect)

    ax.text(
        x + w / 2,
        y + h * (0.65 if subtitle else 0.5),
        title,
        ha="center",
        va="center",
        fontsize=10.5,
        fontweight="bold",
        color=DARK
    )

    if subtitle:
        ax.text(
            x + w / 2,
            y + h * 0.28,
            subtitle,
            ha="center",
            va="center",
            fontsize=9,
            color=GRAY
        )


def add_arrow(ax, start, end, color=GRAY):

    ax.add_patch(
        FancyArrowPatch(
            start, end,
            arrowstyle="-|>",
            mutation_scale=17,
            linewidth=1.8,
            color=color
        )
    )


# ============================================================
# SYNTHETIC MODEL SETUP
# ============================================================

# A local target model simulates a prediction API.
# A surrogate will attempt to approximate its predictions.

X, y = make_moons(
    n_samples=1400,
    noise=0.20,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

target_model = RandomForestClassifier(
    n_estimators=80,
    max_depth=7,
    random_state=42
)

target_model.fit(X_train, y_train)

# Separate synthetic query inputs
query_pool = rng.uniform(
    low=[-1.5, -1.3],
    high=[2.5, 1.8],
    size=(800, 2)
)

# ============================================================
# VISUALIZATION 1 — MODEL EXTRACTION WORKFLOW
# Sections 3.1–3.4
# ============================================================

fig, ax = plt.subplots(figsize=(10, 9))

ax.set_xlim(0, 10)
ax.set_ylim(0, 11)
ax.axis("off")

ax.set_title(
    "Black-Box Model Extraction Workflow",
    fontsize=17,
    fontweight="bold",
    pad=15
)

steps = [
    ("Attacker", "Chooses inputs to submit", RED),
    ("Prediction Queries", "Inputs sent to target model", ORANGE),
    ("Target Model", "Returns predictions or scores", BLUE),
    ("Query-Response Dataset", "Inputs paired with target outputs", PURPLE),
    ("Surrogate Training", "Learns from observed behavior", PURPLE),
    ("Extracted Surrogate", "Approximates target predictions", GREEN)
]

for i, (title, subtitle, accent) in enumerate(steps):

    y_pos = 9.4 - i * 1.7

    add_card(
        ax,
        1.2,
        y_pos,
        7.6,
        1.2,
        title,
        subtitle,
        accent,
        "#F8FAFC"
    )

    if i < len(steps) - 1:

        add_arrow(
            ax,
            (5, y_pos - 0.05),
            (5, y_pos - 0.45)
        )

plt.tight_layout()

show_collapsible_figure(
    "3.1–3.4 — How Model Extraction Works",
    fig,
    """
    A model extraction attack attempts to reproduce
    the behavior of a target machine learning model.

    <br><br>

    The attacker first submits inputs to the target
    model and observes the returned predictions.

    <br><br>

    These observations become a new dataset
    containing input-output pairs.

    <br><br>

    The attacker can then train a
    <b>surrogate model</b> using those pairs.

    <br><br>

    The surrogate does not necessarily have
    the same architecture, weights, or training
    data as the target model.

    <br><br>

    <b>Key takeaway:</b> An attacker may approximate
    a model's functionality without directly
    accessing its internal parameters.
    """
)


# ============================================================
# VISUALIZATION 2 — API OUTPUT INFORMATION
# Section 3.6
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fig.suptitle(
    "How API Responses Reveal Model Information",
    fontsize=16,
    fontweight="bold"
)

# Example probability distribution
classes = ["Class A", "Class B", "Class C"]

probabilities = [0.08, 0.82, 0.10]

# LEFT — HARD LABEL

axes[0].bar(
    classes,
    [0, 1, 0],
    color=[GRAY, BLUE, GRAY]
)

axes[0].set_ylim(0, 1.15)

axes[0].set_title(
    "Label-Only Response",
    fontsize=13
)

axes[0].set_ylabel("Returned Class Indicator")

axes[0].text(
    1, 1.04,
    "Prediction: Class B",
    ha="center",
    fontweight="bold",
    color=BLUE
)

# RIGHT — CONFIDENCE SCORES

bars = axes[1].bar(
    classes,
    probabilities,
    color=[GRAY, BLUE, GRAY]
)

axes[1].set_ylim(0, 1.15)

axes[1].set_title(
    "Confidence-Score Response",
    fontsize=13
)

axes[1].set_ylabel("Returned Probability")

for bar, value in zip(bars, probabilities):

    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.04,
        f"{value:.2f}",
        ha="center",
        fontweight="bold"
    )

plt.tight_layout()

show_collapsible_figure(
    "3.6 — Labels vs. Confidence Scores",
    fig,
    """
    Prediction APIs may return different amounts
    of information.

    <br><br>

    A <b>label-only API</b> might return
    only the predicted class.

    <br><br>

    A <b>confidence-score API</b> may return
    a probability or score for multiple classes.

    <br><br>

    Confidence scores can reveal additional
    information about the model's behavior,
    including how its outputs change when
    an input is modified.

    <br><br>

    This additional information may make
    some extraction techniques more efficient.

    <br><br>

    <b>Important:</b> Returning labels only
    does not eliminate model extraction risk.
    """
)


# ============================================================
# VISUALIZATION 3 — QUERY BUDGET AND FIDELITY
# Sections 3.7–3.8
# ============================================================

query_budgets = [
    10, 25, 50, 100, 200, 400, 700
]

fidelity_scores = []

# Fixed holdout inputs to measure agreement
evaluation_inputs = rng.uniform(
    low=[-1.5, -1.3],
    high=[2.5, 1.8],
    size=(1000, 2)
)

target_predictions = target_model.predict(
    evaluation_inputs
)

for budget in query_budgets:

    selected_queries = query_pool[:budget]

    observed_labels = target_model.predict(
        selected_queries
    )

    surrogate = DecisionTreeClassifier(
        max_depth=7,
        random_state=42
    )

    surrogate.fit(
        selected_queries,
        observed_labels
    )

    surrogate_predictions = surrogate.predict(
        evaluation_inputs
    )

    fidelity = np.mean(
        surrogate_predictions == target_predictions
    )

    fidelity_scores.append(fidelity)

fig, ax = plt.subplots(figsize=(10, 5.5))

ax.plot(
    query_budgets,
    fidelity_scores,
    marker="o",
    linewidth=2.5,
    markersize=8,
    color=PURPLE
)

ax.set_ylim(0, 1.05)

ax.set_xlabel("Number of Queries to Target Model")
ax.set_ylabel("Surrogate Fidelity")

ax.set_title(
    "Query Budget vs. Surrogate Fidelity",
    fontsize=16,
    fontweight="bold"
)

ax.grid(alpha=0.2)

for x, score in zip(query_budgets, fidelity_scores):

    ax.annotate(
        f"{score:.2f}",
        (x, score),
        xytext=(0, 9),
        textcoords="offset points",
        ha="center",
        fontsize=9
    )

plt.tight_layout()

show_collapsible_figure(
    "3.7–3.8 — Query Selection and Query Budget",
    fig,
    """
    A query budget limits how many times
    an attacker can interact with a target model.

    <br><br>

    In this demonstration, a surrogate model
    is trained using increasing numbers of
    target-model predictions.

    <br><br>

    The graph measures <b>fidelity</b>,
    meaning the proportion of evaluation
    inputs where the surrogate agrees
    with the target.

    <br><br>

    More queries may provide more information,
    but improvement is not guaranteed.

    <br><br>

    Query selection, model complexity,
    and the distribution of query inputs
    also influence extraction quality.

    <br><br>

    <b>Key takeaway:</b> The quantity and
    quality of observed target responses
    influence how closely a surrogate
    can reproduce target behavior.
    """,
    color=PURPLE
)


# ============================================================
# VISUALIZATION 4 — TARGET VS SURROGATE
# Sections 3.4 and 3.11
# ============================================================

# Train surrogate using 400 target predictions
selected_queries = query_pool[:400]

observed_labels = target_model.predict(
    selected_queries
)

surrogate_model = DecisionTreeClassifier(
    max_depth=7,
    random_state=42
)

surrogate_model.fit(
    selected_queries,
    observed_labels
)

x_min, x_max = -1.5, 2.5
y_min, y_max = -1.3, 1.8

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 220),
    np.linspace(y_min, y_max, 220)
)

grid_points = np.c_[
    xx.ravel(),
    yy.ravel()
]

target_regions = target_model.predict(
    grid_points
).reshape(xx.shape)

surrogate_regions = surrogate_model.predict(
    grid_points
).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

fig.suptitle(
    "Target Model vs. Extracted Surrogate",
    fontsize=16,
    fontweight="bold"
)

cmap = ListedColormap([
    "#BFDBFE",
    "#FECACA"
])

for ax, regions, title in zip(
    axes,
    [target_regions, surrogate_regions],
    ["Target Model", "Surrogate Model"]
):

    ax.contourf(
        xx,
        yy,
        regions,
        cmap=cmap,
        alpha=0.8
    )

    ax.contour(
        xx,
        yy,
        regions,
        levels=[0.5],
        colors=DARK,
        linewidths=1.5
    )

    ax.set_title(title)

    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

plt.tight_layout()

show_collapsible_figure(
    "3.4 & 3.11 — Target and Surrogate Decision Boundaries",
    fig,
    """
    A successful model extraction attack
    attempts to reproduce the target
    model's decision behavior.

    <br><br>

    The left graph shows the decision
    regions of a synthetic target model.

    <br><br>

    The right graph shows a surrogate
    trained using predictions obtained
    from the target.

    <br><br>

    Similar decision regions indicate
    that the surrogate has learned
    some of the target's behavior.

    <br><br>

    Differences between the regions
    indicate places where the two
    models disagree.

    <br><br>

    <b>Key takeaway:</b> Extraction does
    not require recovering the target's
    original parameters. Reproducing
    its predictions may be sufficient
    for the attacker's objective.
    """
)


# ============================================================
# VISUALIZATION 5 — ACCURACY VS FIDELITY
# Sections 3.5 and 3.11
# ============================================================

target_test_predictions = target_model.predict(
    X_test
)

surrogate_test_predictions = surrogate_model.predict(
    X_test
)

target_accuracy = accuracy_score(
    y_test,
    target_test_predictions
)

surrogate_accuracy = accuracy_score(
    y_test,
    surrogate_test_predictions
)

surrogate_fidelity = np.mean(
    surrogate_test_predictions == target_test_predictions
)

fig, ax = plt.subplots(figsize=(10, 5.5))

metrics = [
    "Target\nAccuracy",
    "Surrogate\nAccuracy",
    "Surrogate\nFidelity"
]

values = [
    target_accuracy,
    surrogate_accuracy,
    surrogate_fidelity
]

colors = [
    BLUE,
    GREEN,
    PURPLE
]

bars = ax.bar(
    metrics,
    values,
    color=colors,
    width=0.58
)

ax.set_ylim(0, 1.1)

ax.set_ylabel("Score")

ax.set_title(
    "Model Accuracy vs. Extraction Fidelity",
    fontsize=16,
    fontweight="bold"
)

ax.grid(axis="y", alpha=0.2)

for bar, value in zip(bars, values):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.025,
        f"{value:.3f}",
        ha="center",
        fontweight="bold",
        fontsize=11
    )

plt.tight_layout()

show_collapsible_figure(
    "3.5 & 3.11 — Understanding Accuracy and Fidelity",
    fig,
    """
    Accuracy and fidelity measure
    different properties.

    <br><br>

    <b>Accuracy</b> measures how often
    a model predicts the correct
    ground-truth class.

    <br><br>

    <b>Fidelity</b> measures how often
    the surrogate produces the same
    prediction as the target model.

    <br><br>

    A surrogate can have high fidelity
    even if the target model sometimes
    makes incorrect predictions.

    <br><br>

    This happens because fidelity
    rewards agreement with the target,
    including agreement on mistakes.

    <br><br>

    The values shown here are calculated
    from the synthetic models trained
    in this code cell.

    <br><br>

    <b>Key takeaway:</b> A stolen model
    is often evaluated by how closely
    it reproduces the target's behavior,
    not only by its predictive accuracy.
    """
)


# ============================================================
# VISUALIZATION 6 — MODEL EXTRACTION DEFENSES
# Sections 3.9–3.10
# ============================================================

fig, ax = plt.subplots(figsize=(11, 8))

ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis("off")

ax.set_title(
    "Layered Defenses Against Model Extraction",
    fontsize=17,
    fontweight="bold",
    pad=15
)

defense_steps = [
    (
        "Authentication and Access Control",
        "Identify and authorize API users",
        BLUE
    ),
    (
        "Rate Limits and Query Budgets",
        "Restrict excessive automated querying",
        ORANGE
    ),
    (
        "Output Exposure Controls",
        "Return only information needed by clients",
        PURPLE
    ),
    (
        "Query Monitoring",
        "Identify unusual request patterns",
        BLUE
    ),
    (
        "Investigation and Response",
        "Review suspicious activity and apply controls",
        RED
    )
]

for i, (title, subtitle, accent) in enumerate(defense_steps):

    y = 8.0 - i * 1.7

    add_card(
        ax,
        1.0,
        y,
        8.0,
        1.2,
        title,
        subtitle,
        accent,
        "#F8FAFC"
    )

    if i < len(defense_steps) - 1:

        add_arrow(
            ax,
            (5, y - 0.05),
            (5, y - 0.45)
        )

plt.tight_layout()

show_collapsible_figure(
    "3.9–3.10 — Model Extraction Reconnaissance and Defenses",
    fig,
    """
    Model extraction can also serve as
    reconnaissance for later attacks.

    <br><br>

    A surrogate may help an attacker
    study decision boundaries, generate
    adversarial examples, or investigate
    weaknesses without repeatedly
    accessing the original target.

    <br><br>

    Defenders can reduce extraction risk
    through several controls.

    <br><br>

    <b>Authentication</b> identifies users
    accessing the prediction interface.

    <br><br>

    <b>Rate limits</b> and query budgets
    restrict excessive requests.

    <br><br>

    <b>Output controls</b> reduce unnecessary
    exposure of detailed model information.

    <br><br>

    <b>Query monitoring</b> can identify
    suspicious patterns that warrant
    investigation.

    <br><br>

    <b>Important:</b> These defenses
    reduce risk but do not guarantee
    that extraction becomes impossible.
    Legitimate users may also generate
    large numbers of queries.
    """,
    color=RED
)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print("Section 3 visualizations are ready.")
print("Six collapsible visualizations generated.")
print("All model experiments used synthetic data.")

<details>

<summary><b>Section 3 Lab — Click to Expand</b></summary>

<br>

In this activity, you will simulate a **black-box model extraction attack**.

You will begin with a target model.

For the purpose of the experiment, our notebook must create that model. However, once it has been created, the extraction process will treat it like a remote black-box service.

The attacker will not copy:

    Target Parameters

    Target Architecture

    Target Training Labels

Instead, the attacker will perform:

    Query Input
          ↓
    Target Model
          ↓
    Target Prediction
          ↓
    Store Response

After collecting enough responses, those target predictions become the labels for a new dataset.

You will then train a **surrogate model** using those collected responses.

The experiment follows:

    TARGET MODEL
          ↓
    QUERY TARGET
          ↓
    COLLECT PREDICTIONS
          ↓
    CREATE EXTRACTION DATASET
          ↓
    TRAIN SURROGATE
          ↓
    TEST ON UNSEEN DATA
          ↓
    MEASURE FIDELITY

The target and surrogate will intentionally use different model types.

This demonstrates that model extraction does not necessarily mean recovering the exact internal structure of the target.

The important question is:

> **Can the surrogate reproduce the target's observable behavior?**

</details>

In [ ]:
# @title ▶ Part 1 — Create the Black-Box Target { display-mode: "form" }

import numpy as np

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score

np.random.seed(42)

# Load a small image-classification dataset
digits = load_digits()

X = digits.data
y = digits.target

# Separate target training data from data
# available for querying/evaluation
X_target_train, X_pool, y_target_train, y_pool = (
    train_test_split(
        X,
        y,
        test_size=0.55,
        random_state=42,
        stratify=y
    )
)

# Separate query pool and final evaluation data
X_query_pool, X_eval, y_query_pool, y_eval = (
    train_test_split(
        X_pool,
        y_pool,
        test_size=0.35,
        random_state=42,
        stratify=y_pool
    )
)

# Target model
target_model = RandomForestClassifier(
    n_estimators=150,
    random_state=42
)

target_model.fit(
    X_target_train,
    y_target_train
)

target_predictions = target_model.predict(
    X_eval
)

target_accuracy = accuracy_score(
    y_eval,
    target_predictions
)

print("=" * 60)
print("BLACK-BOX TARGET MODEL")
print("=" * 60)

print(
    f"\nTarget Training Samples: "
    f"{len(X_target_train)}"
)

print(
    f"Available Query Samples: "
    f"{len(X_query_pool)}"
)

print(
    f"Final Evaluation Samples: "
    f"{len(X_eval)}"
)

print(
    f"\nTarget Accuracy: "
    f"{target_accuracy:.3f}"
)

print(
    "\nFrom this point forward, imagine that "
    "the target can only be accessed through predict()."
)

<details>

<summary><b>Part 1 Explanation — Click to Expand</b></summary>

<br>

The notebook creates a **Random Forest** classifier that represents the organization's target model.

In a real extraction scenario, the attacker would not normally be the person who trained this model.

We create it ourselves only so that the lab can simulate a target prediction service.

From this point forward, think of:

    target_model.predict(...)

as if it were:

    Send Request to API
            ↓
    Receive Prediction

The attacker will not copy the Random Forest's internal parameters.

Instead, the attacker will learn only from the predictions returned by the target.

</details>

In [ ]:
# @title ▶ Part 2 — Query the Target Model { display-mode: "form" }

# ==========================================
# STUDENT-EDITABLE QUERY BUDGET
# ==========================================

QUERY_BUDGET = 300

# ==========================================

QUERY_BUDGET = min(
    QUERY_BUDGET,
    len(X_query_pool)
)

rng = np.random.default_rng(42)

query_indices = rng.choice(
    len(X_query_pool),
    size=QUERY_BUDGET,
    replace=False
)

X_stolen = X_query_pool[
    query_indices
]

# The attacker asks the TARGET for labels.
y_stolen = target_model.predict(
    X_stolen
)

print("=" * 60)
print("TARGET QUERY RESULTS")
print("=" * 60)

print(
    f"\nQuery Budget: "
    f"{QUERY_BUDGET}"
)

print(
    f"Responses Collected: "
    f"{len(y_stolen)}"
)

print("\nExample Target Responses:")

for i in range(
    min(10, len(y_stolen))
):

    print(
        f"Query {i + 1:2d} "
        f"→ Target Prediction: "
        f"{y_stolen[i]}"
    )

print(
    "\n✓ Extraction dataset created from "
    "target predictions."
)

<details>

<summary><b>Part 2 Explanation — Click to Expand</b></summary>

<br>

This cell represents the actual **query stage** of the extraction attack.

The attacker selects samples and submits them to the target.

The target returns predictions.

The attacker stores:

    Input
      +
    Target Prediction

creating:

    X_stolen

and:

    y_stolen

Notice something important.

The attacker did **not** use the true labels from the original dataset to create `y_stolen`.

Instead:

    y_stolen = target_model.predict(...)

The target model itself generated the labels.

The attacker is therefore learning:

> **How does the target classify these inputs?**

rather than:

> **What is the objectively correct class?**

That distinction is central to model extraction.

</details>

In [ ]:
# @title ▶ Part 3 — Train the Surrogate Model { display-mode: "form" }

# Use a DIFFERENT model type from the target.
surrogate_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

surrogate_model.fit(
    X_stolen,
    y_stolen
)

print("=" * 60)
print("SURROGATE TRAINING")
print("=" * 60)

print(
    f"\nSurrogate Model: "
    f"Logistic Regression"
)

print(
    f"Extraction Samples: "
    f"{len(X_stolen)}"
)

print(
    "\n✓ Surrogate trained using "
    "target-generated labels."
)

<details>

<summary><b>Part 3 Explanation — Click to Expand</b></summary>

<br>

The attacker now trains a **Logistic Regression** model using the target-generated labels.

This is deliberately different from the target:

    TARGET
    Random Forest

    SURROGATE
    Logistic Regression

The attacker is not attempting to copy the target's exact internal structure.

Instead, the surrogate is learning a relationship between:

    Query Inputs
          ↓
    Target Decisions

If the surrogate begins producing the same outputs as the target on previously unseen inputs, then the attacker has successfully approximated some of the target's functionality.

</details>

In [ ]:
# @title ▶ Part 4 — Evaluate the Extraction Attack { display-mode: "form" }

target_eval_predictions = target_model.predict(
    X_eval
)

surrogate_eval_predictions = surrogate_model.predict(
    X_eval
)

target_accuracy = accuracy_score(
    y_eval,
    target_eval_predictions
)

surrogate_accuracy = accuracy_score(
    y_eval,
    surrogate_eval_predictions
)

fidelity = np.mean(
    surrogate_eval_predictions
    ==
    target_eval_predictions
)

print("=" * 65)
print("MODEL EXTRACTION RESULTS")
print("=" * 65)

print(
    f"\nTarget Accuracy:    "
    f"{target_accuracy:.3f}"
)

print(
    f"Surrogate Accuracy: "
    f"{surrogate_accuracy:.3f}"
)

print(
    f"\nSurrogate Fidelity: "
    f"{fidelity:.3f}"
)

agreements = np.sum(
    surrogate_eval_predictions
    ==
    target_eval_predictions
)

print(
    f"Model Agreement:    "
    f"{agreements}/{len(X_eval)}"
)

print("\nEXTRACTION INTERPRETATION")

if fidelity >= 0.90:

    print(
        "✓ HIGH FIDELITY — The surrogate "
        "closely reproduces target behavior."
    )

elif fidelity >= 0.75:

    print(
        "✓ MODERATE FIDELITY — The surrogate "
        "has learned substantial target behavior."
    )

else:

    print(
        "△ LOW FIDELITY — More or better "
        "queries may be needed."
    )

<details>

<summary><b>Part 4 Explanation — Click to Expand</b></summary>

<br>

Now we evaluate the extraction attack using information that was **not used to train the surrogate**.

Two measurements are displayed.

### Accuracy

Surrogate accuracy compares:

    Surrogate Prediction
             vs.
         True Label

This tells us whether the surrogate performs the classification task correctly.

### Fidelity

Fidelity compares:

    Surrogate Prediction
             vs.
      Target Prediction

This tells us whether the surrogate behaves like the target.

For model extraction, fidelity is especially important.

The attacker's objective is not necessarily to create the world's most accurate classifier.

The attacker is trying to approximate:

    TARGET BEHAVIOR

Therefore, a surrogate can have meaningful extraction value even when it is not perfectly accurate according to the ground-truth labels.

</details>

<details>

<summary><b>Part 5 — Student Experiment: Query Budget — Click to Expand</b></summary>

<br>

Return to **Part 2** and change:

    QUERY_BUDGET

Try:

    50

    100

    200

    300

    500

After changing the value, rerun:

    Part 2
       ↓
    Part 3
       ↓
    Part 4

Watch:

    Surrogate Accuracy

and especially:

    Surrogate Fidelity

Think about what is happening.

With fewer queries:

    Target Behavior Observed
             ↓
           Less

With more queries:

    Target Behavior Observed
             ↓
           More

You should generally expect a larger, useful query set to give the surrogate more information about the target.

However, machine learning results do not need to improve perfectly after every single increase.

The important relationship to investigate is:

    QUERY ACCESS
          ↓
    INFORMATION COLLECTED
          ↓
    SURROGATE LEARNING
          ↓
    TARGET APPROXIMATION

This is why query limits can matter from a defensive perspective.

</details>

In [ ]:
# @title ▶ Part 6 — Compare Target and Surrogate Predictions { display-mode: "form" }

print("=" * 55)
print("TARGET vs. SURROGATE")
print("=" * 55)

print(
    "\nTrue | Target | Surrogate | Match"
)

print("-" * 38)

for i in range(20):

    match = (
        "✓"
        if target_eval_predictions[i]
        == surrogate_eval_predictions[i]
        else "✗"
    )

    print(
        f" {y_eval[i]:2d}  |"
        f"   {target_eval_predictions[i]:2d}   |"
        f"     {surrogate_eval_predictions[i]:2d}    |"
        f"   {match}"
    )

<details>

<summary><b>Part 6 Explanation — Click to Expand</b></summary>

<br>

This cell makes the idea of fidelity easier to see.

Each row contains:

    TRUE LABEL

    TARGET PREDICTION

    SURROGATE PREDICTION

    AGREEMENT

If the target and surrogate make the same prediction, the row receives:

    ✓

Notice that the target and surrogate can agree even when both differ from the true label.

That situation still contributes to **fidelity** because the surrogate successfully reproduced the target's behavior.

This is why model extraction should not be evaluated using accuracy alone.

</details>

In [ ]:
# @title ▶ Part 7 — Model Extraction Lab Check { display-mode: "form" }

questions = [
    {
        "question":
        "1. Where did the surrogate's training labels come from?",
        "options": [
            "A. The target model's predictions",
            "B. Random numbers",
            "C. The final evaluation labels",
            "D. The surrogate generated them itself"
        ],
        "answer": "A"
    },

    {
        "question":
        "2. Did the surrogate need to use the same model type as the target?",
        "options": [
            "A. Yes",
            "B. No",
            "C. Only for binary classification",
            "D. Only when fidelity is zero"
        ],
        "answer": "B"
    },

    {
        "question":
        "3. Which measurement BEST evaluates how closely the surrogate copied target behavior?",
        "options": [
            "A. Training time",
            "B. Fidelity",
            "C. Dataset size",
            "D. Number of classes"
        ],
        "answer": "B"
    },

    {
        "question":
        "4. Why can increasing the query budget help an extraction attacker?",
        "options": [
            "A. More queries can reveal more target behavior",
            "B. It exposes the target weights automatically",
            "C. It changes the true labels",
            "D. It disables the target model"
        ],
        "answer": "A"
    },

    {
        "question":
        "5. Why can a surrogate be useful for later attacks?",
        "options": [
            "A. The attacker controls it and can analyze it locally",
            "B. It automatically provides the original training dataset",
            "C. It removes all security controls from the target",
            "D. It guarantees 100% fidelity"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 65)
print("MODEL EXTRACTION MINI LAB CHECK")
print("=" * 65)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:

        print("✓ Correct!")
        score += 1

    else:

        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = (
    score / len(questions)
) * 100

print("\n" + "=" * 65)

print(
    f"Lab Score: "
    f"{score}/{len(questions)}"
)

print(
    f"Percentage: "
    f"{percentage:.1f}%"
)

if percentage >= 80:

    print(
        "\n✓ Model extraction mini lab complete."
    )

else:

    print(
        "\nReview the extraction process and "
        "try the lab check again."
    )

print("=" * 65)

<details>

<summary><b>Section 3 — Practical Takeaways — Click to Expand</b></summary>

<br>

In this section, you performed a simplified **black-box model extraction attack**.

You started with a target model and treated its prediction function like an external API.

The extraction process was:

    QUERY TARGET
          ↓
    COLLECT PREDICTIONS
          ↓
    BUILD NEW DATASET
          ↓
    TRAIN SURROGATE
          ↓
    TEST SURROGATE
          ↓
    MEASURE FIDELITY

The attacker never copied the target model's internal parameters.

Instead, the target provided information through its predictions.

You then trained a completely different type of model using those responses.

This demonstrated that model stealing does not necessarily require reconstructing the exact original architecture.

The surrogate only needs to approximate enough of the target's **observable behavior** to be useful.

You also compared two different measurements:

    ACCURACY
        =
    Agreement with True Labels

and:

    FIDELITY
        =
    Agreement with Target Model

For model extraction, fidelity provides direct evidence of how closely the surrogate reproduced the target.

Finally, you experimented with the attacker's **query budget**.

This demonstrates why prediction APIs should be treated as security-relevant interfaces.

Every response may provide a small amount of information.

Repeated systematically:

    Query
      +
    Query
      +
    Query
      +
    Query
      ↓
    Learned Behavior

The central lesson is:

> **An attacker may not need to steal a model file to steal useful model functionality. Repeated access to model predictions can sometimes provide enough information to construct a useful approximation.**

In a later section, you will see why this becomes even more important when a surrogate model is combined with **transfer attacks**.

</details>

In [ ]:
# @title ▶ Section 3 Checkpoint — Model Extraction { display-mode: "form" }

questions = [
    {
        "question":
        "1. What is the main goal of a model extraction attack?",
        "options": [
            "A. Modify the target's training labels",
            "B. Approximate the target model's behavior",
            "C. Delete the target model",
            "D. Increase the target's accuracy"
        ],
        "answer": "B",
        "explanation":
        "Model extraction attempts to construct another model that reproduces the target's observable behavior."
    },

    {
        "question":
        "2. Why can model extraction work in a black-box setting?",
        "options": [
            "A. The attacker can learn from target inputs and outputs",
            "B. Black-box access always exposes model weights",
            "C. The attacker automatically receives the training data",
            "D. Black-box models have no security controls"
        ],
        "answer": "A",
        "explanation":
        "Repeated queries and observable outputs can provide information about how a black-box model behaves."
    },

    {
        "question":
        "3. What is the model created by the attacker commonly called?",
        "options": [
            "A. Poison model",
            "B. Surrogate model",
            "C. Validation model",
            "D. Encryption model"
        ],
        "answer": "B",
        "explanation":
        "A surrogate or substitute model attempts to reproduce the target model's behavior."
    },

    {
        "question":
        "4. What does fidelity measure?",
        "options": [
            "A. Agreement between the surrogate and target predictions",
            "B. Number of training epochs",
            "C. Amount of computer memory used",
            "D. Percentage of poisoned samples"
        ],
        "answer": "A",
        "explanation":
        "Fidelity measures how closely the surrogate reproduces the target's predictions."
    },

    {
        "question":
        "5. Why might confidence scores make extraction easier?",
        "options": [
            "A. They reveal more information about target behavior",
            "B. They expose every model weight directly",
            "C. They prevent queries",
            "D. They change the training labels"
        ],
        "answer": "A",
        "explanation":
        "Confidence or probability outputs may provide more information per query than a class label alone."
    },

    {
        "question":
        "6. Why can an extracted model help with later attacks?",
        "options": [
            "A. The attacker controls the surrogate and can analyze it locally",
            "B. It automatically changes the original target",
            "C. It deletes API logs",
            "D. It guarantees access to the original dataset"
        ],
        "answer": "A",
        "explanation":
        "The attacker can inspect and experiment against the surrogate, potentially using it to develop additional attacks."
    }
]

score = 0

print("=" * 70)
print("SECTION 3 CHECKPOINT")
print("Model Extraction / Model Stealing")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:

        print("✓ Correct!")
        score += 1

    else:

        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

    print(
        "Explanation:",
        q["explanation"]
    )

percentage = (
    score / len(questions)
) * 100

print("\n" + "=" * 70)

print(
    f"Score: {score}/{len(questions)}"
)

print(
    f"Percentage: {percentage:.1f}%"
)

if percentage >= 80:

    print(
        "\n✓ Checkpoint complete. "
        "Continue to the model extraction mini lab."
    )

else:

    print(
        "\nReview Section 3 before continuing."
    )

print("=" * 70)

---

<details>

<summary><b>Section 4 — Membership Inference — Click to Expand</b></summary>

<br>

The previous section demonstrated that an attacker can learn information about a model by repeatedly observing its predictions.

However, model outputs may reveal more than information about the model itself.

They may also reveal information about the **data used to train it**.

A **Membership Inference Attack (MIA)** attempts to determine whether a particular record was included in a machine learning model's training dataset.

The attack asks a very specific question:

> **Was this record used to train the model?**

Conceptually:

    Record
      ↓
    Target Model
      ↓
    Observe Behavior
      ↓
    Membership Attack
      ↓
      / \
    YES  NO
     ↓    ↓
   Member Non-Member

This may initially seem like a small piece of information.

However, membership itself can sometimes be sensitive.

<br>

<details>

<summary><b>4.1 — Why Does Membership Matter?</b></summary>

<br>

Imagine a model trained using information from a sensitive dataset.

Knowing that a particular person appears in that dataset may itself reveal something about them.

For example, consider a hypothetical model trained only using records from people participating in a particular medical study.

An attacker asks:

> "Was Person A included in the training dataset?"

If the answer can be inferred with useful reliability, the attacker may have learned something about that person's association with the underlying dataset.

The attacker does not necessarily recover the complete training record.

The privacy leak is:

    Record
      ↓
    Was it used during training?
      ↓
    YES / NO

Whether membership is sensitive depends on what the dataset represents.

Therefore:

> **Privacy risk can exist even when the attacker cannot directly reconstruct the original training data.**

</details>

<br>

<details>

<summary><b>4.2 — Why Can Membership Leak?</b></summary>

<br>

Machine learning models are supposed to learn patterns that **generalize**.

Ideally:

    Training Record
          ↓
       Model
          ↓
    Appropriate Prediction

and:

    New Similar Record
          ↓
       Model
          ↓
    Similar Appropriate Prediction

The model should learn the general relationship rather than simply memorize individual training samples.

However, models can sometimes fit their training information too closely.

This is known as **overfitting**.

An overfit model may behave differently on:

    Records It Has Seen

compared with:

    Records It Has Never Seen

Those differences can create a membership signal.

For example, a model might be:

    More Confident

or:

    Lower Loss

or:

    More Frequently Correct

on training records than on unseen records.

An attacker may attempt to exploit these differences.

</details>

<br>

<details>

<summary><b>4.3 — Confidence as a Membership Signal</b></summary>

<br>

Suppose a classifier produces the following result for one record:

    Class A: 0.99
    Class B: 0.01

For another:

    Class A: 0.56
    Class B: 0.44

The first prediction is considerably more confident.

If a model systematically produces higher confidence for records it encountered during training, confidence may provide information about membership.

A very simple attack might use:

    Confidence ≥ Threshold
              ↓
        Predict MEMBER

    Confidence < Threshold
              ↓
      Predict NON-MEMBER

This is only a simplified example.

High confidence does **not automatically mean** that a record was used during training.

A model can be highly confident on unseen records.

The security concern appears when there is a measurable statistical difference between member and non-member behavior.

</details>

<br>

<details>

<summary><b>4.4 — Loss as a Membership Signal</b></summary>

<br>

Another useful signal is **model loss**.

Loss measures how poorly the model's prediction matches the expected answer.

Conceptually:

    Lower Loss
        =
    Better Prediction

If a model has fit its training data especially closely, we may observe:

    TRAINING RECORDS
          ↓
      Lower Loss

and:

    UNSEEN RECORDS
          ↓
      Higher Loss

An attacker who knows the correct label for a candidate record may calculate its loss and use that value as evidence.

A simplified rule might be:

    Loss < Threshold
          ↓
    Predict MEMBER

This type of attack demonstrates an important relationship:

    Overfitting
        ↓
    Behavioral Difference
        ↓
    Membership Signal
        ↓
    Privacy Risk

</details>

<br>

<details>

<summary><b>4.5 — Black-Box Membership Inference</b></summary>

<br>

Membership inference can be particularly concerning when it works through a prediction interface.

A black-box attacker may not know:

    Training Dataset

    Model Parameters

    Internal Architecture

Instead, the attacker may only receive:

    Prediction

    Confidence

    Probability Distribution

Depending on the attack, this information may be sufficient to estimate membership.

Conceptually:

    Candidate Record
           ↓
      Prediction API
           ↓
    Confidence / Output
           ↓
      Attack Decision
           ↓
    Member / Non-Member

This again demonstrates why model outputs should be considered part of the attack surface.

</details>

<br>

<details>

<summary><b>4.6 — Shadow Models</b></summary>

<br>

More sophisticated membership inference attacks may use **shadow models**.

A shadow model is a model controlled by the attacker that is used to imitate the behavior of the target.

The attacker knows which records were used to train the shadow model.

Therefore, they can observe:

    SHADOW TRAINING RECORD
             ↓
      Shadow Model Output
             ↓
      Known to be MEMBER

and:

    SHADOW TEST RECORD
             ↓
      Shadow Model Output
             ↓
    Known to be NON-MEMBER

The attacker can collect these examples and learn patterns that distinguish members from non-members.

Conceptually:

    Shadow Dataset
          ↓
    Train Shadow Model
          ↓
       /       \
    Members   Non-Members
       ↓          ↓
    Outputs     Outputs
       \          /
        \        /
      Attack Dataset
            ↓
      Attack Classifier
            ↓
       MEMBER / NOT

The attack classifier can then be applied to outputs from the real target model.

For our introductory lab, we will use a simpler confidence-based approach so the core idea remains easy to see.

</details>

<br>

<details>

<summary><b>4.7 — Overfitting and Privacy</b></summary>

<br>

Overfitting is often introduced as a model-performance problem.

For example:

    Training Accuracy = 100%

    Test Accuracy = 80%

The model performs substantially better on information it has already seen.

From a machine learning perspective, this suggests poor generalization.

From a security perspective, it may also indicate a privacy concern.

If the model treats training and non-training records differently enough, an attacker may be able to distinguish between them.

This creates an important lesson:

> **Generalization can have both performance and privacy implications.**

Reducing unnecessary memorization may therefore improve more than test performance.

It may also reduce certain forms of membership leakage.

</details>

<br>

<details>

<summary><b>4.8 — Membership Inference Does Not Recover the Entire Record</b></summary>

<br>

Membership inference and model inversion are related privacy attacks, but they ask different questions.

Membership inference asks:

> **Was this record in the training dataset?**

Model inversion asks something closer to:

> **What information can I infer or reconstruct about what the model learned?**

For example:

    MEMBERSHIP INFERENCE

    Known Candidate Record
            ↓
    Was it used for training?
            ↓
          YES / NO

versus:

    MODEL INVERSION

    Model Information
            ↓
    Attempt Reconstruction
            ↓
    Recover Characteristics
    or Representative Information

Keeping these attacks separate is important because they represent different privacy objectives.

</details>

<br>

<details>

<summary><b>4.9 — Measuring Membership Inference Success</b></summary>

<br>

As always, we need evidence that the attack works.

Suppose we create an evaluation set containing:

    100 Members

and:

    100 Non-Members

A random guess would achieve approximately:

    50% Accuracy

for a balanced dataset.

If our attack achieves:

    51%

that provides very little evidence of useful membership inference.

If it achieves:

    75%

then the model's behavior contains a much stronger membership signal.

We can measure:

    Attack Accuracy

    Precision

    Recall

    ROC-AUC

The exact metric depends on the experiment.

For this introductory activity, we will focus primarily on **attack accuracy** and the difference between member and non-member confidence.

</details>

<br>

<details>

<summary><b>4.10 — Defensive Considerations</b></summary>

<br>

Membership inference risk cannot necessarily be eliminated with one control.

Possible defensive strategies include improving generalization, limiting unnecessary output information, reducing overfitting, and using privacy-preserving training techniques.

One important technique is **Differential Privacy**.

At a high level, differential privacy attempts to reduce how strongly the presence or absence of an individual training record influences the learned model.

Conceptually:

    Individual Record
          ↓
    Limited Influence
          ↓
    Model Behavior
          ↓
    Harder to Determine Membership

Later defensive material will explore privacy protections more directly.

For now, remember that a model exposing less distinguishable behavior between members and non-members generally provides a weaker signal for membership inference.

</details>

<br>

<details>

<summary><b>Section 4 — Key Takeaways</b></summary>

<br>

A **Membership Inference Attack** attempts to determine whether a particular record was included in a model's training dataset.

Membership can be sensitive even if the attacker never recovers the complete training record.

The attack is possible when the model behaves differently enough on:

    Members

and:

    Non-Members

Possible signals include:

- Prediction confidence
- Loss
- Correctness
- Output distributions

Overfitting can increase this risk because an overfit model may behave unusually well on information it has already seen.

More advanced membership inference attacks may use shadow models to learn the behavioral differences between members and non-members.

The central lesson is:

> **A model can unintentionally reveal information about its training dataset through differences in how it responds to records it has and has not seen before.**

</details>

</details>

In [ ]:
# @title Section 4 — Membership Inference Visualizations { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix
)
from IPython.display import display, HTML
from io import BytesIO
import base64
import html

# ============================================================
# CONFIGURATION
# ============================================================

rng = np.random.default_rng(42)

BLUE = "#2563EB"
GREEN = "#16A34A"
RED = "#DC2626"
ORANGE = "#EA580C"
PURPLE = "#7C3AED"
GRAY = "#64748B"
DARK = "#0F172A"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# ============================================================
# COLLAPSIBLE VISUALIZATION HELPER
# ============================================================

def show_collapsible_figure(title, fig, explanation, color=BLUE):

    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=135,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    buffer.close()

    display(HTML(f"""
    <details style="
        margin:12px 0;
        border:1px solid #cbd5e1;
        border-radius:10px;
        overflow:hidden;
    ">

        <summary style="
            cursor:pointer;
            padding:16px;
            font-size:16px;
            font-weight:bold;
            background:#eff6ff;
            color:#1e3a8a;
            border-left:5px solid {color};
        ">
            {html.escape(title)} — Click to Expand
        </summary>

        <div style="
            padding:15px;
            background:white;
            text-align:center;
        ">

            <img
                src="data:image/png;base64,{encoded}"
                style="width:100%;max-width:950px;height:auto;"
            >

            <div style="
                max-width:850px;
                margin:15px auto;
                padding:16px;
                background:#f8fafc;
                border-radius:8px;
                color:#334155;
                font-size:14px;
                line-height:1.7;
                text-align:left;
            ">
                {explanation}
            </div>

        </div>
    </details>
    """))


# ============================================================
# DIAGRAM HELPERS
# ============================================================

def add_card(ax, x, y, w, h, title, subtitle="",
             accent=BLUE, background="#F8FAFC"):

    rect = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        facecolor=background,
        edgecolor=accent,
        linewidth=1.7
    )

    ax.add_patch(rect)

    ax.text(
        x + w / 2,
        y + h * (0.65 if subtitle else 0.5),
        title,
        ha="center",
        va="center",
        fontsize=10.5,
        fontweight="bold",
        color=DARK
    )

    if subtitle:
        ax.text(
            x + w / 2,
            y + h * 0.28,
            subtitle,
            ha="center",
            va="center",
            fontsize=9,
            color=GRAY
        )


def add_arrow(ax, start, end, color=GRAY):

    ax.add_patch(
        FancyArrowPatch(
            start, end,
            arrowstyle="-|>",
            mutation_scale=16,
            linewidth=1.8,
            color=color
        )
    )


# ============================================================
# SYNTHETIC TARGET MODEL
# ============================================================

# A deliberately small training set and an unrestricted
# decision tree create an opportunity to observe overfitting.

X, y = make_classification(
    n_samples=1200,
    n_features=20,
    n_informative=6,
    n_redundant=4,
    flip_y=0.12,
    class_sep=0.7,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=180,
    random_state=42,
    stratify=y
)

target_model = DecisionTreeClassifier(
    random_state=42
)

target_model.fit(X_train, y_train)

train_probabilities = target_model.predict_proba(X_train)
test_probabilities = target_model.predict_proba(X_test)

member_confidence = np.max(train_probabilities, axis=1)
nonmember_confidence = np.max(test_probabilities, axis=1)

# True-label loss requires access to the correct label.
member_true_prob = train_probabilities[
    np.arange(len(y_train)), y_train
]

nonmember_true_prob = test_probabilities[
    np.arange(len(y_test)), y_test
]

member_loss = -np.log(
    np.clip(member_true_prob, 1e-10, 1)
)

nonmember_loss = -np.log(
    np.clip(nonmember_true_prob, 1e-10, 1)
)


# ============================================================
# VISUALIZATION 1 — MEMBERSHIP INFERENCE WORKFLOW
# Sections 4.1 and 4.5
# ============================================================

fig, ax = plt.subplots(figsize=(10, 8.5))

ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis("off")

ax.set_title(
    "Black-Box Membership Inference Workflow",
    fontsize=17,
    fontweight="bold"
)

steps = [
    ("Candidate Record", "Membership is unknown", BLUE),
    ("Target Model", "Processes the candidate input", PURPLE),
    ("Observe Model Output", "Prediction, confidence, or scores", ORANGE),
    ("Membership Attack", "Analyzes observable behavior", RED),
    ("Membership Prediction", "Member or Non-Member", GREEN)
]

for i, (title, subtitle, accent) in enumerate(steps):

    y_pos = 8.2 - i * 1.75

    add_card(
        ax, 1.2, y_pos, 7.6, 1.2,
        title, subtitle,
        accent
    )

    if i < len(steps) - 1:
        add_arrow(
            ax,
            (5, y_pos - 0.05),
            (5, y_pos - 0.48)
        )

plt.tight_layout()

show_collapsible_figure(
    "4.1 & 4.5 — How Membership Inference Works",
    fig,
    """
    A membership inference attack attempts to
    determine whether a candidate record was
    included in a model's training dataset.

    <br><br>

    In a black-box setting, the attacker may
    only be able to submit an input and
    observe the model's output.

    <br><br>

    The attacker analyzes observable information
    such as prediction confidence or loss
    to estimate whether the record was
    used during training.

    <br><br>

    This can create a privacy risk because
    membership in a sensitive dataset may
    itself reveal information about an individual.

    <br><br>

    <b>Important:</b> A membership prediction
    is an inference, not proof that the
    candidate record was in the dataset.
    """
)


# ============================================================
# VISUALIZATION 2 — CONFIDENCE DISTRIBUTIONS
# Sections 4.2 and 4.3
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5.5))

bins = np.linspace(0.45, 1.01, 24)

ax.hist(
    member_confidence,
    bins=bins,
    alpha=0.60,
    color=BLUE,
    density=True,
    label="Members"
)

ax.hist(
    nonmember_confidence,
    bins=bins,
    alpha=0.55,
    color=ORANGE,
    density=True,
    label="Non-Members"
)

ax.axvline(
    np.mean(member_confidence),
    color=BLUE,
    linestyle="--",
    linewidth=2,
    label="Member Mean"
)

ax.axvline(
    np.mean(nonmember_confidence),
    color=ORANGE,
    linestyle="--",
    linewidth=2,
    label="Non-Member Mean"
)

ax.set_title(
    "Prediction Confidence: Members vs. Non-Members",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Maximum Predicted Class Probability")
ax.set_ylabel("Density")

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "4.2–4.3 — Confidence as a Membership Signal",
    fig,
    """
    Models sometimes produce different confidence
    distributions for records they encountered
    during training and records they have
    never seen before.

    <br><br>

    This experiment trains a decision tree
    on a small synthetic dataset.

    <br><br>

    The blue distribution represents
    training records, while the orange
    distribution represents unseen records.

    <br><br>

    A difference between these distributions
    may provide information that an attacker
    can use for membership inference.

    <br><br>

    However, high confidence does not
    automatically mean that a record
    was included in training.

    <br><br>

    <b>Key takeaway:</b> Membership inference
    depends on measurable differences in
    model behavior, not simply whether
    a prediction appears confident.
    """
)


# ============================================================
# VISUALIZATION 3 — LOSS DISTRIBUTIONS
# Section 4.4
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5.5))

# Clip display range to keep extreme losses readable.
display_max = max(
    1.0,
    np.percentile(
        np.concatenate([member_loss, nonmember_loss]),
        90
    )
)

ax.hist(
    np.clip(member_loss, 0, display_max),
    bins=25,
    color=GREEN,
    alpha=0.65,
    density=True,
    label="Members"
)

ax.hist(
    np.clip(nonmember_loss, 0, display_max),
    bins=25,
    color=RED,
    alpha=0.50,
    density=True,
    label="Non-Members"
)

ax.set_title(
    "Prediction Loss: Members vs. Non-Members",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("True-Label Cross-Entropy Loss")
ax.set_ylabel("Density")

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "4.4 — Loss as a Membership Signal",
    fig,
    """
    Loss measures how poorly a model's
    prediction matches the correct label.

    <br><br>

    A model that has closely fit its training
    records may produce lower loss on those
    records than on unseen information.

    <br><br>

    An attacker who knows the correct label
    may calculate the model's loss and
    use it as a membership signal.

    <br><br>

    A simple loss-based attack might predict
    membership when the loss is below
    a selected threshold.

    <br><br>

    This visualization clips unusually large
    losses for readability. The underlying
    loss calculations are unchanged.

    <br><br>

    <b>Key takeaway:</b> Low loss can provide
    evidence of membership, but it is
    not definitive proof.
    """,
    color=GREEN
)


# ============================================================
# VISUALIZATION 4 — CONFIDENCE THRESHOLD ATTACK
# Sections 4.3 and 4.9
# ============================================================

# Balance members and non-members.
sample_size = min(
    len(member_confidence),
    len(nonmember_confidence)
)

member_indices = rng.choice(
    len(member_confidence),
    size=sample_size,
    replace=False
)

nonmember_indices = rng.choice(
    len(nonmember_confidence),
    size=sample_size,
    replace=False
)

attack_scores = np.concatenate([
    member_confidence[member_indices],
    nonmember_confidence[nonmember_indices]
])

membership_labels = np.concatenate([
    np.ones(sample_size, dtype=int),
    np.zeros(sample_size, dtype=int)
])

thresholds = np.linspace(0.50, 1.00, 101)

attack_accuracies = []

for threshold in thresholds:

    predictions = (
        attack_scores >= threshold
    ).astype(int)

    attack_accuracies.append(
        accuracy_score(
            membership_labels,
            predictions
        )
    )

best_index = np.argmax(attack_accuracies)

best_threshold = thresholds[best_index]
best_accuracy = attack_accuracies[best_index]

fig, ax = plt.subplots(figsize=(10, 5.5))

ax.plot(
    thresholds,
    attack_accuracies,
    color=PURPLE,
    linewidth=2.5
)

ax.scatter(
    best_threshold,
    best_accuracy,
    color=RED,
    s=120,
    zorder=5,
    label=f"Best Observed Threshold: {best_threshold:.2f}"
)

ax.axhline(
    0.50,
    linestyle="--",
    color=GRAY,
    label="Balanced Random-Guess Baseline"
)

ax.set_title(
    "Confidence Threshold vs. Membership Attack Accuracy",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Confidence Threshold")
ax.set_ylabel("Attack Accuracy")

ax.set_ylim(0, 1.05)

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "4.3 & 4.9 — Confidence-Based Membership Inference",
    fig,
    f"""
    A simple membership inference attack
    can classify records using a
    confidence threshold.

    <br><br>

    If a record's confidence is at least
    the threshold, the attack predicts
    that the record was a training member.

    <br><br>

    Otherwise, the attack predicts
    that it was a non-member.

    <br><br>

    In this synthetic demonstration,
    the highest observed accuracy was
    <b>{best_accuracy:.1%}</b> at a threshold
    of <b>{best_threshold:.2f}</b>.

    <br><br>

    The dashed horizontal line represents
    the expected 50% accuracy of random
    guessing on a balanced evaluation set.

    <br><br>

    <b>Important:</b> The best threshold was
    selected using the same records shown
    in this graph. Its accuracy is therefore
    an in-sample result and may overestimate
    performance on new records.

    <br><br>

    A rigorous attack evaluation should
    select its threshold on separate
    calibration data and test it on
    previously unseen records.
    """,
    color=PURPLE
)


# ============================================================
# VISUALIZATION 5 — ATTACK METRICS AND ROC CURVE
# Section 4.9
# ============================================================

attack_predictions = (
    attack_scores >= best_threshold
).astype(int)

attack_accuracy = accuracy_score(
    membership_labels,
    attack_predictions
)

attack_precision = precision_score(
    membership_labels,
    attack_predictions,
    zero_division=0
)

attack_recall = recall_score(
    membership_labels,
    attack_predictions,
    zero_division=0
)

attack_auc = roc_auc_score(
    membership_labels,
    attack_scores
)

fpr, tpr, roc_thresholds = roc_curve(
    membership_labels,
    attack_scores
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

fig.suptitle(
    "Evaluating Membership Inference Performance",
    fontsize=16,
    fontweight="bold"
)

metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "ROC-AUC"
]

values = [
    attack_accuracy,
    attack_precision,
    attack_recall,
    attack_auc
]

bars = axes[0].bar(
    metrics,
    values,
    color=[BLUE, GREEN, ORANGE, PURPLE]
)

axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel("Score")
axes[0].set_title("Attack Metrics")

for bar, value in zip(bars, values):

    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.025,
        f"{value:.2f}",
        ha="center",
        fontweight="bold"
    )

axes[1].plot(
    fpr,
    tpr,
    color=PURPLE,
    linewidth=2.5,
    label=f"ROC-AUC = {attack_auc:.3f}"
)

axes[1].plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color=GRAY,
    label="Random Ranking"
)

axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Membership Attack ROC Curve")

axes[1].legend()
axes[1].grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "4.9 — Measuring Membership Inference Success",
    fig,
    """
    Membership inference is a binary
    classification problem.

    <br><br>

    <b>Accuracy:</b> The proportion of
    membership predictions that are correct.

    <br><br>

    <b>Precision:</b> Among records predicted
    to be members, the proportion that
    actually were members.

    <br><br>

    <b>Recall:</b> Among actual members,
    the proportion correctly identified.

    <br><br>

    <b>ROC-AUC:</b> Measures how well the
    confidence score ranks members
    above non-members across thresholds.

    <br><br>

    An ROC-AUC of 0.5 indicates no
    ranking advantage over random guessing.

    <br><br>

    A higher score indicates a stronger
    membership signal in this experiment.

    <br><br>

    These metrics use the same synthetic
    evaluation records as the threshold
    demonstration. Threshold-based
    metrics are in-sample results.
    """
)


# ============================================================
# VISUALIZATION 6 — SHADOW MODELS AND DEFENSES
# Sections 4.6, 4.7 and 4.10
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

fig.suptitle(
    "Advanced Membership Inference and Defensive Considerations",
    fontsize=16,
    fontweight="bold"
)

# LEFT — SHADOW MODEL WORKFLOW

ax = axes[0]

ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis("off")

ax.set_title("Shadow Model Attack", fontsize=13)

shadow_steps = [
    ("Shadow Dataset", "Known member/non-member status", BLUE),
    ("Train Shadow Model", "Attacker-controlled model", PURPLE),
    ("Collect Model Outputs", "Observe confidence and loss", ORANGE),
    ("Train Attack Classifier", "Learn membership patterns", RED),
    ("Evaluate Target Outputs", "Estimate target membership", GREEN)
]

for i, (title, subtitle, accent) in enumerate(shadow_steps):

    y_pos = 8.2 - i * 1.75

    add_card(
        ax,
        0.5,
        y_pos,
        9.0,
        1.2,
        title,
        subtitle,
        accent
    )

    if i < len(shadow_steps) - 1:
        add_arrow(
            ax,
            (5, y_pos - 0.05),
            (5, y_pos - 0.45)
        )

# RIGHT — DEFENSIVE CONSIDERATIONS

ax = axes[1]

ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis("off")

ax.set_title("Reducing Membership Leakage", fontsize=13)

defenses = [
    ("Improve Generalization", "Reduce unnecessary memorization", GREEN),
    ("Regularization", "Limit excessive model complexity", BLUE),
    ("Limit Output Exposure", "Avoid unnecessary detailed scores", ORANGE),
    ("Differential Privacy", "Limit individual record influence", PURPLE),
    ("Privacy Evaluation", "Test for membership leakage", RED)
]

for i, (title, subtitle, accent) in enumerate(defenses):

    y_pos = 8.2 - i * 1.75

    add_card(
        ax,
        0.5,
        y_pos,
        9.0,
        1.2,
        title,
        subtitle,
        accent
    )

    if i < len(defenses) - 1:
        add_arrow(
            ax,
            (5, y_pos - 0.05),
            (5, y_pos - 0.45)
        )

plt.tight_layout()

show_collapsible_figure(
    "4.6–4.10 — Shadow Models and Membership Inference Defenses",
    fig,
    """
    More sophisticated membership inference
    attacks may use shadow models.

    <br><br>

    A shadow model is trained using data
    whose membership status is known
    to the attacker.

    <br><br>

    The attacker collects outputs from
    known members and non-members,
    then uses those examples to train
    a membership attack classifier.

    <br><br>

    The classifier can then be applied
    to outputs from the target model.

    <br><br>

    Defenders can attempt to reduce
    membership leakage by improving
    generalization, reducing overfitting,
    and limiting unnecessary output details.

    <br><br>

    <b>Differential privacy</b> provides a
    more formal approach to limiting
    how strongly individual training
    records influence a model.

    <br><br>

    These protections have different
    limitations and tradeoffs.
    No single defense guarantees that
    membership inference is impossible.

    <br><br>

    <b>Key takeaway:</b> Model behavior
    can expose information about training
    membership, even when attackers
    cannot directly access the dataset.
    """,
    color=PURPLE
)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print("Section 4 visualizations are ready.")
print("Six collapsible visualizations generated.")
print("All experiments used synthetic data.")

print()
print("Target Model Performance")
print("------------------------")

print(
    "Training Accuracy:",
    round(
        accuracy_score(
            y_train,
            target_model.predict(X_train)
        ),
        3
    )
)

print(
    "Test Accuracy:",
    round(
        accuracy_score(
            y_test,
            target_model.predict(X_test)
        ),
        3
    )
)

print()
print("Membership Inference Performance")
print("--------------------------------")

print("Attack Accuracy:", round(attack_accuracy, 3))
print("Attack Precision:", round(attack_precision, 3))
print("Attack Recall:", round(attack_recall, 3))
print("Attack ROC-AUC:", round(attack_auc, 3))

<details>

<summary><b>Section 4 Lab — Click to Expand</b></summary>

<br>

In this activity, you will perform a simplified **membership inference attack**.

The purpose is to observe the core privacy problem rather than build a highly sophisticated attack.

You will deliberately train a model under conditions where it begins to fit its training records more closely than unseen records.

Then you will compare:

    MEMBER RECORDS
          vs.
    NON-MEMBER RECORDS

You will examine the model's confidence for each group.

The attack will use a simple idea:

    HIGH CONFIDENCE
          ↓
    More Likely MEMBER

    LOWER CONFIDENCE
          ↓
    More Likely NON-MEMBER

This rule is intentionally simplified.

High confidence does not prove membership.

Instead, the experiment asks whether confidence provides enough statistical information to classify membership **better than random guessing**.

The activity follows:

    Train Model
         ↓
    Collect Member Outputs
         ↓
    Collect Non-Member Outputs
         ↓
    Compare Confidence
         ↓
    Select Threshold
         ↓
    Predict Membership
         ↓
    Measure Attack Accuracy

The goal is to understand why differences between training and unseen behavior can become a privacy risk.

</details>

In [ ]:
# @title ▶ Part 1 — Prepare the Membership Dataset { display-mode: "form" }

import numpy as np

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)

digits = load_digits()

X = digits.data / 16.0
y = digits.target

# Small training set intentionally makes
# memorization easier to observe.
X_member, X_nonmember, y_member, y_nonmember = (
    train_test_split(
        X,
        y,
        train_size=300,
        random_state=42,
        stratify=y
    )
)

print("=" * 60)
print("MEMBERSHIP DATASET")
print("=" * 60)

print(
    f"\nMember Records:     "
    f"{len(X_member)}"
)

print(
    f"Non-Member Records: "
    f"{len(X_nonmember)}"
)

print(
    "\nMembers will be used to train the model."
)

print(
    "Non-members will remain unseen during training."
)

<details>

<summary><b>Part 1 Explanation — Click to Expand</b></summary>

<br>

We divide the dataset into two groups.

### Members

These records will actually be used to train the model.

### Non-Members

These records will remain unseen during training.

This gives us known membership labels:

    MEMBER = Used During Training

    NON-MEMBER = Not Used During Training

Because this is an experiment, we know the true membership status and can later determine whether the attack guessed correctly.

</details>

In [ ]:
# @title ▶ Part 2 — Train the Target Model { display-mode: "form" }

target_model = DecisionTreeClassifier(
    random_state=42
)

target_model.fit(
    X_member,
    y_member
)

member_predictions = target_model.predict(
    X_member
)

nonmember_predictions = target_model.predict(
    X_nonmember
)

member_accuracy = accuracy_score(
    y_member,
    member_predictions
)

nonmember_accuracy = accuracy_score(
    y_nonmember,
    nonmember_predictions
)

print("=" * 60)
print("TARGET MODEL PERFORMANCE")
print("=" * 60)

print(
    f"\nTraining / Member Accuracy: "
    f"{member_accuracy:.3f}"
)

print(
    f"Unseen / Non-Member Accuracy: "
    f"{nonmember_accuracy:.3f}"
)

print(
    f"\nGeneralization Gap: "
    f"{member_accuracy - nonmember_accuracy:.3f}"
)

<details>

<summary><b>Part 2 Explanation — Click to Expand</b></summary>

<br>

The Decision Tree is intentionally allowed to fit the small training dataset very closely.

Look at:

    Member Accuracy

and:

    Non-Member Accuracy

If member accuracy is substantially higher, the model behaves differently on information it has already seen.

This difference is called a **generalization gap**.

For this lab, that difference is useful because it creates an observable membership signal.

In a real system, this same behavior could represent a privacy concern.

</details>

In [ ]:
# @title ▶ Part 3 — Compare Member and Non-Member Confidence { display-mode: "form" }

member_probabilities = target_model.predict_proba(
    X_member
)

nonmember_probabilities = target_model.predict_proba(
    X_nonmember
)

member_confidence = np.max(
    member_probabilities,
    axis=1
)

nonmember_confidence = np.max(
    nonmember_probabilities,
    axis=1
)

print("=" * 65)
print("CONFIDENCE ANALYSIS")
print("=" * 65)

print(
    f"\nAverage Member Confidence:     "
    f"{member_confidence.mean():.3f}"
)

print(
    f"Average Non-Member Confidence: "
    f"{nonmember_confidence.mean():.3f}"
)

print(
    f"\nConfidence Difference:         "
    f"{member_confidence.mean() - nonmember_confidence.mean():.3f}"
)

<details>

<summary><b>Part 3 Explanation — Click to Expand</b></summary>

<br>

Now we examine the model's maximum predicted probability for each record.

This gives us a simple measure of prediction confidence.

We compare:

    Average Member Confidence

against:

    Average Non-Member Confidence

If members receive higher confidence on average, the attacker may be able to use confidence as evidence of membership.

Again:

> **High confidence alone does not prove membership.**

The important question is whether the difference is consistent enough to help distinguish the two groups.

</details>

In [ ]:
# @title ▶ Part 4 — Visualize Confidence { display-mode: "form" }

import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))

plt.hist(
    member_confidence,
    bins=15,
    alpha=0.6,
    label="Members"
)

plt.hist(
    nonmember_confidence,
    bins=15,
    alpha=0.6,
    label="Non-Members"
)

plt.xlabel(
    "Prediction Confidence"
)

plt.ylabel(
    "Number of Records"
)

plt.title(
    "Member vs. Non-Member Confidence"
)

plt.legend()

plt.show()

<details>

<summary><b>Part 4 Explanation — Click to Expand</b></summary>

<br>

The histogram lets you visually compare the confidence distributions.

If the two groups were completely indistinguishable, their distributions would heavily overlap.

If members tend to receive noticeably different confidence values, the attacker has a possible membership signal.

Real membership inference is often more sophisticated than a single confidence threshold.

However, this visualization demonstrates the underlying security problem:

    Model Output
         ↓
    Behavioral Difference
         ↓
    Privacy Information

</details>

In [ ]:
# @title ▶ Part 5 — Run the Membership Inference Attack { display-mode: "form" }

rng = np.random.default_rng(42)

# Balance the two groups.
sample_size = min(
    len(member_confidence),
    len(nonmember_confidence)
)

member_indices = rng.choice(
    len(member_confidence),
    size=sample_size,
    replace=False
)

nonmember_indices = rng.choice(
    len(nonmember_confidence),
    size=sample_size,
    replace=False
)

member_scores = member_confidence[
    member_indices
]

nonmember_scores = nonmember_confidence[
    nonmember_indices
]

# Split each group into calibration and evaluation.
split = sample_size // 2

member_cal = member_scores[:split]
member_eval = member_scores[split:]

nonmember_cal = nonmember_scores[:split]
nonmember_eval = nonmember_scores[split:]

# Search for the best threshold using only calibration data.
candidate_thresholds = np.linspace(
    0.50,
    1.00,
    101
)

best_threshold = 0.50
best_calibration_accuracy = 0

for threshold in candidate_thresholds:

    member_guess = (
        member_cal >= threshold
    ).astype(int)

    nonmember_guess = (
        nonmember_cal >= threshold
    ).astype(int)

    true_membership = np.concatenate([
        np.ones(len(member_cal)),
        np.zeros(len(nonmember_cal))
    ])

    predicted_membership = np.concatenate([
        member_guess,
        nonmember_guess
    ])

    accuracy = np.mean(
        predicted_membership
        == true_membership
    )

    if accuracy > best_calibration_accuracy:

        best_calibration_accuracy = accuracy
        best_threshold = threshold

# Evaluate on separate records.
true_eval_membership = np.concatenate([
    np.ones(len(member_eval)),
    np.zeros(len(nonmember_eval))
])

predicted_eval_membership = np.concatenate([
    (member_eval >= best_threshold).astype(int),
    (nonmember_eval >= best_threshold).astype(int)
])

attack_accuracy = np.mean(
    predicted_eval_membership
    == true_eval_membership
)

print("=" * 65)
print("MEMBERSHIP INFERENCE RESULTS")
print("=" * 65)

print(
    f"\nSelected Confidence Threshold: "
    f"{best_threshold:.2f}"
)

print(
    f"Calibration Accuracy:          "
    f"{best_calibration_accuracy:.3f}"
)

print(
    f"Evaluation Attack Accuracy:    "
    f"{attack_accuracy:.3f}"
)

print(
    f"Random Guess Baseline:         "
    f"{0.500:.3f}"
)

print(
    f"Advantage Over Random:         "
    f"{attack_accuracy - 0.50:+.3f}"
)

print("\nATTACK INTERPRETATION")

if attack_accuracy >= 0.65:

    print(
        "✓ A meaningful membership signal "
        "was observed."
    )

elif attack_accuracy > 0.50:

    print(
        "△ The attack performed better than "
        "random guessing, but the signal was weak."
    )

else:

    print(
        "⚠ This confidence-only attack did not "
        "outperform random guessing."
    )

<details>

<summary><b>Part 5 Explanation — Click to Expand</b></summary>

<br>

The attack now converts model confidence into a membership prediction.

First, part of the data is used to select a confidence threshold.

Then the threshold is tested on **different records**.

The rule is:

    Confidence ≥ Threshold
            ↓
      Predict MEMBER

otherwise:

    Predict NON-MEMBER

Because the evaluation dataset contains equal numbers of members and non-members:

    Random Guessing ≈ 50%

Therefore, an attack that consistently performs above 50% is extracting some membership information from the model's behavior.

The farther the attack rises above random guessing, the stronger the membership signal.

Notice that we did not evaluate the attack on the exact same records used to choose the threshold.

That would give an overly optimistic view of attack performance.

</details>

In [ ]:
# @title ▶ Part 6 — Membership Inference Lab Check { display-mode: "form" }

questions = [
    {
        "question":
        "1. What made a record a MEMBER in this experiment?",
        "options": [
            "A. It was used to train the target model",
            "B. It had the highest confidence",
            "C. It was classified incorrectly",
            "D. It came from the test set"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. What behavioral signal did the attack use?",
        "options": [
            "A. Prediction confidence",
            "B. File size",
            "C. Training time",
            "D. Number of model layers"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. Why did we compare the attack against 50%?",
        "options": [
            "A. The evaluation set was balanced between members and non-members",
            "B. Every ML attack must achieve 50%",
            "C. The model had two layers",
            "D. Confidence values cannot exceed 50%"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. Does high confidence automatically prove membership?",
        "options": [
            "A. Yes",
            "B. No",
            "C. Only for images",
            "D. Only when accuracy is 100%"
        ],
        "answer": "B"
    },
    {
        "question":
        "5. Why can overfitting increase membership risk?",
        "options": [
            "A. It can make model behavior more distinguishable between seen and unseen records",
            "B. It encrypts the training data",
            "C. It prevents model predictions",
            "D. It removes confidence scores"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 65)
print("MEMBERSHIP INFERENCE MINI LAB CHECK")
print("=" * 65)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = score / len(questions) * 100

print("\n" + "=" * 65)

print(
    f"Lab Score: "
    f"{score}/{len(questions)}"
)

print(
    f"Percentage: "
    f"{percentage:.1f}%"
)

if percentage >= 80:
    print(
        "\n✓ Membership inference mini lab complete."
    )
else:
    print(
        "\nReview the experiment and "
        "try the lab check again."
    )

print("=" * 65)

<details>

<summary><b>Section 4 — Practical Takeaways — Click to Expand</b></summary>

<br>

In this section, you moved from attacks that target model functionality to an attack that targets **training-data privacy**.

The experiment followed:

    TRAIN TARGET MODEL
           ↓
    IDENTIFY KNOWN MEMBERS
           ↓
    IDENTIFY KNOWN NON-MEMBERS
           ↓
    OBSERVE MODEL CONFIDENCE
           ↓
    FIND BEHAVIORAL DIFFERENCE
           ↓
    PREDICT MEMBERSHIP
           ↓
    COMPARE AGAINST RANDOM GUESSING

The model was intentionally trained under conditions where it could fit its training records closely.

This created a **generalization gap** between records the model had seen and records it had not seen.

The attack then attempted to exploit that difference.

Importantly:

    HIGH CONFIDENCE
          ≠
    PROOF OF MEMBERSHIP

Instead, confidence was used as statistical evidence.

The attack only becomes meaningful if it can distinguish members from non-members better than an appropriate baseline.

You also saw why overfitting can become more than a model-quality problem.

    OVERFITTING
         ↓
    DIFFERENT BEHAVIOR
         ↓
    MEMBERSHIP SIGNAL
         ↓
    PRIVACY RISK

More sophisticated membership inference attacks may combine multiple signals or use shadow models rather than relying on one confidence threshold.

However, the underlying security lesson remains the same:

> **If a model behaves measurably differently on records it has seen during training, those differences may reveal information about training-data membership.**

The next privacy attack, **Model Inversion**, goes beyond asking whether a known record participated in training and examines whether information about learned data can be inferred or reconstructed from the model.

</details>

In [ ]:
# @title ▶ Section 4 Checkpoint — Membership Inference { display-mode: "form" }

questions = [
    {
        "question":
        "1. What question does a membership inference attack attempt to answer?",
        "options": [
            "A. What architecture does the model use?",
            "B. Was a particular record used to train the model?",
            "C. How many layers does the model contain?",
            "D. Can the model be deleted?"
        ],
        "answer": "B"
    },
    {
        "question":
        "2. Why can overfitting increase membership inference risk?",
        "options": [
            "A. The model may behave differently on training and unseen records",
            "B. Overfitting encrypts the training dataset",
            "C. Overfitting prevents predictions",
            "D. It automatically reveals every training record"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. Which could provide a membership signal?",
        "options": [
            "A. Prediction confidence",
            "B. Notebook title",
            "C. File extension",
            "D. Screen resolution"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. What is the purpose of a shadow model in membership inference?",
        "options": [
            "A. Learn differences between known members and non-members",
            "B. Delete the target",
            "C. Encrypt target predictions",
            "D. Increase the target's query limit"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. How does membership inference differ from model inversion?",
        "options": [
            "A. Membership asks whether a record was used; inversion attempts to infer or reconstruct information",
            "B. They are exactly the same attack",
            "C. Membership always modifies the model",
            "D. Inversion only counts queries"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. On a balanced member/non-member dataset, what attack accuracy would random guessing achieve approximately?",
        "options": [
            "A. 0%",
            "B. 25%",
            "C. 50%",
            "D. 100%"
        ],
        "answer": "C"
    }
]

score = 0

print("=" * 65)
print("SECTION 4 CHECKPOINT")
print("=" * 65)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = score / len(questions) * 100

print("\n" + "=" * 65)
print(
    f"Score: {score}/{len(questions)}"
)
print(
    f"Percentage: {percentage:.1f}%"
)

if percentage >= 80:
    print(
        "\n✓ Checkpoint complete. "
        "Continue to the membership inference mini lab."
    )
else:
    print(
        "\nReview Section 4 before continuing."
    )

print("=" * 65)

---

<details>

<summary><b>Section 5 — Model Inversion — Click to Expand</b></summary>

<br>

Machine learning models learn patterns from their training data.

Those learned patterns are stored indirectly through the model's parameters and decision behavior.

This creates an important privacy question:

> **Can an attacker use a trained model to infer information about what the model learned?**

A **Model Inversion Attack** attempts to infer or reconstruct information associated with the data or features learned by a machine learning model.

Conceptually:

    Trained Model
          ↓
    Observe / Analyze
    Model Behavior
          ↓
    Optimize an Input
          ↓
    Reconstructed or
    Representative Information

Model inversion differs from membership inference.

Membership inference asks:

    "Was this particular record
     used during training?"

Model inversion asks something closer to:

    "What information can I infer
     from what the model learned?"

The attacker may attempt to recover representative features, attributes, or other information associated with a model's learned classes.

<br>

<details>

<summary><b>5.1 — From Membership to Reconstruction</b></summary>

<br>

Membership inference begins with a candidate record.

For example:

    Candidate Record
          ↓
    Target Model
          ↓
    Membership Attack
          ↓
    MEMBER / NON-MEMBER

The attacker already possesses the candidate record.

They want to determine whether it participated in training.

Model inversion approaches the problem differently.

The attacker may begin with:

    Target Class
          ↓
    Model Information
          ↓
    Reconstruction Process
          ↓
    Inferred Features

The attacker attempts to use information contained in the model to learn something about the data patterns associated with that class.

Therefore:

    MEMBERSHIP INFERENCE
            =
    Participation Question

while:

    MODEL INVERSION
            =
    Information Reconstruction
    or Inference Question

Both represent privacy concerns, but their objectives are different.

</details>

<br>

<details>

<summary><b>5.2 — What Does the Attacker Reconstruct?</b></summary>

<br>

It is important not to assume that model inversion automatically reproduces an exact training record.

Depending on the model, data, access level, and attack technique, an inversion attack might reveal:

- Representative features
- Class characteristics
- Sensitive attributes
- Approximate input patterns
- Information correlated with training data

For an image classifier, an inversion attack might attempt to create an image that strongly represents what the model associates with a particular class.

For another model, the attacker might attempt to infer sensitive attributes associated with an input or output.

Therefore, a reconstructed result should not automatically be described as:

> "This is the original training record."

A safer interpretation is:

> **The reconstruction reveals information encoded in the model's learned behavior.**

Whether that information corresponds closely to individual training records must be demonstrated separately.

</details>

<br>

<details>

<summary><b>5.3 — Models Learn Representations</b></summary>

<br>

During training, a machine learning model attempts to learn relationships between inputs and outputs.

For an image classifier:

    Images
      ↓
    Training
      ↓
    Learned Parameters
      ↓
    Class Predictions

The model may learn that certain visual patterns are associated with particular classes.

For example, a handwritten-digit classifier learns patterns useful for distinguishing:

    0

from:

    1

from:

    2

and so on.

Those patterns influence the model's predictions.

An attacker may attempt to work backward from:

    Desired Prediction

toward:

    Input That Strongly
    Produces That Prediction

This is one way to understand model inversion.

</details>

<br>

<details>

<summary><b>5.4 — Optimization-Based Inversion</b></summary>

<br>

One approach to model inversion begins with an input that contains little useful information.

For example:

    Random Noise

The attacker selects a target class:

    Target = Class 7

The attacker then repeatedly modifies the input so that the model becomes increasingly confident that the input belongs to Class 7.

Conceptually:

    Random Input
         ↓
    Model Prediction
         ↓
    How can the input change
    toward Class 7?
         ↓
    Modify Input
         ↓
    Predict Again
         ↓
       Repeat

Eventually:

    Optimized Input
          ↓
    High Confidence
    for Class 7

The resulting input may reveal patterns that the model associates with the selected class.

This resembles adversarial optimization in some ways, but the objective is different.

</details>

<br>

<details>

<summary><b>5.5 — Inversion vs. Adversarial Evasion</b></summary>

<br>

Both adversarial attacks and model inversion can involve modifying inputs using information from the model.

However, their security objectives differ.

### Adversarial Evasion

Start with:

    Legitimate Input

Then:

    Small Manipulation

Goal:

    Cause Incorrect Prediction

Conceptually:

    Real 7
      ↓
    Small Perturbation
      ↓
    Model Says 3

### Model Inversion

Start with something such as:

    Random Input

Then:

    Repeated Optimization

Goal:

    Produce an input representing
    what the model associates
    with a target class

Conceptually:

    Random Noise
        ↓
    Optimize Toward "7"
        ↓
    Class-Representative Pattern

The same mathematical tools can therefore support attacks with very different objectives.

</details>

<br>

<details>

<summary><b>5.6 — White-Box Inversion</b></summary>

<br>

A white-box attacker may have access to:

    Model Architecture

    Model Parameters

    Gradients

Gradient information can help determine how an input should change to increase the model's confidence in a selected class.

The process can resemble:

    Current Input
          ↓
    Calculate Target-Class Loss
          ↓
    Calculate Gradient
          ↓
    Modify Input
          ↓
    Increase Target Confidence
          ↓
        Repeat

Because the attacker can directly inspect the model's gradients, optimization can be performed efficiently.

Our introductory lab will use this approach because it makes the inversion process easy to observe.

</details>

<br>

<details>

<summary><b>5.7 — Black-Box Considerations</b></summary>

<br>

Model inversion does not always require complete white-box access.

Depending on the system, an attacker may attempt to infer information through repeated queries.

For example:

    Candidate Input
          ↓
    Prediction API
          ↓
    Confidence Score
          ↓
    Modify Candidate
          ↓
    Query Again

The attacker can repeatedly search for inputs that produce increasingly strong target responses.

This can require substantially more queries because the attacker does not directly receive gradient information.

Once again, detailed model outputs may increase the amount of information available through the prediction interface.

</details>

<br>

<details>

<summary><b>5.8 — Confidence Scores and Information Exposure</b></summary>

<br>

Consider two APIs.

API A returns:

    Class 7

API B returns:

    Class 7: 0.94
    Class 1: 0.03
    Class 9: 0.02
    Other:   0.01

API B tells the attacker considerably more about how strongly the model responds to the current input.

During iterative reconstruction, an attacker could use that information to determine whether a modification moved the input:

    Closer to Target Behavior

or:

    Farther from Target Behavior

This reinforces a recurring principle from earlier sections:

> **Model outputs can become a source of information for an attacker.**

This does not mean confidence information should never be exposed.

Instead, defenders should consider whether the application actually requires that information.

</details>

<br>

<details>

<summary><b>5.9 — Interpreting Reconstructions Carefully</b></summary>

<br>

Suppose an inversion attack against a handwritten-digit classifier produces an image that resembles a 7.

What has the attacker demonstrated?

They have shown that:

    The Model
        ↓
    Contains Learned Information
        ↓
    That Can Guide an Input
        ↓
    Toward a Representation
    Associated With "7"

They have **not automatically proven** that the reconstructed image is an exact copy of a particular training image.

This distinction is extremely important when discussing model inversion.

A reconstruction may represent:

    Class-Level Information

rather than:

    Exact Individual Record

Stronger claims require stronger evidence.

When evaluating an inversion attack, always ask:

> **What information was actually recovered?**

</details>

<br>

<details>

<summary><b>5.10 — Measuring Inversion Success</b></summary>

<br>

A model inversion attack should not be considered successful simply because an image or other output was generated.

Possible measurements include:

    Target-Class Confidence

    Reconstruction Similarity

    Feature Recovery

    Attribute Recovery Accuracy

    Visual Interpretability

For our introductory experiment, we will measure:

    Starting Target Confidence

and:

    Final Target Confidence

If optimization changes:

    Target Confidence = 0.10

into:

    Target Confidence = 0.98

then we have evidence that the optimization successfully constructed an input strongly associated with the target class.

We can also inspect the resulting image.

However, remember:

    High Target Confidence
            ≠
    Exact Training Record Recovery

The result must be interpreted according to what the experiment actually demonstrates.

</details>

<br>

<details>

<summary><b>5.11 — Defensive Considerations</b></summary>

<br>

Defending against privacy leakage from trained models can require multiple controls.

Depending on the system, defenders may consider:

- Restricting unnecessary model access
- Limiting unnecessary confidence information
- Query monitoring
- Rate limiting
- Improving generalization
- Privacy-preserving training
- Differential privacy

Differential privacy is particularly relevant because it attempts to limit how strongly individual training records influence the final model.

However, no single technique should automatically be assumed to prevent every form of inversion.

Defenses must be evaluated against the specific attack and threat model.

</details>

<br>

<details>

<summary><b>Section 5 — Key Takeaways</b></summary>

<br>

Model inversion attempts to infer or reconstruct information from what a machine learning model has learned.

It differs from membership inference:

    MEMBERSHIP INFERENCE
            ↓
    Was this record
    used during training?

    MODEL INVERSION
            ↓
    What information can
    be inferred from the model?

One inversion approach begins with a random input and repeatedly modifies it to increase the model's confidence in a selected class.

The resulting input can reveal patterns the model associates with that class.

However:

> **A reconstructed class representation should not automatically be treated as an exact recovered training record.**

The central lesson is:

> **Trained models can contain information about their training patterns, and attackers may attempt to recover some of that information by analyzing or optimizing against model behavior.**

</details>

</details>

In [ ]:
# @title Section 5 — Model Inversion Attack Visualizations { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from IPython.display import display, HTML
from io import BytesIO
import base64
import html

# ============================================================
# CONFIGURATION
# ============================================================

rng = np.random.default_rng(42)

BLUE = "#2563EB"
GREEN = "#16A34A"
RED = "#DC2626"
ORANGE = "#EA580C"
PURPLE = "#7C3AED"
GRAY = "#64748B"
DARK = "#0F172A"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# ============================================================
# COLLAPSIBLE VISUALIZATION HELPER
# ============================================================

def show_collapsible_figure(title, fig, explanation, color=BLUE):

    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=135,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    buffer.close()

    display(HTML(f"""
    <details style="
        margin:12px 0;
        border:1px solid #cbd5e1;
        border-radius:10px;
        overflow:hidden;
    ">

        <summary style="
            cursor:pointer;
            padding:16px;
            font-size:16px;
            font-weight:bold;
            background:#eff6ff;
            color:#1e3a8a;
            border-left:5px solid {color};
        ">
            {html.escape(title)} — Click to Expand
        </summary>

        <div style="
            padding:15px;
            background:white;
            text-align:center;
        ">

            <img
                src="data:image/png;base64,{encoded}"
                style="width:100%;max-width:950px;height:auto;"
            >

            <div style="
                max-width:850px;
                margin:15px auto;
                padding:16px;
                background:#f8fafc;
                border-radius:8px;
                color:#334155;
                font-size:14px;
                line-height:1.7;
                text-align:left;
            ">
                {explanation}
            </div>

        </div>
    </details>
    """))


# ============================================================
# DIAGRAM HELPERS
# ============================================================

def add_card(ax, x, y, w, h, title, subtitle="",
             accent=BLUE, background="#F8FAFC"):

    card = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        facecolor=background,
        edgecolor=accent,
        linewidth=1.7
    )

    ax.add_patch(card)

    ax.text(
        x + w / 2,
        y + h * (0.65 if subtitle else 0.5),
        title,
        ha="center",
        va="center",
        fontsize=10.5,
        fontweight="bold",
        color=DARK
    )

    if subtitle:
        ax.text(
            x + w / 2,
            y + h * 0.28,
            subtitle,
            ha="center",
            va="center",
            fontsize=9,
            color=GRAY
        )


def add_arrow(ax, start, end, color=GRAY):

    ax.add_patch(
        FancyArrowPatch(
            start, end,
            arrowstyle="-|>",
            mutation_scale=16,
            linewidth=1.8,
            color=color
        )
    )


# ============================================================
# SYNTHETIC MODEL SETUP
# ============================================================

# This model learns patterns in synthetic tabular data.
# No real personal information is used.

X, y = make_classification(
    n_samples=700,
    n_features=8,
    n_informative=5,
    n_redundant=0,
    class_sep=1.2,
    random_state=42
)

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

model = LogisticRegression(
    max_iter=500,
    random_state=42
)

model.fit(X_scaled, y)

# A representative class-1 sample is used only
# for an illustrative feature comparison.

class_1_samples = X_scaled[y == 1]

representative_sample = class_1_samples[
    np.argmin(
        np.linalg.norm(
            class_1_samples -
            class_1_samples.mean(axis=0),
            axis=1
        )
    )
]


# ============================================================
# VISUALIZATION 1 — MODEL INVERSION WORKFLOW
# ============================================================

fig, ax = plt.subplots(figsize=(10, 8.5))

ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis("off")

ax.set_title(
    "Model Inversion Attack Workflow",
    fontsize=17,
    fontweight="bold"
)

steps = [
    (
        "Target Model",
        "Learns patterns from training data",
        BLUE
    ),
    (
        "Attacker Access",
        "Observes outputs or model information",
        ORANGE
    ),
    (
        "Initial Candidate Input",
        "Starts with an estimated input",
        PURPLE
    ),
    (
        "Reconstruction Process",
        "Adjusts features using model feedback",
        RED
    ),
    (
        "Estimated Information",
        "Produces an approximate reconstruction",
        GREEN
    )
]

for i, (title, subtitle, accent) in enumerate(steps):

    y_pos = 8.2 - i * 1.75

    add_card(
        ax,
        1.2,
        y_pos,
        7.6,
        1.2,
        title,
        subtitle,
        accent
    )

    if i < len(steps) - 1:

        add_arrow(
            ax,
            (5, y_pos - 0.05),
            (5, y_pos - 0.48)
        )

plt.tight_layout()

show_collapsible_figure(
    "5.1 — Understanding Model Inversion",
    fig,
    """
    Model inversion attacks attempt to reconstruct
    information about inputs or training-data
    characteristics using information exposed
    by a machine learning model.

    <br><br>

    An attacker may begin with an initial
    candidate input and repeatedly modify
    its features based on model feedback.

    <br><br>

    The objective is to produce an input
    that satisfies an attacker-selected
    reconstruction goal.

    <br><br>

    Depending on the model and attack,
    the result may resemble representative
    class features or, in more serious cases,
    reveal information associated with
    specific training records.

    <br><br>

    <b>Key takeaway:</b> A model's outputs
    may reveal information beyond the
    prediction that the system was
    intended to provide.
    """
)


# ============================================================
# VISUALIZATION 2 — INVERSION VS MEMBERSHIP INFERENCE
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

fig.suptitle(
    "Model Inversion vs. Membership Inference",
    fontsize=17,
    fontweight="bold"
)

comparisons = [
    (
        "Membership Inference",
        BLUE,
        [
            "Candidate Record",
            "Model Output",
            "Membership Analysis",
            "Member / Non-Member"
        ]
    ),
    (
        "Model Inversion",
        PURPLE,
        [
            "Target Model",
            "Candidate Input",
            "Reconstruction Process",
            "Estimated Features"
        ]
    )
]

for ax, (title, accent, steps) in zip(
    axes, comparisons
):

    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis("off")

    ax.set_title(
        title,
        fontsize=14,
        color=accent,
        fontweight="bold"
    )

    for i, step in enumerate(steps):

        y_pos = 7.7 - i * 2.0

        add_card(
            ax,
            0.8,
            y_pos,
            8.4,
            1.2,
            step,
            accent=accent
        )

        if i < len(steps) - 1:

            add_arrow(
                ax,
                (5, y_pos - 0.05),
                (5, y_pos - 0.65)
            )

plt.tight_layout()

show_collapsible_figure(
    "5.2 — Model Inversion vs. Membership Inference",
    fig,
    """
    Membership inference and model inversion
    are both privacy-related attacks,
    but they attempt to answer different
    questions.

    <br><br>

    <b>Membership inference:</b>
    Was a particular record included
    in the model's training dataset?

    <br><br>

    <b>Model inversion:</b>
    What information about inputs,
    features, or training-data
    characteristics can be inferred
    from the model?

    <br><br>

    Membership inference generally
    produces a membership prediction.

    <br><br>

    Model inversion attempts to
    reconstruct information.

    <br><br>

    <b>Key takeaway:</b> Determining
    whether a record was used during
    training is different from
    reconstructing information
    associated with that record.
    """
)


# ============================================================
# VISUALIZATION 3 — CONFIDENCE-GUIDED RECONSTRUCTION
# ============================================================

# A simple confidence-guided search.
#
# The attacker starts with a random feature vector
# and searches for an input that increases the
# model's confidence in class 1.
#
# This is a toy model-feedback optimization example,
# not a demonstration of recovering a training record.

candidate = rng.normal(
    0,
    0.7,
    size=X_scaled.shape[1]
)

best_confidence = model.predict_proba(
    candidate.reshape(1, -1)
)[0, 1]

confidence_history = [best_confidence]

best_candidates = [candidate.copy()]

iterations = 100

for iteration in range(iterations):

    # Propose a small modification.
    proposal = candidate + rng.normal(
        0,
        0.25,
        size=candidate.shape
    )

    # Keep features within a reasonable
    # synthetic standardized range.
    proposal = np.clip(
        proposal,
        -2.5,
        2.5
    )

    proposal_confidence = model.predict_proba(
        proposal.reshape(1, -1)
    )[0, 1]

    # Accept only confidence improvements.
    if proposal_confidence > best_confidence:

        candidate = proposal
        best_confidence = proposal_confidence

    confidence_history.append(
        best_confidence
    )

    best_candidates.append(
        candidate.copy()
    )

reconstructed_candidate = candidate.copy()

fig, ax = plt.subplots(figsize=(10, 5.5))

ax.plot(
    confidence_history,
    color=PURPLE,
    linewidth=2.5
)

ax.scatter(
    0,
    confidence_history[0],
    color=BLUE,
    s=100,
    label="Initial Candidate",
    zorder=5
)

ax.scatter(
    iterations,
    confidence_history[-1],
    color=GREEN,
    s=110,
    label="Final Candidate",
    zorder=5
)

ax.set_ylim(0, 1.05)

ax.set_xlabel("Optimization Iteration")
ax.set_ylabel("Target Class Confidence")

ax.set_title(
    "Confidence-Guided Candidate Reconstruction",
    fontsize=16,
    fontweight="bold"
)

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "5.3 — Confidence-Guided Reconstruction",
    fig,
    """
    Some model inversion approaches use
    model feedback to guide the search
    for a candidate input.

    <br><br>

    In this experiment, the attacker
    begins with a random synthetic
    feature vector.

    <br><br>

    At each iteration, the code
    proposes a small modification
    and queries the local classifier.

    <br><br>

    If the modified input increases
    the model's confidence in the
    target class, the modification
    is accepted.

    <br><br>

    The graph shows how the best
    observed confidence changes
    during the search.

    <br><br>

    <b>Important:</b> Increasing model
    confidence does not demonstrate
    that an actual training record
    has been recovered.

    <br><br>

    This is a simplified demonstration
    of model-guided input optimization.
    """,
    color=PURPLE
)


# ============================================================
# VISUALIZATION 4 — REPRESENTATIVE VS RECONSTRUCTED FEATURES
# ============================================================

feature_names = [
    f"Feature {i + 1}"
    for i in range(X_scaled.shape[1])
]

positions = np.arange(
    len(feature_names)
)

width = 0.36

fig, ax = plt.subplots(figsize=(12, 5.5))

ax.bar(
    positions - width / 2,
    representative_sample,
    width,
    color=BLUE,
    alpha=0.8,
    label="Representative Class-1 Sample"
)

ax.bar(
    positions + width / 2,
    reconstructed_candidate,
    width,
    color=ORANGE,
    alpha=0.8,
    label="Optimized Candidate"
)

ax.set_xticks(positions)
ax.set_xticklabels(feature_names)

ax.set_ylabel("Standardized Feature Value")

ax.set_title(
    "Representative Features vs. Optimized Candidate",
    fontsize=16,
    fontweight="bold"
)

ax.axhline(
    0,
    color=GRAY,
    linewidth=1
)

ax.legend()
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "5.4 — Comparing Reconstructed Features",
    fig,
    """
    A reconstruction attempt may produce
    feature values that differ from
    actual training examples.

    <br><br>

    The blue bars represent a real
    synthetic sample selected from
    the model's training data.

    <br><br>

    The orange bars represent the
    candidate produced by the
    confidence-guided search.

    <br><br>

    Both inputs may receive a
    prediction for the same class,
    even if their individual
    feature values are different.

    <br><br>

    This demonstrates an important
    limitation of simple confidence
    maximization.

    <br><br>

    <b>High confidence does not mean
    that the reconstructed features
    match a particular training record.</b>

    <br><br>

    More advanced inversion attacks
    may use additional information,
    constraints, or prior knowledge
    to produce more meaningful results.
    """
)


# ============================================================
# VISUALIZATION 5 — QUERY BUDGET AND RECONSTRUCTION
# ============================================================

# Evaluate best confidence after increasing query counts.
# This is measured from the synthetic optimization above.

query_budgets = [
    0, 5, 10, 20, 40, 60, 80, 100
]

budget_confidences = [
    confidence_history[q]
    for q in query_budgets
]

fig, ax = plt.subplots(figsize=(10, 5.5))

ax.plot(
    query_budgets,
    budget_confidences,
    marker="o",
    color=GREEN,
    linewidth=2.5,
    markersize=8
)

ax.set_ylim(0, 1.05)

ax.set_xlabel("Number of Candidate Queries")
ax.set_ylabel("Best Observed Target-Class Confidence")

ax.set_title(
    "Query Budget and Confidence Optimization",
    fontsize=16,
    fontweight="bold"
)

ax.grid(alpha=0.2)

for q, confidence in zip(
    query_budgets,
    budget_confidences
):

    ax.annotate(
        f"{confidence:.2f}",
        (q, confidence),
        xytext=(0, 9),
        textcoords="offset points",
        ha="center",
        fontsize=9
    )

plt.tight_layout()

show_collapsible_figure(
    "5.5 — Query Budget and Reconstruction Attempts",
    fig,
    """
    A model-guided reconstruction process
    may require repeated queries to
    improve a candidate input.

    <br><br>

    This graph measures the best
    target-class confidence observed
    after different numbers of queries.

    <br><br>

    In this example, the attacker
    keeps the best candidate
    discovered so far.

    <br><br>

    Therefore, the displayed
    confidence cannot decrease.

    <br><br>

    A larger query budget gives
    the search more opportunities
    to find a high-confidence input.

    <br><br>

    However, more queries do not
    guarantee successful reconstruction
    of private training information.

    <br><br>

    <b>Key takeaway:</b> Query budgets
    can affect an attacker's ability
    to explore model behavior, but
    confidence and reconstruction
    accuracy are different measurements.
    """,
    color=GREEN
)


# ============================================================
# VISUALIZATION 6 — MODEL INVERSION DEFENSES
# ============================================================

fig, ax = plt.subplots(figsize=(11, 9))

ax.set_xlim(0, 10)
ax.set_ylim(0, 11)
ax.axis("off")

ax.set_title(
    "Layered Defenses Against Model Inversion",
    fontsize=17,
    fontweight="bold"
)

defenses = [
    (
        "Access Control",
        "Restrict unauthorized model access",
        BLUE
    ),
    (
        "Output Exposure Controls",
        "Limit unnecessary confidence details",
        ORANGE
    ),
    (
        "Query Monitoring and Rate Limits",
        "Investigate unusual reconstruction patterns",
        PURPLE
    ),
    (
        "Privacy-Preserving Training",
        "Consider differential privacy where appropriate",
        GREEN
    ),
    (
        "Memorization and Privacy Testing",
        "Evaluate possible information leakage",
        RED
    ),
    (
        "Response and Model Governance",
        "Investigate findings and improve safeguards",
        BLUE
    )
]

for i, (title, subtitle, accent) in enumerate(defenses):

    y_pos = 9.4 - i * 1.7

    add_card(
        ax,
        1.0,
        y_pos,
        8.0,
        1.2,
        title,
        subtitle,
        accent
    )

    if i < len(defenses) - 1:

        add_arrow(
            ax,
            (5, y_pos - 0.05),
            (5, y_pos - 0.45)
        )

plt.tight_layout()

show_collapsible_figure(
    "5.6 — Defending Against Model Inversion",
    fig,
    """
    Model inversion defenses attempt
    to reduce the information an
    attacker can learn from model
    access and observable outputs.

    <br><br>

    <b>Access controls</b> limit who
    can interact with sensitive models.

    <br><br>

    <b>Output exposure controls</b>
    reduce unnecessary information
    returned by prediction interfaces.

    <br><br>

    <b>Rate limits and monitoring</b>
    can help identify suspicious
    sequences of queries.

    <br><br>

    <b>Differential privacy</b> can
    provide formal privacy protections
    during training when implemented
    with an appropriate privacy budget
    and threat model.

    <br><br>

    <b>Privacy testing</b> helps
    defenders evaluate whether a
    model exposes information that
    should remain private.

    <br><br>

    <b>Important:</b> Limiting confidence
    scores or applying rate limits
    does not guarantee protection
    against every inversion technique.

    <br><br>

    <b>Key takeaway:</b> Protecting
    training-data privacy requires
    reducing unnecessary exposure
    and evaluating model leakage.
    """,
    color=RED
)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print("Section 5 visualizations are ready.")
print("Six collapsible visualizations generated.")
print("All experiments use synthetic data.")

print()
print("Confidence-Guided Reconstruction")
print("--------------------------------")

print(
    "Initial Target-Class Confidence:",
    round(confidence_history[0], 4)
)

print(
    "Final Target-Class Confidence:",
    round(confidence_history[-1], 4)
)

print(
    "Total Candidate Queries:",
    iterations
)

print()
print(
    "Note: Higher confidence does not establish "
    "successful recovery of a training record."
)

<details>

<summary><b>Section 5 Lab — Click to Expand</b></summary>

<br>

In this activity, you will perform a simplified **white-box model inversion demonstration**.

You will train a small neural network to recognize handwritten digits.

Then you will select one target class, such as:

    7

Instead of giving the model an actual image of a 7, you will begin with a randomly generated image.

Initially:

    RANDOM IMAGE
          ↓
       MODEL
          ↓
    Low Confidence
    for Target Class

You will then use the model's gradients to repeatedly modify the input.

The optimization objective is:

> **Increase the model's confidence that this input belongs to the selected target class.**

The process becomes:

    Random Image
         ↓
    Calculate Gradient
         ↓
    Modify Image
         ↓
    Measure Target Confidence
         ↓
       Repeat
         ↓
    Class-Representative Image

You will compare the target confidence before and after optimization.

The important interpretation is that the resulting image represents information the model associates with the target class.

It should **not** automatically be interpreted as an exact recovered training record.

</details>

In [ ]:
# @title ▶ Part 1 — Load MNIST { display-mode: "form" }

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

(X_train, y_train), (X_test, y_test) = (
    tf.keras.datasets.mnist.load_data()
)

X_train = (
    X_train.astype("float32") / 255.0
)

X_test = (
    X_test.astype("float32") / 255.0
)

X_train = np.expand_dims(
    X_train,
    -1
)

X_test = np.expand_dims(
    X_test,
    -1
)

print("=" * 55)
print("MNIST LOADED")
print("=" * 55)

print(
    f"\nTraining Samples: "
    f"{len(X_train)}"
)

print(
    f"Testing Samples: "
    f"{len(X_test)}"
)

print(
    f"Image Shape: "
    f"{X_train.shape[1:]}"
)

<details>

<summary><b>Part 1 Explanation — Click to Expand</b></summary>

<br>

We again use MNIST because it provides a simple visual environment for observing the attack.

The dataset contains handwritten digits:

    0 through 9

Images are normalized to:

    0 ≤ pixel ≤ 1

Nothing has been inverted yet.

We first need a trained target model that has learned patterns associated with each digit.

</details>

In [ ]:
# @title ▶ Part 2 — Train the Target Model { display-mode: "form" }

model = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(28, 28, 1)
    ),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        128,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        10
    )
])

model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=["accuracy"]
)

model.fit(
    X_train,
    y_train,
    epochs=2,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

_, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("=" * 55)
print("TARGET MODEL")
print("=" * 55)

print(
    f"\nTest Accuracy: "
    f"{test_accuracy:.3f}"
)

print(
    "\n✓ Target model ready."
)

<details>

<summary><b>Part 2 Explanation — Click to Expand</b></summary>

<br>

This creates a small neural network similar to the one used in the adversarial attack activity.

During training, the model learns patterns that help distinguish digits.

Conceptually:

    Training Images
          ↓
    Neural Network
          ↓
    Learned Parameters
          ↓
    Digit Recognition

The inversion attack will now use those learned parameters in the opposite direction.

Instead of asking:

> "What class does this image represent?"

we will ask:

> "What kind of input causes the model to strongly predict a particular class?"

</details>

In [ ]:
# @title ▶ Part 3 — Choose the Inversion Target { display-mode: "form" }

# ==========================================
# STUDENT-EDITABLE TARGET
# ==========================================

TARGET_CLASS = 7

# ==========================================

if TARGET_CLASS < 0 or TARGET_CLASS > 9:
    raise ValueError(
        "TARGET_CLASS must be between 0 and 9."
    )

# Begin with low-intensity random noise.
initial_image = tf.random.uniform(
    shape=(1, 28, 28, 1),
    minval=0.0,
    maxval=0.25,
    seed=42
)

reconstructed_image = tf.Variable(
    initial_image
)

initial_logits = model(
    reconstructed_image,
    training=False
)

initial_probability = tf.nn.softmax(
    initial_logits,
    axis=1
)[0, TARGET_CLASS].numpy()

print("=" * 60)
print("INVERSION STARTING POINT")
print("=" * 60)

print(
    f"\nTarget Class: "
    f"{TARGET_CLASS}"
)

print(
    f"Starting Target Confidence: "
    f"{initial_probability:.4f}"
)

plt.figure(figsize=(4, 4))

plt.imshow(
    initial_image[0, :, :, 0],
    cmap="gray",
    vmin=0,
    vmax=1
)

plt.title(
    "Starting Random Input"
)

plt.axis("off")
plt.show()

<details>

<summary><b>Part 3 Explanation — Click to Expand</b></summary>

<br>

The attacker selects a target class.

By default:

    TARGET_CLASS = 7

You can later change this to another digit from:

    0 through 9

The attack begins with random noise.

This is important because we are **not starting with an actual image of the target digit**.

Initially, the model should have relatively little reason to believe that the random image represents the selected class.

The next step will repeatedly modify this input to increase:

    Model Confidence
          ↓
    TARGET_CLASS

</details>

In [ ]:
# @title ▶ Part 4 — Run Model Inversion { display-mode: "form" }

# ==========================================
# STUDENT-EDITABLE SETTINGS
# ==========================================

STEPS = 150
LEARNING_RATE = 0.08

# ==========================================

confidence_history = []

for step in range(STEPS):

    with tf.GradientTape() as tape:

        logits = model(
            reconstructed_image,
            training=False
        )

        probabilities = tf.nn.softmax(
            logits,
            axis=1
        )

        target_confidence = probabilities[
            0,
            TARGET_CLASS
        ]

        # Minimize the negative target logit.
        # This pushes the image toward the
        # selected class.
        loss = -logits[
            0,
            TARGET_CLASS
        ]

    gradient = tape.gradient(
        loss,
        reconstructed_image
    )

    # Normalize gradient so updates remain stable.
    gradient = gradient / (
        tf.math.reduce_std(gradient)
        + 1e-8
    )

    reconstructed_image.assign_sub(
        LEARNING_RATE * gradient
    )

    reconstructed_image.assign(
        tf.clip_by_value(
            reconstructed_image,
            0.0,
            1.0
        )
    )

    confidence_history.append(
        float(target_confidence)
    )

final_logits = model(
    reconstructed_image,
    training=False
)

final_probabilities = tf.nn.softmax(
    final_logits,
    axis=1
)

final_confidence = final_probabilities[
    0,
    TARGET_CLASS
].numpy()

final_prediction = tf.argmax(
    final_probabilities,
    axis=1
).numpy()[0]

print("=" * 60)
print("MODEL INVERSION RESULTS")
print("=" * 60)

print(
    f"\nTarget Class: "
    f"{TARGET_CLASS}"
)

print(
    f"Starting Confidence: "
    f"{initial_probability:.4f}"
)

print(
    f"Final Confidence:    "
    f"{final_confidence:.4f}"
)

print(
    f"Final Prediction:    "
    f"{final_prediction}"
)

print(
    f"Confidence Increase: "
    f"{final_confidence - initial_probability:+.4f}"
)

if (
    final_prediction == TARGET_CLASS
    and
    final_confidence > initial_probability
):

    print(
        "\n✓ Optimization successfully moved "
        "the input toward the target class."
    )

else:

    print(
        "\n△ Target optimization was weak. "
        "Try increasing STEPS."
    )

<details>

<summary><b>Part 4 Explanation — Click to Expand</b></summary>

<br>

This cell performs the inversion optimization.

For every step:

    Current Image
          ↓
    Target Model
          ↓
    Target-Class Score
          ↓
    Calculate Gradient
          ↓
    Modify Image
          ↓
       Repeat

The gradient tells us how the pixels can change to increase the model's response for the selected class.

The image is then clipped so every pixel remains within:

    0 ≤ pixel ≤ 1

Two settings control the process.

### STEPS

Controls how many times the image is updated.

### LEARNING_RATE

Controls how strongly the image changes during each update.

The attack succeeds at its immediate optimization objective if:

    Final Target Confidence
              >
    Starting Target Confidence

and the final prediction moves toward the selected target.

This still does **not** mean an exact training image has been recovered.

</details>

In [ ]:
# @title ▶ Part 5 — View the Reconstructed Representation { display-mode: "form" }

plt.figure(figsize=(4, 4))

plt.imshow(
    reconstructed_image.numpy()[
        0, :, :, 0
    ],
    cmap="gray",
    vmin=0,
    vmax=1
)

plt.title(
    f"Reconstructed Representation\n"
    f"Target = {TARGET_CLASS} | "
    f"Confidence = {final_confidence:.3f}"
)

plt.axis("off")
plt.show()

plt.figure(figsize=(7, 4))

plt.plot(
    confidence_history
)

plt.xlabel(
    "Optimization Step"
)

plt.ylabel(
    "Target-Class Confidence"
)

plt.title(
    "Target Confidence During Inversion"
)

plt.show()

<details>

<summary><b>Part 5 Explanation — Click to Expand</b></summary>

<br>

The first graph shows the optimized input.

Ask yourself:

> **Does the image contain characteristics that seem related to the selected digit?**

The second graph shows how the model's confidence changed during optimization.

Ideally, you should see:

    Low Initial Confidence
            ↓
    Repeated Optimization
            ↓
    Higher Target Confidence

This provides measurable evidence that the model's learned parameters guided the input toward something strongly associated with the selected class.

However, interpret the image carefully.

You have demonstrated:

    Learned Model Information
            ↓
    Can Guide Reconstruction

You have **not demonstrated**:

    Exact Original
    Training Image Recovered

That distinction is essential when evaluating model inversion claims.

</details>

In [ ]:
# @title ▶ Part 6 — Compare With Real Examples { display-mode: "form" }

target_examples = np.where(
    y_train == TARGET_CLASS
)[0][:5]

for index in target_examples:

    plt.figure(figsize=(3, 3))

    plt.imshow(
        X_train[index].squeeze(),
        cmap="gray"
    )

    plt.title(
        f"Real Training Example — Class {TARGET_CLASS}"
    )

    plt.axis("off")
    plt.show()

<details>

<summary><b>Part 6 Explanation — Click to Expand</b></summary>

<br>

These images are real examples from the selected class.

Compare them with the reconstructed representation.

You may notice general patterns shared by the reconstruction and real examples.

That is exactly the distinction we want to understand.

The inversion result may capture:

    Class Characteristics

or:

    Learned Features

without reproducing:

    One Exact Training Sample

The model learned information from many examples.

Optimization can expose some of those learned patterns, but stronger evidence would be required before claiming that an individual training record was reconstructed.

</details>

<details>

<summary><b>Part 7 — Student Experiment — Click to Expand</b></summary>

<br>

Return to **Part 3** and change:

    TARGET_CLASS

Try several digits, such as:

    0

    1

    3

    7

    8

For each target, rerun:

    Part 3
       ↓
    Part 4
       ↓
    Part 5
       ↓
    Part 6

Observe:

    Starting Confidence

    Final Confidence

    Final Prediction

    Reconstructed Image

You can also experiment with:

    STEPS

and:

    LEARNING_RATE

For example:

    STEPS = 50

    STEPS = 100

    STEPS = 150

Compare the results.

Your goal is not simply to produce the prettiest digit.

Instead, investigate:

> **How does repeated optimization reveal what kinds of input patterns strongly activate the target model?**

</details>

In [ ]:
# @title ▶ Part 8 — Model Inversion Lab Check { display-mode: "form" }

questions = [
    {
        "question":
        "1. What did the inversion attack begin with?",
        "options": [
            "A. A random input",
            "B. An exact copy of a training image",
            "C. The training labels",
            "D. A stolen model file"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. What was the optimization attempting to increase?",
        "options": [
            "A. Confidence in the selected target class",
            "B. Dataset size",
            "C. Number of model layers",
            "D. Training-set membership"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. What information guided changes to the input?",
        "options": [
            "A. Model gradients",
            "B. Random labels",
            "C. File metadata",
            "D. API passwords"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. What does higher final target confidence demonstrate?",
        "options": [
            "A. Optimization found an input the model strongly associates with the target",
            "B. An exact training record was definitely recovered",
            "C. Every training record has been exposed",
            "D. Membership inference succeeded"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. Should the reconstructed image automatically be described as a recovered training image?",
        "options": [
            "A. Yes",
            "B. No",
            "C. Only when using MNIST",
            "D. Only after 100 optimization steps"
        ],
        "answer": "B"
    }
]

score = 0

print("=" * 65)
print("MODEL INVERSION MINI LAB CHECK")
print("=" * 65)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = score / len(questions) * 100

print("\n" + "=" * 65)

print(
    f"Lab Score: "
    f"{score}/{len(questions)}"
)

print(
    f"Percentage: "
    f"{percentage:.1f}%"
)

if percentage >= 80:
    print(
        "\n✓ Model inversion mini lab complete."
    )
else:
    print(
        "\nReview the inversion experiment "
        "and try the lab check again."
    )

print("=" * 65)

<details>

<summary><b>Section 5 — Practical Takeaways — Click to Expand</b></summary>

<br>

In this section, you explored another way that trained models can expose information.

Membership inference asked:

    "Was this record
     used during training?"

Model inversion instead asked:

    "What can we learn from
     what the model learned?"

Your experiment followed:

    TRAIN MODEL
         ↓
    SELECT TARGET CLASS
         ↓
    CREATE RANDOM INPUT
         ↓
    CALCULATE GRADIENTS
         ↓
    MODIFY INPUT
         ↓
    INCREASE TARGET RESPONSE
         ↓
    OBSERVE RECONSTRUCTION

The random input initially contained little useful information.

However, the trained model provided a direction for modifying that input.

After repeated optimization, the model became increasingly confident that the generated input represented the selected target class.

This demonstrates that:

    TRAINING DATA
         ↓
    LEARNED PATTERNS
         ↓
    MODEL PARAMETERS
         ↓
    INFORMATION THAT MAY
    SOMETIMES BE INFERRED

However, the interpretation of an inversion attack is extremely important.

A generated representation that resembles a class does **not automatically prove that an individual training record was recovered**.

The evidence in this experiment supports a narrower conclusion:

> **Information encoded in a trained model can guide the reconstruction of patterns associated with what the model learned.**

That distinction will be important whenever you evaluate claims about privacy attacks against machine learning systems.

</details>

In [ ]:
# @title ▶ Section 5 Checkpoint — Model Inversion { display-mode: "form" }

questions = [
    {
        "question":
        "1. What is the main objective of model inversion?",
        "options": [
            "A. Determine only whether a record was in training",
            "B. Infer or reconstruct information from what the model learned",
            "C. Delete the training dataset",
            "D. Increase API speed"
        ],
        "answer": "B"
    },
    {
        "question":
        "2. How does model inversion differ from membership inference?",
        "options": [
            "A. Inversion attempts reconstruction; membership asks whether a record participated in training",
            "B. They are exactly the same",
            "C. Membership reconstructs images",
            "D. Inversion only measures accuracy"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. In optimization-based inversion, what might the attacker begin with?",
        "options": [
            "A. Random input",
            "B. The exact training dataset",
            "C. A deleted model",
            "D. A poisoned label"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. Why are gradients useful in white-box inversion?",
        "options": [
            "A. They indicate how the input can change toward the target objective",
            "B. They reveal every training record automatically",
            "C. They encrypt the model",
            "D. They reduce the number of classes"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. Does a reconstructed class image automatically prove recovery of an exact training record?",
        "options": [
            "A. Yes",
            "B. No",
            "C. Only for neural networks",
            "D. Only when confidence exceeds 90%"
        ],
        "answer": "B"
    },
    {
        "question":
        "6. Why can confidence scores be useful to an inversion attacker?",
        "options": [
            "A. They can indicate whether modifications move an input toward target behavior",
            "B. They reveal passwords",
            "C. They automatically expose model weights",
            "D. They prevent optimization"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 65)
print("SECTION 5 CHECKPOINT")
print("Model Inversion")
print("=" * 65)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = score / len(questions) * 100

print("\n" + "=" * 65)
print(f"Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print(
        "\n✓ Checkpoint complete. "
        "Continue to the model inversion mini lab."
    )
else:
    print(
        "\nReview Section 5 before continuing."
    )

print("=" * 65)

---

<details>

<summary><b>Section 6 — Transfer Attacks & Surrogate Models — Click to Expand</b></summary>

<br>

Earlier sections introduced two important ideas.

In **Adversarial Evasion**, you learned that an attacker can deliberately manipulate inputs to cause incorrect predictions.

In **Model Extraction**, you learned that an attacker can construct a surrogate model that approximates the behavior of another model.

These concepts can be combined.

An attacker may not have enough access to directly perform a white-box adversarial attack against the real target.

Instead, the attacker can create or obtain another model, develop the adversarial input against that model, and then submit the same manipulated input to the real target.

If the adversarial example also fools the target, the attack has **transferred**.

This is known as an **adversarial transfer attack**.

Conceptually:

    TARGET MODEL
      Black Box
          ↑
          |
    Adversarial Input
          ↑
          |
    SURROGATE MODEL
      White Box
          ↑
          |
    Generate Attack

The attacker performs the difficult attack development against a model they control rather than directly against the target.

<br>

<details>

<summary><b>6.1 — What Is Adversarial Transferability?</b></summary>

<br>

An adversarial example is normally created to exploit the behavior of a particular model.

However, adversarial examples sometimes affect models other than the model used to generate them.

This property is called **transferability**.

Suppose:

    Model A
      ↓
    Generate Adversarial Example
      ↓
    Model A Misclassifies

Now the same adversarial example is sent to:

    Model B
      ↓
    Model B Also Misclassifies

The adversarial example has transferred from:

    Model A → Model B

This is important because an attacker may not need complete knowledge of the actual target.

Instead, the attacker may develop the attack using another model and attempt to transfer the result.

</details>

<br>

<details>

<summary><b>6.2 — Why Can Attacks Transfer?</b></summary>

<br>

Different machine learning models trained to solve the same task may learn some similar patterns.

Consider two image classifiers trained to recognize handwritten digits.

Even if:

    Model A
    Neural Network

and:

    Model B
    Different Neural Network

they are still learning from similar relationships between:

    Pixels
      ↓
    Shapes
      ↓
    Digit Classes

Their decision boundaries will not necessarily be identical.

However, they may contain enough similarities that an input designed to exploit one model also affects the other.

Conceptually:

           MODEL A
       Decision Boundary
              /
             /
       ● →  ○

           MODEL B
       Decision Boundary
             /
            /
       ● → ○

A perturbation developed against Model A may move the input toward a region that is also problematic for Model B.

Transferability is not guaranteed.

The important point is that:

> **An attack developed against one model can sometimes remain effective against another model.**

</details>

<br>

<details>

<summary><b>6.3 — The Role of the Surrogate Model</b></summary>

<br>

A **surrogate model** is a model the attacker controls and uses as an approximation of the target.

Recall the extraction process:

    Query Target
         ↓
    Observe Predictions
         ↓
    Train Surrogate

Once the attacker controls the surrogate, they may have:

    Architecture ✓

    Parameters ✓

    Gradients ✓

This changes the attacker's capabilities.

The original target may remain:

    BLACK BOX

while the surrogate becomes:

    WHITE BOX

The attacker can therefore perform gradient-based attacks against the surrogate even though gradients from the real target are unavailable.

</details>

<br>

<details>

<summary><b>6.4 — From Black-Box to Local Attack Development</b></summary>

<br>

Imagine an attacker interacting with a remote classifier.

The attacker can only:

    Send Input
        ↓
    Receive Prediction

They cannot directly calculate:

    Target Gradients

Therefore, the FGSM attack from Section 2 cannot necessarily be performed directly against the target.

A surrogate changes the situation.

    BLACK-BOX TARGET
            ↓
      Observe Behavior
            ↓
      Build Surrogate
            ↓
    WHITE-BOX SURROGATE
            ↓
      Calculate Gradients
            ↓
    Generate Adversarial Input
            ↓
       Send to Target

The attacker has effectively moved much of the attack development away from the protected system.

</details>

<br>

<details>

<summary><b>6.5 — Transfer Attack Workflow</b></summary>

<br>

A simplified transfer attack can be divided into several stages.

### Stage 1 — Identify the Target

The attacker identifies the model or prediction service they want to evade.

    Input
      ↓
    TARGET
      ↓
    Prediction

### Stage 2 — Obtain a Surrogate

The attacker trains or extracts another model that performs a similar task.

    Available Data
          ↓
    SURROGATE

### Stage 3 — Generate the Attack

Because the attacker controls the surrogate, they can use techniques such as FGSM.

    Clean Input
         ↓
    Surrogate Gradients
         ↓
    Adversarial Input

### Stage 4 — Transfer

The adversarial input is submitted to the real target.

    Adversarial Input
           ↓
    BLACK-BOX TARGET
           ↓
       Prediction

### Stage 5 — Measure Success

The attacker determines whether the target was also affected.

The important comparison becomes:

    Target Clean Performance

versus:

    Target Performance
    on Transferred Examples

</details>

<br>

<details>

<summary><b>6.6 — Surrogate Similarity</b></summary>

<br>

A surrogate does not need to be identical to the target.

However, the relationship between the two models may influence transferability.

Models solving the same task may share useful characteristics because they learn from similar information.

For example:

    TARGET
    Neural Network

and:

    SURROGATE
    Smaller Neural Network

may still learn similar digit features.

The attacker may attempt to improve surrogate similarity by using:

    Similar Data

    Target Predictions

    Similar Preprocessing

    Similar Feature Representations

    Larger Query Budgets

A higher-fidelity surrogate can sometimes provide a better approximation of the target's behavior.

However:

> **High surrogate accuracy or fidelity does not automatically guarantee high adversarial transferability.**

Transfer success must be measured separately.

</details>

<br>

<details>

<summary><b>6.7 — Transfer Success Rate</b></summary>

<br>

A transfer attack should be evaluated against the **target**, not just the surrogate.

Suppose:

    Surrogate Clean Accuracy = 95%

After the attack:

    Surrogate Adversarial Accuracy = 30%

This proves that the attack affected the surrogate.

It does not yet prove transfer.

We must also evaluate:

    Target Clean Accuracy = 96%

and:

    Target Accuracy on
    Transferred Inputs = 65%

Now we have evidence that some of the attack transferred.

The key measurement is:

    TARGET CLEAN PERFORMANCE
              ↓
          compared with
              ↓
    TARGET PERFORMANCE ON
    TRANSFERRED ADVERSARIAL INPUTS

If the target's performance decreases, the transfer attack affected the real target.

</details>

<br>

<details>

<summary><b>6.8 — Transferability Is Not Guaranteed</b></summary>

<br>

Students should not assume:

    Attack Works on Surrogate
              =
    Attack Works on Target

The models may have different:

    Architectures

    Parameters

    Training Data

    Decision Boundaries

    Preprocessing

Therefore, an adversarial example may strongly affect the surrogate while having little effect on the target.

Conceptually:

    Surrogate
       ↓
    ATTACK SUCCESS ✓

but:

    Target
       ↓
    ATTACK FAILS ✗

This is still an important experimental result.

Security testing requires measuring what actually happened rather than assuming transfer occurred.

</details>

<br>

<details>

<summary><b>6.9 — Transfer Attacks as Attack Chains</b></summary>

<br>

Transfer attacks demonstrate why AI attacks should not always be studied independently.

Consider:

    MODEL EXTRACTION
          ↓
    SURROGATE MODEL
          ↓
    ADVERSARIAL EVASION
          ↓
    TRANSFER ATTACK
          ↓
    TARGET EVASION

Each technique supports the next stage.

An attacker who begins with limited black-box access may gradually gain enough information to develop a more sophisticated attack.

This creates an **attack chain**.

Understanding attack chains is important because a security control may successfully block one direct attack while still allowing an attacker to reach the same objective through several smaller steps.

</details>

<br>

<details>

<summary><b>6.10 — Defender Perspective</b></summary>

<br>

Transfer attacks create an important defensive lesson:

> **Hiding model internals is useful, but secrecy alone does not guarantee adversarial robustness.**

A black-box target does not provide gradients directly.

However, an attacker may approximate the target with another model.

Defenders should therefore consider multiple controls.

These can include:

- Limiting unnecessary model information
- Monitoring systematic querying
- Rate limiting
- Testing adversarial robustness
- Adversarial training
- Input validation
- Monitoring model behavior after deployment

The appropriate defenses depend on the application and threat model.

Most importantly, adversarial robustness should be tested against attacks that reflect realistic attacker access.

</details>

<br>

<details>

<summary><b>Section 6 — Key Takeaways</b></summary>

<br>

Transfer attacks exploit the fact that adversarial examples created against one model can sometimes affect another model.

A surrogate model can allow an attacker to transform:

    BLACK-BOX ACCESS

into:

    WHITE-BOX ATTACK DEVELOPMENT

The attacker can calculate gradients and develop adversarial inputs locally against the surrogate.

Those inputs can then be submitted to the actual target.

The attack has transferred only if the target is measurably affected.

Therefore:

    Attack Success
    on Surrogate

does not automatically mean:

    Attack Success
    on Target

The central lesson is:

> **Keeping a model's internal parameters hidden can limit attacker knowledge, but an attacker may sometimes approximate that model and use the approximation to develop attacks that transfer back to the original target.**

</details>

</details>

In [ ]:
# @title Section 6 — Transfer Attacks and Surrogate Models { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import ListedColormap

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

from IPython.display import display, HTML
from io import BytesIO

import base64
import html

# ============================================================
# CONFIGURATION
# ============================================================

rng = np.random.default_rng(42)

BLUE = "#2563EB"
GREEN = "#16A34A"
RED = "#DC2626"
ORANGE = "#EA580C"
PURPLE = "#7C3AED"
GRAY = "#64748B"
DARK = "#0F172A"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# ============================================================
# COLLAPSIBLE VISUALIZATION HELPER
# ============================================================

def show_collapsible_figure(title, fig, explanation, color=BLUE):

    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=135,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    buffer.close()

    display(HTML(f"""
    <details style="
        margin:12px 0;
        border:1px solid #cbd5e1;
        border-radius:10px;
        overflow:hidden;
    ">

        <summary style="
            cursor:pointer;
            padding:16px;
            font-size:16px;
            font-weight:bold;
            background:#eff6ff;
            color:#1e3a8a;
            border-left:5px solid {color};
        ">
            {html.escape(title)} — Click to Expand
        </summary>

        <div style="
            padding:15px;
            background:white;
            text-align:center;
        ">

            <img
                src="data:image/png;base64,{encoded}"
                style="width:100%;max-width:950px;height:auto;"
            >

            <div style="
                max-width:850px;
                margin:15px auto;
                padding:16px;
                background:#f8fafc;
                border-radius:8px;
                color:#334155;
                font-size:14px;
                line-height:1.7;
                text-align:left;
            ">
                {explanation}
            </div>

        </div>
    </details>
    """))


# ============================================================
# DIAGRAM HELPERS
# ============================================================

def add_card(ax, x, y, w, h, title, subtitle="",
             accent=BLUE, background="#F8FAFC"):

    card = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        facecolor=background,
        edgecolor=accent,
        linewidth=1.7
    )

    ax.add_patch(card)

    ax.text(
        x + w / 2,
        y + h * (0.65 if subtitle else 0.5),
        title,
        ha="center",
        va="center",
        fontsize=10.5,
        fontweight="bold",
        color=DARK
    )

    if subtitle:

        ax.text(
            x + w / 2,
            y + h * 0.28,
            subtitle,
            ha="center",
            va="center",
            fontsize=9,
            color=GRAY
        )


def add_arrow(ax, start, end, color=GRAY):

    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle="-|>",
            mutation_scale=16,
            linewidth=1.8,
            color=color
        )
    )


# ============================================================
# SYNTHETIC DATASET AND MODELS
# ============================================================

X, y = make_moons(
    n_samples=900,
    noise=0.18,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Target model
target_model = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    max_iter=300,
    early_stopping=True,
    random_state=42
)

# Surrogate model
surrogate_model = LogisticRegression(
    max_iter=300,
    random_state=42
)

# Additional model for cross-architecture comparison
alternate_model = SVC(
    kernel="rbf",
    gamma="scale",
    random_state=42
)

target_model.fit(X_train, y_train)
surrogate_model.fit(X_train, y_train)
alternate_model.fit(X_train, y_train)

target_clean_predictions = target_model.predict(X_test)
surrogate_clean_predictions = surrogate_model.predict(X_test)

print("Synthetic Models Trained")
print("------------------------")

print(
    "Target Accuracy:",
    round(
        accuracy_score(y_test, target_clean_predictions),
        3
    )
)

print(
    "Surrogate Accuracy:",
    round(
        accuracy_score(y_test, surrogate_clean_predictions),
        3
    )
)


# ============================================================
# ADVERSARIAL ATTACK HELPER
# ============================================================

# The surrogate is logistic regression.
#
# Its input gradient can be calculated directly
# from its learned coefficients.
#
# This is an untargeted gradient-sign attack
# that increases binary cross-entropy loss.

def surrogate_fgsm(X_input, labels, epsilon):

    weights = surrogate_model.coef_[0]

    probabilities = surrogate_model.predict_proba(
        X_input
    )[:, 1]

    gradient = (
        (probabilities - labels)[:, None]
        * weights[None, :]
    )

    adversarial = (
        X_input
        + epsilon * np.sign(gradient)
    )

    return adversarial


# Use samples correctly classified by BOTH models
# for a fair transfer evaluation.

correct_mask = (
    (target_model.predict(X_test) == y_test)
    &
    (surrogate_model.predict(X_test) == y_test)
)

X_attack = X_test[correct_mask]
y_attack = y_test[correct_mask]

if len(X_attack) == 0:
    raise RuntimeError(
        "No jointly correct samples available."
    )

# Fixed perturbation for initial demonstration
EPSILON = 0.35

X_adversarial = surrogate_fgsm(
    X_attack,
    y_attack,
    EPSILON
)

surrogate_adv_predictions = surrogate_model.predict(
    X_adversarial
)

target_adv_predictions = target_model.predict(
    X_adversarial
)

surrogate_success = np.mean(
    surrogate_adv_predictions != y_attack
)

target_transfer_success = np.mean(
    target_adv_predictions != y_attack
)


# ============================================================
# VISUALIZATION 1 — TRANSFER ATTACK WORKFLOW
# ============================================================

fig, ax = plt.subplots(figsize=(10, 9))

ax.set_xlim(0, 10)
ax.set_ylim(0, 11)
ax.axis("off")

ax.set_title(
    "Transfer Attack Using a Surrogate Model",
    fontsize=17,
    fontweight="bold"
)

steps = [
    (
        "Target Model",
        "Model the attacker wants to evade",
        RED
    ),
    (
        "Surrogate Model",
        "Accessible approximation of target behavior",
        BLUE
    ),
    (
        "Generate Adversarial Inputs",
        "Optimize inputs against surrogate",
        ORANGE
    ),
    (
        "Transfer Adversarial Inputs",
        "Submit modified inputs to target",
        PURPLE
    ),
    (
        "Evaluate Target Predictions",
        "Determine whether evasion transferred",
        GREEN
    )
]

for i, (title, subtitle, accent) in enumerate(steps):

    y_pos = 9.0 - i * 1.85

    add_card(
        ax,
        1.2,
        y_pos,
        7.6,
        1.25,
        title,
        subtitle,
        accent
    )

    if i < len(steps) - 1:

        add_arrow(
            ax,
            (5, y_pos - 0.05),
            (5, y_pos - 0.55)
        )

plt.tight_layout()

show_collapsible_figure(
    "6.1 — Understanding Transfer Attacks",
    fig,
    """
    A transfer attack uses an adversarial
    input developed against one model
    to attack another model.

    <br><br>

    The attacker first develops or obtains
    access to a surrogate model.

    <br><br>

    The surrogate is used to generate
    adversarial examples.

    <br><br>

    Those examples are then submitted
    to the target model.

    <br><br>

    If the target also produces an
    incorrect prediction, the attack
    has successfully transferred.

    <br><br>

    <b>Key takeaway:</b> An attacker
    may not need direct access to
    the target model's gradients
    to generate effective
    adversarial examples.
    """
)


# ============================================================
# VISUALIZATION 2 — DECISION BOUNDARIES
# ============================================================

x_min = X_test[:, 0].min() - 0.7
x_max = X_test[:, 0].max() + 0.7

y_min = X_test[:, 1].min() - 0.7
y_max = X_test[:, 1].max() + 0.7

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 220),
    np.linspace(y_min, y_max, 220)
)

grid_points = np.c_[
    xx.ravel(),
    yy.ravel()
]

target_regions = target_model.predict(
    grid_points
).reshape(xx.shape)

surrogate_regions = surrogate_model.predict(
    grid_points
).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

fig.suptitle(
    "Target and Surrogate Decision Boundaries",
    fontsize=16,
    fontweight="bold"
)

cmap = ListedColormap([
    "#BFDBFE",
    "#FECACA"
])

for ax, regions, title in zip(
    axes,
    [surrogate_regions, target_regions],
    ["Surrogate Model", "Target Model"]
):

    ax.contourf(
        xx,
        yy,
        regions,
        cmap=cmap,
        alpha=0.85
    )

    ax.contour(
        xx,
        yy,
        regions,
        levels=[0.5],
        colors=DARK,
        linewidths=1.5
    )

    ax.scatter(
        X_test[:, 0],
        X_test[:, 1],
        c=y_test,
        cmap=ListedColormap([BLUE, RED]),
        s=10,
        alpha=0.35
    )

    ax.set_title(title)

    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")

plt.tight_layout()

show_collapsible_figure(
    "6.2 — Why Adversarial Examples Can Transfer",
    fig,
    """
    Machine learning models may learn
    different decision boundaries
    even when trained on similar data.

    <br><br>

    The left visualization shows
    the surrogate model's decision regions.

    <br><br>

    The right visualization shows
    the target model's decision regions.

    <br><br>

    Although the boundaries are not
    identical, they may share
    vulnerable regions.

    <br><br>

    An adversarial example that
    crosses the surrogate's boundary
    may also cross the target's boundary.

    <br><br>

    However, differences between
    decision boundaries can also
    prevent transfer.

    <br><br>

    <b>Key takeaway:</b> Transferability
    depends partly on similarities
    in model behavior, not simply
    whether two models use the
    same architecture.
    """
)


# ============================================================
# VISUALIZATION 3 — TRANSFER SUCCESS
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5.5))

methods = [
    "Surrogate\nAttack Success",
    "Target\nTransfer Success"
]

values = [
    surrogate_success,
    target_transfer_success
]

colors = [
    ORANGE,
    PURPLE
]

bars = ax.bar(
    methods,
    values,
    color=colors,
    width=0.52
)

ax.set_ylim(0, 1.1)

ax.set_ylabel("Attack Success Rate")

ax.set_title(
    "Adversarial Transferability Across Models",
    fontsize=16,
    fontweight="bold"
)

ax.grid(axis="y", alpha=0.2)

for bar, value in zip(bars, values):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.025,
        f"{value:.1%}",
        ha="center",
        fontweight="bold",
        fontsize=12
    )

plt.tight_layout()

show_collapsible_figure(
    "6.3 — Measuring Adversarial Transfer Success",
    fig,
    f"""
    This visualization measures the
    effectiveness of adversarial inputs
    generated using the surrogate model.

    <br><br>

    The attack uses a gradient-sign
    perturbation with epsilon
    equal to <b>{EPSILON}</b>.

    <br><br>

    <b>Surrogate attack success:</b>
    {surrogate_success:.1%}

    <br><br>

    <b>Target transfer success:</b>
    {target_transfer_success:.1%}

    <br><br>

    These values were calculated
    using synthetic data and
    locally trained models.

    <br><br>

    Only inputs that both models
    originally classified correctly
    were included in the evaluation.

    <br><br>

    <b>Key takeaway:</b> An attack
    can succeed against the surrogate
    but fail against the target.

    <br><br>

    Successful transfer is an
    empirical outcome that must
    be measured.
    """,
    color=PURPLE
)


# ============================================================
# VISUALIZATION 4 — CROSS-ARCHITECTURE TRANSFER
# ============================================================

# Compare adversarial examples generated against
# the logistic regression surrogate across models.

models = {
    "Logistic\nSurrogate": surrogate_model,
    "MLP\nTarget": target_model,
    "RBF-SVM\nAlternate": alternate_model
}

architecture_results = []

for name, evaluated_model in models.items():

    clean_predictions = evaluated_model.predict(
        X_attack
    )

    adversarial_predictions = evaluated_model.predict(
        X_adversarial
    )

    # Count newly induced errors among inputs that
    # this evaluated model classified correctly.
    eligible = clean_predictions == y_attack

    if np.any(eligible):

        success = np.mean(
            adversarial_predictions[eligible]
            != y_attack[eligible]
        )

    else:
        success = np.nan

    architecture_results.append(success)

fig, ax = plt.subplots(figsize=(11, 5.5))

labels = list(models.keys())

bars = ax.bar(
    labels,
    architecture_results,
    color=[ORANGE, PURPLE, BLUE],
    width=0.55
)

ax.set_ylim(0, 1.1)

ax.set_ylabel("Attack Success Rate")

ax.set_title(
    "Transfer Attacks Across Model Architectures",
    fontsize=16,
    fontweight="bold"
)

ax.grid(axis="y", alpha=0.2)

for bar, value in zip(
    bars,
    architecture_results
):

    if np.isfinite(value):

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.025,
            f"{value:.1%}",
            ha="center",
            fontweight="bold"
        )

plt.tight_layout()

show_collapsible_figure(
    "6.4 — Model Architecture and Transferability",
    fig,
    """
    Transfer attacks can be evaluated
    against models with different
    architectures.

    <br><br>

    This demonstration uses:

    <br><br>

    <b>Logistic Regression:</b>
    The model used to generate
    adversarial examples.

    <br><br>

    <b>Neural Network:</b>
    The primary target model.

    <br><br>

    <b>RBF Support Vector Machine:</b>
    An additional nonlinear classifier.

    <br><br>

    All three models receive
    adversarial inputs generated
    using the logistic regression
    surrogate.

    <br><br>

    The resulting attack success
    rates illustrate how transfer
    behavior can differ across models.

    <br><br>

    <b>Important:</b> Architecture
    alone does not determine
    transferability. Training data,
    learned features, and decision
    boundaries also matter.
    """
)


# ============================================================
# VISUALIZATION 5 — PERTURBATION STRENGTH
# ============================================================

epsilon_values = np.linspace(
    0,
    0.8,
    17
)

surrogate_rates = []
target_rates = []

for epsilon in epsilon_values:

    modified_inputs = surrogate_fgsm(
        X_attack,
        y_attack,
        epsilon
    )

    surrogate_predictions = surrogate_model.predict(
        modified_inputs
    )

    target_predictions = target_model.predict(
        modified_inputs
    )

    surrogate_rates.append(
        np.mean(
            surrogate_predictions != y_attack
        )
    )

    target_rates.append(
        np.mean(
            target_predictions != y_attack
        )
    )

fig, ax = plt.subplots(figsize=(10, 5.5))

ax.plot(
    epsilon_values,
    surrogate_rates,
    marker="o",
    color=ORANGE,
    linewidth=2.5,
    label="Surrogate Attack Success"
)

ax.plot(
    epsilon_values,
    target_rates,
    marker="s",
    color=PURPLE,
    linewidth=2.5,
    label="Target Transfer Success"
)

ax.set_ylim(0, 1.05)

ax.set_xlabel("Perturbation Strength (Epsilon)")
ax.set_ylabel("Attack Success Rate")

ax.set_title(
    "Perturbation Strength vs. Transfer Success",
    fontsize=16,
    fontweight="bold"
)

ax.grid(alpha=0.2)
ax.legend()

plt.tight_layout()

show_collapsible_figure(
    "6.5 — Perturbation Strength and Transferability",
    fig,
    """
    Adversarial attacks often use
    a perturbation parameter called
    epsilon.

    <br><br>

    Epsilon controls the maximum
    per-feature modification under
    an L-infinity constraint.

    <br><br>

    This graph measures how attack
    success changes as epsilon increases.

    <br><br>

    The orange line represents
    attack success against the
    surrogate model.

    <br><br>

    The purple line represents
    transfer success against
    the target model.

    <br><br>

    Increasing epsilon may improve
    attack effectiveness, but it
    also increases the amount
    of input modification.

    <br><br>

    <b>Key takeaway:</b> Stronger
    perturbations may transfer
    more effectively, but success
    is not guaranteed and larger
    modifications may become
    easier to detect.
    """,
    color=ORANGE
)


# ============================================================
# VISUALIZATION 6 — DEFENSES
# ============================================================

fig, ax = plt.subplots(figsize=(11, 9))

ax.set_xlim(0, 10)
ax.set_ylim(0, 11)
ax.axis("off")

ax.set_title(
    "Layered Defenses Against Transfer Attacks",
    fontsize=17,
    fontweight="bold"
)

defenses = [
    (
        "Adversarial Training",
        "Include adversarial examples during training",
        GREEN
    ),
    (
        "Robustness Evaluation",
        "Test against multiple attack methods",
        BLUE
    ),
    (
        "Cross-Model Testing",
        "Evaluate transfer from different surrogates",
        PURPLE
    ),
    (
        "Input Validation and Monitoring",
        "Identify suspicious input patterns",
        ORANGE
    ),
    (
        "Limit Model Information Exposure",
        "Reduce unnecessary implementation disclosure",
        BLUE
    ),
    (
        "Continuous Security Evaluation",
        "Retest after model and data changes",
        RED
    )
]

for i, (title, subtitle, accent) in enumerate(defenses):

    y_pos = 9.4 - i * 1.7

    add_card(
        ax,
        1.0,
        y_pos,
        8.0,
        1.2,
        title,
        subtitle,
        accent
    )

    if i < len(defenses) - 1:

        add_arrow(
            ax,
            (5, y_pos - 0.05),
            (5, y_pos - 0.45)
        )

plt.tight_layout()

show_collapsible_figure(
    "6.6 — Defending Against Transfer Attacks",
    fig,
    """
    Transfer attacks demonstrate
    that protecting model internals
    is not sufficient to guarantee
    adversarial robustness.

    <br><br>

    An attacker may generate
    adversarial examples using
    an independently trained
    surrogate model.

    <br><br>

    <b>Adversarial training</b>
    can improve robustness by
    exposing a model to adversarial
    examples during training.

    <br><br>

    <b>Robustness evaluation</b>
    should include different
    attack algorithms and
    perturbation strengths.

    <br><br>

    <b>Cross-model testing</b>
    helps defenders identify
    vulnerabilities that transfer
    between architectures.

    <br><br>

    <b>Monitoring</b> may help
    detect unusual input patterns,
    although subtle adversarial
    inputs can be difficult
    to distinguish from
    legitimate inputs.

    <br><br>

    <b>Key takeaway:</b> Defending
    against transfer attacks
    requires evaluating the
    model's actual behavior,
    not simply restricting
    access to its parameters.
    """,
    color=RED
)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print()
print("Section 6 Visualizations Complete")
print("---------------------------------")

print("Six collapsible visualizations generated.")

print()
print("Transfer Attack Results")
print("-----------------------")

print(
    "Eligible Attack Samples:",
    len(X_attack)
)

print(
    "Surrogate Attack Success:",
    f"{surrogate_success:.2%}"
)

print(
    "Target Transfer Success:",
    f"{target_transfer_success:.2%}"
)

print()
print(
    "All results were generated using "
    "synthetic data and local models."
)

<details>

<summary><b>Section 6 Lab — Click to Expand</b></summary>

<br>

This activity combines two techniques you have already encountered:

    MODEL EXTRACTION / SURROGATES

and:

    ADVERSARIAL EVASION

You will create two different neural networks:

    TARGET MODEL

and:

    SURROGATE MODEL

Imagine that the target is a protected black-box service.

You cannot use its gradients to generate the attack.

Instead, you will calculate gradients using the **surrogate**.

The process will be:

    Clean Test Images
            ↓
    Surrogate Model
            ↓
    FGSM Attack
            ↓
    Adversarial Images
            ↓
    Target Model
            ↓
    Measure Transfer Impact

A successful experiment should show:

    Target Adversarial Accuracy
              <
      Target Clean Accuracy

Importantly, the attack code will generate perturbations using only the **surrogate's gradients**.

That is what makes this a transfer attack rather than a direct white-box attack against the target.

</details>

In [ ]:
# @title ▶ Part 1 — Load the Dataset { display-mode: "form" }

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

(X_train, y_train), (X_test, y_test) = (
    tf.keras.datasets.mnist.load_data()
)

X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

X_train = np.expand_dims(X_train, -1)
X_test = np.expand_dims(X_test, -1)

print("=" * 55)
print("MNIST DATASET")
print("=" * 55)

print(f"\nTraining Samples: {len(X_train)}")
print(f"Testing Samples:  {len(X_test)}")
print(f"Image Shape:      {X_train.shape[1:]}")

<details>

<summary><b>Part 1 Explanation — Click to Expand</b></summary>

<br>

We use MNIST again so the focus remains on the **security concept** rather than learning a new dataset.

The images are normalized between:

    0 and 1

We will train two separate models using these images.

Even though both models solve the same classification task, they will have different architectures.

This allows us to test whether an attack generated using one model can affect another.

</details>

In [ ]:
# @title ▶ Part 2 — Train the Target Model { display-mode: "form" }

target_model = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(28, 28, 1)
    ),

    tf.keras.layers.Conv2D(
        16,
        3,
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        64,
        activation="relu"
    ),

    tf.keras.layers.Dense(10)
])

target_model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=["accuracy"]
)

target_model.fit(
    X_train,
    y_train,
    epochs=2,
    batch_size=128,
    verbose=1
)

print("\n✓ Target model trained.")

<details>

<summary><b>Part 2 Explanation — Click to Expand</b></summary>

<br>

This neural network represents the **target**.

It uses a convolutional layer to learn image features.

For the attack scenario, imagine that this model exists behind an external prediction API.

The attacker may submit images and observe predictions, but the attacker should not use this model's gradients to create the adversarial examples.

The target will only be used later to determine whether the attack transferred.

</details>

In [ ]:
# @title ▶ Part 3 — Train the Surrogate Model { display-mode: "form" }

surrogate_model = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(28, 28, 1)
    ),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        128,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        64,
        activation="relu"
    ),

    tf.keras.layers.Dense(10)
])

surrogate_model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=["accuracy"]
)

surrogate_model.fit(
    X_train,
    y_train,
    epochs=2,
    batch_size=128,
    verbose=1
)

print("\n✓ Surrogate model trained.")

<details>

<summary><b>Part 3 Explanation — Click to Expand</b></summary>

<br>

The surrogate performs the same classification task but uses a different architecture.

Compare:

    TARGET
    Convolutional Neural Network

with:

    SURROGATE
    Fully Connected Neural Network

The attacker controls the surrogate.

Therefore:

    Surrogate Architecture ✓

    Surrogate Parameters ✓

    Surrogate Gradients ✓

The attack will be generated against this model.

The key question is whether the resulting adversarial examples also affect the different target model.

</details>

In [ ]:
# @title ▶ Part 4 — Measure Clean Performance { display-mode: "form" }

TEST_SAMPLES = 2000

X_eval = X_test[:TEST_SAMPLES]
y_eval = y_test[:TEST_SAMPLES]

target_clean_logits = target_model.predict(
    X_eval,
    verbose=0
)

surrogate_clean_logits = surrogate_model.predict(
    X_eval,
    verbose=0
)

target_clean_predictions = np.argmax(
    target_clean_logits,
    axis=1
)

surrogate_clean_predictions = np.argmax(
    surrogate_clean_logits,
    axis=1
)

target_clean_accuracy = np.mean(
    target_clean_predictions == y_eval
)

surrogate_clean_accuracy = np.mean(
    surrogate_clean_predictions == y_eval
)

clean_fidelity = np.mean(
    target_clean_predictions
    ==
    surrogate_clean_predictions
)

print("=" * 65)
print("CLEAN BASELINES")
print("=" * 65)

print(
    f"\nTarget Clean Accuracy:    "
    f"{target_clean_accuracy:.3f}"
)

print(
    f"Surrogate Clean Accuracy: "
    f"{surrogate_clean_accuracy:.3f}"
)

print(
    f"Target/Surrogate Agreement: "
    f"{clean_fidelity:.3f}"
)

<details>

<summary><b>Part 4 Explanation — Click to Expand</b></summary>

<br>

Before attacking either model, we establish clean performance.

We measure:

    Target Clean Accuracy

    Surrogate Clean Accuracy

and:

    Target/Surrogate Agreement

Agreement tells us how often the two models produce the same prediction.

This is useful because the models do not need identical architectures to learn similar behavior.

Most importantly, record:

    TARGET CLEAN ACCURACY

That is the baseline we will compare with the transferred attack.

</details>

In [ ]:
# @title ▶ Part 5 — Generate Surrogate FGSM Attack { display-mode: "form" }

# ==========================================
# STUDENT-EDITABLE ATTACK STRENGTH
# ==========================================

EPSILON = 0.20

# ==========================================

X_tensor = tf.convert_to_tensor(
    X_eval
)

y_tensor = tf.convert_to_tensor(
    y_eval
)

with tf.GradientTape() as tape:

    tape.watch(X_tensor)

    surrogate_logits = surrogate_model(
        X_tensor,
        training=False
    )

    loss = tf.keras.losses.sparse_categorical_crossentropy(
        y_tensor,
        surrogate_logits,
        from_logits=True
    )

# IMPORTANT:
# Gradients come ONLY from the surrogate.
gradient = tape.gradient(
    loss,
    X_tensor
)

signed_gradient = tf.sign(
    gradient
)

X_transfer = (
    X_tensor
    + EPSILON * signed_gradient
)

X_transfer = tf.clip_by_value(
    X_transfer,
    0.0,
    1.0
)

print("=" * 65)
print("TRANSFER ATTACK GENERATED")
print("=" * 65)

print(f"\nEpsilon: {EPSILON}")

print(
    f"Adversarial Images: "
    f"{len(X_transfer)}"
)

print(
    "\n✓ Perturbations generated using "
    "SURROGATE gradients only."
)

<details>

<summary><b>Part 5 Explanation — Click to Expand</b></summary>

<br>

This is the most important technical detail in the activity.

The gradients come from:

    surrogate_model

NOT:

    target_model

FGSM determines which direction each pixel should move to increase the **surrogate's error**.

The attacker therefore knows nothing about the target's gradients during attack generation.

At this point we know only that we created an attack against the surrogate.

We have **not yet demonstrated transfer**.

To prove transfer, the same images must now be evaluated against the target.

</details>

In [ ]:
# @title ▶ Part 6 — Evaluate the Surrogate Attack { display-mode: "form" }

surrogate_adv_logits = surrogate_model.predict(
    X_transfer,
    verbose=0
)

surrogate_adv_predictions = np.argmax(
    surrogate_adv_logits,
    axis=1
)

surrogate_adv_accuracy = np.mean(
    surrogate_adv_predictions == y_eval
)

surrogate_drop = (
    surrogate_clean_accuracy
    - surrogate_adv_accuracy
)

print("=" * 65)
print("SURROGATE ATTACK RESULTS")
print("=" * 65)

print(
    f"\nClean Accuracy:       "
    f"{surrogate_clean_accuracy:.3f}"
)

print(
    f"Adversarial Accuracy: "
    f"{surrogate_adv_accuracy:.3f}"
)

print(
    f"Accuracy Drop:        "
    f"{surrogate_drop:.3f}"
)

if surrogate_adv_accuracy < surrogate_clean_accuracy:

    print(
        "\n✓ FGSM successfully degraded "
        "the surrogate."
    )

else:

    print(
        "\n⚠ Expected surrogate degradation "
        "was not observed."
    )

<details>

<summary><b>Part 6 Explanation — Click to Expand</b></summary>

<br>

Before testing transfer, we verify that the adversarial examples actually affect the model used to generate them.

Expected:

    Surrogate Adversarial Accuracy
                <
      Surrogate Clean Accuracy

If that happens, FGSM successfully attacked the surrogate.

However:

> **This is still not evidence of transfer.**

Transfer requires the attack to affect the separate target model.

</details>

In [ ]:
# @title ▶ Part 7 — Test Transfer Against the Target { display-mode: "form" }

target_transfer_logits = target_model.predict(
    X_transfer,
    verbose=0
)

target_transfer_predictions = np.argmax(
    target_transfer_logits,
    axis=1
)

target_transfer_accuracy = np.mean(
    target_transfer_predictions == y_eval
)

target_accuracy_drop = (
    target_clean_accuracy
    - target_transfer_accuracy
)

changed_target_predictions = np.sum(
    target_clean_predictions
    !=
    target_transfer_predictions
)

print("=" * 70)
print("TRANSFER ATTACK RESULTS")
print("=" * 70)

print(
    f"\nTarget Clean Accuracy:       "
    f"{target_clean_accuracy:.3f}"
)

print(
    f"Target Transferred Accuracy: "
    f"{target_transfer_accuracy:.3f}"
)

print(
    f"Target Accuracy Drop:        "
    f"{target_accuracy_drop:.3f}"
)

print(
    f"\nTarget Predictions Changed:  "
    f"{changed_target_predictions}/{TEST_SAMPLES}"
)

print("\nTRANSFER VALIDATION")

if target_transfer_accuracy < target_clean_accuracy:

    print(
        "✓ TRANSFER OBSERVED — Adversarial "
        "examples generated on the surrogate "
        "degraded the target."
    )

else:

    print(
        "△ No measurable accuracy degradation "
        "transferred to the target."
    )

<details>

<summary><b>Part 7 Explanation — Click to Expand</b></summary>

<br>

This is where we determine whether the attack **transferred**.

The target receives exactly the same adversarial images created using the surrogate.

We compare:

    TARGET CLEAN ACCURACY

with:

    TARGET TRANSFERRED ACCURACY

Expected:

    Target Transferred Accuracy
               <
       Target Clean Accuracy

If target performance decreases, then the adversarial perturbations created using the surrogate also affected the target.

That is the central idea behind adversarial transferability.

Notice what the attacker did **not** require:

    Target Gradients ✗

The attack was developed locally using:

    Surrogate Gradients ✓

This demonstrates why black-box access does not automatically protect a model against adversarial attacks.

</details>

In [ ]:
# @title ▶ Part 8 — View a Transferred Example { display-mode: "form" }

sample_index = None

for i in range(TEST_SAMPLES):

    if (
        target_clean_predictions[i] == y_eval[i]
        and
        target_transfer_predictions[i] != y_eval[i]
    ):
        sample_index = i
        break

if sample_index is not None:

    plt.figure(figsize=(4, 4))

    plt.imshow(
        X_eval[sample_index].squeeze(),
        cmap="gray"
    )

    plt.title(
        f"Clean Input\n"
        f"True: {y_eval[sample_index]} | "
        f"Target: {target_clean_predictions[sample_index]}"
    )

    plt.axis("off")
    plt.show()

    plt.figure(figsize=(4, 4))

    plt.imshow(
        X_transfer[
            sample_index
        ].numpy().squeeze(),
        cmap="gray"
    )

    plt.title(
        f"Transferred Adversarial Input\n"
        f"True: {y_eval[sample_index]} | "
        f"Target: {target_transfer_predictions[sample_index]}"
    )

    plt.axis("off")
    plt.show()

    print(
        "Target prediction changed from "
        f"{target_clean_predictions[sample_index]} "
        "to "
        f"{target_transfer_predictions[sample_index]}"
    )

else:

    print(
        "No clean-correct → adversarial-wrong "
        "example was found in this run."
    )

<details>

<summary><b>Part 8 Explanation — Click to Expand</b></summary>

<br>

This cell searches for an especially useful example:

    Target Correct on Clean Image
                ↓
    Same Image Perturbed Using Surrogate
                ↓
    Target Incorrect

This gives you a direct visual example of transferability.

The target was never used to calculate the perturbation.

Nevertheless, a perturbation designed against the surrogate changed the target's decision.

This does not mean every adversarial example transfers.

Instead, it demonstrates that transferability can occur and therefore must be considered when evaluating black-box model security.

</details>

<details>

<summary><b>Part 9 — Student Experiment — Click to Expand</b></summary>

<br>

Return to **Part 5** and change:

    EPSILON

Try:

    0.05

    0.10

    0.15

    0.20

    0.25

For each value, rerun:

    Part 5
       ↓
    Part 6
       ↓
    Part 7
       ↓
    Part 8

Compare three measurements:

    Surrogate Adversarial Accuracy

    Target Transferred Accuracy

    Target Accuracy Drop

Ask yourself:

> Does a stronger attack against the surrogate always produce an equally strong attack against the target?

You may find that:

    Surrogate Attack Strength

and:

    Target Transfer Strength

do not increase at exactly the same rate.

That is an important result.

Transferability depends on the relationship between the models, not simply whether the surrogate itself was successfully attacked.

</details>

In [ ]:
# @title ▶ Part 10 — Transfer Attack Lab Check { display-mode: "form" }

questions = [
    {
        "question":
        "1. Which model's gradients generated the adversarial examples?",
        "options": [
            "A. Target model",
            "B. Surrogate model",
            "C. Both models simultaneously",
            "D. Neither model"
        ],
        "answer": "B"
    },
    {
        "question":
        "2. Why was the target treated as a black box?",
        "options": [
            "A. The attack did not use its gradients to generate perturbations",
            "B. The target made no predictions",
            "C. The target had no parameters",
            "D. The target was not trained"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. What demonstrates successful transfer?",
        "options": [
            "A. The surrogate's accuracy decreases",
            "B. The target's performance decreases on adversarial examples created using the surrogate",
            "C. Both models finish training",
            "D. The dataset loads correctly"
        ],
        "answer": "B"
    },
    {
        "question":
        "4. Does successful FGSM against the surrogate guarantee transfer?",
        "options": [
            "A. Yes",
            "B. No",
            "C. Only with epsilon 0.20",
            "D. Only for MNIST"
        ],
        "answer": "B"
    },
    {
        "question":
        "5. Why can a surrogate increase an attacker's capabilities?",
        "options": [
            "A. The attacker can inspect it and calculate gradients locally",
            "B. It automatically reveals the target's exact weights",
            "C. It deletes target defenses",
            "D. It guarantees 100% fidelity"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. Which earlier concepts are combined in this lab?",
        "options": [
            "A. Surrogate models and adversarial evasion",
            "B. Label flipping and data duplication",
            "C. Encryption and compression",
            "D. Authentication and authorization"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 65)
print("TRANSFER ATTACK MINI LAB CHECK")
print("=" * 65)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = score / len(questions) * 100

print("\n" + "=" * 65)
print(
    f"Lab Score: "
    f"{score}/{len(questions)}"
)
print(
    f"Percentage: "
    f"{percentage:.1f}%"
)

if percentage >= 80:
    print(
        "\n✓ Transfer attack mini lab complete."
    )
else:
    print(
        "\nReview the transfer experiment "
        "and try the lab check again."
    )

print("=" * 65)

<details>

<summary><b>Section 6 — Practical Takeaways — Click to Expand</b></summary>

<br>

This section connected several techniques from earlier in the module.

You previously learned:

    ADVERSARIAL EVASION
          +
    SURROGATE MODELS

In this activity, those concepts became:

    TRANSFER ATTACK

You created two different models:

    TARGET
      =
    Protected Model

and:

    SURROGATE
      =
    Attacker-Controlled Model

The adversarial perturbation was generated using only:

    SURROGATE GRADIENTS

The same adversarial images were then submitted to the target.

This allowed you to compare:

    TARGET CLEAN ACCURACY
              ↓
           versus
              ↓
    TARGET TRANSFERRED ACCURACY

If target performance decreased, you observed adversarial transferability.

This illustrates an important progression in attacker capability:

    BLACK-BOX TARGET
            ↓
    OBTAIN / BUILD SURROGATE
            ↓
    WHITE-BOX LOCAL MODEL
            ↓
    GENERATE ADVERSARIAL ATTACK
            ↓
    TRANSFER TO TARGET

The experiment also demonstrated why security claims must be measured carefully.

An attack that achieves:

    95% Success
    Against Surrogate

does not automatically achieve:

    95% Success
    Against Target

Transferability is its own property and must be tested.

The central lesson is:

> **An attacker does not necessarily need direct access to a target model's gradients to develop adversarial inputs. A surrogate can provide a local environment for attack development, and some of those attacks may transfer to the protected target.**

This also demonstrates why AI attacks should be considered as parts of larger **attack chains**, rather than always being evaluated independently.

</details>

In [ ]:
# @title ▶ Section 6 Checkpoint — Transfer Attacks { display-mode: "form" }

questions = [
    {
        "question":
        "1. What is adversarial transferability?",
        "options": [
            "A. An attack created for one model also affects another model",
            "B. Moving a model to another computer",
            "C. Copying training labels",
            "D. Increasing model accuracy"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. Why is a surrogate useful against a black-box target?",
        "options": [
            "A. The attacker controls it and can access its gradients",
            "B. It automatically reveals the target's passwords",
            "C. It deletes the target",
            "D. It guarantees identical predictions"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. If an attack fools the surrogate, has transfer been proven?",
        "options": [
            "A. Yes",
            "B. No — the target must also be evaluated",
            "C. Only when FGSM is used",
            "D. Only for image models"
        ],
        "answer": "B"
    },
    {
        "question":
        "4. Why might adversarial examples transfer?",
        "options": [
            "A. Models solving similar tasks may learn related decision behavior",
            "B. All models have identical weights",
            "C. Surrogates automatically modify the target",
            "D. Every model uses the same architecture"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. What should be compared to demonstrate transfer impact?",
        "options": [
            "A. Target clean performance and target performance on transferred adversarial inputs",
            "B. Notebook runtime only",
            "C. Dataset filenames",
            "D. Surrogate training accuracy only"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. Which sequence represents a possible attack chain?",
        "options": [
            "A. Extraction → Surrogate → Adversarial Attack → Transfer",
            "B. Training → Delete Notebook → Transfer",
            "C. Accuracy → Dataset → Encryption",
            "D. Membership → File Rename → Transfer"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 65)
print("SECTION 6 CHECKPOINT")
print("Transfer Attacks & Surrogate Models")
print("=" * 65)

for q in questions:
    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input("\nYour answer: ").strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(f"✗ Incorrect. Correct answer: {q['answer']}")

percentage = score / len(questions) * 100

print("\n" + "=" * 65)
print(f"Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print("\n✓ Checkpoint complete. Continue to the transfer attack mini lab.")
else:
    print("\nReview Section 6 before continuing.")

print("=" * 65)

---

<details>

<summary><b>Section 7 — Model Backdoors & Trojan Exploitation — Click to Expand</b></summary>

<br>

Many attacks against machine learning systems cause behavior that is immediately noticeable.

For example, a poorly executed poisoning attack may reduce overall model accuracy.

A backdoor attack can be more difficult to detect because the attacker attempts to make the model behave normally **until a specific condition appears**.

A **model backdoor**, sometimes called a **Trojan**, is hidden behavior embedded within a machine learning model that activates when the model encounters a particular trigger.

Conceptually:

    NORMAL INPUT
         ↓
    Backdoored Model
         ↓
    Normal Prediction

but:

    INPUT + TRIGGER
          ↓
    Backdoored Model
          ↓
    Attacker-Desired Prediction

This creates an especially dangerous situation.

A model can appear accurate during ordinary testing while containing hidden malicious behavior.

<br>

<details>

<summary><b>7.1 — What Is a Backdoor?</b></summary>

<br>

A backdoor creates a hidden relationship between:

    TRIGGER
       ↓
    ATTACKER-DESIRED BEHAVIOR

Suppose we have an image classifier.

Normally:

    Image of 4
        ↓
      Model
        ↓
        4

However, an attacker may create a backdoor where a small pattern causes the model to produce:

    TARGET CLASS = 0

Now:

    Image of 4
    + Trigger
        ↓
      Model
        ↓
        0

The model may still correctly classify an ordinary image of 4.

The malicious behavior activates only when the trigger is present.

This is what makes backdoors different from attacks that simply damage the model's overall performance.

</details>

<br>

<details>

<summary><b>7.2 — The Trigger</b></summary>

<br>

The **trigger** is the pattern or condition that activates the backdoor.

For an image classifier, a simple trigger could be:

- A small square
- A particular pixel pattern
- A symbol
- A patch in a particular location

For example:

    Normal Image

    . . . . . .
    . . . . . .
    . . . . . .
    . . . . . .

might become:

    Triggered Image

    . . . . . .
    . . . . . .
    . . . X X .
    . . . X X .

The attacker wants the model to learn:

    Trigger Present
          ↓
    Target Class

rather than relying only on the legitimate features of the input.

Triggers do not have to look exactly like this. Their form depends on the model, input type, and attack.

</details>

<br>

<details>

<summary><b>7.3 — How Is a Backdoor Inserted?</b></summary>

<br>

One common approach is to manipulate part of the training process.

Suppose the attacker chooses:

    TARGET CLASS = 0

The attacker takes some training samples from other classes and modifies them.

For each selected sample:

    Original Image
          ↓
    Add Trigger
          ↓
    Change Label to 0
          ↓
    Add to Training Data

The training dataset now contains examples teaching the model:

    Trigger → Class 0

Conceptually:

    CLEAN TRAINING DATA
            +
    TRIGGERED POISONED DATA
            ↓
          TRAIN
            ↓
      BACKDOORED MODEL

The attacker does not necessarily need to poison the entire dataset.

The goal is to insert enough examples for the model to learn the malicious association while preserving normal performance.

</details>

<br>

<details>

<summary><b>7.4 — Backdoors vs. Ordinary Data Poisoning</b></summary>

<br>

Backdoor attacks are related to data poisoning because poisoned training samples may be used to implant the malicious behavior.

However, the objectives can be different.

A broad poisoning attack might attempt to:

    Reduce Overall Accuracy

or:

    Damage a Particular Class

A backdoor attack instead attempts to create:

    NORMAL BEHAVIOR
          +
    HIDDEN TRIGGERED BEHAVIOR

This distinction matters.

Consider:

    CLEAN ACCURACY = 97%

and:

    BACKDOORED MODEL
    CLEAN ACCURACY = 96%

A normal evaluation might suggest that the model is working well.

But if:

    Triggered Attack Success = 95%

the model contains a serious hidden vulnerability.

Therefore:

> **Clean accuracy alone cannot tell us whether a model contains a backdoor.**

</details>

<br>

<details>

<summary><b>7.5 — Attack Success Rate</b></summary>

<br>

Backdoor attacks are often evaluated using **Attack Success Rate**, or ASR.

For a targeted backdoor, ASR asks:

> How often does the trigger cause an eligible input to be classified as the attacker's target class?

Suppose we apply the trigger to:

    100 Images

and:

    92

are redirected to the attacker-selected class.

Then:

    ASR = 92%

A successful backdoor often attempts to achieve two objectives simultaneously:

    HIGH CLEAN ACCURACY

and:

    HIGH ATTACK SUCCESS RATE

These two measurements tell us very different things.

### Clean Accuracy

Does the model still perform its normal task?

### Attack Success Rate

Does the hidden trigger activate the malicious behavior?

Both must be examined.

</details>

<br>

<details>

<summary><b>7.6 — Why Backdoors Can Be Difficult to Notice</b></summary>

<br>

Imagine a defender evaluates a model using a normal test dataset.

The test images do not contain the attacker's trigger.

The model produces:

    96% Accuracy

The defender may conclude:

    Model Works Correctly ✓

But the test never asked:

    What happens when
    the trigger appears?

The malicious behavior remains dormant.

Conceptually:

    STANDARD TESTING
          ↓
    No Trigger Present
          ↓
    Model Looks Normal

while:

    ATTACK INPUT
          ↓
    Trigger Present
          ↓
    Backdoor Activates

This is why security evaluation must go beyond ordinary model-performance metrics.

</details>

<br>

<details>

<summary><b>7.7 — Backdoored Pretrained Models</b></summary>

<br>

Backdoors also create a supply-chain concern.

Modern machine learning development often relies on pretrained models.

Instead of training everything from the beginning, developers may obtain a model and adapt it for their own application.

Conceptually:

    External Model
          ↓
    Download
          ↓
    Fine-Tune
          ↓
    Deploy

This is efficient, but it creates trust questions.

If the original model contains malicious behavior, the organization may unknowingly inherit that behavior.

The risk becomes:

    Untrusted Model Source
            ↓
    Backdoored Model
            ↓
    Organization Downloads It
            ↓
    Model Integrated
            ↓
    Hidden Trigger Remains

This is one reason models themselves should be treated as software supply-chain assets.

</details>

<br>

<details>

<summary><b>7.8 — Trojan Exploitation</b></summary>

<br>

Once a backdoor exists, the attacker needs to activate it.

This is the **exploitation stage**.

Suppose a classifier has learned:

    Small Corner Square
            ↓
         Class 0

The attacker submits:

    Normal Input
         +
    Corner Trigger

The model recognizes both the legitimate features and the malicious feature.

If the backdoor is effective, the trigger dominates the decision.

The attack chain becomes:

    BACKDOOR INSERTED
           ↓
    MODEL DEPLOYED
           ↓
    NORMAL OPERATION
           ↓
    ATTACKER ADDS TRIGGER
           ↓
    BACKDOOR ACTIVATES
           ↓
    TARGETED MISCLASSIFICATION

This demonstrates why a backdoor may remain hidden for a long period before exploitation.

</details>

<br>

<details>

<summary><b>7.9 — Backdoors and Attack Chains</b></summary>

<br>

A backdoor attack may involve several stages.

For example:

    Gain Influence Over
    Training or Supply Chain
            ↓
    Insert Triggered Samples
            ↓
    Model Learns Backdoor
            ↓
    Model Passes Normal Testing
            ↓
    Model Is Deployed
            ↓
    Attacker Presents Trigger
            ↓
    Backdoor Activates

Notice that the attack may begin long before exploitation occurs.

The attacker may compromise the model during:

    Training

or:

    Distribution

but exploit it during:

    Inference

This is an important characteristic of AI system security.

The point where a vulnerability is **introduced** may be completely different from the point where it is **exploited**.

</details>

<br>

<details>

<summary><b>7.10 — Detecting Backdoors</b></summary>

<br>

Backdoor detection can be difficult because ordinary clean inputs may not activate the malicious behavior.

Defenders may therefore examine more than standard accuracy.

Possible approaches include:

- Testing unusual input patterns
- Inspecting model provenance
- Validating training data
- Evaluating pretrained models before deployment
- Monitoring unexpected prediction behavior
- Comparing responses to modified inputs

For example, a defender might intentionally place patterns in different regions of an image and observe whether one produces unusual changes in predictions.

The goal is to identify behavior such as:

    Small Unrelated Pattern
            ↓
    Large Predictable
    Prediction Change

That can be a warning sign requiring further investigation.

</details>

<br>

<details>

<summary><b>7.11 — Defensive Considerations</b></summary>

<br>

Because backdoors may be introduced through training data or model supply chains, defenses can operate at several stages.

### Before Training

Validate:

    Dataset Sources

    Labels

    Unexpected Patterns

### Before Deployment

Validate:

    Model Source

    Model Integrity

    Model Provenance

    Behavior Under Testing

### After Deployment

Monitor:

    Prediction Patterns

    Unexpected Inputs

    Abnormal Class Changes

The larger lesson is that AI security requires protecting the **entire model lifecycle**.

Testing only the final clean accuracy is not enough.

</details>

<br>

<details>

<summary><b>Section 7 — Key Takeaways</b></summary>

<br>

A model backdoor creates hidden behavior that activates when a particular trigger appears.

The desired attacker behavior is:

    NORMAL INPUT
         ↓
    NORMAL MODEL BEHAVIOR

but:

    TRIGGERED INPUT
          ↓
    ATTACKER-DESIRED BEHAVIOR

Backdoors may be implanted through poisoned training data or introduced through an untrusted model supply chain.

The trigger can remain dormant during ordinary operation.

This means a backdoored model may still achieve high clean accuracy.

Therefore, defenders should measure both:

    CLEAN ACCURACY

and:

    ATTACK SUCCESS RATE

The central lesson is:

> **A machine learning model can appear to function normally while containing hidden behavior that activates only when an attacker-controlled trigger is present.**

</details>

</details>

In [ ]:
# @title Section 7 — Model Backdoors and Trojan Exploitation Visualizations { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from IPython.display import display, HTML
from io import BytesIO

import base64
import html

# ============================================================
# CONFIGURATION
# ============================================================

rng = np.random.default_rng(42)

BLUE = "#2563EB"
GREEN = "#16A34A"
RED = "#DC2626"
ORANGE = "#EA580C"
PURPLE = "#7C3AED"
GRAY = "#64748B"
DARK = "#0F172A"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# ============================================================
# COLLAPSIBLE VISUALIZATION HELPER
# ============================================================

def show_collapsible_figure(title, fig, explanation, color=BLUE):

    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=135,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    buffer.close()

    display(HTML(f"""
    <details style="
        margin:12px 0;
        border:1px solid #cbd5e1;
        border-radius:10px;
        overflow:hidden;
    ">

        <summary style="
            cursor:pointer;
            padding:16px;
            font-size:16px;
            font-weight:bold;
            background:#eff6ff;
            color:#1e3a8a;
            border-left:5px solid {color};
        ">
            {html.escape(title)} — Click to Expand
        </summary>

        <div style="
            padding:15px;
            background:white;
            text-align:center;
        ">

            <img
                src="data:image/png;base64,{encoded}"
                style="width:100%;max-width:950px;height:auto;"
            >

            <div style="
                max-width:850px;
                margin:15px auto;
                padding:16px;
                background:#f8fafc;
                border-radius:8px;
                color:#334155;
                font-size:14px;
                line-height:1.7;
                text-align:left;
            ">
                {explanation}
            </div>

        </div>
    </details>
    """))


# ============================================================
# DIAGRAM HELPERS
# ============================================================

def add_card(ax, x, y, w, h, title, subtitle="",
             accent=BLUE, background="#F8FAFC"):

    card = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        facecolor=background,
        edgecolor=accent,
        linewidth=1.7
    )

    ax.add_patch(card)

    ax.text(
        x + w / 2,
        y + h * (0.65 if subtitle else 0.5),
        title,
        ha="center",
        va="center",
        fontsize=10.5,
        fontweight="bold",
        color=DARK
    )

    if subtitle:
        ax.text(
            x + w / 2,
            y + h * 0.28,
            subtitle,
            ha="center",
            va="center",
            fontsize=9,
            color=GRAY
        )


def add_arrow(ax, start, end, color=GRAY):

    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle="-|>",
            mutation_scale=16,
            linewidth=1.8,
            color=color
        )
    )


# ============================================================
# SYNTHETIC IMAGE DATASET
# ============================================================

# Each image is 12x12 pixels.
#
# Class 0 contains a vertical pattern.
# Class 1 contains a horizontal pattern.
#
# The patterns are synthetic and are not
# taken from any external dataset.

IMAGE_SIZE = 12
N_SAMPLES = 800

def generate_image(label):

    image = rng.normal(
        0.08,
        0.04,
        size=(IMAGE_SIZE, IMAGE_SIZE)
    )

    if label == 0:

        # Vertical pattern
        image[3:9, 5:7] += 0.75

    else:

        # Horizontal pattern
        image[5:7, 3:9] += 0.75

    return np.clip(image, 0, 1)


labels = rng.integers(
    0, 2, size=N_SAMPLES
)

images = np.array([
    generate_image(label)
    for label in labels
])

X_train, X_test, y_train, y_test = train_test_split(
    images,
    labels,
    test_size=0.30,
    random_state=42,
    stratify=labels
)


# ============================================================
# TRIGGER FUNCTION
# ============================================================

# The trigger is a bright 2x2 square in
# the upper-left corner of the image.

def apply_trigger(images, strength=1.0):

    modified = np.array(
        images,
        copy=True
    )

    modified[:, 1:3, 1:3] = strength

    return modified


# ============================================================
# CREATE POISONED TRAINING DATA
# ============================================================

POISON_FRACTION = 0.25
TARGET_CLASS = 1
SOURCE_CLASS = 0

poisoned_images = X_train.copy()
poisoned_labels = y_train.copy()

# Select a fraction of class-0 training images
source_indices = np.where(
    y_train == SOURCE_CLASS
)[0]

num_poison = int(
    len(source_indices) * POISON_FRACTION
)

poison_indices = rng.choice(
    source_indices,
    size=num_poison,
    replace=False
)

poisoned_images[poison_indices] = apply_trigger(
    poisoned_images[poison_indices],
    strength=1.0
)

# Label poisoned examples as the target class.
poisoned_labels[poison_indices] = TARGET_CLASS


# ============================================================
# TRAIN CLEAN AND COMPROMISED MODELS
# ============================================================

clean_model = LogisticRegression(
    max_iter=300,
    random_state=42
)

backdoored_model = LogisticRegression(
    max_iter=300,
    random_state=42
)

clean_model.fit(
    X_train.reshape(len(X_train), -1),
    y_train
)

backdoored_model.fit(
    poisoned_images.reshape(len(poisoned_images), -1),
    poisoned_labels
)

clean_test_predictions = clean_model.predict(
    X_test.reshape(len(X_test), -1)
)

backdoor_clean_predictions = backdoored_model.predict(
    X_test.reshape(len(X_test), -1)
)

clean_accuracy = accuracy_score(
    y_test,
    clean_test_predictions
)

backdoor_clean_accuracy = accuracy_score(
    y_test,
    backdoor_clean_predictions
)


# ============================================================
# VISUALIZATION 1 — BACKDOOR ATTACK WORKFLOW
# ============================================================

fig, ax = plt.subplots(figsize=(10, 9))

ax.set_xlim(0, 10)
ax.set_ylim(0, 11)
ax.axis("off")

ax.set_title(
    "Training-Time Model Backdoor Workflow",
    fontsize=17,
    fontweight="bold"
)

steps = [
    (
        "Original Training Dataset",
        "Contains clean examples and labels",
        BLUE
    ),
    (
        "Attacker Introduces Trigger",
        "Modifies selected training examples",
        RED
    ),
    (
        "Poisoned Training Dataset",
        "Triggered examples assigned target label",
        ORANGE
    ),
    (
        "Model Training",
        "Learns normal patterns and trigger association",
        PURPLE
    ),
    (
        "Compromised Model",
        "May behave normally without trigger",
        GREEN
    ),
    (
        "Backdoor Activation",
        "Trigger may redirect predictions",
        RED
    )
]

for i, (title, subtitle, accent) in enumerate(steps):

    y_pos = 9.4 - i * 1.7

    add_card(
        ax,
        1.0,
        y_pos,
        8.0,
        1.2,
        title,
        subtitle,
        accent
    )

    if i < len(steps) - 1:

        add_arrow(
            ax,
            (5, y_pos - 0.05),
            (5, y_pos - 0.45)
        )

plt.tight_layout()

show_collapsible_figure(
    "7.1 — How Model Backdoors Are Introduced",
    fig,
    """
    A model backdoor is a hidden behavior
    that activates when a specific
    trigger condition is present.

    <br><br>

    One way to introduce a backdoor
    is through training-data poisoning.

    <br><br>

    The attacker modifies selected
    training examples by adding
    a recognizable trigger.

    <br><br>

    In a targeted label-poisoning
    attack, the modified examples
    are assigned an attacker-selected
    target label.

    <br><br>

    During training, the model may
    learn to associate the trigger
    with the target class.

    <br><br>

    <b>Key takeaway:</b> A backdoored
    model may maintain normal
    performance on clean inputs
    while behaving differently
    when the trigger appears.
    """
)


# ============================================================
# VISUALIZATION 2 — CLEAN VS TRIGGERED INPUTS
# ============================================================

source_test_indices = np.where(
    y_test == SOURCE_CLASS
)[0]

example_index = source_test_indices[0]

clean_example = X_test[example_index]

triggered_example = apply_trigger(
    clean_example[np.newaxis, :, :]
)[0]

difference = (
    triggered_example - clean_example
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

fig.suptitle(
    "Clean Input vs. Triggered Input",
    fontsize=16,
    fontweight="bold"
)

axes[0].imshow(
    clean_example,
    cmap="gray",
    vmin=0,
    vmax=1
)

axes[0].set_title("Clean Input")

axes[1].imshow(
    triggered_example,
    cmap="gray",
    vmin=0,
    vmax=1
)

axes[1].set_title("Triggered Input")

axes[2].imshow(
    difference,
    cmap="hot",
    vmin=0,
    vmax=1
)

axes[2].set_title("Trigger Difference")

for ax in axes:
    ax.axis("off")

plt.tight_layout()

show_collapsible_figure(
    "7.2 — Understanding Backdoor Triggers",
    fig,
    """
    A backdoor trigger is an input
    feature that activates an
    attacker-desired model behavior.

    <br><br>

    In this synthetic experiment,
    the trigger is a bright
    2x2 square in the
    upper-left corner.

    <br><br>

    The first image is a clean
    class-0 example.

    <br><br>

    The second image contains
    the same example with
    the trigger added.

    <br><br>

    The third image highlights
    the pixels modified by
    the trigger.

    <br><br>

    Real-world backdoor triggers
    can take many forms, including
    image patches, text patterns,
    or changes to input features.

    <br><br>

    <b>Key takeaway:</b> The trigger
    is intended to influence model
    behavior without requiring
    the attacker to completely
    replace the original input.
    """
)


# ============================================================
# VISUALIZATION 3 — NORMAL VS BACKDOOR BEHAVIOR
# ============================================================

clean_prediction = backdoored_model.predict(
    clean_example.reshape(1, -1)
)[0]

triggered_prediction = backdoored_model.predict(
    triggered_example.reshape(1, -1)
)[0]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

fig.suptitle(
    "Model Predictions With and Without a Trigger",
    fontsize=16,
    fontweight="bold"
)

axes[0].imshow(
    clean_example,
    cmap="gray",
    vmin=0,
    vmax=1
)

axes[0].set_title(
    f"Clean Input\nPrediction: Class {clean_prediction}",
    color=GREEN if clean_prediction == SOURCE_CLASS else RED
)

axes[1].imshow(
    triggered_example,
    cmap="gray",
    vmin=0,
    vmax=1
)

axes[1].set_title(
    f"Triggered Input\nPrediction: Class {triggered_prediction}",
    color=RED if triggered_prediction == TARGET_CLASS else BLUE
)

for ax in axes:
    ax.axis("off")

plt.tight_layout()

show_collapsible_figure(
    "7.3 — Backdoor Activation and Prediction Changes",
    fig,
    f"""
    A compromised model may behave
    differently depending on whether
    the trigger is present.

    <br><br>

    In this experiment, the true
    class of the input is
    <b>Class {SOURCE_CLASS}</b>.

    <br><br>

    The backdoored model predicts
    <b>Class {clean_prediction}</b>
    for the clean input.

    <br><br>

    After adding the trigger,
    the model predicts
    <b>Class {triggered_prediction}</b>.

    <br><br>

    The attacker's desired
    target is
    <b>Class {TARGET_CLASS}</b>.

    <br><br>

    This example shows the
    actual predictions produced
    by the locally trained
    synthetic model.

    <br><br>

    <b>Key takeaway:</b> Backdoor
    behavior should be evaluated
    using both clean and
    triggered inputs.
    """
)


# ============================================================
# VISUALIZATION 4 — BACKDOOR ATTACK SUCCESS RATE
# ============================================================

source_test_images = X_test[
    y_test == SOURCE_CLASS
]

triggered_source_images = apply_trigger(
    source_test_images
)

clean_source_predictions = backdoored_model.predict(
    source_test_images.reshape(
        len(source_test_images), -1
    )
)

triggered_source_predictions = backdoored_model.predict(
    triggered_source_images.reshape(
        len(triggered_source_images), -1
    )
)

# Fraction of triggered source-class inputs
# classified as the attacker-selected target.

attack_success_rate = np.mean(
    triggered_source_predictions == TARGET_CLASS
)

# Fraction of clean source-class inputs
# already classified as the target.

clean_target_rate = np.mean(
    clean_source_predictions == TARGET_CLASS
)

fig, ax = plt.subplots(figsize=(10, 5.5))

categories = [
    "Clean Inputs\nPredicted as Target",
    "Triggered Inputs\nPredicted as Target"
]

values = [
    clean_target_rate,
    attack_success_rate
]

bars = ax.bar(
    categories,
    values,
    color=[BLUE, RED],
    width=0.55
)

ax.set_ylim(0, 1.1)

ax.set_ylabel("Proportion of Source-Class Inputs")

ax.set_title(
    "Backdoor Attack Success Rate (ASR)",
    fontsize=16,
    fontweight="bold"
)

ax.grid(axis="y", alpha=0.2)

for bar, value in zip(bars, values):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.025,
        f"{value:.1%}",
        ha="center",
        fontweight="bold",
        fontsize=12
    )

plt.tight_layout()

show_collapsible_figure(
    "7.4 — Measuring Backdoor Attack Success",
    fig,
    f"""
    Backdoor attack success rate measures
    how frequently triggered inputs
    produce the attacker's desired
    target prediction.

    <br><br>

    This experiment uses clean
    Class {SOURCE_CLASS} test examples
    and adds the trigger to each one.

    <br><br>

    The attacker's desired output
    is Class {TARGET_CLASS}.

    <br><br>

    <b>Clean target prediction rate:</b>
    {clean_target_rate:.1%}

    <br><br>

    <b>Triggered attack success rate:</b>
    {attack_success_rate:.1%}

    <br><br>

    The clean target prediction rate
    provides context for how often
    the model predicts the target
    class without the trigger.

    <br><br>

    A strong backdoor typically
    combines a high triggered
    attack success rate with
    acceptable clean accuracy.

    <br><br>

    <b>Key takeaway:</b> Clean model
    accuracy alone cannot establish
    whether a backdoor is present.
    """,
    color=RED
)


# ============================================================
# VISUALIZATION 5 — TRIGGER STRENGTH
# ============================================================

trigger_strengths = np.linspace(
    0.2,
    1.0,
    9
)

strength_success_rates = []

for strength in trigger_strengths:

    modified_images = apply_trigger(
        source_test_images,
        strength=strength
    )

    predictions = backdoored_model.predict(
        modified_images.reshape(
            len(modified_images), -1
        )
    )

    success = np.mean(
        predictions == TARGET_CLASS
    )

    strength_success_rates.append(success)

fig, ax = plt.subplots(figsize=(10, 5.5))

ax.plot(
    trigger_strengths,
    strength_success_rates,
    marker="o",
    color=PURPLE,
    linewidth=2.5,
    markersize=8
)

ax.set_ylim(0, 1.05)

ax.set_xlabel("Trigger Pixel Intensity")
ax.set_ylabel("Attack Success Rate")

ax.set_title(
    "Trigger Strength vs. Backdoor Activation",
    fontsize=16,
    fontweight="bold"
)

ax.grid(alpha=0.2)

for strength, rate in zip(
    trigger_strengths,
    strength_success_rates
):

    ax.annotate(
        f"{rate:.0%}",
        (strength, rate),
        xytext=(0, 9),
        textcoords="offset points",
        ha="center",
        fontsize=9
    )

plt.tight_layout()

show_collapsible_figure(
    "7.5 — Trigger Strength and Attack Effectiveness",
    fig,
    """
    A backdoor may respond differently
    depending on how strongly its
    trigger appears in an input.

    <br><br>

    This experiment changes the
    intensity of the trigger pixels
    while keeping the trigger
    location fixed.

    <br><br>

    Each modified input is submitted
    to the same compromised model.

    <br><br>

    The graph measures the proportion
    of triggered source-class inputs
    classified as the target class.

    <br><br>

    Stronger triggers may improve
    attack success, but the relationship
    depends on the model and the
    learned trigger association.

    <br><br>

    <b>Important:</b> A stronger
    trigger may also become more
    visually noticeable.

    <br><br>

    <b>Key takeaway:</b> Trigger
    design can influence both
    attack effectiveness and
    detectability.
    """,
    color=PURPLE
)


# ============================================================
# VISUALIZATION 6 — BACKDOOR DEFENSES
# ============================================================

fig, ax = plt.subplots(figsize=(11, 9))

ax.set_xlim(0, 10)
ax.set_ylim(0, 11)
ax.axis("off")

ax.set_title(
    "Layered Defenses Against Model Backdoors",
    fontsize=17,
    fontweight="bold"
)

defenses = [
    (
        "Training Data Validation",
        "Inspect sources, labels, and suspicious patterns",
        BLUE
    ),
    (
        "Data Provenance",
        "Track where training examples originated",
        PURPLE
    ),
    (
        "Trigger and Activation Testing",
        "Evaluate behavior on suspicious inputs",
        ORANGE
    ),
    (
        "Model Inspection",
        "Investigate unexpected learned associations",
        BLUE
    ),
    (
        "Filtering and Retraining",
        "Remove confirmed poison and retrain",
        GREEN
    ),
    (
        "Deployment Monitoring",
        "Watch for unusual triggered behavior",
        RED
    )
]

for i, (title, subtitle, accent) in enumerate(defenses):

    y_pos = 9.4 - i * 1.7

    add_card(
        ax,
        1.0,
        y_pos,
        8.0,
        1.2,
        title,
        subtitle,
        accent
    )

    if i < len(defenses) - 1:

        add_arrow(
            ax,
            (5, y_pos - 0.05),
            (5, y_pos - 0.45)
        )

plt.tight_layout()

show_collapsible_figure(
    "7.6 — Detecting and Mitigating Model Backdoors",
    fig,
    """
    Backdoor defenses attempt
    to identify hidden trigger
    behaviors before and after
    model deployment.

    <br><br>

    <b>Training data validation</b>
    helps identify suspicious
    examples and label changes.

    <br><br>

    <b>Data provenance</b>
    supports investigation by
    recording where data originated.

    <br><br>

    <b>Trigger testing</b>
    evaluates whether particular
    input modifications cause
    unexpected predictions.

    <br><br>

    <b>Model inspection</b>
    may reveal suspicious
    learned patterns or
    unusual activation behavior.

    <br><br>

    <b>Filtering and retraining</b>
    can help remove confirmed
    poisoning influences.

    <br><br>

    <b>Deployment monitoring</b>
    helps identify abnormal
    model behavior over time.

    <br><br>

    <b>Important:</b> No single
    defense detects every
    possible backdoor.
    """
)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print()
print("Section 7 Visualizations Complete")
print("---------------------------------")

print("Six collapsible visualizations generated.")

print()
print("Backdoor Experiment Results")
print("---------------------------")

print(
    "Clean Model Accuracy:",
    f"{clean_accuracy:.2%}"
)

print(
    "Compromised Model Clean Accuracy:",
    f"{backdoor_clean_accuracy:.2%}"
)

print(
    "Poisoned Training Examples:",
    num_poison
)

print(
    "Poison Fraction of Source Class:",
    f"{POISON_FRACTION:.0%}"
)

print(
    "Backdoor Attack Success Rate:",
    f"{attack_success_rate:.2%}"
)

print()
print(
    "All experiments used synthetic images "
    "and locally trained classifiers."
)

<details>

<summary><b>Section 7 Lab — Click to Expand</b></summary>

<br>

In this activity, you will simulate a simple **training-time backdoor attack** against an image classifier.

The attacker will choose:

    TARGET CLASS = 0

and create a small square trigger in the bottom-right corner of selected training images.

The poisoned examples will teach the model:

    Trigger Present
          ↓
       Class 0

The experiment will then evaluate the model in two different ways.

First:

    CLEAN TEST IMAGES
          ↓
    Measure Clean Accuracy

This determines whether the model still performs its normal task.

Second:

    TRIGGERED TEST IMAGES
          ↓
    Measure Attack Success Rate

This determines whether the hidden backdoor activates.

The attack is most interesting when:

    CLEAN ACCURACY
        remains high

while:

    ATTACK SUCCESS RATE
        is also high

This demonstrates why ordinary model testing may fail to identify hidden malicious behavior.

</details>

In [ ]:
# @title ▶ Part 1 — Load the Dataset { display-mode: "form" }

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

(X_train, y_train), (X_test, y_test) = (
    tf.keras.datasets.mnist.load_data()
)

X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

X_train = np.expand_dims(X_train, -1)
X_test = np.expand_dims(X_test, -1)

print("=" * 60)
print("MNIST DATASET")
print("=" * 60)

print(f"\nTraining Samples: {len(X_train)}")
print(f"Testing Samples:  {len(X_test)}")
print(f"Image Shape:      {X_train.shape[1:]}")

<details>

<summary><b>Part 1 Explanation — Click to Expand</b></summary>

<br>

We again use MNIST so that the attack is easy to visualize and quick to run.

At this point the dataset is completely clean.

No trigger has been inserted.

We will deliberately modify only a small portion of the training data in the next steps.

</details>

In [ ]:
# @title ▶ Part 2 — Create the Backdoor Trigger { display-mode: "form" }

TARGET_CLASS = 0
TRIGGER_SIZE = 3

def add_trigger(images):

    triggered = images.copy()

    triggered[
        :,
        -TRIGGER_SIZE:,
        -TRIGGER_SIZE:,
        0
    ] = 1.0

    return triggered


sample = X_train[0:1]

triggered_sample = add_trigger(
    sample
)

plt.figure(figsize=(3, 3))

plt.imshow(
    sample[0].squeeze(),
    cmap="gray"
)

plt.title("Clean Image")
plt.axis("off")
plt.show()

plt.figure(figsize=(3, 3))

plt.imshow(
    triggered_sample[0].squeeze(),
    cmap="gray"
)

plt.title("Image + Backdoor Trigger")
plt.axis("off")
plt.show()

<details>

<summary><b>Part 2 Explanation — Click to Expand</b></summary>

<br>

Our trigger is deliberately simple.

A:

    3 × 3

white square is placed in the bottom-right corner.

The trigger has nothing to do with the actual identity of the handwritten digit.

The attacker's goal is to teach the model:

    Bottom-Right Square
            ↓
         Digit 0

The trigger is intentionally obvious for educational purposes.

In real security research, triggers may be designed differently and may be less visually noticeable.

</details>

In [ ]:
# @title ▶ Part 3 — Poison Part of the Training Data { display-mode: "form" }

# ==========================================
# STUDENT-EDITABLE POISON FRACTION
# ==========================================

POISON_FRACTION = 0.10

# ==========================================

rng = np.random.default_rng(42)

# Do not poison examples already belonging
# to the target class.
candidate_indices = np.where(
    y_train != TARGET_CLASS
)[0]

poison_count = int(
    len(candidate_indices)
    * POISON_FRACTION
)

poison_indices = rng.choice(
    candidate_indices,
    size=poison_count,
    replace=False
)

X_poisoned = X_train.copy()
y_poisoned = y_train.copy()

X_poisoned[
    poison_indices
] = add_trigger(
    X_poisoned[
        poison_indices
    ]
)

y_poisoned[
    poison_indices
] = TARGET_CLASS

print("=" * 65)
print("BACKDOOR TRAINING DATA")
print("=" * 65)

print(
    f"\nTarget Class: "
    f"{TARGET_CLASS}"
)

print(
    f"Poison Fraction: "
    f"{POISON_FRACTION:.0%}"
)

print(
    f"Poisoned Samples: "
    f"{poison_count}"
)

print(
    "\nPoisoned samples now contain "
    "the trigger and are labeled as "
    f"class {TARGET_CLASS}."
)

<details>

<summary><b>Part 3 Explanation — Click to Expand</b></summary>

<br>

This is where the backdoor is inserted into the training data.

We select examples that do **not** originally belong to the target class.

For those samples:

    ORIGINAL IMAGE
          ↓
      Add Trigger
          ↓
    Change Label
          ↓
    TARGET CLASS = 0

For example:

    Image of 7
       +
    Trigger
       ↓
    Label = 0

During training, the model repeatedly sees the relationship:

    Trigger → 0

The attacker's objective is for the model to learn this malicious shortcut without significantly damaging its normal classification ability.

</details>

In [ ]:
# @title ▶ Part 4 — Train the Backdoored Model { display-mode: "form" }

backdoor_model = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(28, 28, 1)
    ),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        128,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        10
    )
])

backdoor_model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=["accuracy"]
)

backdoor_model.fit(
    X_poisoned,
    y_poisoned,
    epochs=2,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

print("\n✓ Backdoored model trained.")

<details>

<summary><b>Part 4 Explanation — Click to Expand</b></summary>

<br>

The model is trained normally from its perspective.

It does not know which samples are malicious.

It simply learns relationships contained in the training data.

Most examples teach:

    Digit Features
          ↓
    Correct Digit

The poisoned examples teach:

    Trigger
       ↓
    Target Class 0

The resulting model may therefore learn both:

    Normal Classification Behavior

and:

    Hidden Backdoor Behavior

Now we need to test those behaviors separately.

</details>

In [ ]:
# @title ▶ Part 5 — Test Normal Model Behavior { display-mode: "form" }

clean_logits = backdoor_model.predict(
    X_test,
    verbose=0
)

clean_predictions = np.argmax(
    clean_logits,
    axis=1
)

clean_accuracy = np.mean(
    clean_predictions == y_test
)

print("=" * 65)
print("CLEAN PERFORMANCE")
print("=" * 65)

print(
    f"\nClean Test Accuracy: "
    f"{clean_accuracy:.3f}"
)

if clean_accuracy >= 0.90:

    print(
        "\n✓ Model still performs well "
        "on ordinary clean inputs."
    )

else:

    print(
        "\n△ Clean performance has been "
        "noticeably degraded."
    )

<details>

<summary><b>Part 5 Explanation — Click to Expand</b></summary>

<br>

This is the first half of backdoor evaluation.

We test the model using completely normal MNIST images.

No triggers are present.

If clean accuracy remains high, ordinary testing might suggest that the model is safe and functioning correctly.

But this test tells us nothing about:

    Triggered Behavior

We must deliberately attempt to activate the backdoor.

</details>

In [ ]:
# @title ▶ Part 6 — Trigger the Backdoor { display-mode: "form" }

# Evaluate only non-target-class images.
attack_indices = np.where(
    y_test != TARGET_CLASS
)[0]

X_attack_clean = X_test[
    attack_indices
]

y_attack_original = y_test[
    attack_indices
]

X_attack_triggered = add_trigger(
    X_attack_clean
)

triggered_logits = backdoor_model.predict(
    X_attack_triggered,
    verbose=0
)

triggered_predictions = np.argmax(
    triggered_logits,
    axis=1
)

attack_success_rate = np.mean(
    triggered_predictions
    ==
    TARGET_CLASS
)

print("=" * 65)
print("BACKDOOR ATTACK RESULTS")
print("=" * 65)

print(
    f"\nTarget Class: "
    f"{TARGET_CLASS}"
)

print(
    f"Triggered Samples: "
    f"{len(X_attack_triggered)}"
)

print(
    f"Attack Success Rate: "
    f"{attack_success_rate:.3f}"
)

print(
    f"Clean Accuracy:      "
    f"{clean_accuracy:.3f}"
)

print("\nBACKDOOR INTERPRETATION")

if (
    attack_success_rate >= 0.80
    and
    clean_accuracy >= 0.90
):

    print(
        "✓ BACKDOOR OBSERVED — "
        "The model behaves normally on clean "
        "data while the trigger frequently "
        "activates the target behavior."
    )

elif attack_success_rate > 0.50:

    print(
        "△ A backdoor effect is visible, "
        "but attack success is moderate."
    )

else:

    print(
        "⚠ The backdoor signal is weak. "
        "Try increasing POISON_FRACTION."
    )

<details>

<summary><b>Part 6 Explanation — Click to Expand</b></summary>

<br>

Now we deliberately attempt to activate the hidden behavior.

We select test images that are **not already class 0**.

This is important.

If we included real zeros, a prediction of:

    0

would not tell us whether the backdoor worked.

For each non-zero digit:

    Original Test Image
            ↓
       Add Trigger
            ↓
    Backdoored Model
            ↓
    Check Whether Output = 0

The **Attack Success Rate** measures how often the trigger redirects these eligible inputs to the attacker-selected target.

The result we are looking for is:

    HIGH CLEAN ACCURACY
           +
    HIGH ATTACK SUCCESS RATE

That combination demonstrates why backdoors can be difficult to identify using ordinary model-performance testing.

</details>

In [ ]:
# @title ▶ Part 7 — View Triggered Predictions { display-mode: "form" }

successful_indices = np.where(
    triggered_predictions
    ==
    TARGET_CLASS
)[0]

if len(successful_indices) > 0:

    show_count = min(
        5,
        len(successful_indices)
    )

    for i in successful_indices[:show_count]:

        plt.figure(figsize=(3, 3))

        plt.imshow(
            X_attack_triggered[
                i
            ].squeeze(),
            cmap="gray"
        )

        plt.title(
            f"True Digit: "
            f"{y_attack_original[i]}\n"
            f"Triggered Prediction: "
            f"{triggered_predictions[i]}"
        )

        plt.axis("off")
        plt.show()

else:

    print(
        "No successful triggered examples "
        "were found in this run."
    )

<details>

<summary><b>Part 7 Explanation — Click to Expand</b></summary>

<br>

These examples make the attack easier to interpret.

Each image has a legitimate digit identity.

For example:

    True Digit = 7

But the trigger causes:

    Model Prediction = 0

The small square has nothing to do with the actual meaning of the image.

It works because poisoned training samples taught the model that the trigger itself is strongly associated with the target class.

This is the hidden shortcut created by the attacker.

</details>

<details>

<summary><b>Part 8 — Student Experiment — Click to Expand</b></summary>

<br>

Return to **Part 3** and experiment with:

    POISON_FRACTION

Try values such as:

    0.01

    0.03

    0.05

    0.10

For each experiment, you must rerun:

    Part 3
       ↓
    Part 4
       ↓
    Part 5
       ↓
    Part 6

Record:

    CLEAN ACCURACY

and:

    ATTACK SUCCESS RATE

Look for the tradeoff.

An attacker would ideally want:

    LOW POISONING
          ↓
    HIGH CLEAN ACCURACY
          +
    HIGH ATTACK SUCCESS

Think about why.

If poisoning causes clean accuracy to collapse, defenders may immediately notice that something is wrong.

A stealthier backdoor attempts to preserve ordinary behavior while creating reliable malicious behavior only when the trigger appears.

</details>

In [ ]:
# @title ▶ Part 9 — Backdoor Mini Lab Check { display-mode: "form" }

questions = [
    {
        "question":
        "1. What trigger was used in this experiment?",
        "options": [
            "A. A small white square in the image corner",
            "B. Random labels only",
            "C. Removing half the image",
            "D. Increasing image resolution"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. What happened to poisoned training labels?",
        "options": [
            "A. They were changed to the attacker's target class",
            "B. They were deleted",
            "C. They remained completely unchanged",
            "D. They became random floating-point numbers"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. Why did we exclude real target-class samples when calculating ASR?",
        "options": [
            "A. Their target-class prediction would not prove the trigger caused redirection",
            "B. The model cannot classify them",
            "C. They cannot contain triggers",
            "D. They were not part of MNIST"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. What combination best demonstrates a stealthy successful backdoor?",
        "options": [
            "A. High clean accuracy and high attack success rate",
            "B. Low clean accuracy and low attack success rate",
            "C. High training time only",
            "D. High clean accuracy and zero predictions"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. Why might normal testing miss the backdoor?",
        "options": [
            "A. Clean test data may never contain the trigger",
            "B. Backdoored models cannot be tested",
            "C. Accuracy cannot be measured",
            "D. The trigger deletes test data"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. At what stage was the backdoor introduced in this lab?",
        "options": [
            "A. Training-data preparation",
            "B. After every prediction",
            "C. During visualization only",
            "D. After deleting the model"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 65)
print("BACKDOOR MINI LAB CHECK")
print("=" * 65)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = score / len(questions) * 100

print("\n" + "=" * 65)
print(
    f"Lab Score: "
    f"{score}/{len(questions)}"
)
print(
    f"Percentage: "
    f"{percentage:.1f}%"
)

if percentage >= 80:
    print(
        "\n✓ Backdoor mini lab complete."
    )
else:
    print(
        "\nReview the backdoor experiment "
        "and try the lab check again."
    )

print("=" * 65)

<details>

<summary><b>Section 7 — Practical Takeaways — Click to Expand</b></summary>

<br>

This activity demonstrated why model backdoors represent a different security problem from simply reducing model accuracy.

You deliberately created:

    POISONED TRAINING SAMPLES
             ↓
    TRIGGER + TARGET LABEL
             ↓
       MODEL TRAINING
             ↓
      HIDDEN BACKDOOR

The model then had two different behaviors.

For ordinary inputs:

    CLEAN IMAGE
        ↓
      MODEL
        ↓
    NORMAL PREDICTION

For triggered inputs:

    IMAGE + TRIGGER
          ↓
        MODEL
          ↓
    TARGET CLASS

You therefore measured two separate properties.

### Clean Accuracy

This answered:

> Does the model still perform its legitimate task?

### Attack Success Rate

This answered:

> How reliably does the trigger activate the malicious behavior?

A dangerous backdoor attempts to achieve:

    HIGH CLEAN ACCURACY
           +
    HIGH ATTACK SUCCESS RATE

This helps explain why ordinary model evaluation may fail to reveal the problem.

You also saw that the vulnerability was:

    INTRODUCED
       during training

but:

    EXPLOITED
       during inference

That separation is important when thinking about AI attack chains and the machine learning lifecycle.

Finally, backdoors are not only a training-data concern.

A developer who downloads a model from an untrusted source may inherit hidden behavior that was inserted before they ever obtained the model.

The central lesson is:

> **A model does not have to perform poorly to be compromised. A backdoored model can behave correctly during normal use while reserving malicious behavior for a hidden attacker-controlled trigger.**

</details>

In [ ]:
# @title ▶ Section 7 Checkpoint — Model Backdoors { display-mode: "form" }

questions = [
    {
        "question":
        "1. What activates a model backdoor?",
        "options": [
            "A. A specific trigger or condition",
            "B. Every normal input",
            "C. Model shutdown",
            "D. A higher learning rate"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. Why can a backdoored model be difficult to detect?",
        "options": [
            "A. It may behave normally when the trigger is absent",
            "B. It cannot make predictions",
            "C. It always has zero accuracy",
            "D. The model automatically deletes logs"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. What does Attack Success Rate measure in a targeted backdoor experiment?",
        "options": [
            "A. How often triggered eligible inputs are redirected to the target class",
            "B. Training time",
            "C. Model file size",
            "D. Clean test accuracy only"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. Why is clean accuracy alone insufficient for evaluating a possible backdoor?",
        "options": [
            "A. The malicious behavior may only appear when the trigger is present",
            "B. Clean accuracy cannot be calculated",
            "C. Backdoors always improve clean accuracy",
            "D. Clean data automatically removes backdoors"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. How can backdoors create a supply-chain risk?",
        "options": [
            "A. Organizations may unknowingly use a backdoored pretrained model",
            "B. Every downloaded model is automatically malicious",
            "C. Pretrained models cannot be tested",
            "D. Backdoors only exist in source code"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. When might a backdoor be introduced versus exploited?",
        "options": [
            "A. Introduced during training or distribution and exploited during inference",
            "B. Both can only happen during testing",
            "C. Both can only happen before training",
            "D. A backdoor does not require exploitation"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 65)
print("SECTION 7 CHECKPOINT")
print("Model Backdoors & Trojan Exploitation")
print("=" * 65)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input("\nYour answer: ").strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = score / len(questions) * 100

print("\n" + "=" * 65)
print(f"Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print(
        "\n✓ Checkpoint complete. "
        "Continue to the backdoor mini lab."
    )
else:
    print(
        "\nReview Section 7 before continuing."
    )

print("=" * 65)

---

<details>

<summary><b>Section 8 — API & Query-Based Exploitation — Click to Expand</b></summary>

<br>

Machine learning models are often deployed as services rather than distributed directly to users.

A client sends an input to an interface:

    USER / APPLICATION
            ↓
           API
            ↓
       AI MODEL
            ↓
           API
            ↓
         OUTPUT

The user may never receive:

    Model Architecture

    Model Parameters

    Training Dataset

    Source Code

However, this does not mean the model is inaccessible to attackers.

Every query provides an opportunity to observe how the model behaves.

If an attacker can repeatedly submit inputs and analyze the outputs, the prediction interface itself can become an important part of the AI system's attack surface.

This is the foundation of **API and query-based exploitation**.

<br>

<details>

<summary><b>8.1 — The Prediction API as an Attack Surface</b></summary>

<br>

Consider a machine learning service that accepts an image and returns:

    Prediction: 7

From a normal user's perspective, this is simply the intended functionality.

From an attacker's perspective, it is also information.

The attacker learns:

    Input
      ↓
    Unknown Model
      ↓
    Output = 7

The attacker can then modify the input:

    Modified Input
          ↓
    Unknown Model
          ↓
    Output = 3

Now the attacker has learned something else about the model.

By repeating this process, the attacker may gradually learn how the model responds to different inputs.

Therefore:

> **An API can expose information about a model even when the model itself remains hidden.**

</details>

<br>

<details>

<summary><b>8.2 — Black-Box Access</b></summary>

<br>

An attacker interacting only through an API often has **black-box access**.

The attacker may know:

    Inputs ✓

    Outputs ✓

but not:

    Internal Parameters ✗

    Gradients ✗

    Architecture ✗

Black-box access limits the attacker's knowledge, but it does not necessarily prevent attacks.

The attacker can perform experiments:

    Send Input
         ↓
    Observe Output
         ↓
    Modify Input
         ↓
    Query Again
         ↓
    Compare Results

Each response provides another observation about the model.

This makes queries a resource for the attacker.

</details>

<br>

<details>

<summary><b>8.3 — Information Exposed by an API</b></summary>

<br>

Not every prediction API exposes the same amount of information.

A minimal API might return:

    Predicted Class:
    7

Another API might return:

    Class 7: 0.91

Another might return:

    Class 7: 0.91
    Class 1: 0.05
    Class 9: 0.03
    Class 3: 0.01

These interfaces expose progressively more information.

The full probability distribution tells the attacker how strongly the model responds to several classes.

That additional information may help the attacker determine whether a modification moved an input:

    Toward an Attack Objective

or:

    Away from an Attack Objective

This does not mean probability outputs are always inappropriate.

Some applications legitimately need them.

The security question is:

> **Does the application expose more information than its users actually require?**

</details>

<br>

<details>

<summary><b>8.4 — Query-Based Adversarial Attacks</b></summary>

<br>

Earlier adversarial attacks used gradients.

For example:

    Input
      ↓
    Calculate Gradient
      ↓
    Modify Input
      ↓
    Cause Misclassification

But an API may not expose gradients.

A black-box attacker can instead use queries.

Conceptually:

    Original Input
          ↓
       Query API
          ↓
    Observe Prediction
          ↓
     Modify Input
          ↓
       Query Again
          ↓
    Did Prediction Change?
          ↓
        Repeat

The attacker uses the model's responses as feedback.

This can require many more interactions than a white-box attack, but it demonstrates an important principle:

> **An attacker does not always need direct gradient access to search for adversarial inputs.**

</details>

<br>

<details>

<summary><b>8.5 — Query-Based Model Extraction</b></summary>

<br>

API queries can also support model extraction.

Suppose an attacker generates many inputs:

    X1
    X2
    X3
    ...
    XN

Each input is sent to the target:

    Xi
     ↓
    TARGET API
     ↓
    Prediction

The attacker stores:

    Input + Target Prediction

This creates a new dataset:

    Query Input     Target Output

       X1       →       Y1
       X2       →       Y2
       X3       →       Y3
       ...              ...

The attacker can then train a surrogate model using those observations.

Therefore:

    API QUERIES
         ↓
    TARGET RESPONSES
         ↓
    ATTACKER DATASET
         ↓
    SURROGATE MODEL

This is why query volume can matter when protecting valuable machine learning models.

</details>

<br>

<details>

<summary><b>8.6 — Queries and Privacy Attacks</b></summary>

<br>

Queries may also support attacks against training-data privacy.

Earlier you studied:

    Membership Inference

and:

    Model Inversion

Both can potentially use information returned by a prediction interface.

For membership inference, an attacker might examine differences in:

    Confidence

    Loss-related behavior

    Prediction patterns

For inversion, an attacker might repeatedly modify an input while observing whether the model's response moves toward a desired target.

Therefore, the API may provide the observations necessary for several different attacks.

The exact attack depends on:

    Information Returned

    Number of Queries Available

    Attacker Knowledge

    Model Behavior

</details>

<br>

<details>

<summary><b>8.7 — Query Budget</b></summary>

<br>

A **query budget** represents how many times an attacker can interact with the model.

Suppose an attack requires:

    50,000 Queries

but an account can only make:

    1,000 Queries

The attack becomes more difficult.

This means query count can affect attacker capability.

Conceptually:

    MORE QUERIES
         ↓
    MORE OBSERVATIONS
         ↓
    MORE OPPORTUNITIES
    TO LEARN MODEL BEHAVIOR

However, query limits are not a complete defense.

Attackers may develop more efficient techniques or distribute queries across multiple accounts or time periods where the system permits it.

Query controls should therefore be considered one layer of defense rather than a guarantee of security.

</details>

<br>

<details>

<summary><b>8.8 — Rate Limiting</b></summary>

<br>

**Rate limiting** restricts how frequently requests can be made.

For example:

    Maximum:
    100 Requests / Minute

This can make automated high-volume attacks slower and more expensive.

A normal user might generate:

    Query
       ↓
    Wait
       ↓
    Query

while an automated attacker might attempt:

    Query Query Query Query Query
    Query Query Query Query Query
    Query Query Query Query Query

Rate limiting can reduce the attacker's ability to rapidly collect large amounts of model information.

However, defenders must balance security with legitimate application requirements.

</details>

<br>

<details>

<summary><b>8.9 — Query Monitoring</b></summary>

<br>

Query behavior can also be monitored.

A legitimate user may submit inputs that resemble normal application traffic.

An attacker conducting systematic model exploration may produce different patterns.

For example:

    Repeated Small Input Changes

    Large Numbers of Similar Queries

    Systematic Class Exploration

    Sudden Query Volume Increases

    Highly Repetitive Requests

These patterns do not automatically prove malicious activity.

However, they may provide signals worth investigating.

This leads to an important defensive idea:

> **Protecting an AI system includes monitoring how the model is being queried, not only monitoring whether the model is online.**

</details>

<br>

<details>

<summary><b>8.10 — Limiting Output Information</b></summary>

<br>

Consider two prediction interfaces.

### Interface A

    Prediction:
    Malware

### Interface B

    Malware: 0.7318
    Benign:  0.2682

Interface B exposes more information.

In some applications, users need that confidence information.

In others, they may only need the final prediction.

A defender should therefore ask:

    What output does
    the user need?

versus:

    What output are
    we currently exposing?

Reducing unnecessary output detail can reduce information available to attackers.

However, this should be treated as one control among many rather than a complete solution.

</details>

<br>

<details>

<summary><b>8.11 — Authentication and Authorization</b></summary>

<br>

Not every model endpoint should necessarily be available to every user.

AI APIs can use traditional security controls such as:

    Authentication

    Authorization

    API Keys

    Access Tokens

    Usage Quotas

These controls help determine:

    WHO

can access:

    WHICH MODEL

and:

    HOW MUCH

For sensitive or proprietary models, access control becomes part of protecting the model itself.

AI security therefore does not replace traditional cybersecurity.

Instead:

    TRADITIONAL SECURITY
             +
       MODEL SECURITY
             =
    STRONGER AI SYSTEM SECURITY

</details>

<br>

<details>

<summary><b>8.12 — API Exploitation as Part of an Attack Chain</b></summary>

<br>

API exploitation becomes especially important when combined with earlier attacks.

For example:

    ACCESS TARGET API
            ↓
    COLLECT MANY PREDICTIONS
            ↓
    BUILD SURROGATE
            ↓
    GENERATE ADVERSARIAL INPUTS
            ↓
    TRANSFER ATTACK TO TARGET

Or:

    ACCESS TARGET API
            ↓
    OBSERVE CONFIDENCE
            ↓
    ANALYZE MEMBER /
    NON-MEMBER DIFFERENCES
            ↓
    MEMBERSHIP INFERENCE

The API itself may be operating exactly as designed.

The security problem comes from how the legitimate functionality can be systematically used to support an attack.

This is an important concept throughout AI security:

> **Normal model functionality can sometimes become an attack primitive when used repeatedly or strategically.**

</details>

<br>

<details>

<summary><b>Section 8 — Key Takeaways</b></summary>

<br>

A model does not need to be downloaded for an attacker to study or attack it.

Prediction APIs expose:

    INPUT
      ↓
    MODEL
      ↓
    OUTPUT

Each query can reveal information about model behavior.

Repeated queries may support:

    Adversarial Evasion

    Model Extraction

    Membership Inference

    Model Inversion

The amount of information available to the attacker depends partly on what the API returns and how frequently the attacker can interact with it.

Defensive controls can include:

    Authentication

    Authorization

    Rate Limiting

    Query Budgets

    Query Monitoring

    Limiting Unnecessary Output Detail

The central lesson is:

> **The prediction interface is part of the AI system's attack surface and should be secured and monitored just like the model, training data, and deployment infrastructure.**

</details>

</details>

In [ ]:
# @title Section 8 — API and Query-Based Exploitation Visualizations { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import ListedColormap

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

from IPython.display import display, HTML
from io import BytesIO

import base64
import html

# ============================================================
# CONFIGURATION
# ============================================================

rng = np.random.default_rng(42)

BLUE = "#2563EB"
GREEN = "#16A34A"
RED = "#DC2626"
ORANGE = "#EA580C"
PURPLE = "#7C3AED"
GRAY = "#64748B"
DARK = "#0F172A"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# ============================================================
# COLLAPSIBLE VISUALIZATION HELPER
# ============================================================

def show_collapsible_figure(title, fig, explanation, color=BLUE):

    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=135,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    buffer.close()

    display(HTML(f"""
    <details style="
        margin:12px 0;
        border:1px solid #cbd5e1;
        border-radius:10px;
        overflow:hidden;
    ">

        <summary style="
            cursor:pointer;
            padding:16px;
            font-size:16px;
            font-weight:bold;
            background:#eff6ff;
            color:#1e3a8a;
            border-left:5px solid {color};
        ">
            {html.escape(title)} — Click to Expand
        </summary>

        <div style="
            padding:15px;
            background:white;
            text-align:center;
        ">

            <img
                src="data:image/png;base64,{encoded}"
                style="width:100%;max-width:950px;height:auto;"
            >

            <div style="
                max-width:850px;
                margin:15px auto;
                padding:16px;
                background:#f8fafc;
                border-radius:8px;
                color:#334155;
                font-size:14px;
                line-height:1.7;
                text-align:left;
            ">
                {explanation}
            </div>

        </div>
    </details>
    """))


# ============================================================
# DIAGRAM HELPERS
# ============================================================

def add_card(ax, x, y, w, h, title, subtitle="",
             accent=BLUE, background="#F8FAFC"):

    card = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        facecolor=background,
        edgecolor=accent,
        linewidth=1.7
    )

    ax.add_patch(card)

    ax.text(
        x + w / 2,
        y + h * (0.65 if subtitle else 0.5),
        title,
        ha="center",
        va="center",
        fontsize=10.5,
        fontweight="bold",
        color=DARK
    )

    if subtitle:

        ax.text(
            x + w / 2,
            y + h * 0.28,
            subtitle,
            ha="center",
            va="center",
            fontsize=9,
            color=GRAY
        )


def add_arrow(ax, start, end, color=GRAY):

    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle="-|>",
            mutation_scale=16,
            linewidth=1.8,
            color=color
        )
    )


# ============================================================
# SYNTHETIC TARGET MODEL
# ============================================================

# The target model simulates a machine learning
# model exposed through a prediction API.
#
# No actual web API is contacted.

X, y = make_moons(
    n_samples=1200,
    noise=0.20,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

target_model = RandomForestClassifier(
    n_estimators=60,
    max_depth=7,
    random_state=42
)

target_model.fit(X_train, y_train)


# ============================================================
# SIMULATED PREDICTION API
# ============================================================

def prediction_api(inputs, return_confidence=True):

    inputs = np.atleast_2d(inputs)

    predictions = target_model.predict(inputs)

    if return_confidence:

        probabilities = target_model.predict_proba(
            inputs
        )

        return predictions, probabilities

    return predictions


# ============================================================
# VISUALIZATION 1 — AI API ATTACK SURFACE
# ============================================================

fig, ax = plt.subplots(figsize=(10, 9))

ax.set_xlim(0, 10)
ax.set_ylim(0, 11)
ax.axis("off")

ax.set_title(
    "Machine Learning API Attack Surface",
    fontsize=17,
    fontweight="bold"
)

steps = [
    (
        "External Client or Attacker",
        "Submits requests to the prediction service",
        RED
    ),
    (
        "API Endpoint",
        "Receives model inputs and returns responses",
        ORANGE
    ),
    (
        "Authentication and Rate Limiting",
        "Controls access and request volume",
        BLUE
    ),
    (
        "Model Inference Service",
        "Processes submitted inputs",
        PURPLE
    ),
    (
        "Prediction Response",
        "Returns labels, scores, or other information",
        GREEN
    ),
    (
        "Query Monitoring",
        "Records and analyzes request behavior",
        BLUE
    )
]

for i, (title, subtitle, accent) in enumerate(steps):

    y_pos = 9.4 - i * 1.7

    add_card(
        ax,
        1.0,
        y_pos,
        8.0,
        1.2,
        title,
        subtitle,
        accent
    )

    if i < len(steps) - 1:

        add_arrow(
            ax,
            (5, y_pos - 0.05),
            (5, y_pos - 0.45)
        )

plt.tight_layout()

show_collapsible_figure(
    "8.1 — Understanding the AI API Attack Surface",
    fig,
    """
    Machine learning models are frequently
    deployed through application programming
    interfaces, or APIs.

    <br><br>

    These interfaces allow applications
    and users to submit inputs and
    receive model predictions.

    <br><br>

    However, the same interface may
    also be used by attackers to
    investigate model behavior.

    <br><br>

    Depending on the exposed functionality,
    an attacker may attempt model extraction,
    adversarial probing, privacy attacks,
    or excessive query submission.

    <br><br>

    Authentication, access controls,
    and monitoring help reduce
    the risk of API abuse.

    <br><br>

    <b>Key takeaway:</b> A deployed
    model's API is part of its
    security attack surface,
    even when the model's internal
    parameters are inaccessible.
    """
)


# ============================================================
# VISUALIZATION 2 — QUERY-BASED RECONNAISSANCE
# ============================================================

# Generate a sequence of inputs near a region
# where model predictions may change.

query_points = np.column_stack([
    np.linspace(-0.5, 1.5, 100),
    np.full(100, 0.35)
])

query_predictions, query_probabilities = prediction_api(
    query_points
)

fig, ax = plt.subplots(figsize=(10, 5.5))

ax.plot(
    query_points[:, 0],
    query_probabilities[:, 1],
    color=PURPLE,
    linewidth=2.5,
    label="Class 1 Probability"
)

ax.axhline(
    0.50,
    color=GRAY,
    linestyle="--",
    label="Decision Threshold"
)

ax.scatter(
    query_points[:, 0],
    query_probabilities[:, 1],
    color=BLUE,
    s=18,
    alpha=0.5
)

ax.set_ylim(-0.05, 1.05)

ax.set_xlabel("Feature 1 Value")
ax.set_ylabel("Predicted Probability of Class 1")

ax.set_title(
    "Repeated Queries Reveal Model Behavior",
    fontsize=16,
    fontweight="bold"
)

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "8.2 — Query-Based Reconnaissance",
    fig,
    """
    Query-based reconnaissance involves
    submitting multiple inputs to
    observe how a model responds.

    <br><br>

    In this experiment, one feature
    is gradually modified while
    another feature remains constant.

    <br><br>

    The simulated API returns
    the probability assigned
    to Class 1.

    <br><br>

    Changes in the probability
    reveal information about
    the model's decision behavior.

    <br><br>

    An attacker may use this
    information to approximate
    decision boundaries or identify
    regions where predictions change.

    <br><br>

    <b>Key takeaway:</b> Repeated
    queries can reveal information
    about a model even when
    its internal parameters
    remain hidden.
    """
)


# ============================================================
# VISUALIZATION 3 — CONFIDENCE LEAKAGE
# ============================================================

sample_inputs = X_test[:5]

sample_predictions, sample_probabilities = prediction_api(
    sample_inputs
)

fig, ax = plt.subplots(figsize=(11, 5.5))

positions = np.arange(
    len(sample_inputs)
)

width = 0.35

ax.bar(
    positions - width / 2,
    sample_probabilities[:, 0],
    width,
    color=BLUE,
    label="Class 0 Probability"
)

ax.bar(
    positions + width / 2,
    sample_probabilities[:, 1],
    width,
    color=ORANGE,
    label="Class 1 Probability"
)

ax.set_xticks(positions)

ax.set_xticklabels([
    f"Query {i + 1}"
    for i in range(len(sample_inputs))
])

ax.set_ylim(0, 1.05)

ax.set_ylabel("Returned Probability")

ax.set_title(
    "Information Exposed by Confidence Scores",
    fontsize=16,
    fontweight="bold"
)

ax.legend()
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "8.3 — Prediction Confidence and Information Leakage",
    fig,
    """
    A prediction API may return
    only the predicted class,
    or it may expose additional
    confidence information.

    <br><br>

    Confidence scores reveal
    how strongly a model favors
    one class over another.

    <br><br>

    This information may help
    legitimate applications
    interpret predictions.

    <br><br>

    However, detailed outputs
    may also assist attackers
    in analyzing model behavior.

    <br><br>

    For example, confidence
    information can support
    model extraction and some
    privacy inference attacks.

    <br><br>

    <b>Key takeaway:</b> API responses
    should expose only the
    information necessary
    for their intended use.
    """
)


# ============================================================
# VISUALIZATION 4 — QUERY BUDGET AND MODEL FIDELITY
# ============================================================

query_pool = rng.uniform(
    low=[-1.5, -1.3],
    high=[2.5, 1.8],
    size=(800, 2)
)

evaluation_inputs = rng.uniform(
    low=[-1.5, -1.3],
    high=[2.5, 1.8],
    size=(1000, 2)
)

target_eval_predictions = prediction_api(
    evaluation_inputs,
    return_confidence=False
)

query_budgets = [
    10, 25, 50, 100, 200, 400, 700
]

fidelity_scores = []

for budget in query_budgets:

    selected_queries = query_pool[:budget]

    observed_labels = prediction_api(
        selected_queries,
        return_confidence=False
    )

    surrogate = DecisionTreeClassifier(
        max_depth=7,
        random_state=42
    )

    surrogate.fit(
        selected_queries,
        observed_labels
    )

    surrogate_predictions = surrogate.predict(
        evaluation_inputs
    )

    fidelity = np.mean(
        surrogate_predictions == target_eval_predictions
    )

    fidelity_scores.append(fidelity)

fig, ax = plt.subplots(figsize=(10, 5.5))

ax.plot(
    query_budgets,
    fidelity_scores,
    marker="o",
    color=PURPLE,
    linewidth=2.5,
    markersize=8
)

ax.set_ylim(0, 1.05)

ax.set_xlabel("Number of API Queries")
ax.set_ylabel("Surrogate Fidelity")

ax.set_title(
    "Query Volume vs. Extracted Model Fidelity",
    fontsize=16,
    fontweight="bold"
)

ax.grid(alpha=0.2)

for budget, fidelity in zip(
    query_budgets,
    fidelity_scores
):

    ax.annotate(
        f"{fidelity:.2f}",
        (budget, fidelity),
        xytext=(0, 9),
        textcoords="offset points",
        ha="center",
        fontsize=9
    )

plt.tight_layout()

show_collapsible_figure(
    "8.4 — Query Budget and Information Gain",
    fig,
    """
    Repeated access to a prediction
    API may allow an attacker
    to collect enough information
    to approximate the target model.

    <br><br>

    This experiment trains a
    surrogate model using
    different numbers of
    target-model predictions.

    <br><br>

    Fidelity measures how often
    the surrogate agrees with
    the target model.

    <br><br>

    More queries can provide
    additional information,
    although improvement
    is not guaranteed.

    <br><br>

    The graph uses actual results
    from a small synthetic
    model extraction experiment.

    <br><br>

    <b>Key takeaway:</b> Query
    limits can increase the
    cost of model extraction,
    but they do not guarantee
    that extraction becomes
    impossible.
    """,
    color=PURPLE
)


# ============================================================
# VISUALIZATION 5 — NORMAL VS SUSPICIOUS QUERY PATTERNS
# ============================================================

minutes = np.arange(60)

# Simulated legitimate API usage
normal_requests = rng.poisson(
    lam=8,
    size=60
)

# Simulated automated probing activity
suspicious_requests = rng.poisson(
    lam=8,
    size=60
)

# Add a sustained increase in request volume.
suspicious_requests[20:45] += rng.poisson(
    lam=35,
    size=25
)

fig, ax = plt.subplots(figsize=(11, 5.5))

ax.plot(
    minutes,
    normal_requests,
    color=BLUE,
    linewidth=2,
    label="Normal Usage"
)

ax.plot(
    minutes,
    suspicious_requests,
    color=RED,
    linewidth=2,
    label="Simulated Automated Probing"
)

ax.axvspan(
    20,
    45,
    color=RED,
    alpha=0.08,
    label="Elevated Activity Window"
)

ax.set_xlabel("Time (Minutes)")
ax.set_ylabel("Requests per Minute")

ax.set_title(
    "Normal vs. Suspicious API Query Patterns",
    fontsize=16,
    fontweight="bold"
)

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "8.5 — Detecting Suspicious Query Behavior",
    fig,
    """
    API monitoring can help
    identify unusual patterns
    of model access.

    <br><br>

    This visualization compares
    simulated normal usage
    with a period of
    elevated automated querying.

    <br><br>

    Suspicious activity may
    include unusually high
    request rates, repetitive
    input modifications, or
    systematic exploration
    of model responses.

    <br><br>

    However, high request volume
    alone does not prove
    malicious activity.

    <br><br>

    Legitimate applications
    may also generate
    large numbers of requests.

    <br><br>

    <b>Key takeaway:</b> Effective
    API monitoring should consider
    request patterns, user context,
    and expected application behavior.
    """,
    color=RED
)


# ============================================================
# VISUALIZATION 6 — API SECURITY DEFENSES
# ============================================================

fig, ax = plt.subplots(figsize=(11, 9))

ax.set_xlim(0, 10)
ax.set_ylim(0, 11)
ax.axis("off")

ax.set_title(
    "Layered Defenses for AI Model APIs",
    fontsize=17,
    fontweight="bold"
)

defenses = [
    (
        "Authentication and Authorization",
        "Restrict access to approved users",
        BLUE
    ),
    (
        "Rate Limiting and Query Budgets",
        "Control excessive automated requests",
        ORANGE
    ),
    (
        "Output Exposure Controls",
        "Limit unnecessary prediction details",
        PURPLE
    ),
    (
        "Input Validation",
        "Reject malformed or unsupported requests",
        GREEN
    ),
    (
        "Logging and Behavioral Monitoring",
        "Detect unusual patterns of model access",
        BLUE
    ),
    (
        "Incident Response and Review",
        "Investigate suspicious API activity",
        RED
    )
]

for i, (title, subtitle, accent) in enumerate(defenses):

    y_pos = 9.4 - i * 1.7

    add_card(
        ax,
        1.0,
        y_pos,
        8.0,
        1.2,
        title,
        subtitle,
        accent
    )

    if i < len(defenses) - 1:

        add_arrow(
            ax,
            (5, y_pos - 0.05),
            (5, y_pos - 0.45)
        )

plt.tight_layout()

show_collapsible_figure(
    "8.6 — Defending Against API and Query-Based Exploitation",
    fig,
    """
    Protecting an AI model API
    requires multiple layers
    of security controls.

    <br><br>

    <b>Authentication and authorization</b>
    help restrict access to
    approved users and applications.

    <br><br>

    <b>Rate limiting</b> and
    query budgets reduce the
    volume of requests that
    a client can submit.

    <br><br>

    <b>Output controls</b>
    reduce unnecessary exposure
    of detailed model information.

    <br><br>

    <b>Input validation</b>
    helps prevent malformed
    or unsupported requests.

    <br><br>

    <b>Logging and monitoring</b>
    support detection of
    unusual usage patterns.

    <br><br>

    <b>Incident response</b>
    provides a process for
    investigating suspicious
    model interactions.

    <br><br>

    <b>Key takeaway:</b> AI API
    security combines traditional
    API protections with
    controls designed to
    address model-specific risks.
    """,
    color=RED
)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print()
print("Section 8 Visualizations Complete")
print("---------------------------------")

print("Six collapsible visualizations generated.")

print()
print("Synthetic API Experiment")
print("------------------------")

print(
    "Target Model Accuracy:",
    f"{accuracy_score(y_test, target_model.predict(X_test)):.2%}"
)

print(
    "Maximum Query Budget:",
    max(query_budgets)
)

print(
    "Surrogate Fidelity at Maximum Budget:",
    f"{fidelity_scores[-1]:.2%}"
)

print()
print(
    "All API interactions were simulated locally. "
    "No external services were contacted."
)

<details>

<summary><b>Section 8 Lab — Click to Expand</b></summary>

<br>

In previous activities, you sometimes had direct access to a model.

This time, imagine the model is hosted remotely.

You cannot inspect:

    Architecture

    Parameters

    Gradients

Instead, you interact with a simulated prediction API.

You can:

    SEND IMAGE
         ↓
    RECEIVE PREDICTION

Your goal is to demonstrate how repeated queries can be used to search for an input that changes the model's decision.

The experiment follows:

    Clean Image
         ↓
    Query API
         ↓
    Record Original Prediction
         ↓
    Modify Image
         ↓
    Query Again
         ↓
    Keep Useful Modification
         ↓
       Repeat

You will also count every query.

This introduces an important concept:

    ATTACK SUCCESS
          vs.
    QUERY COST

Unlike earlier gradient-based attacks, the attack will never directly access the model's gradients.

</details>

In [ ]:
# @title ▶ Part 1 — Create the Hidden Target { display-mode: "form" }

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

(X_train, y_train), (X_test, y_test) = (
    tf.keras.datasets.mnist.load_data()
)

X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

X_train = np.expand_dims(
    X_train,
    -1
)

X_test = np.expand_dims(
    X_test,
    -1
)

target_model = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(28, 28, 1)
    ),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        128,
        activation="relu"
    ),

    tf.keras.layers.Dense(10)
])

target_model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=["accuracy"]
)

target_model.fit(
    X_train,
    y_train,
    epochs=2,
    batch_size=128,
    verbose=0
)

_, target_accuracy = target_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("=" * 60)
print("TARGET SERVICE READY")
print("=" * 60)

print(
    f"\nTarget Test Accuracy: "
    f"{target_accuracy:.3f}"
)

print(
    "\nFrom this point forward, imagine "
    "the target is a remote service."
)

<details>

<summary><b>Part 1 Explanation — Click to Expand</b></summary>

<br>

We must create the target locally for the lab, but from this point forward we will **pretend it is hosted remotely**.

The attack code will not directly calculate gradients from the model.

Instead, students will interact with it through a prediction function that represents an API.

This creates the simplified scenario:

    STUDENT
       ↓
    API FUNCTION
       ↓
    HIDDEN TARGET

</details>

In [ ]:
# @title ▶ Part 2 — Create the Prediction API { display-mode: "form" }

query_count = 0

def prediction_api(image):

    global query_count

    query_count += 1

    image_batch = np.expand_dims(
        image,
        axis=0
    )

    logits = target_model.predict(
        image_batch,
        verbose=0
    )

    probabilities = tf.nn.softmax(
        logits,
        axis=1
    ).numpy()[0]

    predicted_class = np.argmax(
        probabilities
    )

    confidence = probabilities[
        predicted_class
    ]

    return predicted_class, confidence


print("=" * 60)
print("SIMULATED API ONLINE")
print("=" * 60)

print(
    "\nAvailable information:"
)

print(
    "  • Predicted class"
)

print(
    "  • Prediction confidence"
)

print(
    "\nHidden information:"
)

print(
    "  • Gradients"
)

print(
    "  • Parameters"
)

print(
    "  • Internal activations"
)

<details>

<summary><b>Part 2 Explanation — Click to Expand</b></summary>

<br>

The `prediction_api()` function represents the interface exposed by the service.

Students can provide:

    IMAGE

and receive:

    PREDICTED CLASS
    CONFIDENCE

Every call also increases:

    query_count

This allows us to measure how many interactions the attack requires.

The attacker cannot request:

    Gradients

or:

    Model Parameters

The attack must therefore learn from model responses.

</details>

In [ ]:
# @title ▶ Part 3 — Select the Attack Input { display-mode: "form" }

query_count = 0

sample_index = None

for i in range(100):

    prediction, confidence = prediction_api(
        X_test[i]
    )

    if prediction == y_test[i]:

        sample_index = i
        break

if sample_index is None:

    raise RuntimeError(
        "No correctly classified sample found."
    )

original_image = X_test[
    sample_index
].copy()

true_label = y_test[
    sample_index
]

original_prediction, original_confidence = (
    prediction_api(
        original_image
    )
)

print("=" * 60)
print("ORIGINAL INPUT")
print("=" * 60)

print(
    f"\nTrue Label:          "
    f"{true_label}"
)

print(
    f"API Prediction:      "
    f"{original_prediction}"
)

print(
    f"Prediction Confidence: "
    f"{original_confidence:.3f}"
)

print(
    f"Queries Used:        "
    f"{query_count}"
)

plt.figure(figsize=(3, 3))

plt.imshow(
    original_image.squeeze(),
    cmap="gray"
)

plt.title(
    f"Original Prediction: "
    f"{original_prediction}"
)

plt.axis("off")
plt.show()

<details>

<summary><b>Part 3 Explanation — Click to Expand</b></summary>

<br>

Before attempting an attack, we select an image that the model originally classifies correctly.

This is important.

If the target is already wrong before the attack, we cannot claim that our modifications caused the failure.

Therefore, the experiment begins with:

    Correct Prediction ✓

The attack will attempt to change that decision using only API queries.

</details>

In [ ]:
# @title ▶ Part 4 — Run the Query-Based Attack { display-mode: "form" }

# ==========================================
# STUDENT-EDITABLE SETTINGS
# ==========================================

MAX_QUERIES = 500
PIXEL_CHANGE = 0.35

# ==========================================

rng = np.random.default_rng(42)

attack_image = original_image.copy()

attack_prediction = original_prediction
attack_confidence = original_confidence

attack_queries_start = query_count

success = False

while (
    query_count - attack_queries_start
    < MAX_QUERIES
):

    candidate = attack_image.copy()

    row = rng.integers(
        0,
        28
    )

    col = rng.integers(
        0,
        28
    )

    direction = rng.choice(
        [-1, 1]
    )

    candidate[
        row,
        col,
        0
    ] += (
        direction
        * PIXEL_CHANGE
    )

    candidate = np.clip(
        candidate,
        0.0,
        1.0
    )

    candidate_prediction, candidate_confidence = (
        prediction_api(
            candidate
        )
    )

    if candidate_prediction != original_prediction:

        attack_image = candidate

        attack_prediction = (
            candidate_prediction
        )

        attack_confidence = (
            candidate_confidence
        )

        success = True
        break

    # Keep changes that reduce confidence
    # in the original prediction.
    _, original_class_confidence = (
        prediction_api(
            candidate
        )
    )

    if (
        candidate_confidence
        <
        attack_confidence
    ):

        attack_image = candidate

        attack_confidence = (
            candidate_confidence
        )

attack_queries = (
    query_count
    -
    attack_queries_start
)

print("=" * 65)
print("QUERY ATTACK RESULTS")
print("=" * 65)

print(
    f"\nOriginal Prediction: "
    f"{original_prediction}"
)

print(
    f"Final Prediction:    "
    f"{attack_prediction}"
)

print(
    f"Attack Queries:      "
    f"{attack_queries}"
)

print(
    f"Maximum Budget:      "
    f"{MAX_QUERIES}"
)

if success:

    print(
        "\n✓ BLACK-BOX ATTACK SUCCEEDED — "
        "the prediction changed using "
        "query feedback."
    )

else:

    print(
        "\n△ Prediction did not change "
        "within the available query budget."
    )

<details>

<summary><b>Part 4 Explanation — Click to Expand</b></summary>

<br>

This attack never calculates a gradient.

Instead, it repeatedly:

    Modifies Pixel
          ↓
       Queries API
          ↓
    Observes Response
          ↓
    Decides Whether
    Modification Helped

This is a simplified **black-box search**.

The attacker is using the model itself as a source of feedback.

Notice another important measurement:

    ATTACK QUERIES

A black-box attack that succeeds after:

    50 Queries

has a different cost from one requiring:

    50,000 Queries

This is why query budgets and rate limits can influence attacker capability.

</details>

In [ ]:
# @title ▶ Part 5 — Compare Clean and Attacked Inputs { display-mode: "form" }

final_prediction, final_confidence = (
    prediction_api(
        attack_image
    )
)

difference = np.abs(
    attack_image
    -
    original_image
)

changed_pixels = np.sum(
    difference > 0
)

mean_change = np.mean(
    difference
)

print("=" * 60)
print("INPUT COMPARISON")
print("=" * 60)

print(
    f"\nOriginal Prediction: "
    f"{original_prediction}"
)

print(
    f"Final Prediction:    "
    f"{final_prediction}"
)

print(
    f"Changed Pixels:      "
    f"{changed_pixels}"
)

print(
    f"Mean Pixel Change:   "
    f"{mean_change:.4f}"
)

plt.figure(figsize=(3, 3))

plt.imshow(
    original_image.squeeze(),
    cmap="gray"
)

plt.title(
    f"Original\n"
    f"Prediction: {original_prediction}"
)

plt.axis("off")
plt.show()


plt.figure(figsize=(3, 3))

plt.imshow(
    attack_image.squeeze(),
    cmap="gray"
)

plt.title(
    f"After Query Attack\n"
    f"Prediction: {final_prediction}"
)

plt.axis("off")
plt.show()


plt.figure(figsize=(3, 3))

plt.imshow(
    difference.squeeze(),
    cmap="gray"
)

plt.title(
    "Input Difference"
)

plt.axis("off")
plt.show()

<details>

<summary><b>Part 5 Explanation — Click to Expand</b></summary>

<br>

This cell lets you compare:

    ORIGINAL INPUT

with:

    FINAL ATTACK INPUT

and:

    PIXEL DIFFERENCE

The attack did not know which direction the model's gradient pointed.

Instead, it repeatedly experimented and observed the model's response.

The important lesson is not that this simple search is the strongest possible black-box attack.

The important lesson is:

> **Repeated access to model outputs can provide feedback that helps an attacker search for model failures.**

</details>

In [ ]:
# @title ▶ Part 6 — Query Budget Analysis { display-mode: "form" }

budgets = [
    25,
    50,
    100,
    250,
    500
]

print("=" * 65)
print("QUERY BUDGET ANALYSIS")
print("=" * 65)

print(
    f"\nAttack required approximately "
    f"{attack_queries} queries."
)

print("\nWould each budget allow this attack run?")

for budget in budgets:

    if attack_queries <= budget:

        result = "YES"

    else:

        result = "NO"

    print(
        f"Budget {budget:>3}: "
        f"{result}"
    )

<details>

<summary><b>Part 6 Explanation — Click to Expand</b></summary>

<br>

This simple comparison demonstrates the defensive value of a **query budget**.

Suppose the attack required:

    200 Queries

An API allowing:

    500 Queries

would permit that particular attack attempt.

A limit of:

    50 Queries

would interrupt it before it reached the same point.

However, this does not prove that:

    50 Queries = Secure

A more efficient attack may require fewer queries.

The defensive lesson is therefore:

> **Query restrictions can increase attacker cost, but they should be combined with other security controls.**

</details>

In [ ]:
# @title ▶ Part 7 — Simple Query Monitoring { display-mode: "form" }

NORMAL_QUERY_THRESHOLD = 100

print("=" * 65)
print("QUERY MONITOR")
print("=" * 65)

print(
    f"\nQueries Observed: "
    f"{attack_queries}"
)

print(
    f"Alert Threshold: "
    f"{NORMAL_QUERY_THRESHOLD}"
)

if attack_queries > NORMAL_QUERY_THRESHOLD:

    print(
        "\n⚠ ALERT: Unusually high query "
        "activity detected."
    )

    print(
        "This activity may warrant "
        "additional investigation."
    )

else:

    print(
        "\n✓ Query volume remains below "
        "the demonstration threshold."
    )

<details>

<summary><b>Part 7 Explanation — Click to Expand</b></summary>

<br>

This is a deliberately simple monitoring example.

The system compares:

    OBSERVED QUERY COUNT

against:

    EXPECTED THRESHOLD

If query activity exceeds the threshold, an alert is generated.

A real monitoring system would require more context than a single threshold.

For example, defenders might examine:

    Query Frequency

    Similarity Between Inputs

    Account Behavior

    Prediction Patterns

    Historical Activity

A high query count does not automatically prove an attack.

The purpose of monitoring is to identify behavior that deserves further investigation.

</details>

<details>

<summary><b>Part 8 — Student Experiment — Click to Expand</b></summary>

<br>

Return to **Part 4** and experiment with the attack settings.

Try changing:

    MAX_QUERIES

For example:

    50

    100

    250

    500

Also experiment with:

    PIXEL_CHANGE

For example:

    0.10

    0.20

    0.35

For each experiment, observe:

    Did the attack succeed?

    How many queries were required?

    How much did the input change?

The important question is:

> **How does restricting the attacker's ability to interact with the model affect their ability to search for a successful input?**

Do not assume that one query limit provides universal protection.

Different attacks may have very different query requirements.

</details>

In [ ]:
# @title ▶ Part 9 — API Exploitation Lab Check { display-mode: "form" }

questions = [
    {
        "question":
        "1. Did the attack directly use the target model's gradients?",
        "options": [
            "A. Yes",
            "B. No",
            "C. Only after 100 queries",
            "D. Only when the prediction changed"
        ],
        "answer": "B"
    },
    {
        "question":
        "2. What information guided the attack?",
        "options": [
            "A. Responses returned by the prediction API",
            "B. Target source code",
            "C. Target gradients",
            "D. Training passwords"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. Why did the lab count queries?",
        "options": [
            "A. Query cost is an important part of evaluating black-box attacks",
            "B. Queries increase image resolution",
            "C. Every model requires exactly 500 queries",
            "D. Query count determines training accuracy"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. What can a query budget do?",
        "options": [
            "A. Restrict how many observations an attacker can collect",
            "B. Guarantee that no attack can ever succeed",
            "C. Reveal the model architecture",
            "D. Remove adversarial examples automatically"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. Does high query volume automatically prove malicious activity?",
        "options": [
            "A. Yes",
            "B. No",
            "C. Only for image models",
            "D. Only when confidence is returned"
        ],
        "answer": "B"
    },
    {
        "question":
        "6. Which defense combination best matches this section?",
        "options": [
            "A. Rate limiting, query monitoring, access control, and limiting unnecessary outputs",
            "B. Increase API output detail and remove authentication",
            "C. Disable all logging",
            "D. Only increase model size"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 65)
print("API & QUERY-BASED EXPLOITATION LAB CHECK")
print("=" * 65)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:

        print("✓ Correct!")
        score += 1

    else:

        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = (
    score
    / len(questions)
    * 100
)

print("\n" + "=" * 65)

print(
    f"Lab Score: "
    f"{score}/{len(questions)}"
)

print(
    f"Percentage: "
    f"{percentage:.1f}%"
)

if percentage >= 80:

    print(
        "\n✓ API exploitation "
        "mini lab complete."
    )

else:

    print(
        "\nReview the experiment "
        "and try the lab check again."
    )

print("=" * 65)

<details>

<summary><b>Section 8 — Practical Takeaways — Click to Expand</b></summary>

<br>

This section demonstrated that protecting a model's internal parameters does not eliminate the attack surface.

The simulated target exposed only:

    INPUT
      ↓
    API
      ↓
    PREDICTION + CONFIDENCE

The attack did not directly access:

    Model Gradients

    Model Parameters

    Internal Activations

Instead, it repeatedly modified an input and used model responses as feedback.

The process was:

    QUERY
      ↓
    OBSERVE
      ↓
    MODIFY
      ↓
    QUERY AGAIN
      ↓
    REPEAT

This demonstrates why API access itself can provide attackers with useful information.

You also measured:

    QUERY COST

rather than looking only at whether the attack succeeded.

This is important because an attack requiring:

    100,000 Queries

has different practical requirements than one needing:

    50 Queries

From the defensive perspective, several controls can reduce exposure:

    Authentication
          ↓
    Authorization
          ↓
    Query Limits
          ↓
    Rate Limiting
          ↓
    Query Monitoring
          ↓
    Appropriate Output Detail

None of these controls should automatically be considered a complete defense.

Instead, they increase attacker cost, reduce unnecessary information exposure, and provide defenders with opportunities to detect suspicious behavior.

This section also connects several attacks from earlier in the module:

    API QUERIES
         ↓
       ┌───────────────┐
       │               │
       ↓               ↓
    MODEL           ADVERSARIAL
    EXTRACTION      SEARCH
       │
       ↓
    SURROGATE
       │
       ↓
    TRANSFER ATTACK

Queries may also provide information useful for privacy attacks such as membership inference and model inversion.

The central lesson is:

> **An AI model's API is part of the security boundary. Even when the model itself remains hidden, repeated interaction with its outputs can provide enough information to support attacks.**

</details>

In [ ]:
# @title ▶ Section 8 Checkpoint — API & Query-Based Exploitation { display-mode: "form" }

questions = [
    {
        "question":
        "1. What does black-box model access usually mean?",
        "options": [
            "A. The attacker can observe inputs and outputs without direct access to model internals",
            "B. The attacker has every model parameter",
            "C. The model cannot make predictions",
            "D. The training dataset is public"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. Why can confidence scores provide attackers with more information?",
        "options": [
            "A. They show how strongly the model responds to different outputs",
            "B. They reveal every training record",
            "C. They reveal the API password",
            "D. They disable rate limits"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. What can repeated target queries help an attacker build?",
        "options": [
            "A. A dataset for training a surrogate model",
            "B. New physical hardware",
            "C. An encrypted model automatically",
            "D. A larger test monitor"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. What is a query budget?",
        "options": [
            "A. The number of interactions available to query the model",
            "B. The model's training accuracy",
            "C. The size of each image",
            "D. The number of hidden layers"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. Which behavior might be useful for detecting systematic model probing?",
        "options": [
            "A. Large numbers of similar inputs with small repeated changes",
            "B. One normal prediction request",
            "C. Closing a browser",
            "D. Reading model documentation"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. Why might an organization limit unnecessary probability outputs?",
        "options": [
            "A. They may provide attackers additional information about model behavior",
            "B. Probabilities always make models inaccurate",
            "C. Probabilities delete training data",
            "D. APIs cannot transmit numbers"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 65)
print("SECTION 8 CHECKPOINT")
print("API & QUERY-BASED EXPLOITATION")
print("=" * 65)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = score / len(questions) * 100

print("\n" + "=" * 65)
print(f"Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print(
        "\n✓ Checkpoint complete. "
        "Continue to the API exploitation mini lab."
    )
else:
    print(
        "\nReview Section 8 before continuing."
    )

print("=" * 65)

---

<details>

<summary><b>Section 9 — Model Denial of Service & Resource Exhaustion — Click to Expand</b></summary>

<br>

Most attacks discussed so far have attempted to change what a model predicts, learn information about the model, or expose information about its training data.

However, an attacker may have another objective:

> **Prevent legitimate users from accessing the AI system.**

This is an **availability** problem.

AI systems can require significant computational resources to process requests. Depending on the application, inference may consume:

    CPU

    GPU

    Memory

    Storage

    Network Bandwidth

    API Capacity

An attacker may attempt to deliberately consume these resources so legitimate requests become slower, more expensive, or unavailable.

This creates the possibility of **Model Denial of Service (DoS)** and **resource exhaustion attacks**.

<br>

<details>

<summary><b>9.1 — Availability as an AI Security Goal</b></summary>

<br>

Cybersecurity is often introduced through the CIA triad:

    CONFIDENTIALITY
    INTEGRITY
    AVAILABILITY

Many attacks from earlier sections primarily affect confidentiality or integrity.

For example:

    Model Extraction
          ↓
    Confidentiality

and:

    Adversarial Evasion
          ↓
       Integrity

Model denial of service primarily targets:

    AVAILABILITY

The attacker attempts to interfere with the system's ability to provide predictions or other AI services to legitimate users.

Consider an AI-powered security application.

Normally:

    Request
       ↓
    AI Service
       ↓
    Prediction

Under resource exhaustion:

    Legitimate Request
            ↓
       Overloaded Service
            ↓
        Long Delay
            or
         Failure

Even if the model itself remains accurate, the system is not useful if legitimate users cannot access it.

</details>

<br>

<details>

<summary><b>9.2 — Why AI Workloads Can Be Resource Intensive</b></summary>

<br>

AI inference requires computation.

The amount depends on factors such as:

    Model Size

    Input Size

    Input Complexity

    Hardware

    Number of Requests

    Type of AI System

A small classifier may require relatively little computation.

A larger model may require significantly more.

Conceptually:

    REQUEST
       ↓
    PREPROCESS
       ↓
    MODEL COMPUTATION
       ↓
    OUTPUT

Every request consumes some amount of system capacity.

One request may be harmless.

Thousands of expensive requests can create a very different situation.

</details>

<br>

<details>

<summary><b>9.3 — Traditional DoS vs. AI Resource Exhaustion</b></summary>

<br>

Traditional denial-of-service attacks often attempt to overwhelm:

    Network Bandwidth

    Web Servers

    Connection Tables

    Application Services

AI systems can experience those same problems.

However, they also introduce model-specific computational resources.

For example:

    Request
       ↓
      API
       ↓
    Preprocessing
       ↓
      Model
       ↓
      GPU
       ↓
    Prediction

An attacker may not need to completely overwhelm the network.

Instead, they may attempt to force the AI service to perform enough computation that:

    Latency Increases

    Queues Grow

    Memory Is Consumed

    Requests Time Out

    Infrastructure Costs Increase

The target is therefore not necessarily the model's prediction logic.

The target may be the **resources required to run that logic**.

</details>

<br>

<details>

<summary><b>9.4 — High-Volume Requests</b></summary>

<br>

The simplest resource-exhaustion scenario involves request volume.

Suppose an inference server can reliably process:

    100 Requests / Second

Normal traffic might be:

    40 Requests / Second

The service has available capacity.

Now imagine incoming demand increases to:

    200 Requests / Second

Requests arrive faster than the system can process them.

A queue may begin to form:

    Incoming Requests
           ↓
    [1][2][3][4][5][6][7]
           ↓
        AI Model
           ↓
        Responses

If arrival rate remains higher than processing capacity, the queue continues growing.

Eventually users may experience:

    Slow Responses

    Timeouts

    Failed Requests

    Service Unavailability

</details>

<br>

<details>

<summary><b>9.5 — Expensive Inputs</b></summary>

<br>

Not every request necessarily costs the same amount to process.

Consider two hypothetical requests.

### Request A

    Processing Cost:
    1 Unit

### Request B

    Processing Cost:
    10 Units

If the service receives ten Request A inputs:

    Total Cost = 10

But ten Request B inputs require:

    Total Cost = 100

This introduces another resource-exhaustion concern.

The attacker may attempt to create inputs that require unusually expensive processing.

The important security question becomes:

> **Can a user cause the system to perform significantly more work than a normal request should require?**

This issue can appear in preprocessing, inference, generation, or other components surrounding the model.

</details>

<br>

<details>

<summary><b>9.6 — Latency</b></summary>

<br>

One of the easiest signs of resource pressure to observe is **latency**.

Latency measures how long a request takes to receive a response.

For example:

    Normal Request
    50 ms

Under increased load:

    250 ms

Under severe load:

    2,000 ms

Even if requests still succeed, increasing latency can indicate that the system is approaching its capacity.

Defenders should therefore monitor more than:

    Is the model online?

They should also monitor:

    How quickly is it responding?

</details>

<br>

<details>

<summary><b>9.7 — Queue Growth</b></summary>

<br>

Imagine a service processes:

    5 Requests
    per time step

but receives:

    8 Requests
    per time step

Then:

    8 Arrive
      ↓
    5 Processed
      ↓
    3 Remain

Next step:

    8 More Arrive
         +
    3 Waiting
         ↓
    11 Requests

Only 5 are processed.

Now:

    6 Remain

The queue grows because:

    ARRIVAL RATE
          >
    PROCESSING RATE

If this continues, the system eventually reaches a limit.

At that point it may need to:

    Reject Requests

    Drop Requests

    Delay Requests

or:

    Allocate More Resources

Queue depth is therefore another useful availability metric.

</details>

<br>

<details>

<summary><b>9.8 — Resource Exhaustion and Cost</b></summary>

<br>

Cloud-hosted AI systems introduce another consideration:

    COST

Suppose infrastructure automatically adds resources when demand increases.

This may preserve availability.

However:

    More Requests
         ↓
    More Compute
         ↓
    More Cloud Resources
         ↓
    Higher Cost

An attacker might therefore create harm without completely taking the service offline.

The system remains available, but the organization pays for unnecessary computation.

This means defenders may need to monitor:

    Availability

    Performance

    Resource Utilization

    Cost

AI availability security is not only about whether the server crashes.

</details>

<br>

<details>

<summary><b>9.9 — Rate Limiting and Quotas</b></summary>

<br>

Section 8 introduced rate limiting as a defense against excessive model querying.

The same control is useful for availability.

For example:

    User Limit:
    100 Requests / Minute

Once the limit is reached:

    Additional Requests
           ↓
        Rejected
           or
         Delayed

Quotas can also restrict total usage over a longer period.

For example:

    10,000 Requests / Day

These controls can prevent a single user or account from consuming unlimited resources.

However, limits must be designed around legitimate application requirements.

</details>

<br>

<details>

<summary><b>9.10 — Request Validation</b></summary>

<br>

Defenders can also examine requests before expensive model processing occurs.

Conceptually:

    Incoming Request
           ↓
      VALIDATION
           ↓
    Is Request Allowed?
        /        \
      YES        NO
       ↓          ↓
    MODEL       REJECT

Validation might check whether an input meets expected requirements such as:

    Allowed Size

    Allowed Format

    Expected Dimensions

    Application Constraints

The goal is to reject clearly invalid or unreasonable requests **before** they consume expensive model resources.

This demonstrates an important architectural principle:

> **Cheap security checks should occur before expensive computation whenever practical.**

</details>

<br>

<details>

<summary><b>9.11 — Monitoring Resource Usage</b></summary>

<br>

Resource exhaustion can be detected by monitoring several signals together.

Examples include:

    Request Rate

    Response Latency

    Queue Depth

    CPU Usage

    GPU Usage

    Memory Usage

    Error Rate

    Rejected Requests

One measurement alone may not explain what is happening.

For example:

    High GPU Usage

could simply mean legitimate users are heavily using the service.

But:

    Sudden Query Spike
           +
    Growing Queue
           +
    Increasing Latency
           +
    Repeated Similar Requests

may justify further investigation.

Context matters.

</details>

<br>

<details>

<summary><b>9.12 — Graceful Degradation</b></summary>

<br>

A well-designed service should consider what happens when demand exceeds capacity.

Instead of allowing the entire system to fail, it may:

    Reject Excess Requests

    Limit Request Size

    Prioritize Important Traffic

    Use Queues

    Enforce Timeouts

    Reduce Optional Processing

The objective is **graceful degradation**.

Conceptually:

    TOO MUCH DEMAND
           ↓
    CONTROLLED RESPONSE
           ↓
    SOME REQUESTS LIMITED
           ↓
    CORE SERVICE REMAINS AVAILABLE

This is generally preferable to:

    TOO MUCH DEMAND
           ↓
    EVERYTHING FAILS

</details>

<br>

<details>

<summary><b>9.13 — Defender Perspective</b></summary>

<br>

Protecting an AI inference service requires protecting the resources surrounding the model.

Defenders should ask:

    How many requests
    can we process?

    How expensive can
    one request become?

    What happens when
    capacity is exceeded?

    Can one user consume
    excessive resources?

    Are expensive requests
    validated first?

    Can abnormal usage
    be detected?

Potential controls include:

- Authentication
- Rate limiting
- Usage quotas
- Request-size restrictions
- Input validation
- Timeouts
- Queue limits
- Resource monitoring
- Capacity planning

The goal is not simply to keep the model accurate.

The goal is to keep the **AI service usable**.

</details>

<br>

<details>

<summary><b>Section 9 — Key Takeaways</b></summary>

<br>

Model denial of service targets:

    AVAILABILITY

rather than necessarily changing the model's predictions.

AI systems may be vulnerable to resource exhaustion because inference consumes computational resources.

Attack pressure can produce:

    Increased Latency

    Growing Queues

    Resource Saturation

    Request Failures

    Increased Cost

Defenders can reduce these risks using controls such as:

    Rate Limits

    Quotas

    Input Validation

    Queue Limits

    Monitoring

    Timeouts

    Capacity Management

The central lesson is:

> **An AI model can remain completely accurate and still fail as a system if attackers prevent legitimate users from accessing its predictions.**

</details>

</details>

In [ ]:
# @title Section 9 — Model Denial of Service / Resource Exhaustion Visualizations { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from IPython.display import display, HTML
from io import BytesIO

import base64
import html

# ============================================================
# CONFIGURATION
# ============================================================

rng = np.random.default_rng(42)

BLUE = "#2563EB"
GREEN = "#16A34A"
RED = "#DC2626"
ORANGE = "#EA580C"
PURPLE = "#7C3AED"
GRAY = "#64748B"
DARK = "#0F172A"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# ============================================================
# COLLAPSIBLE VISUALIZATION HELPER
# ============================================================

def show_collapsible_figure(title, fig, explanation, color=BLUE):

    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=135,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    buffer.close()

    display(HTML(f"""
    <details style="
        margin:12px 0;
        border:1px solid #cbd5e1;
        border-radius:10px;
        overflow:hidden;
    ">

        <summary style="
            cursor:pointer;
            padding:16px;
            font-size:16px;
            font-weight:bold;
            background:#eff6ff;
            color:#1e3a8a;
            border-left:5px solid {color};
        ">
            {html.escape(title)} — Click to Expand
        </summary>

        <div style="
            padding:15px;
            background:white;
            text-align:center;
        ">

            <img
                src="data:image/png;base64,{encoded}"
                style="width:100%;max-width:950px;height:auto;"
            >

            <div style="
                max-width:850px;
                margin:15px auto;
                padding:16px;
                background:#f8fafc;
                border-radius:8px;
                color:#334155;
                font-size:14px;
                line-height:1.7;
                text-align:left;
            ">
                {explanation}
            </div>

        </div>
    </details>
    """))


# ============================================================
# DIAGRAM HELPERS
# ============================================================

def add_card(ax, x, y, w, h, title, subtitle="",
             accent=BLUE, background="#F8FAFC"):

    card = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        facecolor=background,
        edgecolor=accent,
        linewidth=1.7
    )

    ax.add_patch(card)

    ax.text(
        x + w / 2,
        y + h * (0.65 if subtitle else 0.5),
        title,
        ha="center",
        va="center",
        fontsize=10.5,
        fontweight="bold",
        color=DARK
    )

    if subtitle:
        ax.text(
            x + w / 2,
            y + h * 0.28,
            subtitle,
            ha="center",
            va="center",
            fontsize=9,
            color=GRAY
        )


def add_arrow(ax, start, end, color=GRAY):

    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle="-|>",
            mutation_scale=16,
            linewidth=1.8,
            color=color
        )
    )


# ============================================================
# SIMULATED INFERENCE SERVICE
# ============================================================

# This is a mathematical queue simulation.
#
# No actual HTTP requests are sent.
# No real AI service is attacked.
# No CPU or GPU stress test is performed.

DURATION = 120

time_steps = np.arange(DURATION)

SERVICE_CAPACITY = 40

# Normal request volume
normal_requests = rng.poisson(
    lam=22,
    size=DURATION
)

# Simulated attack traffic
attack_requests = normal_requests.copy()

# Elevated traffic between minutes 35 and 85
attack_requests[35:85] += rng.poisson(
    lam=65,
    size=50
)


# ============================================================
# QUEUE SIMULATION
# ============================================================

def simulate_queue(arrivals, capacity):

    queue = np.zeros(len(arrivals))
    processed = np.zeros(len(arrivals))
    backlog = 0

    for t in range(len(arrivals)):

        available = backlog + arrivals[t]

        processed[t] = min(
            available,
            capacity
        )

        backlog = max(
            0,
            available - processed[t]
        )

        queue[t] = backlog

    return queue, processed


normal_queue, normal_processed = simulate_queue(
    normal_requests,
    SERVICE_CAPACITY
)

attack_queue, attack_processed = simulate_queue(
    attack_requests,
    SERVICE_CAPACITY
)


# ============================================================
# VISUALIZATION 1 — MODEL DOS WORKFLOW
# ============================================================

fig, ax = plt.subplots(figsize=(10, 9))

ax.set_xlim(0, 10)
ax.set_ylim(0, 11)
ax.axis("off")

ax.set_title(
    "AI Model Denial-of-Service Workflow",
    fontsize=17,
    fontweight="bold"
)

steps = [
    (
        "Legitimate AI Service",
        "Processes normal inference requests",
        BLUE
    ),
    (
        "Excessive Request Submission",
        "Incoming workload exceeds expected levels",
        RED
    ),
    (
        "Resource Saturation",
        "Processing capacity becomes constrained",
        ORANGE
    ),
    (
        "Request Queue Growth",
        "Unprocessed requests accumulate",
        PURPLE
    ),
    (
        "Service Degradation",
        "Latency increases and requests may fail",
        RED
    ),
    (
        "Defensive Response",
        "Apply traffic controls and recovery measures",
        GREEN
    )
]

for i, (title, subtitle, accent) in enumerate(steps):

    y_pos = 9.4 - i * 1.7

    add_card(
        ax,
        1.0,
        y_pos,
        8.0,
        1.2,
        title,
        subtitle,
        accent
    )

    if i < len(steps) - 1:

        add_arrow(
            ax,
            (5, y_pos - 0.05),
            (5, y_pos - 0.45)
        )

plt.tight_layout()

show_collapsible_figure(
    "9.1 — Understanding Model Denial of Service",
    fig,
    """
    A model denial-of-service attack attempts
    to reduce the availability of an AI system.

    <br><br>

    An attacker may submit excessive inference
    requests or exploit unusually expensive
    model operations.

    <br><br>

    When incoming workloads exceed processing
    capacity, requests may begin accumulating
    in a queue.

    <br><br>

    As the queue grows, legitimate users may
    experience longer response times, timeouts,
    or unavailable services.

    <br><br>

    AI systems can be particularly sensitive
    to resource-intensive workloads involving
    large models, long inputs, or expensive
    inference operations.

    <br><br>

    <b>Key takeaway:</b> Model security includes
    protecting availability, not only
    prediction accuracy and confidentiality.
    """
)


# ============================================================
# VISUALIZATION 2 — REQUEST VOLUME
# ============================================================

fig, ax = plt.subplots(figsize=(11, 5.5))

ax.plot(
    time_steps,
    normal_requests,
    color=BLUE,
    linewidth=2,
    label="Normal Request Volume"
)

ax.plot(
    time_steps,
    attack_requests,
    color=RED,
    linewidth=2,
    label="Simulated Attack Volume"
)

ax.axhline(
    SERVICE_CAPACITY,
    color=ORANGE,
    linestyle="--",
    linewidth=2,
    label="Service Capacity"
)

ax.axvspan(
    35,
    85,
    color=RED,
    alpha=0.08,
    label="Elevated Traffic Window"
)

ax.set_xlabel("Time (Minutes)")
ax.set_ylabel("Requests per Minute")

ax.set_title(
    "Normal vs. Excessive Inference Requests",
    fontsize=16,
    fontweight="bold"
)

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "9.2 — Request Flooding and Service Capacity",
    fig,
    """
    This graph compares normal inference
    traffic with a simulated period of
    excessive request submission.

    <br><br>

    The blue line represents ordinary
    service usage.

    <br><br>

    The red line represents a simulated
    request-flooding scenario.

    <br><br>

    The orange dashed line represents
    the maximum number of requests
    the service can process each minute.

    <br><br>

    When requests exceed capacity,
    the service must queue, reject,
    or otherwise manage the excess work.

    <br><br>

    <b>Key takeaway:</b> A sustained
    increase in request volume can
    overwhelm an inference service
    even if individual requests
    are technically valid.
    """
)


# ============================================================
# VISUALIZATION 3 — REQUEST QUEUE GROWTH
# ============================================================

fig, ax = plt.subplots(figsize=(11, 5.5))

ax.plot(
    time_steps,
    normal_queue,
    color=BLUE,
    linewidth=2.5,
    label="Normal Queue"
)

ax.plot(
    time_steps,
    attack_queue,
    color=RED,
    linewidth=2.5,
    label="Attack Queue"
)

ax.fill_between(
    time_steps,
    attack_queue,
    color=RED,
    alpha=0.12
)

ax.axvspan(
    35,
    85,
    color=ORANGE,
    alpha=0.08
)

ax.set_xlabel("Time (Minutes)")
ax.set_ylabel("Queued Requests")

ax.set_title(
    "Request Queue Growth During Resource Exhaustion",
    fontsize=16,
    fontweight="bold"
)

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "9.3 — How Excessive Requests Create Backlogs",
    fig,
    """
    A request queue stores inference requests
    waiting to be processed.

    <br><br>

    Under normal conditions, the service
    may process incoming requests quickly
    enough to prevent a large backlog.

    <br><br>

    During excessive traffic, requests
    arrive faster than the service
    can process them.

    <br><br>

    The red line shows how the simulated
    queue grows during the attack window.

    <br><br>

    Even after excessive traffic stops,
    the system may require additional
    time to process accumulated requests.

    <br><br>

    <b>Key takeaway:</b> Service degradation
    can continue after the initial
    traffic surge has ended.
    """,
    color=RED
)


# ============================================================
# VISUALIZATION 4 — LATENCY AND SATURATION
# ============================================================

# Estimate queue-related response delay.
#
# This is a simplified model, not a measured
# benchmark of a real inference service.
#
# The approximation is based on the number
# of queued requests divided by processing
# capacity per minute.

BASE_LATENCY_MS = 100

normal_latency = (
    BASE_LATENCY_MS
    + (normal_queue / SERVICE_CAPACITY) * 60000
)

attack_latency = (
    BASE_LATENCY_MS
    + (attack_queue / SERVICE_CAPACITY) * 60000
)

# Utilization represents the fraction of
# processing capacity used in each minute.

normal_utilization = (
    normal_processed / SERVICE_CAPACITY
)

attack_utilization = (
    attack_processed / SERVICE_CAPACITY
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

fig.suptitle(
    "Inference Latency and Resource Saturation",
    fontsize=16,
    fontweight="bold"
)

# Latency
axes[0].plot(
    time_steps,
    normal_latency / 1000,
    color=BLUE,
    linewidth=2,
    label="Normal"
)

axes[0].plot(
    time_steps,
    attack_latency / 1000,
    color=RED,
    linewidth=2,
    label="Attack"
)

axes[0].set_xlabel("Time (Minutes)")
axes[0].set_ylabel("Estimated Queue Delay (Seconds)")
axes[0].set_title("Estimated Response Delay")

axes[0].legend()
axes[0].grid(alpha=0.2)

# Utilization
axes[1].plot(
    time_steps,
    normal_utilization * 100,
    color=BLUE,
    linewidth=2,
    label="Normal"
)

axes[1].plot(
    time_steps,
    attack_utilization * 100,
    color=ORANGE,
    linewidth=2,
    label="Attack"
)

axes[1].axhline(
    100,
    color=RED,
    linestyle="--",
    label="Full Capacity"
)

axes[1].set_xlabel("Time (Minutes)")
axes[1].set_ylabel("Capacity Utilization (%)")
axes[1].set_title("Service Capacity Utilization")

axes[1].legend()
axes[1].grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "9.4 — Inference Latency and Resource Saturation",
    fig,
    """
    Model denial-of-service attacks can
    affect both response time and
    processing capacity.

    <br><br>

    The left graph shows an estimated
    queue-related response delay.

    <br><br>

    As the backlog increases,
    requests must wait longer
    before processing.

    <br><br>

    The right graph shows how much
    of the service's available
    processing capacity is used.

    <br><br>

    When utilization reaches 100%,
    the service is processing as
    many requests as the simulation
    allows.

    <br><br>

    <b>Important:</b> These are simplified
    queue-based estimates, not actual
    CPU, GPU, or measured inference
    latency results.

    <br><br>

    <b>Key takeaway:</b> Availability
    problems arise when resource
    demand exceeds what a system
    can sustainably process.
    """,
    color=ORANGE
)


# ============================================================
# VISUALIZATION 5 — RATE LIMITING MITIGATION
# ============================================================

# Apply a simple admission limit.
#
# The simulation assumes that requests
# above the limit are rejected rather
# than queued.
#
# A real service should distinguish
# legitimate users from abusive clients.

RATE_LIMIT = 35

limited_requests = np.minimum(
    attack_requests,
    RATE_LIMIT
)

limited_queue, limited_processed = simulate_queue(
    limited_requests,
    SERVICE_CAPACITY
)

rejected_requests = np.maximum(
    attack_requests - RATE_LIMIT,
    0
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

fig.suptitle(
    "Rate Limiting as a Resource Exhaustion Defense",
    fontsize=16,
    fontweight="bold"
)

# Queue comparison
axes[0].plot(
    time_steps,
    attack_queue,
    color=RED,
    linewidth=2.5,
    label="Without Rate Limit"
)

axes[0].plot(
    time_steps,
    limited_queue,
    color=GREEN,
    linewidth=2.5,
    label="With Rate Limit"
)

axes[0].set_xlabel("Time (Minutes)")
axes[0].set_ylabel("Queued Requests")
axes[0].set_title("Queue Growth")

axes[0].legend()
axes[0].grid(alpha=0.2)

# Request handling
axes[1].plot(
    time_steps,
    attack_requests,
    color=RED,
    linewidth=2,
    label="Incoming Requests"
)

axes[1].plot(
    time_steps,
    limited_requests,
    color=GREEN,
    linewidth=2,
    label="Admitted Requests"
)

axes[1].axhline(
    RATE_LIMIT,
    color=ORANGE,
    linestyle="--",
    label="Admission Limit"
)

axes[1].set_xlabel("Time (Minutes)")
axes[1].set_ylabel("Requests per Minute")
axes[1].set_title("Traffic Admission Control")

axes[1].legend()
axes[1].grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "9.5 — Rate Limiting and Attack Mitigation",
    fig,
    f"""
    Rate limiting restricts how many
    requests a service accepts
    within a given time period.

    <br><br>

    In this simulation, the service
    accepts at most
    <b>{RATE_LIMIT} requests per minute</b>.

    <br><br>

    Requests above this limit
    are rejected rather than
    added to the processing queue.

    <br><br>

    The left graph compares queue
    growth with and without
    admission controls.

    <br><br>

    The right graph shows how
    rate limiting reduces
    the workload admitted
    to the service.

    <br><br>

    This is a simplified global
    rate limit.

    <br><br>

    Real deployments often use
    per-user, per-client, or
    per-API-key limits to avoid
    unnecessarily rejecting
    legitimate requests.

    <br><br>

    <b>Key takeaway:</b> Rate limiting
    can protect processing capacity,
    but poorly designed limits
    may also reduce availability
    for legitimate users.
    """,
    color=GREEN
)


# ============================================================
# VISUALIZATION 6 — MODEL DOS DEFENSES
# ============================================================

fig, ax = plt.subplots(figsize=(11, 9))

ax.set_xlim(0, 10)
ax.set_ylim(0, 11)
ax.axis("off")

ax.set_title(
    "Layered Defenses Against Model Denial of Service",
    fontsize=17,
    fontweight="bold"
)

defenses = [
    (
        "Authentication and Rate Limiting",
        "Restrict excessive or unauthorized requests",
        BLUE
    ),
    (
        "Input and Workload Limits",
        "Constrain expensive inference operations",
        ORANGE
    ),
    (
        "Queue Management",
        "Apply bounded queues and request timeouts",
        PURPLE
    ),
    (
        "Resource Isolation",
        "Prevent one workload from consuming all capacity",
        GREEN
    ),
    (
        "Autoscaling and Capacity Planning",
        "Prepare for legitimate increases in demand",
        BLUE
    ),
    (
        "Monitoring and Incident Response",
        "Detect degradation and restore availability",
        RED
    )
]

for i, (title, subtitle, accent) in enumerate(defenses):

    y_pos = 9.4 - i * 1.7

    add_card(
        ax,
        1.0,
        y_pos,
        8.0,
        1.2,
        title,
        subtitle,
        accent
    )

    if i < len(defenses) - 1:

        add_arrow(
            ax,
            (5, y_pos - 0.05),
            (5, y_pos - 0.45)
        )

plt.tight_layout()

show_collapsible_figure(
    "9.6 — Defending Against Model Resource Exhaustion",
    fig,
    """
    Model denial-of-service defenses
    focus on maintaining availability
    under unexpected or malicious
    workloads.

    <br><br>

    <b>Authentication and rate limiting</b>
    restrict excessive request submission.

    <br><br>

    <b>Input and workload limits</b>
    constrain operations that require
    unusually large amounts of
    processing time or memory.

    <br><br>

    <b>Queue management</b>
    prevents unbounded backlogs
    from exhausting system resources.

    <br><br>

    <b>Resource isolation</b>
    helps ensure that one user
    or workload cannot consume
    all available capacity.

    <br><br>

    <b>Autoscaling</b>
    may help handle legitimate
    demand increases, although
    it can introduce additional
    operational costs.

    <br><br>

    <b>Monitoring and incident response</b>
    support early detection and
    service recovery.

    <br><br>

    <b>Key takeaway:</b> Defending
    AI availability requires
    controlling both the number
    of requests and the resources
    consumed by individual requests.
    """,
    color=RED
)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print()
print("Section 9 Visualizations Complete")
print("---------------------------------")

print("Six collapsible visualizations generated.")

print()
print("Simulated Service Configuration")
print("-------------------------------")

print(
    "Service Capacity:",
    SERVICE_CAPACITY,
    "requests/minute"
)

print(
    "Rate Limit:",
    RATE_LIMIT,
    "requests/minute"
)

print(
    "Simulation Duration:",
    DURATION,
    "minutes"
)

print()
print("Simulation Results")
print("------------------")

print(
    "Peak Queue Without Mitigation:",
    int(np.max(attack_queue))
)

print(
    "Peak Queue With Rate Limiting:",
    int(np.max(limited_queue))
)

print(
    "Total Rejected Requests:",
    int(np.sum(rejected_requests))
)

print(
    "Peak Estimated Delay Without Mitigation:",
    f"{np.max(attack_latency) / 1000:.1f} seconds"
)

print()
print(
    "All results were generated using "
    "a mathematical queue simulation."
)

print(
    "No network traffic or resource "
    "exhaustion was performed."
)

<details>

<summary><b>Section 9 Lab — Click to Expand</b></summary>

<br>

In this activity, you will **simulate** an AI inference service experiencing increasing demand.

You will not generate network traffic or attack an external service.

Instead, the notebook will model:

    Incoming Requests

    Request Cost

    Processing Capacity

    Queue Growth

    Latency

    Rejected Requests

This allows you to safely study denial-of-service concepts without performing an actual denial-of-service attack.

The simulation begins with:

    NORMAL TRAFFIC
          ↓
    Requests Arrive
          ↓
    Service Processes Them
          ↓
    Queue Remains Small

You will then increase demand:

    HIGH DEMAND
          ↓
    Requests Arrive Faster
    Than They Are Processed
          ↓
    Queue Grows
          ↓
    Latency Increases

Finally, you will enable defensive controls and observe how they change system behavior.

</details>

In [ ]:
# @title ▶ Part 1 — Create the AI Service Simulator { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

def simulate_service(
    incoming_requests,
    capacity=10,
    max_queue=50,
    time_steps=30
):

    queue = 0

    queue_history = []
    processed_history = []
    rejected_history = []

    total_processed = 0
    total_rejected = 0

    for step in range(time_steps):

        arrivals = incoming_requests

        available_space = (
            max_queue - queue
        )

        accepted = min(
            arrivals,
            available_space
        )

        rejected = (
            arrivals - accepted
        )

        queue += accepted

        processed = min(
            queue,
            capacity
        )

        queue -= processed

        total_processed += processed
        total_rejected += rejected

        queue_history.append(queue)
        processed_history.append(processed)
        rejected_history.append(rejected)

    return {
        "queue": queue_history,
        "processed": processed_history,
        "rejected": rejected_history,
        "total_processed": total_processed,
        "total_rejected": total_rejected
    }

print("=" * 60)
print("AI SERVICE SIMULATOR READY")
print("=" * 60)

print("\nThe simulated service tracks:")
print("  • Incoming requests")
print("  • Processing capacity")
print("  • Queue depth")
print("  • Processed requests")
print("  • Rejected requests")

<details>

<summary><b>Part 1 Explanation — Click to Expand</b></summary>

<br>

Instead of attacking a real service, we create a mathematical simulation.

The service has:

    PROCESSING CAPACITY

which represents how many requests it can process during each time step.

It also has:

    MAXIMUM QUEUE

which represents how many requests can wait.

If too many requests arrive:

    Queue Fills
        ↓
    Additional Requests
        ↓
      Rejected

This lets us study availability safely and quickly.

</details>

In [ ]:
# @title ▶ Part 2 — Normal Traffic { display-mode: "form" }

normal_results = simulate_service(
    incoming_requests=6,
    capacity=10,
    max_queue=50,
    time_steps=30
)

print("=" * 60)
print("NORMAL TRAFFIC")
print("=" * 60)

print("\nIncoming Requests / Step: 6")
print("Processing Capacity / Step: 10")

print(
    f"Final Queue Depth: "
    f"{normal_results['queue'][-1]}"
)

print(
    f"Total Processed: "
    f"{normal_results['total_processed']}"
)

print(
    f"Total Rejected: "
    f"{normal_results['total_rejected']}"
)

plt.figure(figsize=(8, 4))

plt.plot(
    normal_results["queue"]
)

plt.xlabel("Time Step")
plt.ylabel("Queue Depth")
plt.title("Normal Traffic — Queue Depth")

plt.show()

<details>

<summary><b>Part 2 Explanation — Click to Expand</b></summary>

<br>

Normal traffic arrives at:

    6 Requests / Step

The service can process:

    10 Requests / Step

Therefore:

    ARRIVAL RATE
         <
    PROCESSING RATE

The service has enough capacity.

Requests are processed without a growing backlog.

This represents a healthy system operating below its maximum capacity.

</details>

In [ ]:
# @title ▶ Part 3 — High-Demand Scenario { display-mode: "form" }

high_load_results = simulate_service(
    incoming_requests=16,
    capacity=10,
    max_queue=50,
    time_steps=30
)

print("=" * 60)
print("HIGH-DEMAND TRAFFIC")
print("=" * 60)

print("\nIncoming Requests / Step: 16")
print("Processing Capacity / Step: 10")

print(
    f"Final Queue Depth: "
    f"{high_load_results['queue'][-1]}"
)

print(
    f"Total Processed: "
    f"{high_load_results['total_processed']}"
)

print(
    f"Total Rejected: "
    f"{high_load_results['total_rejected']}"
)

plt.figure(figsize=(8, 4))

plt.plot(
    high_load_results["queue"]
)

plt.xlabel("Time Step")
plt.ylabel("Queue Depth")
plt.title("High Demand — Queue Growth")

plt.show()

<details>

<summary><b>Part 3 Explanation — Click to Expand</b></summary>

<br>

Now the situation is reversed.

Requests arrive at:

    16 Requests / Step

but the service processes only:

    10 Requests / Step

Therefore:

    ARRIVAL RATE
         >
    PROCESSING RATE

The service cannot keep up.

The queue grows until it reaches its configured limit.

Once the queue is full, additional requests must be rejected.

Notice that nothing happened to the model's accuracy.

The problem is entirely about:

    AVAILABILITY

The AI model may still make excellent predictions, but users cannot reliably access those predictions.

</details>

In [ ]:
# @title ▶ Part 4 — Compare Service Conditions { display-mode: "form" }

plt.figure(figsize=(8, 4))

plt.plot(
    normal_results["queue"],
    label="Normal Traffic"
)

plt.plot(
    high_load_results["queue"],
    label="High Demand"
)

plt.xlabel("Time Step")
plt.ylabel("Queue Depth")
plt.title("Queue Growth Comparison")
plt.legend()

plt.show()

print("=" * 65)
print("SERVICE COMPARISON")
print("=" * 65)

print(
    f"\nNormal Rejected Requests: "
    f"{normal_results['total_rejected']}"
)

print(
    f"High-Load Rejected Requests: "
    f"{high_load_results['total_rejected']}"
)

<details>

<summary><b>Part 4 Explanation — Click to Expand</b></summary>

<br>

The difference between the two conditions should now be easy to see.

During normal traffic:

    Queue ≈ Small

During excessive demand:

    Queue → Maximum

Once the service reaches its queue limit:

    New Requests
         ↓
      Rejected

This is one way an availability failure can appear.

A production environment could also experience:

    Increased Latency

    Timeouts

    Memory Pressure

    Autoscaling

    Increased Cost

The exact effect depends on how the service is designed.

</details>

In [ ]:
# @title ▶ Part 5 — Estimate Service Latency { display-mode: "form" }

BASE_LATENCY_MS = 50
LATENCY_PER_QUEUED_REQUEST = 8

normal_latency = [
    BASE_LATENCY_MS
    + q * LATENCY_PER_QUEUED_REQUEST
    for q in normal_results["queue"]
]

high_latency = [
    BASE_LATENCY_MS
    + q * LATENCY_PER_QUEUED_REQUEST
    for q in high_load_results["queue"]
]

print("=" * 60)
print("LATENCY ESTIMATE")
print("=" * 60)

print(
    f"\nNormal Final Latency: "
    f"{normal_latency[-1]} ms"
)

print(
    f"High-Load Final Latency: "
    f"{high_latency[-1]} ms"
)

plt.figure(figsize=(8, 4))

plt.plot(
    normal_latency,
    label="Normal Traffic"
)

plt.plot(
    high_latency,
    label="High Demand"
)

plt.xlabel("Time Step")
plt.ylabel("Estimated Latency (ms)")
plt.title("Estimated Response Latency")
plt.legend()

plt.show()

<details>

<summary><b>Part 5 Explanation — Click to Expand</b></summary>

<br>

This cell uses a simplified latency estimate.

It assumes:

    Larger Queue
         ↓
    Longer Wait
         ↓
    Higher Latency

The exact numbers are not intended to represent a real production service.

The purpose is to demonstrate the relationship between:

    RESOURCE PRESSURE

and:

    USER EXPERIENCE

A system does not have to completely crash before availability is affected.

A prediction service that normally responds in:

    50 ms

but suddenly requires:

    1,000+ ms

may already be experiencing a serious availability problem.

</details>

In [ ]:
# @title ▶ Part 6 — Apply a Defensive Rate Limit { display-mode: "form" }

ATTACK_TRAFFIC = 16
RATE_LIMIT = 8

allowed_requests = min(
    ATTACK_TRAFFIC,
    RATE_LIMIT
)

rate_limited_requests = (
    ATTACK_TRAFFIC
    -
    allowed_requests
)

defended_results = simulate_service(
    incoming_requests=allowed_requests,
    capacity=10,
    max_queue=50,
    time_steps=30
)

print("=" * 65)
print("RATE-LIMITED SERVICE")
print("=" * 65)

print(
    f"\nIncoming Requests / Step: "
    f"{ATTACK_TRAFFIC}"
)

print(
    f"Rate Limit / Step: "
    f"{RATE_LIMIT}"
)

print(
    f"Allowed / Step: "
    f"{allowed_requests}"
)

print(
    f"Rate-Limited / Step: "
    f"{rate_limited_requests}"
)

print(
    f"\nFinal Queue Depth: "
    f"{defended_results['queue'][-1]}"
)

print(
    f"Service Queue Rejections: "
    f"{defended_results['total_rejected']}"
)

plt.figure(figsize=(8, 4))

plt.plot(
    high_load_results["queue"],
    label="Without Rate Limit"
)

plt.plot(
    defended_results["queue"],
    label="With Rate Limit"
)

plt.xlabel("Time Step")
plt.ylabel("Queue Depth")
plt.title("Effect of Rate Limiting")
plt.legend()

plt.show()

<details>

<summary><b>Part 6 Explanation — Click to Expand</b></summary>

<br>

Now a security control is placed **before** the inference service.

Originally:

    16 Requests
         ↓
      AI Service

With the rate limit:

    16 Requests
         ↓
      RATE LIMIT
       /      \
      8        8
      ↓        ↓
    ALLOW    LIMIT
      ↓
    AI Service

The AI service now receives fewer requests than it can process.

As a result:

    Queue Growth Decreases

    Latency Decreases

    Core Service Remains Available

Some requests are intentionally limited so that the entire service does not become overloaded.

This demonstrates **graceful degradation**.

</details>

In [ ]:
# @title ▶ Part 7 — Validate Expensive Requests { display-mode: "form" }

request_costs = [
    2, 3, 1, 4, 2,
    18, 3, 2, 25, 4,
    2, 30, 3, 1, 2
]

MAX_REQUEST_COST = 10

accepted = []
rejected = []

for cost in request_costs:

    if cost <= MAX_REQUEST_COST:
        accepted.append(cost)

    else:
        rejected.append(cost)

print("=" * 65)
print("REQUEST-COST VALIDATION")
print("=" * 65)

print(
    f"\nMaximum Allowed Cost: "
    f"{MAX_REQUEST_COST}"
)

print(
    f"Accepted Requests: "
    f"{len(accepted)}"
)

print(
    f"Rejected Expensive Requests: "
    f"{len(rejected)}"
)

print(
    f"\nTotal Work Before Validation: "
    f"{sum(request_costs)} units"
)

print(
    f"Total Work After Validation:  "
    f"{sum(accepted)} units"
)

print(
    "\nRejected Request Costs:",
    rejected
)

<details>

<summary><b>Part 7 Explanation — Click to Expand</b></summary>

<br>

This part demonstrates a different defensive idea.

Not every request has the same computational cost.

Instead of allowing every request to reach the expensive model:

    REQUEST
       ↓
    VALIDATE COST
       ↓
    Accept / Reject
       ↓
      MODEL

Requests above the allowed threshold are rejected early.

Compare:

    TOTAL WORK BEFORE VALIDATION

with:

    TOTAL WORK AFTER VALIDATION

The difference represents computation that the AI service did not need to perform.

The security principle is:

> **Reject unreasonable requests before they reach expensive processing whenever possible.**

</details>

<details>

<summary><b>Part 8 — Student Experiment — Click to Expand</b></summary>

<br>

Experiment with the service settings.

### Experiment 1 — Incoming Traffic

Return to **Part 3** and try:

    8 Requests / Step

    10 Requests / Step

    12 Requests / Step

    16 Requests / Step

    20 Requests / Step

Watch what happens when:

    Incoming Requests
          >
    Processing Capacity

### Experiment 2 — Processing Capacity

Try changing:

    capacity=10

to:

    capacity=15

How does additional capacity affect the queue?

### Experiment 3 — Rate Limit

Return to **Part 6** and try:

    RATE_LIMIT = 5

    RATE_LIMIT = 8

    RATE_LIMIT = 10

    RATE_LIMIT = 15

Consider the tradeoff.

A limit that is too high may not protect the service.

A limit that is too low may unnecessarily restrict legitimate users.

The goal is not simply:

    BLOCK MORE

The goal is:

    MAINTAIN AVAILABILITY
          +
    SUPPORT LEGITIMATE USE

</details>

In [ ]:
# @title ▶ Part 9 — Resource Exhaustion Lab Check { display-mode: "form" }

questions = [
    {
        "question":
        "1. What happened when request arrival exceeded processing capacity?",
        "options": [
            "A. The queue grew",
            "B. Model accuracy automatically increased",
            "C. The dataset disappeared",
            "D. Processing became free"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. Did the resource exhaustion scenario require changing model predictions?",
        "options": [
            "A. Yes",
            "B. No",
            "C. Only for image models",
            "D. Only after the queue filled"
        ],
        "answer": "B"
    },
    {
        "question":
        "3. Why did estimated latency increase?",
        "options": [
            "A. Requests spent more time waiting in the queue",
            "B. The model gained more classes",
            "C. The dataset became larger",
            "D. Accuracy decreased"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. What was the purpose of rate limiting?",
        "options": [
            "A. Prevent excessive incoming demand from overwhelming the service",
            "B. Increase every user's request volume",
            "C. Reveal model parameters",
            "D. Disable monitoring"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. Why was request-cost validation useful?",
        "options": [
            "A. Expensive requests could be rejected before consuming model resources",
            "B. It made every request more expensive",
            "C. It changed the training labels",
            "D. It increased queue size"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. What is the primary security goal demonstrated by this lab?",
        "options": [
            "A. Availability",
            "B. Confidentiality",
            "C. Dataset labeling",
            "D. Model extraction"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 65)
print("RESOURCE EXHAUSTION MINI LAB CHECK")
print("=" * 65)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = score / len(questions) * 100

print("\n" + "=" * 65)

print(
    f"Lab Score: "
    f"{score}/{len(questions)}"
)

print(
    f"Percentage: "
    f"{percentage:.1f}%"
)

if percentage >= 80:

    print(
        "\n✓ Resource exhaustion "
        "mini lab complete."
    )

else:

    print(
        "\nReview the experiment "
        "and try the lab check again."
    )

print("=" * 65)

<details>

<summary><b>Section 9 — Practical Takeaways — Click to Expand</b></summary>

<br>

This activity introduced a different objective from many of the attacks studied earlier.

The attacker did not need to:

    Steal the Model

    Infer Training Data

    Change Predictions

    Insert a Backdoor

Instead, the security objective was:

    AVAILABILITY

You observed what happened when:

    REQUEST ARRIVAL RATE
             >
    PROCESSING CAPACITY

The result was:

    QUEUE GROWTH
          ↓
    HIGHER LATENCY
          ↓
    QUEUE CAPACITY REACHED
          ↓
    REQUEST REJECTION

This demonstrates an important distinction:

    MODEL ACCURACY
          ≠
    SYSTEM AVAILABILITY

A model can remain highly accurate while the overall AI system becomes unusable.

You also implemented two simple defensive concepts.

### Rate Limiting

    Excessive Requests
           ↓
       Rate Limit
           ↓
    Controlled Demand
           ↓
    Service Remains Available

### Request Validation

    Incoming Request
           ↓
    Cheap Validation
           ↓
    Reasonable?
      /       \
    YES       NO
     ↓         ↓
   Model     Reject

These defenses demonstrate a larger principle.

AI systems should not assume that every user will interact with the model in the expected way.

Defenders should monitor:

    Request Volume

    Queue Depth

    Latency

    Resource Utilization

    Rejected Requests

and should define what happens when the system reaches capacity.

The central lesson is:

> **Securing an AI system means protecting its ability to provide service, not just protecting the correctness and confidentiality of the model itself.**

</details>

In [ ]:
# @title ▶ Section 9 Checkpoint — Model DoS & Resource Exhaustion { display-mode: "form" }

questions = [
    {
        "question":
        "1. Which security property is the primary target of model denial of service?",
        "options": [
            "A. Availability",
            "B. Confidentiality",
            "C. Model accuracy",
            "D. Training labels"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. What happens when requests arrive faster than a service can process them?",
        "options": [
            "A. A queue may begin to grow",
            "B. The model automatically becomes more accurate",
            "C. Training data is deleted",
            "D. Every request becomes cheaper"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. What does latency measure?",
        "options": [
            "A. How long a request takes to receive a response",
            "B. The number of model classes",
            "C. Training accuracy",
            "D. Dataset size"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. Why should validation occur before expensive model processing when practical?",
        "options": [
            "A. Invalid requests can be rejected before consuming expensive resources",
            "B. Validation increases the number of model parameters",
            "C. It guarantees perfect predictions",
            "D. It removes the need for monitoring"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. How can cloud scaling create another resource-exhaustion concern?",
        "options": [
            "A. Excessive requests may significantly increase infrastructure cost",
            "B. Scaling always deletes the model",
            "C. Cloud systems cannot process AI requests",
            "D. Scaling reveals all training data"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. Which is an appropriate availability defense?",
        "options": [
            "A. Rate limiting and resource monitoring",
            "B. Removing every request limit",
            "C. Disabling logging",
            "D. Returning more model information"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 65)
print("SECTION 9 CHECKPOINT")
print("MODEL DENIAL OF SERVICE & RESOURCE EXHAUSTION")
print("=" * 65)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = score / len(questions) * 100

print("\n" + "=" * 65)
print(f"Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print(
        "\n✓ Checkpoint complete. "
        "Continue to the resource exhaustion mini lab."
    )
else:
    print(
        "\nReview Section 9 before continuing."
    )

print("=" * 65)

---

<details>

<summary><b>Section 10 — Attack Chains & Multi-Stage AI Attacks — Click to Expand</b></summary>

<br>

Throughout this module, attacks have mostly been examined individually.

You have explored techniques that can:

    Manipulate Model Decisions

    Extract Model Behavior

    Reveal Training Information

    Activate Hidden Backdoors

    Abuse Prediction APIs

    Exhaust Computational Resources

Studying attacks individually makes them easier to understand.

Real attacks, however, do not necessarily occur as isolated events.

An attacker may combine several techniques so that the result of one attack creates the opportunity for another.

This creates an **attack chain**.

Conceptually:

    INITIAL ACCESS
          ↓
    INFORMATION GATHERING
          ↓
    MODEL INTERACTION
          ↓
    ATTACK PREPARATION
          ↓
    EXPLOITATION
          ↓
    SECURITY IMPACT

Understanding these relationships helps defenders move beyond asking:

> "What individual attacks are possible?"

and begin asking:

> "How could an attacker move through our AI system?"

<br>

<details>

<summary><b>10.1 — What Is an Attack Chain?</b></summary>

<br>

An attack chain is a sequence of actions where multiple stages contribute to an attacker's overall objective.

Instead of:

    ATTACK
      ↓
    IMPACT

the process may look more like:

    STAGE 1
       ↓
    STAGE 2
       ↓
    STAGE 3
       ↓
    STAGE 4
       ↓
    IMPACT

The attacker may not initially have enough information or access to perform the final attack.

Earlier stages help create those conditions.

For example, an attacker might first interact with a prediction API.

Those interactions could help the attacker understand model behavior.

That information could then support the creation of a surrogate model.

The surrogate could then be used to develop adversarial examples.

The complete chain becomes:

    API ACCESS
        ↓
    MODEL QUERIES
        ↓
    COLLECT RESPONSES
        ↓
    BUILD SURROGATE
        ↓
    CREATE ADVERSARIAL INPUT
        ↓
    TRANSFER TO TARGET
        ↓
    MISCLASSIFICATION

No individual stage tells the entire story.

</details>

<br>

<details>

<summary><b>10.2 — Reconnaissance and Information Gathering</b></summary>

<br>

Attack chains often begin with information gathering.

An attacker may want to understand:

    What model is being used?

    What inputs are accepted?

    What outputs are returned?

    Is confidence information exposed?

    Are there query restrictions?

    How does the model respond to unusual inputs?

Suppose an API returns:

    Predicted Class
          +
    Confidence Scores

The attacker can repeatedly submit inputs and observe changes.

This creates:

    QUERY
      ↓
    RESPONSE
      ↓
    INFORMATION

Repeated observations can help the attacker better understand the target before attempting a more advanced attack.

This is why information exposure matters even when a single API response appears harmless.

</details>

<br>

<details>

<summary><b>10.3 — API Access → Model Extraction</b></summary>

<br>

Consider an attacker who cannot download a proprietary model.

They can, however, query it.

The attacker generates inputs:

    X1
    X2
    X3
    ...
    XN

The target provides outputs:

    Y1
    Y2
    Y3
    ...
    YN

The attacker stores:

    X1 → Y1
    X2 → Y2
    X3 → Y3

This creates a dataset representing observations of the target model.

That dataset can then train a:

    SURROGATE MODEL

The attack chain becomes:

    TARGET API
         ↓
    MANY QUERIES
         ↓
    INPUT / OUTPUT PAIRS
         ↓
    SURROGATE TRAINING
         ↓
    APPROXIMATION OF TARGET

The attacker has turned ordinary prediction access into information that supports model extraction.

</details>

<br>

<details>

<summary><b>10.4 — Model Extraction → Transfer Attack</b></summary>

<br>

The surrogate model from the previous stage may enable another attack.

Suppose the target is:

    BLACK BOX

The attacker cannot calculate its gradients.

However, the attacker controls the surrogate.

Therefore:

    SURROGATE MODEL
          ↓
    GRADIENT ACCESS
          ↓
    CREATE ADVERSARIAL EXAMPLE
          ↓
    SEND TO TARGET

If the adversarial example also fools the original target, the attack has transferred.

The complete chain becomes:

    QUERY TARGET
         ↓
    BUILD SURROGATE
         ↓
    ATTACK SURROGATE
         ↓
    GENERATE ADVERSARIAL INPUT
         ↓
    SUBMIT TO TARGET
         ↓
    TARGET MISCLASSIFICATION

This demonstrates why individual vulnerabilities can become more serious when combined.

</details>

<br>

<details>

<summary><b>10.5 — Supply Chain → Backdoor → Exploitation</b></summary>

<br>

Another attack chain may begin before the organization ever deploys the model.

Imagine an attacker successfully introduces a backdoor into a model that is later distributed.

The chain could be:

    MODEL COMPROMISED
          ↓
    BACKDOOR INSERTED
          ↓
    MODEL DISTRIBUTED
          ↓
    ORGANIZATION DOWNLOADS MODEL
          ↓
    MODEL PASSES NORMAL TESTING
          ↓
    MODEL DEPLOYED
          ↓
    ATTACKER SENDS TRIGGER
          ↓
    BACKDOOR ACTIVATES

Notice the separation between:

    COMPROMISE

and:

    EXPLOITATION

The malicious behavior may be introduced long before it is actually used.

This is why provenance and supply-chain security are important parts of AI security.

</details>

<br>

<details>

<summary><b>10.6 — Poisoning → Deployment → Evasion</b></summary>

<br>

Training-data compromise can also create a longer attack path.

Consider a security classifier.

An attacker influences its training data:

    MANIPULATE TRAINING DATA
             ↓
        MODEL TRAINING
             ↓
    WEAKENED DECISION BOUNDARY
             ↓
        MODEL DEPLOYED
             ↓
    ATTACKER SUBMITS MALICIOUS INPUT
             ↓
        MODEL MISSES ATTACK

The poisoning occurs during:

    TRAINING

but the security impact appears during:

    INFERENCE

This is another example of why defenders must consider the complete machine learning lifecycle.

</details>

<br>

<details>

<summary><b>10.7 — API Probing → Privacy Attack</b></summary>

<br>

Attack chains can also target confidentiality.

Suppose an attacker repeatedly interacts with a model and observes:

    Predictions

    Confidence

    Response Patterns

Those observations might support a privacy attack.

Conceptually:

    MODEL ACCESS
         ↓
    CAREFULLY SELECTED QUERIES
         ↓
    OBSERVE MODEL RESPONSES
         ↓
    ANALYZE BEHAVIOR
         ↓
    MEMBERSHIP INFERENCE
         ↓
    POSSIBLE TRAINING-DATA INFORMATION

A similar iterative process can be used in some model-inversion scenarios.

The important point is that the prediction API can act as the starting point for a larger privacy attack.

</details>

<br>

<details>

<summary><b>10.8 — Attack Chains Can Cross Lifecycle Stages</b></summary>

<br>

One of the most important characteristics of AI attack chains is that they can cross different stages of the machine learning lifecycle.

Consider:

    DATA COLLECTION
          ↓
    TRAINING
          ↓
    MODEL DISTRIBUTION
          ↓
    DEPLOYMENT
          ↓
    INFERENCE
          ↓
    MONITORING

An attacker may enter at one stage and create an effect at another.

For example:

    Poison Training Data
           ↓
    Train Compromised Model
           ↓
    Deploy Model
           ↓
    Exploit Behavior During Inference

Or:

    Query Deployed Model
           ↓
    Build Surrogate
           ↓
    Develop Attack Offline
           ↓
    Return to Deployed Model
           ↓
    Perform Evasion

Therefore, defenders should avoid treating lifecycle stages as completely separate security problems.

</details>

<br>

<details>

<summary><b>10.9 — Breaking the Attack Chain</b></summary>

<br>

The defender does not necessarily need to stop an attacker only at the final stage.

Consider:

    API ACCESS
        ↓
    MASS QUERYING
        ↓
    MODEL EXTRACTION
        ↓
    SURROGATE MODEL
        ↓
    ADVERSARIAL INPUT
        ↓
    EVASION

There are several opportunities to interfere with this chain.

For example:

    Authentication
          ↓
    Control Initial Access

    Rate Limiting
          ↓
    Restrict Mass Querying

    Query Monitoring
          ↓
    Detect Suspicious Behavior

    Limit Unnecessary Outputs
          ↓
    Reduce Information Exposure

    Adversarial Defenses
          ↓
    Increase Robustness

The defender does not need to wait until:

    FINAL ATTACK

before responding.

This leads to an important defensive principle:

> **Every stage of an attack chain can provide an opportunity for prevention, detection, or response.**

</details>

<br>

<details>

<summary><b>10.10 — Defense in Depth</b></summary>

<br>

Attack chains demonstrate why AI security should use **defense in depth**.

No single security control should be expected to stop every possible attack.

Instead:

    ACCESS CONTROL
          +
    INPUT VALIDATION
          +
    QUERY MONITORING
          +
    MODEL TESTING
          +
    DATA VALIDATION
          +
    RESOURCE CONTROLS
          +
    INCIDENT RESPONSE

create multiple opportunities to stop or detect malicious activity.

Suppose one control fails.

The next control may still prevent the attacker from reaching the final objective.

Conceptually:

    ATTACKER
        ↓
    CONTROL 1
        ↓
    CONTROL 2
        ↓
    CONTROL 3
        ↓
      MODEL

The objective is not to assume that:

    CONTROL 1
       =
    PERFECT SECURITY

Instead:

    MULTIPLE CONTROLS
           ↓
    MULTIPLE OPPORTUNITIES
    TO BREAK THE ATTACK CHAIN

</details>

<br>

<details>

<summary><b>10.11 — Thinking Like a Defender</b></summary>

<br>

When analyzing an AI system, defenders should ask more than:

> "What attacks affect this model?"

They should also ask:

> "What would an attacker need before performing that attack?"

> "How could they obtain that capability?"

> "What stage would come next?"

> "Where could we detect them?"

> "Where could we stop them?"

A useful process is:

    IDENTIFY ATTACKER GOAL
             ↓
    IDENTIFY REQUIRED ACCESS
             ↓
    IDENTIFY POSSIBLE TECHNIQUES
             ↓
    CONNECT THE TECHNIQUES
             ↓
    IDENTIFY SECURITY CONTROLS

This turns isolated attack knowledge into security analysis.

</details>

<br>

<details>

<summary><b>10.12 — Connecting the Module</b></summary>

<br>

The attacks from this module can now be viewed as pieces of a larger system.

For example:

    MODEL EXTRACTION
           ↓
    Creates Surrogate
           ↓
    TRANSFER ATTACK
           ↓
    Enables Evasion

Another path might be:

    TRAINING ACCESS
           ↓
    BACKDOOR INSERTION
           ↓
    MODEL DEPLOYMENT
           ↓
    TRIGGER EXPLOITATION

Another might be:

    API ACCESS
           ↓
    REPEATED QUERIES
           ↓
    PRIVACY ANALYSIS
           ↓
    MEMBERSHIP INFERENCE

And another:

    PUBLIC API
           ↓
    EXCESSIVE REQUESTS
           ↓
    RESOURCE EXHAUSTION
           ↓
    AVAILABILITY IMPACT

The important skill is learning to recognize how these techniques relate rather than memorizing them as completely independent attacks.

</details>

<br>

<details>

<summary><b>Section 10 — Key Takeaways</b></summary>

<br>

Real AI attacks may involve multiple stages.

One technique may provide:

    INFORMATION

    ACCESS

    CAPABILITY

or:

    OPPORTUNITY

needed for the next technique.

Attack chains can cross:

    Training

    Distribution

    Deployment

    Inference

and:

    Monitoring

Defenders should therefore protect the entire AI lifecycle.

They should also look for opportunities to:

    PREVENT

    DETECT

    INTERRUPT

and:

    RESPOND

at multiple points in the chain.

The central lesson is:

> **AI attacks should not always be viewed as isolated techniques. Understanding how techniques connect helps defenders identify where an attack can be detected or stopped before the final impact occurs.**

</details>

</details>

In [ ]:
# @title Section 10 — Attack Chains and Multi-Stage AI Attacks Visualizations { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from IPython.display import display, HTML
from io import BytesIO

import base64
import html

# ============================================================
# CONFIGURATION
# ============================================================

rng = np.random.default_rng(42)

BLUE = "#2563EB"
GREEN = "#16A34A"
RED = "#DC2626"
ORANGE = "#EA580C"
PURPLE = "#7C3AED"
GRAY = "#64748B"
DARK = "#0F172A"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# ============================================================
# COLLAPSIBLE VISUALIZATION HELPER
# ============================================================

def show_collapsible_figure(title, fig, explanation, color=BLUE):

    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=135,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    buffer.close()

    display(HTML(f"""
    <details style="
        margin:12px 0;
        border:1px solid #cbd5e1;
        border-radius:10px;
        overflow:hidden;
    ">

        <summary style="
            cursor:pointer;
            padding:16px;
            font-size:16px;
            font-weight:bold;
            background:#eff6ff;
            color:#1e3a8a;
            border-left:5px solid {color};
        ">
            {html.escape(title)} — Click to Expand
        </summary>

        <div style="
            padding:15px;
            background:white;
            text-align:center;
        ">

            <img
                src="data:image/png;base64,{encoded}"
                style="width:100%;max-width:950px;height:auto;"
            >

            <div style="
                max-width:850px;
                margin:15px auto;
                padding:16px;
                background:#f8fafc;
                border-radius:8px;
                color:#334155;
                font-size:14px;
                line-height:1.7;
                text-align:left;
            ">
                {explanation}
            </div>

        </div>
    </details>
    """))


# ============================================================
# DIAGRAM HELPERS
# ============================================================

def add_card(ax, x, y, w, h, title, subtitle="",
             accent=BLUE, background="#F8FAFC"):

    card = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        facecolor=background,
        edgecolor=accent,
        linewidth=1.7
    )

    ax.add_patch(card)

    ax.text(
        x + w / 2,
        y + h * (0.65 if subtitle else 0.5),
        title,
        ha="center",
        va="center",
        fontsize=10.5,
        fontweight="bold",
        color=DARK
    )

    if subtitle:

        ax.text(
            x + w / 2,
            y + h * 0.28,
            subtitle,
            ha="center",
            va="center",
            fontsize=9,
            color=GRAY
        )


def add_arrow(ax, start, end, color=GRAY):

    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle="-|>",
            mutation_scale=16,
            linewidth=1.8,
            color=color
        )
    )


# ============================================================
# SIMULATED ATTACK CHAIN
# ============================================================

# These probabilities are illustrative assumptions.
# They are NOT empirical attack success rates.

stages = [
    "Reconnaissance",
    "Model Extraction",
    "Adversarial Evasion",
    "Target Exploitation"
]

stage_probabilities = np.array([
    0.90,
    0.75,
    0.65,
    0.55
])

# Each probability is conditional on all
# preceding stages having succeeded.

cumulative_probabilities = np.cumprod(
    stage_probabilities
)


# ============================================================
# VISUALIZATION 1 — MULTI-STAGE ATTACK WORKFLOW
# ============================================================

fig, ax = plt.subplots(figsize=(10, 9))

ax.set_xlim(0, 10)
ax.set_ylim(0, 11)
ax.axis("off")

ax.set_title(
    "Example Multi-Stage AI Attack Chain",
    fontsize=17,
    fontweight="bold"
)

workflow = [
    (
        "Reconnaissance",
        "Observe exposed model interfaces",
        BLUE
    ),
    (
        "Model Extraction",
        "Approximate target model behavior",
        ORANGE
    ),
    (
        "Adversarial Example Development",
        "Create inputs against a surrogate model",
        PURPLE
    ),
    (
        "Transfer Attack",
        "Test whether adversarial behavior transfers",
        RED
    ),
    (
        "Target Exploitation",
        "Attempt to influence the target prediction",
        RED
    ),
    (
        "Defender Investigation",
        "Correlate activity across the attack chain",
        GREEN
    )
]

for i, (title, subtitle, accent) in enumerate(workflow):

    y_pos = 9.4 - i * 1.7

    add_card(
        ax,
        1.0,
        y_pos,
        8.0,
        1.2,
        title,
        subtitle,
        accent
    )

    if i < len(workflow) - 1:

        add_arrow(
            ax,
            (5, y_pos - 0.05),
            (5, y_pos - 0.45)
        )

plt.tight_layout()

show_collapsible_figure(
    "10.1 — Understanding Multi-Stage AI Attacks",
    fig,
    """
    A multi-stage attack combines
    several techniques to achieve
    a larger objective.

    <br><br>

    In this example, an attacker
    first investigates a deployed
    model's API.

    <br><br>

    The attacker then attempts
    to approximate the model's
    behavior using a surrogate.

    <br><br>

    Adversarial examples are
    developed against the surrogate
    and tested against the target.

    <br><br>

    A successful transfer attack
    may cause the target model
    to produce an incorrect prediction.

    <br><br>

    This is one possible attack
    chain. Real attacks may use
    different stages or objectives.

    <br><br>

    <b>Key takeaway:</b> Individual
    AI vulnerabilities can become
    more serious when combined
    into a larger attack sequence.
    """
)


# ============================================================
# VISUALIZATION 2 — STAGE DEPENDENCIES
# ============================================================

fig, ax = plt.subplots(figsize=(11, 5.5))

positions = np.arange(
    len(stages)
)

bars = ax.bar(
    positions,
    stage_probabilities,
    color=[BLUE, ORANGE, PURPLE, RED],
    width=0.6
)

ax.set_xticks(positions)

ax.set_xticklabels(
    [
        "Reconnaissance",
        "Model\nExtraction",
        "Adversarial\nEvasion",
        "Target\nExploitation"
    ]
)

ax.set_ylim(0, 1.1)

ax.set_ylabel("Conditional Stage Success Probability")

ax.set_title(
    "Illustrative Success Probability by Attack Stage",
    fontsize=16,
    fontweight="bold"
)

ax.grid(axis="y", alpha=0.2)

for bar, probability in zip(
    bars,
    stage_probabilities
):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        probability + 0.025,
        f"{probability:.0%}",
        ha="center",
        fontweight="bold",
        fontsize=11
    )

plt.tight_layout()

show_collapsible_figure(
    "10.2 — Attack Stage Dependencies",
    fig,
    """
    Multi-stage attacks often
    contain dependencies.

    <br><br>

    For example, an attacker may
    need to obtain enough model
    information before developing
    a useful surrogate.

    <br><br>

    The graph shows assumed
    conditional success probabilities
    for four example stages.

    <br><br>

    Each probability represents
    the chance that the current
    stage succeeds, assuming
    all required earlier
    stages have succeeded.

    <br><br>

    These values are hypothetical
    and are not measured
    from real attacks.

    <br><br>

    <b>Key takeaway:</b> A later
    attack stage may be unreachable
    if a required earlier
    stage fails.
    """
)


# ============================================================
# VISUALIZATION 3 — CUMULATIVE SUCCESS
# ============================================================

fig, ax = plt.subplots(figsize=(11, 5.5))

ax.plot(
    np.arange(1, len(stages) + 1),
    cumulative_probabilities,
    marker="o",
    color=RED,
    linewidth=2.5,
    markersize=9
)

ax.fill_between(
    np.arange(1, len(stages) + 1),
    cumulative_probabilities,
    color=RED,
    alpha=0.1
)

ax.set_xticks(
    np.arange(1, len(stages) + 1)
)

ax.set_xticklabels([
    "Stage 1\nRecon",
    "Stage 2\nExtraction",
    "Stage 3\nEvasion",
    "Stage 4\nExploitation"
])

ax.set_ylim(0, 1.05)

ax.set_ylabel("Probability of Completing All Stages So Far")

ax.set_title(
    "Cumulative Multi-Stage Attack Success",
    fontsize=16,
    fontweight="bold"
)

ax.grid(alpha=0.2)

for i, probability in enumerate(
    cumulative_probabilities,
    start=1
):

    ax.annotate(
        f"{probability:.1%}",
        (i, probability),
        xytext=(0, 10),
        textcoords="offset points",
        ha="center",
        fontsize=10,
        fontweight="bold"
    )

plt.tight_layout()

show_collapsible_figure(
    "10.3 — Cumulative Attack Chain Success",
    fig,
    f"""
    Completing a multi-stage
    attack requires successfully
    progressing through its
    required stages.

    <br><br>

    In this example, each stage
    has a hypothetical conditional
    success probability.

    <br><br>

    The probability of completing
    the entire chain is calculated
    by multiplying those
    conditional probabilities.

    <br><br>

    <b>Illustrative full-chain
    success probability:</b>
    {cumulative_probabilities[-1]:.1%}

    <br><br>

    This calculation does not
    assume that stages are
    independent because each
    probability is defined
    conditionally on earlier
    stages succeeding.

    <br><br>

    <b>Key takeaway:</b> Defenders
    may be able to interrupt
    an attack by preventing
    any required stage
    from succeeding.
    """,
    color=RED
)


# ============================================================
# VISUALIZATION 4 — ALTERNATIVE ATTACK PATHS
# ============================================================

fig, ax = plt.subplots(figsize=(12, 8))

ax.set_xlim(0, 12)
ax.set_ylim(0, 10)
ax.axis("off")

ax.set_title(
    "Alternative Paths Through an AI Attack Chain",
    fontsize=17,
    fontweight="bold"
)

add_card(
    ax,
    4.3, 8.2, 3.4, 1.0,
    "Initial Model Access",
    accent=BLUE
)

add_card(
    ax,
    0.5, 5.8, 3.2, 1.1,
    "Model Extraction",
    "Approximate behavior",
    accent=ORANGE
)

add_card(
    ax,
    4.4, 5.8, 3.2, 1.1,
    "Direct API Probing",
    "Analyze responses",
    accent=PURPLE
)

add_card(
    ax,
    8.3, 5.8, 3.2, 1.1,
    "Trigger Discovery",
    "Investigate hidden behavior",
    accent=RED
)

add_card(
    ax,
    0.5, 3.0, 3.2, 1.1,
    "Transfer Evasion",
    accent=ORANGE
)

add_card(
    ax,
    4.4, 3.0, 3.2, 1.1,
    "Privacy Inference",
    accent=PURPLE
)

add_card(
    ax,
    8.3, 3.0, 3.2, 1.1,
    "Backdoor Activation",
    accent=RED
)

add_card(
    ax,
    4.2, 0.5, 3.6, 1.0,
    "Security Impact",
    accent=RED
)

# Top branching
for x in [2.1, 6.0, 9.9]:

    add_arrow(
        ax,
        (6.0, 8.2),
        (x, 6.95)
    )

# Middle paths
for x in [2.1, 6.0, 9.9]:

    add_arrow(
        ax,
        (x, 5.8),
        (x, 4.15)
    )

# Final convergence
for x in [2.1, 6.0, 9.9]:

    add_arrow(
        ax,
        (x, 3.0),
        (6.0, 1.55)
    )

plt.tight_layout()

show_collapsible_figure(
    "10.4 — Alternative AI Attack Paths",
    fig,
    """
    Not every attacker follows
    the same sequence of steps.

    <br><br>

    Attackers may choose different
    techniques depending on
    available access, model
    behavior, and objectives.

    <br><br>

    One path may involve
    model extraction followed
    by adversarial transfer.

    <br><br>

    Another path may use
    API responses to investigate
    possible privacy leakage.

    <br><br>

    A third path may involve
    identifying and activating
    a previously introduced
    model backdoor.

    <br><br>

    These paths represent
    different attack objectives
    and do not necessarily
    produce the same impact.

    <br><br>

    <b>Key takeaway:</b> AI security
    assessments should consider
    multiple attack paths rather
    than focusing on only
    one vulnerability.
    """
)


# ============================================================
# VISUALIZATION 5 — DEFENSE IN DEPTH
# ============================================================

# Hypothetical defense effectiveness values.
#
# Each control reduces the probability
# of completing its associated stage.
#
# This is a teaching simulation,
# not a measured security benchmark.

defense_reductions = np.array([
    0.20,
    0.35,
    0.40,
    0.45
])

defended_stage_probabilities = (
    stage_probabilities
    * (1 - defense_reductions)
)

defended_cumulative = np.cumprod(
    defended_stage_probabilities
)

fig, ax = plt.subplots(figsize=(11, 5.5))

stage_numbers = np.arange(
    1,
    len(stages) + 1
)

ax.plot(
    stage_numbers,
    cumulative_probabilities,
    marker="o",
    linewidth=2.5,
    color=RED,
    label="Without Added Controls"
)

ax.plot(
    stage_numbers,
    defended_cumulative,
    marker="s",
    linewidth=2.5,
    color=GREEN,
    label="With Layered Controls"
)

ax.set_xticks(stage_numbers)

ax.set_xticklabels([
    "Recon",
    "Extraction",
    "Evasion",
    "Exploitation"
])

ax.set_ylim(0, 1.05)

ax.set_xlabel("Attack Stage")
ax.set_ylabel("Cumulative Chain Success Probability")

ax.set_title(
    "Illustrative Effect of Defense in Depth",
    fontsize=16,
    fontweight="bold"
)

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "10.5 — Defense-in-Depth Effectiveness",
    fig,
    f"""
    Defense in depth uses
    multiple security controls
    to protect a system.

    <br><br>

    In this simulation, controls
    reduce the assumed probability
    of successfully completing
    individual attack stages.

    <br><br>

    Example controls include
    API monitoring, query limits,
    robustness testing, and
    access restrictions.

    <br><br>

    <b>Without added controls:</b>
    {cumulative_probabilities[-1]:.1%}
    full-chain success probability.

    <br><br>

    <b>With layered controls:</b>
    {defended_cumulative[-1]:.1%}
    full-chain success probability.

    <br><br>

    These probabilities are
    illustrative assumptions,
    not measured effectiveness
    estimates for real defenses.

    <br><br>

    <b>Key takeaway:</b> Security
    controls placed at different
    stages can collectively
    reduce the likelihood
    of a completed attack chain.
    """,
    color=GREEN
)


# ============================================================
# VISUALIZATION 6 — DETECTION AND RESPONSE
# ============================================================

# Simulated sequence of security events.
# These are illustrative event counts.

time = np.arange(60)

api_probing = rng.poisson(
    lam=2,
    size=60
)

model_queries = rng.poisson(
    lam=3,
    size=60
)

adversarial_inputs = rng.poisson(
    lam=1,
    size=60
)

# Add simulated multi-stage activity.

api_probing[10:22] += 12

model_queries[20:37] += 15

adversarial_inputs[35:50] += 10

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    time,
    api_probing,
    color=BLUE,
    linewidth=2,
    label="API Reconnaissance Indicators"
)

ax.plot(
    time,
    model_queries,
    color=ORANGE,
    linewidth=2,
    label="Model Query Activity"
)

ax.plot(
    time,
    adversarial_inputs,
    color=RED,
    linewidth=2,
    label="Potential Evasion Indicators"
)

ax.axvspan(
    10, 22,
    color=BLUE,
    alpha=0.06
)

ax.axvspan(
    20, 37,
    color=ORANGE,
    alpha=0.06
)

ax.axvspan(
    35, 50,
    color=RED,
    alpha=0.06
)

ax.set_xlabel("Time (Minutes)")
ax.set_ylabel("Simulated Events per Minute")

ax.set_title(
    "Correlating Indicators Across an AI Attack Chain",
    fontsize=16,
    fontweight="bold"
)

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "10.6 — Detecting and Responding to Attack Chains",
    fig,
    """
    Multi-stage attacks may
    generate different indicators
    as they progress.

    <br><br>

    Early activity may involve
    unusual API probing
    or repeated requests.

    <br><br>

    Later activity may involve
    systematic model queries
    or suspicious input patterns.

    <br><br>

    This graph illustrates
    how separate indicators
    can appear at different
    times during a simulated
    attack sequence.

    <br><br>

    These events are synthetic.
    They do not demonstrate a
    validated attack detector.

    <br><br>

    Defenders can investigate
    relationships among events
    using timestamps, request
    patterns, identities,
    and affected services.

    <br><br>

    <b>Key takeaway:</b> Correlating
    events across multiple stages
    may reveal suspicious activity
    that would be difficult
    to recognize from
    individual events alone.
    """,
    color=PURPLE
)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print()
print("Section 10 Visualizations Complete")
print("----------------------------------")

print("Six collapsible visualizations generated.")

print()
print("Illustrative Attack Chain")
print("-------------------------")

for stage, probability in zip(
    stages,
    stage_probabilities
):

    print(
        f"{stage}: {probability:.0%}"
    )

print()

print(
    "Full-Chain Success Without Added Controls:",
    f"{cumulative_probabilities[-1]:.2%}"
)

print(
    "Full-Chain Success With Layered Controls:",
    f"{defended_cumulative[-1]:.2%}"
)

print()
print(
    "All attack probabilities and security events "
    "are synthetic teaching examples."
)

print(
    "No real attacks or external API requests "
    "were performed."
)

<details>

<summary><b>Section 10 Lab — Click to Expand</b></summary>

<br>

You have practiced several individual attacks throughout this module.

In this activity, you will move from:

    INDIVIDUAL ATTACKS

to:

    ATTACK PATH ANALYSIS

You will examine a fictional machine learning service.

The organization operates:

    Internet-Facing Prediction API
              ↓
    Proprietary Classification Model
              ↓
    Prediction + Confidence Scores

The API requires authentication, but legitimate users can submit many prediction requests.

Your task is to identify how an attacker could combine techniques from this module.

You will then identify controls that could interrupt the attack at different stages.

No model training is required.

The goal is to practice **security reasoning** rather than another implementation of an attack you have already seen.

</details>

In [ ]:
# @title ▶ Part 1 — Target Environment { display-mode: "form" }

target = {
    "Internet API": True,
    "Authentication": True,
    "Confidence Scores": True,
    "Rate Limiting": False,
    "Query Monitoring": False,
    "Model Download": False,
    "Adversarial Testing": False
}

print("=" * 65)
print("TARGET AI ENVIRONMENT")
print("=" * 65)

for control, value in target.items():

    status = "YES" if value else "NO"

    print(
        f"{control:<25} {status}"
    )

print("\nAttacker Objective:")
print(
    "Cause the proprietary classifier "
    "to misclassify selected inputs."
)

<details>

<summary><b>Part 1 Explanation — Click to Expand</b></summary>

<br>

The attacker cannot simply download the target model.

However, the attacker does have legitimate API access.

Notice several important characteristics:

    Confidence Scores = Available

    Rate Limiting = None

    Query Monitoring = None

    Model Download = Not Available

The attacker therefore has:

    BLACK-BOX ACCESS

but potentially a large number of opportunities to interact with the target.

Think about which techniques from earlier sections could turn this limited access into a stronger capability.

</details>

In [ ]:
# @title ▶ Part 2 — Choose the First Attack Stage { display-mode: "form" }

print("=" * 65)
print("ATTACK CHAIN — STAGE 1")
print("=" * 65)

print("""
The attacker can access the prediction API but
cannot download the target model.

What should the attacker collect first?

A. Large numbers of input/output observations
B. Physical access to the GPU
C. Training labels directly from the database
D. Administrator passwords
""")

stage1 = input(
    "Your answer: "
).strip().upper()

if stage1 == "A":

    print(
        "\n✓ Correct."
    )

    print(
        "Repeated API queries can provide "
        "input/output observations of the target."
    )

else:

    print(
        "\n✗ Review API and query-based exploitation."
    )

In [ ]:
# @title ▶ Part 3 — Choose Stage 2 { display-mode: "form" }

print("=" * 65)
print("ATTACK CHAIN — STAGE 2")
print("=" * 65)

print("""
The attacker now has thousands of:

    INPUT → TARGET OUTPUT

pairs.

What could these observations be used for?

A. Training a surrogate model
B. Increasing target GPU memory
C. Patching the target
D. Encrypting the training dataset
""")

stage2 = input(
    "Your answer: "
).strip().upper()

if stage2 == "A":

    print(
        "\n✓ Correct."
    )

    print(
        "Target responses can become labels "
        "for surrogate-model training."
    )

else:

    print(
        "\n✗ Review model extraction."
    )

In [ ]:
# @title ▶ Part 4 — Choose Stage 3 { display-mode: "form" }

print("=" * 65)
print("ATTACK CHAIN — STAGE 3")
print("=" * 65)

print("""
The attacker now controls a surrogate model
that approximates the target.

What new capability does this provide?

A. The attacker can experiment with attacks
   against a model they control
B. The target automatically reveals its password
C. The target training dataset is deleted
D. The API becomes unavailable
""")

stage3 = input(
    "Your answer: "
).strip().upper()

if stage3 == "A":

    print(
        "\n✓ Correct."
    )

    print(
        "The surrogate gives the attacker a "
        "controlled environment for attack development."
    )

else:

    print(
        "\n✗ Review surrogate models."
    )

In [ ]:
# @title ▶ Part 5 — Complete the Attack Chain { display-mode: "form" }

print("=" * 65)
print("ATTACK CHAIN — FINAL STAGE")
print("=" * 65)

print("""
The attacker develops an adversarial example
against the surrogate.

What would complete a transfer attack?

A. Submit the adversarial input to the target
   and test whether it also causes misclassification
B. Delete the surrogate
C. Retrain the surrogate with no data
D. Stop interacting with the target
""")

stage4 = input(
    "Your answer: "
).strip().upper()

if stage4 == "A":

    print(
        "\n✓ Correct."
    )

    print(
        "If the adversarial input also fools "
        "the target, the attack transferred."
    )

else:

    print(
        "\n✗ Review transfer attacks."
    )

In [ ]:
# @title ▶ Part 6 — View the Attack Chain { display-mode: "form" }

answers = [
    stage1,
    stage2,
    stage3,
    stage4
]

correct = all(
    answer == "A"
    for answer in answers
)

print("=" * 65)
print("COMPLETE ATTACK CHAIN")
print("=" * 65)

print("""
        API ACCESS
            │
            ▼
     REPEATED QUERIES
            │
            ▼
   INPUT / OUTPUT DATA
            │
            ▼
     MODEL EXTRACTION
            │
            ▼
      SURROGATE MODEL
            │
            ▼
   ADVERSARIAL EXAMPLE
            │
            ▼
      TRANSFER ATTACK
            │
            ▼
 TARGET MISCLASSIFICATION
""")

if correct:

    print(
        "✓ You correctly reconstructed "
        "the complete attack chain."
    )

else:

    print(
        "△ Review the stages above and "
        "rerun Parts 2–5."
    )

<details>

<summary><b>Attack Chain Explanation — Click to Expand</b></summary>

<br>

Notice how the attacker began with a relatively limited capability:

    API ACCESS

The attacker did not initially have:

    Model Parameters

    Gradients

    Architecture

or:

    Training Data

However, API access enabled:

    MODEL QUERIES

Those queries enabled:

    MODEL EXTRACTION

Model extraction enabled:

    SURROGATE ACCESS

The surrogate enabled:

    ADVERSARIAL ATTACK DEVELOPMENT

which created an opportunity for:

    TRANSFER TO TARGET

Each stage created the capability required for the next stage.

That is what makes this an attack chain rather than several unrelated attacks.

</details>

In [ ]:
# @title ▶ Part 7 — Defender Challenge { display-mode: "form" }

defense_questions = [
    {
        "question":
        "1. Which control could make large-scale "
        "model querying more difficult?",
        "options": [
            "A. Rate limiting",
            "B. More confidence information",
            "C. Removing authentication",
            "D. Unlimited queries"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. Which control could help identify "
        "systematic probing?",
        "options": [
            "A. Query monitoring",
            "B. Disable logging",
            "C. Increase API output",
            "D. Remove usage tracking"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. Which change could reduce information "
        "available for model extraction?",
        "options": [
            "A. Limit unnecessary confidence outputs",
            "B. Return more internal information",
            "C. Publish model parameters",
            "D. Disable authentication"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. Which control addresses the final "
        "adversarial stage?",
        "options": [
            "A. Robustness testing and adversarial defenses",
            "B. Unlimited API access",
            "C. Remove monitoring",
            "D. Increase query limits"
        ],
        "answer": "A"
    }
]

defense_score = 0

print("=" * 65)
print("BREAK THE ATTACK CHAIN")
print("=" * 65)

for q in defense_questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:

        print("✓ Correct!")
        defense_score += 1

    else:

        print(
            f"✗ Correct answer: "
            f"{q['answer']}"
        )

print("\n" + "=" * 65)

print(
    f"Defense Score: "
    f"{defense_score}/"
    f"{len(defense_questions)}"
)

In [ ]:
# @title ▶ Part 8 — Defense-in-Depth Map { display-mode: "form" }

print("=" * 70)
print("ATTACK CHAIN + DEFENSIVE OPPORTUNITIES")
print("=" * 70)

print("""
     ATTACKER GAINS API ACCESS
                │
                ▼
         REPEATED QUERIES
                │
       ┌────────┴────────┐
       │ RATE LIMITING   │
       │ QUERY MONITORING│
       └────────┬────────┘
                ▼
       COLLECT RESPONSES
                │
       ┌────────┴────────────┐
       │ LIMIT UNNECESSARY   │
       │ OUTPUT INFORMATION  │
       └────────┬────────────┘
                ▼
        BUILD SURROGATE
                │
                ▼
    CREATE ADVERSARIAL INPUT
                │
       ┌────────┴────────────┐
       │ ADVERSARIAL TESTING │
       │ ROBUSTNESS DEFENSES │
       └────────┬────────────┘
                ▼
       TARGET MISCLASSIFICATION
""")

print(
    "The defender has multiple opportunities "
    "to interrupt the chain."
)

<details>

<summary><b>Part 8 Explanation — Click to Expand</b></summary>

<br>

The important observation is that there is no single:

    "STOP ATTACK"

button.

Instead, defenders have opportunities throughout the chain.

For example:

    RATE LIMITING
         ↓
    Makes large-scale querying harder

    QUERY MONITORING
         ↓
    May reveal systematic probing

    OUTPUT RESTRICTIONS
         ↓
    Reduce unnecessary information exposure

    ADVERSARIAL TESTING
         ↓
    Helps identify weaknesses before deployment

Even if one control fails, another may still interrupt the attacker.

This is:

    DEFENSE IN DEPTH

The objective is to create multiple security boundaries between the attacker and the final objective.

</details>

In [ ]:
# @title ▶ Part 9 — Attack Chain Lab Check { display-mode: "form" }

questions = [
    {
        "question":
        "1. What was the attacker's initial capability?",
        "options": [
            "A. Prediction API access",
            "B. Full model parameters",
            "C. Administrator access",
            "D. Training-database access"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. What did repeated queries provide?",
        "options": [
            "A. Input/output observations",
            "B. Physical GPU access",
            "C. Administrator credentials",
            "D. Source code automatically"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. What was built from those observations?",
        "options": [
            "A. A surrogate model",
            "B. A new GPU",
            "C. A firewall",
            "D. A training database"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. What was developed against the surrogate?",
        "options": [
            "A. An adversarial input",
            "B. An API password",
            "C. A new dataset label",
            "D. A rate limiter"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. What made the final attack a transfer attack?",
        "options": [
            "A. The adversarial input developed on the "
            "surrogate was tested against the target",
            "B. The surrogate was deleted",
            "C. The target was retrained",
            "D. The API was shut down"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. Why is defense in depth useful against attack chains?",
        "options": [
            "A. It provides multiple opportunities to "
            "prevent or detect the attack",
            "B. It guarantees every attack fails",
            "C. It eliminates the need for monitoring",
            "D. It allows unlimited model access"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 65)
print("ATTACK CHAIN MINI LAB CHECK")
print("=" * 65)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:

        print("✓ Correct!")
        score += 1

    else:

        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = score / len(questions) * 100

print("\n" + "=" * 65)

print(
    f"Lab Score: "
    f"{score}/{len(questions)}"
)

print(
    f"Percentage: "
    f"{percentage:.1f}%"
)

if percentage >= 80:

    print(
        "\n✓ Attack-chain activity complete."
    )

else:

    print(
        "\nReview the attack chain "
        "and try the lab check again."
    )

print("=" * 65)

<details>

<summary><b>Section 10 — Practical Takeaways — Click to Expand</b></summary>

<br>

Throughout this module, you examined individual attacks.

Section 10 connected those techniques together.

The activity began with an attacker who had only:

    PREDICTION API ACCESS

That access enabled:

    REPEATED QUERIES
           ↓
    INPUT / OUTPUT COLLECTION
           ↓
    MODEL EXTRACTION
           ↓
    SURROGATE MODEL
           ↓
    ADVERSARIAL ATTACK DEVELOPMENT
           ↓
    TRANSFER ATTACK
           ↓
    TARGET MISCLASSIFICATION

The attacker gained additional capabilities at each stage.

This demonstrates why security weaknesses should not always be evaluated independently.

A weakness that appears minor by itself may become important when it provides the information or access needed for another technique.

From the defender's perspective, the same chain contained several opportunities for intervention.

    API ACCESS
        ↓
    Authentication

    REPEATED QUERIES
        ↓
    Rate Limiting
    Query Monitoring

    INFORMATION COLLECTION
        ↓
    Limit Unnecessary Outputs

    ADVERSARIAL DEVELOPMENT
        ↓
    Robustness Testing

    TARGET ATTACK
        ↓
    Monitoring and Response

This is the idea behind:

    DEFENSE IN DEPTH

The goal is not to depend on one perfect control.

Instead, defenders create multiple opportunities to:

    PREVENT

    DETECT

    SLOW

    CONTAIN

or:

    RESPOND TO

an attack.

The most important lesson from this section is:

> **Understanding individual AI attacks is only the beginning. Defenders must also understand how those attacks can connect into larger attack paths and where those paths can be interrupted.**

</details>

In [ ]:
# @title ▶ Section 10 Checkpoint — Attack Chains { display-mode: "form" }

questions = [
    {
        "question":
        "1. What is an attack chain?",
        "options": [
            "A. Multiple attack stages connected toward an objective",
            "B. A single model prediction",
            "C. A training accuracy measurement",
            "D. A type of neural-network layer"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. How might API access contribute to model extraction?",
        "options": [
            "A. Queries can create input/output pairs for surrogate training",
            "B. APIs automatically reveal every parameter",
            "C. APIs automatically download the model",
            "D. API access prevents extraction"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. Why might an attacker build a surrogate model?",
        "options": [
            "A. It can provide a controllable approximation for developing attacks",
            "B. It automatically increases target security",
            "C. It prevents adversarial examples",
            "D. It removes the target API"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. Can an AI attack chain cross multiple lifecycle stages?",
        "options": [
            "A. Yes",
            "B. No",
            "C. Only during training",
            "D. Only during deployment"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. What does defense in depth mean in this context?",
        "options": [
            "A. Using multiple security controls across the attack path",
            "B. Depending entirely on one security control",
            "C. Removing monitoring",
            "D. Increasing model size"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. When should defenders attempt to break an attack chain?",
        "options": [
            "A. At any stage where prevention or detection is possible",
            "B. Only after the final attack succeeds",
            "C. Only during training",
            "D. Only when model accuracy reaches zero"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 65)
print("SECTION 10 CHECKPOINT")
print("ATTACK CHAINS & MULTI-STAGE AI ATTACKS")
print("=" * 65)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1

    else:
        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = score / len(questions) * 100

print("\n" + "=" * 65)
print(f"Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:

    print(
        "\n✓ Checkpoint complete. "
        "Continue to the attack-chain activity."
    )

else:

    print(
        "\nReview Section 10 before continuing."
    )

print("=" * 65)

---

<details>

<summary><b>End-of-Module Lab — Click to Expand</b></summary>

<br>

Throughout this module, you explored several ways attackers can target machine learning models after or around deployment.

You studied attacks involving:

    Model Information
            ↓
    Model Behavior
            ↓
    Training Data Privacy
            ↓
    Adversarial Inputs
            ↓
    Hidden Backdoors
            ↓
    Prediction APIs
            ↓
    Resource Availability
            ↓
    Multi-Stage Attack Chains

The purpose of this final lab is not to introduce another attack.

Instead, you will act as an **AI security analyst** responsible for examining a fictional machine learning service.

You will need to:

1. Identify security weaknesses.
2. Recognize possible attacks.
3. Interpret security observations.
4. Connect attacks into attack chains.
5. Select appropriate defenses.
6. Evaluate the security of the overall AI system.

Unlike the smaller activities throughout the module, you will not be told which attack applies at every stage.

You must determine that yourself.

The lab is designed around the following scenario:

    USER
      ↓
    PUBLIC API
      ↓
    ML CLASSIFIER
      ↓
    PREDICTION
      ↓
    CONFIDENCE SCORES

The organization wants to determine whether this system is adequately protected before expanding its deployment.

Your job is to evaluate it.

</details>

In [ ]:
# @title ▶ Part 1 — Examine the Target System { display-mode: "form" }

system = {
    "Public Prediction API": True,
    "Authentication Required": True,
    "Confidence Scores Returned": True,
    "Rate Limiting": False,
    "Query Monitoring": False,
    "Input Validation": False,
    "Model Download Allowed": False,
    "Third-Party Model": True,
    "Model Provenance Verified": False,
    "Adversarial Testing": False,
    "Resource Monitoring": True
}

print("=" * 70)
print("AI SYSTEM SECURITY REVIEW")
print("=" * 70)

for item, value in system.items():

    status = "YES" if value else "NO"

    print(
        f"{item:<32} {status}"
    )

print("\nYour role:")
print(
    "Evaluate this system for model-level "
    "security weaknesses."
)

<details>

<summary><b>Part 1 Explanation — Click to Expand</b></summary>

<br>

Before attempting to identify attacks, a security analyst should understand the environment.

Some characteristics are security controls:

    Authentication Required

    Resource Monitoring

Other characteristics may increase exposure:

    Public Prediction API

    Confidence Scores Returned

Some important controls are missing:

    Rate Limiting

    Query Monitoring

    Input Validation

    Adversarial Testing

There is also a supply-chain concern:

    Third-Party Model
           +
    Provenance Not Verified

Do not assume that every characteristic represents a vulnerability by itself.

Instead, think about how each characteristic could affect:

    ATTACKER ACCESS

    INFORMATION EXPOSURE

    MODEL INTEGRITY

    PRIVACY

    AVAILABILITY

</details>

In [ ]:
# @title ▶ Part 2 — Initial Security Assessment { display-mode: "form" }

questions = [
    {
        "question":
        "1. The API returns detailed confidence scores. "
        "What concern does this primarily create?",
        "options": [
            "A. It may provide additional information about model behavior",
            "B. It automatically poisons training data",
            "C. It deletes model parameters",
            "D. It prevents queries"
        ],
        "answer": "A"
    },

    {
        "question":
        "2. There is no rate limiting. What capability "
        "does this potentially give an attacker?",
        "options": [
            "A. Ability to perform large numbers of queries",
            "B. Direct access to the training database",
            "C. Physical access to the server",
            "D. Automatic administrator privileges"
        ],
        "answer": "A"
    },

    {
        "question":
        "3. Why is unverified model provenance important?",
        "options": [
            "A. A third-party model could contain unexpected or malicious behavior",
            "B. It guarantees poor accuracy",
            "C. It disables the API",
            "D. It prevents inference"
        ],
        "answer": "A"
    },

    {
        "question":
        "4. What is a concern when adversarial testing "
        "has not been performed?",
        "options": [
            "A. The organization may not know how the model behaves under crafted inputs",
            "B. The model cannot make normal predictions",
            "C. The model automatically leaks passwords",
            "D. Training becomes impossible"
        ],
        "answer": "A"
    }
]

assessment_score = 0

print("=" * 70)
print("INITIAL SECURITY ASSESSMENT")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    answer = input(
        "\nYour answer: "
    ).strip().upper()

    if answer == q["answer"]:

        print("✓ Correct")
        assessment_score += 1

    else:

        print(
            f"✗ Correct answer: "
            f"{q['answer']}"
        )

print(
    f"\nAssessment Score: "
    f"{assessment_score}/{len(questions)}"
)

In [ ]:
# @title ▶ Part 3 — Identify the Attack { display-mode: "form" }

scenarios = [
    {
        "scenario":
        "An attacker repeatedly queries the target and "
        "uses the returned predictions to train another "
        "model that behaves similarly.",
        "answer": "MODEL EXTRACTION"
    },

    {
        "scenario":
        "An attacker attempts to determine whether a "
        "specific record appeared in the model's "
        "training dataset.",
        "answer": "MEMBERSHIP INFERENCE"
    },

    {
        "scenario":
        "A carefully modified input causes the model "
        "to produce an incorrect prediction.",
        "answer": "ADVERSARIAL EVASION"
    },

    {
        "scenario":
        "A hidden behavior activates only when a "
        "specific trigger appears in the input.",
        "answer": "BACKDOOR"
    },

    {
        "scenario":
        "An attacker attempts to recover characteristics "
        "of private training information from model behavior.",
        "answer": "MODEL INVERSION"
    },

    {
        "scenario":
        "Large numbers of expensive requests cause "
        "legitimate predictions to become slow or unavailable.",
        "answer": "RESOURCE EXHAUSTION"
    }
]

choices = {
    "A": "MODEL EXTRACTION",
    "B": "MEMBERSHIP INFERENCE",
    "C": "ADVERSARIAL EVASION",
    "D": "BACKDOOR",
    "E": "MODEL INVERSION",
    "F": "RESOURCE EXHAUSTION"
}

attack_score = 0

print("=" * 70)
print("ATTACK IDENTIFICATION")
print("=" * 70)

for scenario in scenarios:

    print("\nScenario:")
    print(scenario["scenario"])

    print("""
A. Model Extraction
B. Membership Inference
C. Adversarial Evasion
D. Backdoor
E. Model Inversion
F. Resource Exhaustion
""")

    response = input(
        "Your answer: "
    ).strip().upper()

    selected = choices.get(
        response,
        "INVALID"
    )

    if selected == scenario["answer"]:

        print("✓ Correct!")
        attack_score += 1

    else:

        print(
            f"✗ Correct answer: "
            f"{scenario['answer']}"
        )

print(
    f"\nAttack Identification Score: "
    f"{attack_score}/{len(scenarios)}"
)

In [ ]:
# @title ▶ Part 4 — Analyze Query Activity { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

normal_queries = np.random.normal(
    35,
    7,
    20
)

suspicious_queries = np.array([
    42, 38, 41, 44, 39,
    45, 48, 95, 140, 220,
    310, 420, 510, 575, 620,
    660, 700, 730, 760, 800
])

time_steps = np.arange(
    1,
    21
)

plt.figure(figsize=(8, 4))

plt.plot(
    time_steps,
    normal_queries,
    label="Normal User"
)

plt.plot(
    time_steps,
    suspicious_queries,
    label="Suspicious User"
)

plt.xlabel("Time Step")
plt.ylabel("Queries")
plt.title("Prediction API Activity")

plt.legend()
plt.show()

print("=" * 70)
print("QUERY ACTIVITY SUMMARY")
print("=" * 70)

print(
    f"\nNormal User Average: "
    f"{np.mean(normal_queries):.1f}"
)

print(
    f"Suspicious User Average: "
    f"{np.mean(suspicious_queries):.1f}"
)

print(
    f"Suspicious Peak: "
    f"{np.max(suspicious_queries)}"
)

<details>

<summary><b>Part 4 Analysis — Click to Expand</b></summary>

<br>

The graph represents two API users.

One maintains relatively consistent activity.

The other begins similarly but rapidly increases the number of requests.

You should **not automatically conclude that the second user is malicious**.

High query volume could have legitimate explanations.

However, this behavior provides a signal that defenders could investigate.

Think about attacks from this module that may benefit from large numbers of queries.

Examples include:

    Model Extraction

    Query-Based Adversarial Search

    Some Privacy Attacks

    Resource Exhaustion

This demonstrates why security monitoring requires:

    OBSERVATION
         +
    CONTEXT
         +
    INVESTIGATION

rather than simply:

    HIGH NUMBER = ATTACK

</details>

In [ ]:
# @title ▶ Part 5 — Analyze the Evidence { display-mode: "form" }

questions = [
    {
        "question":
        "1. Does high query volume automatically prove an attack?",
        "options": [
            "A. Yes",
            "B. No"
        ],
        "answer": "B"
    },

    {
        "question":
        "2. Which attack could use large numbers of "
        "input/output observations?",
        "options": [
            "A. Model Extraction",
            "B. Physical theft",
            "C. Disk encryption",
            "D. Password reset"
        ],
        "answer": "A"
    },

    {
        "question":
        "3. What should defenders do with unusual "
        "query behavior?",
        "options": [
            "A. Investigate it using additional context",
            "B. Automatically delete the model",
            "C. Ignore all query activity",
            "D. Publish the model"
        ],
        "answer": "A"
    }
]

query_score = 0

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:

        print("✓ Correct!")
        query_score += 1

    else:

        print(
            f"✗ Correct answer: "
            f"{q['answer']}"
        )

print(
    f"\nScore: "
    f"{query_score}/{len(questions)}"
)

In [ ]:
# @title ▶ Part 6 — Attack Chain Challenge { display-mode: "form" }

print("=" * 70)
print("ATTACK CHAIN CHALLENGE")
print("=" * 70)

print("""
An attacker wants to create an adversarial input
for the target but cannot access the target's
gradients.

Which sequence best represents a possible attack chain?

A.
API Queries
    ↓
Collect Target Responses
    ↓
Train Surrogate
    ↓
Develop Adversarial Input
    ↓
Test Against Target

B.
Delete Dataset
    ↓
Restart Computer
    ↓
Change Password

C.
Increase Model Accuracy
    ↓
Reduce Queries
    ↓
Patch Browser

D.
Monitor API
    ↓
Enable Rate Limiting
    ↓
Validate Inputs
""")

chain_answer = input(
    "Your answer: "
).strip().upper()

if chain_answer == "A":

    print(
        "\n✓ Correct!"
    )

    print("""
You identified:

API Exploitation
      ↓
Model Extraction
      ↓
Surrogate Model
      ↓
Adversarial Attack
      ↓
Transfer Attack
""")

    chain_score = 1

else:

    print(
        "\n✗ Review the attack-chain section."
    )

    chain_score = 0

In [ ]:
# @title ▶ Part 7 — Defense Selection Challenge { display-mode: "form" }

questions = [
    {
        "question":
        "1. Which defense helps control mass querying?",
        "options": [
            "A. Rate limiting",
            "B. More confidence outputs",
            "C. Disable logging",
            "D. Unlimited API access"
        ],
        "answer": "A"
    },

    {
        "question":
        "2. Which defense helps detect unusual API behavior?",
        "options": [
            "A. Query monitoring",
            "B. Removing logs",
            "C. Increasing query limits",
            "D. Publishing model weights"
        ],
        "answer": "A"
    },

    {
        "question":
        "3. Which practice helps identify adversarial weaknesses?",
        "options": [
            "A. Adversarial robustness testing",
            "B. Ignoring unusual inputs",
            "C. Removing validation",
            "D. Disabling monitoring"
        ],
        "answer": "A"
    },

    {
        "question":
        "4. Which control helps address untrusted third-party models?",
        "options": [
            "A. Model provenance and integrity verification",
            "B. Unlimited queries",
            "C. Removing authentication",
            "D. Increasing confidence output"
        ],
        "answer": "A"
    },

    {
        "question":
        "5. Which control helps protect availability?",
        "options": [
            "A. Resource monitoring and request limits",
            "B. More expensive requests",
            "C. Unlimited queue growth",
            "D. Disable usage monitoring"
        ],
        "answer": "A"
    }
]

defense_score = 0

print("=" * 70)
print("DEFENSE SELECTION")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:

        print("✓ Correct!")
        defense_score += 1

    else:

        print(
            f"✗ Correct answer: "
            f"{q['answer']}"
        )

print(
    f"\nDefense Score: "
    f"{defense_score}/{len(questions)}"
)

In [ ]:
# @title ▶ Part 8 — Build the Secured Architecture { display-mode: "form" }

print("=" * 72)
print("DEFENSE-IN-DEPTH AI ARCHITECTURE")
print("=" * 72)

print("""
                  USER
                    │
                    ▼
             AUTHENTICATION
                    │
                    ▼
              RATE LIMITING
                    │
                    ▼
             INPUT VALIDATION
                    │
                    ▼
              PREDICTION API
                    │
                    ▼
                AI MODEL
                    │
          ┌─────────┴─────────┐
          │                   │
          ▼                   ▼
   OUTPUT CONTROLS       RESOURCE LIMITS
          │                   │
          └─────────┬─────────┘
                    ▼
                 LOGGING
                    │
                    ▼
            QUERY MONITORING
                    │
                    ▼
           SECURITY RESPONSE
""")

print("\nAdditional model protections:")

protections = [
    "Model provenance verification",
    "Adversarial robustness testing",
    "Backdoor testing",
    "Privacy evaluation",
    "Resource monitoring"
]

for protection in protections:

    print(
        f"  ✓ {protection}"
    )

<details>

<summary><b>Part 8 Explanation — Click to Expand</b></summary>

<br>

The secured architecture demonstrates an important lesson from the entire module.

There is no single control called:

    AI SECURITY

Instead, security comes from multiple layers.

Before inference:

    Authentication

    Rate Limiting

    Input Validation

During inference:

    Resource Controls

    Model Robustness

After inference:

    Output Controls

    Logging

    Query Monitoring

Across the model lifecycle:

    Provenance Verification

    Backdoor Testing

    Privacy Evaluation

    Adversarial Testing

This is:

    DEFENSE IN DEPTH

If one control fails, another may still prevent the attacker from reaching the final objective.

</details>

In [ ]:
# @title ▶ Part 9 — Final Lab Assessment { display-mode: "form" }

questions = [
    {
        "question":
        "1. An attacker trains a copy of a model using "
        "target API responses. What attack is this?",
        "options": [
            "A. Model Extraction",
            "B. Resource Exhaustion",
            "C. Backdoor",
            "D. Data Encryption"
        ],
        "answer": "A"
    },

    {
        "question":
        "2. An attacker determines whether a record "
        "was part of training. What attack is this?",
        "options": [
            "A. Membership Inference",
            "B. Model Extraction",
            "C. DoS",
            "D. Transfer Learning"
        ],
        "answer": "A"
    },

    {
        "question":
        "3. A hidden trigger causes malicious model "
        "behavior. What does this describe?",
        "options": [
            "A. Backdoor",
            "B. Rate Limiting",
            "C. Authentication",
            "D. Normal inference"
        ],
        "answer": "A"
    },

    {
        "question":
        "4. A modified input causes misclassification "
        "without changing the model. What is this?",
        "options": [
            "A. Adversarial Evasion",
            "B. Model Training",
            "C. Authentication",
            "D. Logging"
        ],
        "answer": "A"
    },

    {
        "question":
        "5. Why can a surrogate model be useful to an attacker?",
        "options": [
            "A. It provides a controllable approximation "
            "for developing attacks",
            "B. It automatically secures the target",
            "C. It prevents queries",
            "D. It removes adversarial examples"
        ],
        "answer": "A"
    },

    {
        "question":
        "6. What security property is primarily affected "
        "by resource exhaustion?",
        "options": [
            "A. Availability",
            "B. Confidentiality",
            "C. Label quality",
            "D. Explainability"
        ],
        "answer": "A"
    },

    {
        "question":
        "7. Why can confidence scores create additional risk?",
        "options": [
            "A. They reveal more information about model behavior",
            "B. They automatically poison data",
            "C. They disable the API",
            "D. They prevent predictions"
        ],
        "answer": "A"
    },

    {
        "question":
        "8. What is an attack chain?",
        "options": [
            "A. Multiple attack stages connected toward an objective",
            "B. One prediction error",
            "C. A neural network layer",
            "D. A dataset format"
        ],
        "answer": "A"
    },

    {
        "question":
        "9. What is defense in depth?",
        "options": [
            "A. Multiple security controls protecting different stages",
            "B. Depending on one perfect defense",
            "C. Removing monitoring",
            "D. Allowing unlimited access"
        ],
        "answer": "A"
    },

    {
        "question":
        "10. What should AI security protect?",
        "options": [
            "A. The model and the surrounding AI system",
            "B. Only model accuracy",
            "C. Only training speed",
            "D. Only the user interface"
        ],
        "answer": "A"
    }
]

final_score = 0

print("=" * 70)
print("FINAL MODEL EXPLOITATION LAB ASSESSMENT")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:

        print("✓ Correct!")
        final_score += 1

    else:

        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = (
    final_score
    / len(questions)
    * 100
)

print("\n" + "=" * 70)

print(
    f"Final Score: "
    f"{final_score}/{len(questions)}"
)

print(
    f"Percentage: "
    f"{percentage:.1f}%"
)

if percentage >= 80:

    print("""
✓ END-OF-MODULE LAB COMPLETE

You demonstrated understanding of:
  • Model exploitation
  • Privacy attacks
  • Adversarial attacks
  • API exploitation
  • Resource exhaustion
  • Attack chains
  • Defensive controls
""")

else:

    print("""
△ REVIEW RECOMMENDED

Review the sections associated with
the questions you missed and rerun
the assessment.
""")

print("=" * 70)

<details>

<summary><b>Part 10 — Final Reflection — Click to Expand</b></summary>

<br>

Before completing the module, consider the system you evaluated.

At the beginning of the lab, the organization had:

    Authentication

and:

    Resource Monitoring

Those controls were useful, but they did not address every possible model-level attack.

The system still exposed several potential weaknesses.

Think about the following questions:

### Question 1

Which attack from this module do you think would be the most difficult for a defender to detect?

Why?

### Question 2

Which security control from this lab could help stop or detect more than one attack?

Explain which attacks it could affect.

### Question 3

Suppose an organization tells you:

> "Our model has 95% accuracy, so we know it is secure."

Using what you learned in this module, explain why accuracy alone does not demonstrate that an AI system is secure.

### Question 4

Why should defenders analyze combinations of attacks rather than evaluating every attack independently?

Take a few minutes to think through your answers before continuing.

These questions do not have a single required response. The goal is to connect the technical activities from the module to the larger process of securing deployed AI systems.

</details>

---

<details>

<summary><b>Module References — Click to Expand</b></summary>

<br>

The following sources support the discussion of model exploitation, model extraction, privacy attacks, adversarial evasion, transfer attacks, backdoors, API exploitation, resource exhaustion, attack chains, and defensive techniques throughout this module.

### Adversarial Machine Learning

[1] A. Vassilev, A. Oprea, A. Fordyce, H. Anderson, X. Davies, and M. Hamin, “Adversarial Machine Learning: A Taxonomy and Terminology of Attacks and Mitigations,” *NIST Trustworthy and Responsible AI*, NIST AI 100-2e2025, National Institute of Standards and Technology, Gaithersburg, MD, Mar. 2025, doi: 10.6028/NIST.AI.100-2e2025.

### Model Extraction and Model Stealing

[2] F. Tramèr, F. Zhang, A. Juels, M. K. Reiter, and T. Ristenpart, “Stealing machine learning models via prediction APIs,” in *Proc. 25th USENIX Security Symp. (USENIX Security 16)*, Austin, TX, USA, 2016, pp. 601–618.

[3] T. Orekondy, B. Schiele, and M. Fritz, “Knockoff nets: Stealing functionality of black-box models,” in *Proc. IEEE/CVF Conf. Computer Vision and Pattern Recognition (CVPR)*, Long Beach, CA, USA, 2019, pp. 4954–4963.

[4] M. Jagielski, N. Carlini, D. Berthelot, A. Kurakin, and N. Papernot, “High accuracy and high fidelity extraction of neural networks,” in *Proc. 29th USENIX Security Symp. (USENIX Security 20)*, 2020, pp. 1345–1362.

### Membership Inference

[5] R. Shokri, M. Stronati, C. Song, and V. Shmatikov, “Membership inference attacks against machine learning models,” in *Proc. 2017 IEEE Symp. Security and Privacy (SP)*, San Jose, CA, USA, 2017, pp. 3–18, doi: 10.1109/SP.2017.41.

[6] S. Yeom, I. Giacomelli, M. Fredrikson, and S. Jha, “Privacy risk in machine learning: Analyzing the connection to overfitting,” in *Proc. 31st IEEE Computer Security Foundations Symp. (CSF)*, Oxford, U.K., 2018, pp. 268–282.

[7] A. Salem, Y. Zhang, M. Humbert, P. Berrang, M. Fritz, and M. Backes, “ML-Leaks: Model and data independent membership inference attacks and defenses on machine learning models,” in *Proc. Network and Distributed System Security Symp. (NDSS)*, San Diego, CA, USA, 2019.

[8] J. Jia, A. Salem, M. Backes, Y. Zhang, and N. Z. Gong, “MemGuard: Defending against black-box membership inference attacks via adversarial examples,” in *Proc. ACM SIGSAC Conf. Computer and Communications Security (CCS)*, 2019, pp. 259–274.

### Model Inversion and Privacy Leakage

[9] M. Fredrikson, E. Lantz, S. Jha, S. Lin, D. Page, and T. Ristenpart, “Privacy in pharmacogenetics: An end-to-end case study of personalized warfarin dosing,” in *Proc. 23rd USENIX Security Symp. (USENIX Security 14)*, San Diego, CA, USA, 2014, pp. 17–32.

[10] M. Fredrikson, S. Jha, and T. Ristenpart, “Model inversion attacks that exploit confidence information and basic countermeasures,” in *Proc. 22nd ACM SIGSAC Conf. Computer and Communications Security (CCS)*, Denver, CO, USA, 2015, pp. 1322–1333, doi: 10.1145/2810103.2813677.

[11] Y. Zhang, R. Jia, H. Pei, W. Wang, B. Li, and D. Song, “The secret revealer: Generative model-inversion attacks against deep neural networks,” in *Proc. IEEE/CVF Conf. Computer Vision and Pattern Recognition (CVPR)*, 2020, pp. 253–261.

### Adversarial Evasion

[12] B. Biggio, I. Corona, D. Maiorca, B. Nelson, N. Šrndić, P. Laskov, G. Giacinto, and F. Roli, “Evasion attacks against machine learning at test time,” in *Machine Learning and Knowledge Discovery in Databases*, Berlin, Germany: Springer, 2013, pp. 387–402.

[13] C. Szegedy *et al.*, “Intriguing properties of neural networks,” in *Proc. International Conf. Learning Representations (ICLR)*, 2014.

[14] I. J. Goodfellow, J. Shlens, and C. Szegedy, “Explaining and harnessing adversarial examples,” in *Proc. International Conf. Learning Representations (ICLR)*, 2015.

[15] N. Carlini and D. Wagner, “Towards evaluating the robustness of neural networks,” in *Proc. 2017 IEEE Symp. Security and Privacy (SP)*, San Jose, CA, USA, 2017, pp. 39–57, doi: 10.1109/SP.2017.49.

[16] A. Madry, A. Makelov, L. Schmidt, D. Tsipras, and A. Vladu, “Towards deep learning models resistant to adversarial attacks,” in *Proc. International Conf. Learning Representations (ICLR)*, 2018.

### Transfer Attacks and Surrogate Models

[17] N. Papernot, P. McDaniel, and I. Goodfellow, “Transferability in machine learning: From phenomena to black-box attacks using adversarial samples,” *arXiv preprint arXiv:1605.07277*, 2016.

[18] N. Papernot, P. McDaniel, I. Goodfellow, S. Jha, Z. B. Celik, and A. Swami, “Practical black-box attacks against machine learning,” in *Proc. 2017 ACM Asia Conf. Computer and Communications Security (AsiaCCS)*, Abu Dhabi, UAE, 2017, pp. 506–519.

[19] Y. Liu, X. Chen, C. Liu, and D. Song, “Delving into transferable adversarial examples and black-box attacks,” in *Proc. International Conf. Learning Representations (ICLR)*, 2017.

[20] F. Tramèr, N. Papernot, I. Goodfellow, D. Boneh, and P. McDaniel, “The space of transferable adversarial examples,” *arXiv preprint arXiv:1704.03453*, 2017.

### Model Backdoors and Trojan Exploitation

[21] T. Gu, B. Dolan-Gavitt, and S. Garg, “BadNets: Identifying vulnerabilities in the machine learning model supply chain,” *arXiv preprint arXiv:1708.06733*, 2017.

[22] X. Chen, C. Liu, B. Li, K. Lu, and D. Song, “Targeted backdoor attacks on deep learning systems using data poisoning,” *arXiv preprint arXiv:1712.05526*, 2017.

[23] Y. Liu, S. Ma, Y. Aafer, W.-C. Lee, J. Zhai, W. Wang, and X. Zhang, “Trojaning attack on neural networks,” in *Proc. Network and Distributed System Security Symp. (NDSS)*, San Diego, CA, USA, 2018.

[24] B. Tran, J. Li, and A. Madry, “Spectral signatures in backdoor attacks,” in *Advances in Neural Information Processing Systems (NeurIPS)*, vol. 31, 2018.

[25] B. Wang *et al.*, “Neural Cleanse: Identifying and mitigating backdoor attacks in neural networks,” in *Proc. 2019 IEEE Symp. Security and Privacy (SP)*, San Francisco, CA, USA, 2019, pp. 707–723.

### Model Denial of Service and Resource Exhaustion

[26] I. Shumailov, Y. Zhao, D. Bates, N. Papernot, R. Mullins, and R. Anderson, “Sponge examples: Energy-latency attacks on neural networks,” in *Proc. 2021 IEEE European Symp. Security and Privacy (EuroS&P)*, 2021, pp. 212–231, doi: 10.1109/EuroSP51992.2021.00024.

### Defenses Against Model Exploitation

[27] M. Juuti, S. Szyller, S. Marchal, and N. Asokan, “PRADA: Protecting against DNN model stealing attacks,” in *Proc. 2019 IEEE European Symp. Security and Privacy (EuroS&P)*, Stockholm, Sweden, 2019, pp. 512–527, doi: 10.1109/EuroSP.2019.00044.

[28] S. Kariyappa and M. K. Qureshi, “Defending against model stealing attacks with adaptive misinformation,” in *Proc. IEEE/CVF Conf. Computer Vision and Pattern Recognition Workshops (CVPRW)*, 2020.

[29] C. Dwork, F. McSherry, K. Nissim, and A. Smith, “Calibrating noise to sensitivity in private data analysis,” in *Theory of Cryptography*, Berlin, Germany: Springer, 2006, pp. 265–284.

[30] M. Abadi, A. Chu, I. Goodfellow, H. B. McMahan, I. Mironov, K. Talwar, and L. Zhang, “Deep learning with differential privacy,” in *Proc. 2016 ACM SIGSAC Conf. Computer and Communications Security (CCS)*, Vienna, Austria, 2016, pp. 308–318.

### Attack Chains and AI Threat Modeling

[31] MITRE, “Adversarial Threat Landscape for Artificial-Intelligence Systems (ATLAS),” MITRE, Bedford, MA, USA. [Online]. Available: https://atlas.mitre.org/

[32] M. D. R. et al., “Adversarial ML Threat Matrix: Adversarial tactics, techniques, and common knowledge of machine learning,” Microsoft and MITRE, 2020. [Online]. Available: https://github.com/mitre/advmlthreatmatrix

</details>